# 导入库

In [ ]:
# ==================== 单元格 1: 导入库 ====================
"""
TFT异常检测和变点检测主流程
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch import Trainer

from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import train_test_split


plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
import json


In [ ]:
from tqdm.auto import tqdm

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# 检查版本
print("📦 检查环境版本...")
import torch
# import lightning.pytorch as 
import lightning.pytorch as pl
try:
    import pytorch_forecasting as pf
    print(f"✅ PyTorch: {torch.__version__}")
    print(f"✅ PyTorch Lightning: {pl.__version__}")
    print(f"✅ PyTorch Forecasting: {pf.__version__}")
    
    # 检查版本兼容性
    pl_major = int(pl.__version__.split('.')[0])
    if pl_major >= 2:
        print("⚠️  警告: PyTorch Lightning 2.x 可能存在兼容性问题")
        print("   建议版本: pytorch-lightning<2.0.0")
except ImportError as e:
    print(f"❌ 导入错误: {e}")
    # print("请运行: pip install pytorch-forecasting pytorch-lightning==1.9.5")
    sys.exit(1)

# 加载数据

In [ ]:

# ==================== 单元格 2: 加载数据 ====================
"""
加载模拟数据、运营日历、真实数据
"""
import os
# 数据路径（根据实际情况修改）

DATA_PATH = r'C:\tongji\0 code\00_data\03_TFT'
RAW_DATA_PATH = r'C:\tongji\0 code\00_data\01_EDA'


# 加载数据
print("📂 加载数据...")



transform_info = pd.read_csv(DATA_PATH +os.sep+ '变换信息汇总.csv')

df_official_pretrain = pd.read_csv(RAW_DATA_PATH +os.sep+ 'temp_all_official_standart_time_type_pretrain.csv')# 无crisis官方运营日历
df_official_real = pd.read_csv(RAW_DATA_PATH +os.sep+ 'temp_all_official_standart_time_type.csv')# 真实官方运营日历
df_features = pd.read_csv(RAW_DATA_PATH +os.sep+ 'Time_Series_Features_2024-12-05 00 to 2025-11-27 00_15min.csv')# 真实数据集
df_raw = pd.read_csv(RAW_DATA_PATH +os.sep+ 'temp_all_standart_time_2cleaned.csv')# 真实原始数据



In [ ]:
# df_synthetic = pd.read_csv(DATA_PATH +os.sep+ 'df_synthetic_abnormal_v3.csv')
# df_synthetic_labels = pd.read_csv(DATA_PATH +os.sep+ 'df_synthetic_labels_v3.csv')

In [ ]:

# %%
df_normal = pd.read_csv(r'C:\tongji\0 code\00_data\02_stimulate\g_step6_inverse'+os.sep+'synthetic_physical.csv')
df_normal['timestamp'] = pd.to_datetime(df_normal['timestamp'] )
# ★ 注意: 不再 set_index, 保持 timestamp 为列
# df_normal = df_normal.set_index('timestamp')

crisis_df =  pd.read_csv(r'C:\tongji\0 code\00_data\01_EDA\0_crisis_event'+os.sep+'crisis_event_pool.csv')
# print('df_normal',df_normal.columns)
# abnormal_point = sorted(list(pd.to_datetime(df_official[df_official['category'] == "crisis"]['timestamp'])))[1:]
# crisis_events = sorted(list(df_official[df_official['category'] == "crisis"]['timestamp']))
# del crisis_events[4]
# del crisis_events[0:2]

# %%


In [ ]:
# df_synthetic['timestamp'] = pd.to_datetime(df_synthetic['timestamp'])
# df_synthetic_labels['timestamp'] = pd.to_datetime(df_synthetic_labels['timestamp'])
df_official_pretrain['timestamp'] = pd.to_datetime(df_official_pretrain['timestamp'])
df_official_real['timestamp'] = pd.to_datetime(df_official_real['timestamp'])
df_features.rename(columns={'Unnamed: 0': 'timestamp'}, inplace=True)
df_features['timestamp'] = pd.to_datetime(df_features['timestamp'])
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

In [ ]:
transform_info.drop(columns=['类型','原始偏度','变换偏度','偏度改善','状态','零值指示','规则覆盖'], inplace=True)

In [ ]:
# mask_normal = (df_synthetic_labels['is_changepoint'] == 0) & (df_synthetic_labels['is_anomaly'] == 0)
# df_synthetic_labels.loc[mask_normal, 'label'] = 'N'
# df_synthetic_labels.loc[df_synthetic_labels['is_anomaly'] ==1,'label'] = 'AP'
# df_synthetic_labels.loc[df_synthetic_labels['is_changepoint'] ==1,'label'] = 'CP'


In [ ]:
# 导入预训练官方事件
df_official_pretrain['event'] = df_official_pretrain['official_author']+' ' + df_official_pretrain['category']
# df_official_pretrain.drop(columns=['official_author','category'], inplace=True)
df_official_pretrain = df_official_pretrain[['event','timestamp']]
df_official_pretrain['timestamp'] = pd.to_datetime(df_official_pretrain['timestamp'])

In [ ]:
# 导入真实官方事件
df_official_real['event'] = df_official_real['official_author']+' ' + df_official_real['category']
# df_official_real.drop(columns=['official_author','category'], inplace=True)
df_official_real = df_official_real[['event','timestamp']]
df_official_real['timestamp'] = pd.to_datetime(df_official_real['timestamp'])

In [ ]:
# 导入真实特征集
# df_features['timestamp'] = pd.to_datetime(df_features.index)
df_features['comp_ratio_post'] = 1-df_features['comp_ratio_post']
df_features['comp_ratio_comment'] = 1-df_features['comp_ratio_comment']
df_features.rename(columns={'origin_ratio': 'retweet_ratio_post'}, inplace=True)
df_features = df_features.drop(['total_medium_post','total_medium_comment','unique_users_post', 'unique_users_comment', 'semantic_shift_cross'],axis=1) 

In [ ]:
print("✅ 数据加载完成")
print(f"  df_normal: {df_normal.shape}, "
      f"{df_normal['timestamp'].min()} ~ {df_normal['timestamp'].max()}")
print(f"  df_official_pretrain: {len(df_official_pretrain)} rows (无crisis)")
print(f"  df_official_real: {len(df_official_real)} rows (含crisis)")
print(f"  crisis_df: {len(crisis_df)} events")

In [ ]:
print(f"Training Data Info: {df_normal.info()}")
# print(f"Labels Distribution Info: {df_synthetic_labels.info()}")
print(f"Inference Data Info: {df_raw.info()}")



# TFT engine

## 导入

In [ ]:
# %% [markdown]
# # TFT + LightGBM 舆情异常检测
# ## 第一阶段：数据处理

# %% 
# 导入必要的库
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 导入自定义模块
from TFT_utils import load_config, setup_logger
from TFT_transform import (
    TFTDataProcessor, 
    FeatureTransformer,
    TFTDataset
)
from target_builder import TargetBuilder
from tft_full_period_utils import create_held_out_split, compute_baseline_from_held_out, verify_baseline_bias


# 设置日志
logger = setup_logger("TFT_Main")

# %%
# 加载配置
config = load_config("TFT_config.yaml")
print("配置加载成功")
print(f"跳过的变换方法: {config['data']['skip_transforms']}")
# print(f"静态属性映射: {config['data']['static_suffix_mapping']}")


## 验证

In [ ]:
# ==================== 确认 TFT_config.yaml 修改 ====================
import yaml

with open("TFT_config.yaml", 'r', encoding='utf-8') as f:
    config_raw = yaml.safe_load(f)

# 打印关键配置
print("=" * 60)
print("TFT 模型配置")
print("=" * 60)

# 目标变量
print("\n📌 目标变量:")
for mn, mc in config_raw.get('tft_models', {}).items():
    print(f"  [{mn}] target={mc.get('target')}, "
          f"encoder={mc.get('encoder_length')}, "
          f"decoder={mc.get('decoder_length')}")

# 特征列表
print("\n📌 时变未知实数变量:")
data_cfg = config_raw.get('data', {})
unknown_reals = data_cfg.get('time_varying_unknown_reals', [])
print(f"  共 {len(unknown_reals)} 个: {unknown_reals}")

# 已知实数变量
known_reals = data_cfg.get('time_varying_known_reals', [])
print(f"\n📌 时变已知实数变量: {known_reals}")

# skip_transforms
print(f"\n📌 skip_transforms: {data_cfg.get('skip_transforms', [])}")

# static 映射
print(f"\n📌 static_suffix_mapping: {data_cfg.get('static_suffix_mapping', {})}")

# TargetBuilder 相关
print(f"\n📌 当前 target_builder FEATURES 列表:")

print(f"  {TargetBuilder.FEATURES}")
print(f"  Z_CLIP = {TargetBuilder.Z_CLIP}")

## 数据处理

In [ ]:

processor = TFTDataProcessor(config)

tft_dataset = processor.fit_transform(
    df_synthetic=df_normal,
    transform_info=transform_info,
    df_official=df_official_pretrain       # ← 无 crisis 日历
)
df_full = tft_dataset.data.copy()


In [ ]:

tb = TargetBuilder(n_components_post=2, n_components_comment=2)
tb.fit(df_full)
df_full = tb.transform(df_full)

# 验证目标列已生成
for target_col in ['comment_pc1', 'post_pc2']:
    assert target_col in df_full.columns, f"❌ {target_col} 未生成!"
    print(f"✅ {target_col}: median={df_full[target_col].median():.4f}, "
          f"P95={df_full[target_col].quantile(0.95):.4f}")

# 打印loadings
tb.print_loadings(top_k=3)

### 检查

In [ ]:
# ==================== 全面确认检查 ====================
import yaml

with open("TFT_config.yaml", 'r', encoding='utf-8') as f:
    config_raw = yaml.safe_load(f)

print("=" * 70)
print("  分组PCA TargetBuilder 兼容性检查")
print("=" * 70)

# 1. TFT target 是否在 df_full 中
print("\n📌 TFT target 变量存在性:")
for mn, mc in config_raw.get('tft_models', {}).items():
    target = mc.get('target', '')
    exists = target in df_full.columns
    if exists:
        s = df_full[target]
        print(f"  ✅ [{mn}] target='{target}' | "
              f"median={s.median():.4f}, std={s.std():.4f}, "
              f"P5={s.quantile(0.05):.4f}, P95={s.quantile(0.95):.4f}")
    else:
        print(f"  ❌ [{mn}] target='{target}' 不存在!")

# 2. 分组PCA组件
print(f"\n📌 PCA 组件:")
print(f"  pca_post:    n_components={tb.pca_post.n_components_}, "
      f"features={len(tb.POST_FEATURES)}")
print(f"  pca_comment: n_components={tb.pca_comment.n_components_}, "
      f"features={len(tb.COMMENT_FEATURES)}")

# 3. 旧列名不存在（确认不再生成全局 pc{i}_score）
old_cols = [f'pc{i}_score' for i in range(1, 6)]
for c in old_cols:
    if c in df_full.columns:
        print(f"  ⚠️ 旧列 {c} 仍存在（应已移除）")
    else:
        print(f"  ✅ 旧列 {c} 已移除")

# 4. 新列名存在
new_cols_expected = ([f'post_pc{i}' for i in range(1, 6)] + 
                     [f'comment_pc{i}' for i in range(1, 6)])
for c in new_cols_expected:
    if c in df_full.columns:
        print(f"  ✅ {c} 存在")

# 5. comment_pc1 与 post_pc2 的相关性
corr = df_full['comment_pc1'].corr(df_full['post_pc2'])
print(f"\n📌 comment_pc1 ↔ post_pc2 相关性: {corr:.4f}")
if abs(corr) < 0.3:
    print("  ✅ 低相关性，两个TFT模型可以捕捉不同维度的异常")
else:
    print(f"  ⚠️ 相关性={corr:.4f}，较高，两个模型可能有冗余")

# 6. 基线统计
print(f"\n📌 基线统计:")
print(f"  quiet_energy_median = {tb.quiet_energy_median:.4f}")
print(f"  quiet_energy_mad    = {tb.quiet_energy_mad:.4f}")
print(f"  quiet_energy_p95    = {tb.quiet_energy_p95:.4f}")

# 7. Loadings
tb.print_loadings(top_k=3)

print(f"\n{'='*70}")
print("  ✅ 检查完成，如果没有 ❌，可以继续下一步")
print(f"{'='*70}")

## 切分

In [ ]:
# ==================== Step 3~6: Held-out + Baseline + 保存 ====================

from tft_full_period_utils import create_held_out_split, compute_baseline_from_held_out, verify_baseline_bias

# --- Step 3: 创建 Held-out ---
held_out_mask, held_mask, split_info = create_held_out_split(
        df_full,
        time_col='timestamp',
        block_days=4,  # Held-out长度（天）
        min_block_windows=96,   # ≥ 24h, TFT encoder 最小长度
    )
df_train = df_full[~held_mask].copy()
df_held_out = df_full[held_out_mask].copy()
held_out_time_indices = set(df_full.loc[held_mask, 'time_idx'].values)
print(f"训练集: {len(df_train)}, Held-out: {held_mask.sum()}")


In [ ]:
# 验证 held-out 和训练集无交集
train_times = set(df_train['time_idx'].values)

train_times = set(df_train['time_idx'].values)
overlap = train_times & held_out_time_indices
assert len(overlap) == 0, f"训练集和 held-out 有 {len(overlap)} 个重叠 time_idx!"
print(f"  ✅ 训练集和 held-out 无重叠")

In [ ]:
df_train.columns

# TFT train

## 干净数据训练

In [ ]:
from TFT_tft_engine import TFTEngine


In [ ]:


# --- Step 4: 训练 TFT ---
engine_PC = TFTEngine('TFT_config.yaml')

for model_name in engine_PC.config['tft_models']:

    print(f"  训练 [{model_name}]  ({len(df_train)} windows)")
    print(f"  target = {engine_PC.config['tft_models'][model_name]['target']}")
    print(f"  {'─'*50}")
    # if engine_PC.config['tft_models'][model_name]['target'] == "comment_pc1":
    #     continue
    # else:
    engine_PC.build_and_fit(
        model_name=model_name,
        df=df_train,
        max_epochs=300,
        quiet_end_idx=None
    )



## save

In [ ]:


# --- Step 6: 冻结 + 保存 ---
import pickle, os

SAVE_DIR = "./checkpoints/pretrained_planPC"

for model_name in engine_PC.models:
    engine_PC.freeze_layers(model_name)

engine_PC.save(f"{SAVE_DIR}/tft_engine")
tb.save(f"{SAVE_DIR}/target_builder.pkl")

os.makedirs(SAVE_DIR, exist_ok=True)
with open(f"{SAVE_DIR}/processor.pkl", 'wb') as f:
    pickle.dump(processor, f)

print(f"\n🎯 全部保存到: {SAVE_DIR}")


In [ ]:
# 保存 processor (部署时需要)
import pickle, os
proc_path = os.path.join(SAVE_DIR, "processor.pkl")
with open(proc_path, 'wb') as f:
    pickle.dump(processor, f)
print(f"  ✅ Processor saved to {proc_path}")

# 保存 split_info (可追溯)
info_path = os.path.join(SAVE_DIR, "split_info.pkl")
with open(info_path, 'wb') as f:
    pickle.dump(split_info, f)

## load

In [ ]:
from TFT_tft_engine import TFTEngine
import pickle, os


SAVE_DIR = "./checkpoints/pretrained_planB"
engine_PC=TFTEngine.load(f"{SAVE_DIR}/tft_engine",config_path="TFT_config.yaml")
print(f"加载的模型: {list(engine_PC.models.keys())}")


## 创建baseline

In [ ]:

# --- Step 5: 推理全量 + Held-out Baseline ---
results_PC = {}
for model_name in engine_PC.models:
    print(f"\n推理: {model_name}")
    res = engine_PC.analyze_rolling(model_name, df_full, baseline_end_idx=None)
    baseline = compute_baseline_from_held_out(res, held_out_time_indices)
    engine_PC.baselines[model_name] = baseline
    results_PC[model_name] = res
    print(f"  Baseline: residual μ={baseline['residual_mean']:.4f}, "
          f"σ={baseline['residual_std']:.4f}, P95={baseline['residual_p95']:.4f}")


In [ ]:
results_PC

In [ ]:
print(f"\n{'='*70}")
print(f"  ✅ 全周期TFT训练完成")
print(f"{'='*70}")
print(f"  TFT模型数:     {len(engine_PC.models)}")
for mn in engine_PC.models:
      bl = engine_PC.baselines.get(mn, {})
      print(f"    [{mn}] baseline σ={bl.get('residual_std', '?'):.4f} "
            f"(n={bl.get('n_samples', '?')}, source={bl.get('source', '?')})")
print(f"  Held-out:       {split_info['n_held_out']} windows "
      f"({split_info['held_out_ratio']:.1%})")
# print(f"  全量重训:       {'是' if retrain_on_full else '否'}")
print(f"  保存路径:       {SAVE_DIR}")
print(f"  Processor:      global_min_time={processor.global_min_time}")
print(f"  TargetBuilder:  quiet_energy_p95={tb.quiet_energy_p95:.4f}")
print(f"{'='*70}")


## 验证

In [ ]:
print("\n验证 engine_PC:")
for mn in engine_PC.models:
    model = engine_PC.models[mn]
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  [{mn}] trainable={n_trainable}/{n_total} "
          f"({n_trainable/n_total:.1%})")
    
    bl = engine_PC.baselines[mn]
    print(f"         baseline: n={bl['n_samples']}, "
          f"residual_std={bl['residual_std']:.4f}")

# 确认 processor_full 的 time_idx 基准与原始一致
print(f"\n  processor global_min_time: {processor.global_min_time}")
print(f"  df_normal min timestamp:   {df_normal['timestamp'].min()}")


### 1

In [ ]:
# ============================================================
# 对比分析：TFT 在干净数据 vs 真实数据上的表现
# ============================================================
# 前提：engine_PC, tb, processor, results_PC 已就绪（干净数据的推理结果）
# df_features: 真实数据, df_normal: 干净数据

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

print("=" * 70)
print("  Phase 1: 真实数据 TFT 推理")
print("=" * 70)

# 1. 处理真实数据（沿用已有processor，只重fit feature_transformer）
start_time_real = pd.to_datetime('2025-02-18 00:00:00')
warmup_windows = 960

# 用冷启动期数据重fit（与正式检测一致）
fit_end = start_time_real + pd.Timedelta(minutes=15 * warmup_windows)
df_warmup_fit = df_features[
    (df_features['timestamp'] >= start_time_real) &
    (df_features['timestamp'] < fit_end)
].copy()

import copy
processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_warmup_fit)

tft_ds_real = processor_real.transform(df_features, df_official_real)
df_real_transformed = tft_ds_real.data.copy()
df_real_transformed.fillna(0, inplace=True)
df_real_transformed.replace([np.inf, -np.inf], 0, inplace=True)
df_real_transformed = tb.transform(df_real_transformed)

print(f"真实数据: {df_real_transformed.shape}, "
      f"{df_real_transformed['timestamp'].min()} ~ {df_real_transformed['timestamp'].max()}")

# 2. 推理真实数据
results_real = {}
for mn in engine_PC.models:
    print(f"  推理真实数据: {mn}")
    res = engine_PC.analyze_rolling(mn, df_real_transformed, baseline_end_idx=None)
    results_real[mn] = res
    print(f"    metrics shape: {res['metrics'].shape}")

# 3. 干净数据推理结果已在 results_PC 中
print(f"\n干净数据推理结果: {list(results_PC.keys())}")
for mn in results_PC:
    print(f"  [{mn}] metrics shape: {results_PC[mn]['metrics'].shape}")

两个模型捕捉不同尺度的模式：comment_pc1 用较短上下文（ATT=108）捕捉评论域的短期突变，post_pc2 用较长上下文（ATT=192）捕捉帖子域的趋势偏移。这意味着异常注入策略应至少包含两类形态——脉冲型（短时异常点）和漂移型（持续性变点）。

In [ ]:
# ============================================================
# Phase 2: 残差分布对比（干净 vs 真实）
# ============================================================

fig, axes = plt.subplots(len(engine_PC.models), 3, figsize=(20, 5 * len(engine_PC.models)))
if len(engine_PC.models) == 1:
    axes = axes[np.newaxis, :]

for i, mn in enumerate(engine_PC.models):
    bl = engine_PC.baselines[mn]
    target = engine_PC.config['tft_models'][mn]['target']
    
    res_clean = results_PC[mn]['metrics']
    res_real = results_real[mn]['metrics']
    
    resid_clean = res_clean['residual'].values
    resid_real = res_real['residual'].values
    
    # --- 2a. 残差直方图 ---
    ax = axes[i, 0]
    ax.hist(resid_clean, bins=100, alpha=0.5, density=True, label='干净数据', color='steelblue')
    ax.hist(resid_real, bins=100, alpha=0.5, density=True, label='真实数据', color='coral')
    ax.axvline(bl['residual_p95'], color='r', ls='--', alpha=0.5, label=f"P95={bl['residual_p95']:.3f}")
    ax.axvline(-bl['residual_p95'], color='r', ls='--', alpha=0.5)
    ax.set_title(f'{mn} ({target})\n残差分布对比')
    ax.legend(fontsize=8)
    ax.set_xlabel('Residual')
    
    # --- 2b. 残差绝对值时序（真实数据）---
    ax = axes[i, 1]
    time_idx_real = res_real['time_idx'].values
    resid_abs_real = np.abs(resid_real)
    ax.plot(time_idx_real, resid_abs_real, linewidth=0.3, alpha=0.6, color='coral')
    ax.axhline(bl['residual_p95'], color='r', ls='--', alpha=0.5, label='baseline P95')
    ax.axhline(bl['residual_p95'] * 2, color='darkred', ls=':', alpha=0.5, label='2× P95')
    
    # 标出超阈值区域
    exceed_mask = resid_abs_real > bl['residual_p95'] * 1.5
    ax.scatter(time_idx_real[exceed_mask], resid_abs_real[exceed_mask], 
               c='red', s=3, alpha=0.8, zorder=5, label=f'exceed 1.5×P95 (n={exceed_mask.sum()})')
    ax.set_title(f'{mn}: 真实数据 |residual| 时序')
    ax.legend(fontsize=7)
    ax.set_xlabel('time_idx')
    
    # --- 2c. QQ-plot ---
    ax = axes[i, 2]
    clean_sorted = np.sort(np.abs(resid_clean))
    real_sorted = np.sort(np.abs(resid_real))
    n_min = min(len(clean_sorted), len(real_sorted))
    q_clean = np.quantile(clean_sorted, np.linspace(0, 1, 200))
    q_real = np.quantile(real_sorted, np.linspace(0, 1, 200))
    ax.scatter(q_clean, q_real, s=5, alpha=0.6)
    max_val = max(q_clean.max(), q_real.max())
    ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='y=x')
    ax.set_xlabel('干净数据 |residual| quantile')
    ax.set_ylabel('真实数据 |residual| quantile')
    ax.set_title(f'{mn}: QQ-Plot (|residual|)')
    ax.legend()

plt.suptitle('TFT残差：干净数据 vs 真实数据', fontsize=14)
plt.tight_layout()
plt.savefig('./output/compare_residual_clean_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- 数值汇总 ----
print("\n📊 残差统计汇总:")
print(f"{'Model':30s} | {'数据':6s} | {'mean':>8s} | {'std':>8s} | {'P50':>8s} | {'P95':>8s} | {'P99':>8s} | {'>1.5×P95':>10s}")
print("-" * 100)
for mn in engine_PC.models:
    bl = engine_PC.baselines[mn]
    for label, res in [('干净', results_PC[mn]), ('真实', results_real[mn])]:
        r = np.abs(res['metrics']['residual'].values)
        exceed = (r > bl['residual_p95'] * 1.5).sum()
        print(f"{mn:30s} | {label:6s} | {r.mean():8.4f} | {r.std():8.4f} | "
              f"{np.median(r):8.4f} | {np.quantile(r, 0.95):8.4f} | "
              f"{np.quantile(r, 0.99):8.4f} | {exceed:>10d}")

In [ ]:
# ============================================================
# Phase 3: 识别真实数据中的异常区间
# ============================================================

def detect_anomaly_segments(res_metrics, baseline, model_name,
                            k_threshold=1.5, min_consec=2, merge_gap=4):
    """
    从TFT残差中检测异常段
    k_threshold: P95的倍数
    min_consec: 最少连续超阈值窗口数
    merge_gap: 合并间距（窗口数）
    """
    resid_abs = np.abs(res_metrics['residual'].values)
    time_idx = res_metrics['time_idx'].values
    threshold = baseline['residual_p95'] * k_threshold
    
    exceed = resid_abs > threshold
    
    # 找连续段
    segments = []
    in_seg = False
    seg_start = 0
    gap_count = 0
    
    for i in range(len(exceed)):
        if exceed[i]:
            if not in_seg:
                seg_start = i
                in_seg = True
            gap_count = 0
        else:
            if in_seg:
                gap_count += 1
                if gap_count > merge_gap:
                    seg_end = i - gap_count
                    if seg_end - seg_start + 1 >= min_consec:
                        segments.append({
                            'start_idx': int(time_idx[seg_start]),
                            'end_idx': int(time_idx[seg_end]),
                            'length': seg_end - seg_start + 1,
                            'max_residual': float(resid_abs[seg_start:seg_end+1].max()),
                            'mean_residual': float(resid_abs[seg_start:seg_end+1].mean()),
                            'ratio_to_p95': float(resid_abs[seg_start:seg_end+1].max() / baseline['residual_p95']),
                        })
                    in_seg = False
    
    # 处理末尾
    if in_seg:
        seg_end = len(exceed) - 1
        if seg_end - seg_start + 1 >= min_consec:
            segments.append({
                'start_idx': int(time_idx[seg_start]),
                'end_idx': int(time_idx[seg_end]),
                'length': seg_end - seg_start + 1,
                'max_residual': float(resid_abs[seg_start:seg_end+1].max()),
                'mean_residual': float(resid_abs[seg_start:seg_end+1].mean()),
                'ratio_to_p95': float(resid_abs[seg_start:seg_end+1].max() / baseline['residual_p95']),
            })
    
    return pd.DataFrame(segments)


# 对每个模型检测异常段
all_segments = {}
for mn in engine_PC.models:
    bl = engine_PC.baselines[mn]
    target = engine_PC.config['tft_models'][mn]['target']
    
    segs = detect_anomaly_segments(
        results_real[mn]['metrics'], bl, mn,
        k_threshold=1.5, min_consec=3, merge_gap=4
    )
    all_segments[mn] = segs
    
    print(f"\n📌 [{mn}] target={target}")
    print(f"   检测到 {len(segs)} 个异常段")
    if len(segs) > 0:
        print(segs.to_string(index=False))

# 把 time_idx 转回 timestamp
print("\n\n📌 异常区间时间映射:")
for mn, segs in all_segments.items():
    if len(segs) == 0:
        continue
    print(f"\n  [{mn}]")
    for _, seg in segs.iterrows():
        # 从 df_real_transformed 中查找对应 timestamp
        ts_start = df_real_transformed.loc[
            df_real_transformed['time_idx'] == seg['start_idx'], 'timestamp'
        ]
        ts_end = df_real_transformed.loc[
            df_real_transformed['time_idx'] == seg['end_idx'], 'timestamp'
        ]
        ts_s = ts_start.values[0] if len(ts_start) > 0 else '?'
        ts_e = ts_end.values[0] if len(ts_end) > 0 else '?'
        duration_h = seg['length'] * 15 / 60
        print(f"    {ts_s} ~ {ts_e} | "
              f"持续{duration_h:.1f}h ({seg['length']}窗口) | "
              f"max_ratio={seg['ratio_to_p95']:.2f}×P95")

In [ ]:
# ============================================================
# Phase 4: 异常区间的形态分析（用于反推注入策略）
# ============================================================

def analyze_anomaly_morphology(df_transformed, res_metrics, segments_df, 
                                baseline, tb, model_name):
    """
    分析每个异常段的特征形态：
    - 哪些原始特征驱动了异常？
    - 是尖峰(spike)还是持续偏移(shift)？
    - 是单特征还是多特征协同？
    """
    if len(segments_df) == 0:
        return []
    
    results = []
    
    for seg_idx, seg in segments_df.iterrows():
        t_start = seg['start_idx']
        t_end = seg['end_idx']
        
        # 取异常段数据
        seg_mask = (df_transformed['time_idx'] >= t_start) & \
                   (df_transformed['time_idx'] <= t_end)
        df_seg = df_transformed[seg_mask]
        
        # 取异常段前一个等长的"对照"段
        seg_len = t_end - t_start + 1
        pre_start = max(t_start - seg_len * 2, df_transformed['time_idx'].min())
        pre_end = t_start - 1
        pre_mask = (df_transformed['time_idx'] >= pre_start) & \
                   (df_transformed['time_idx'] <= pre_end)
        df_pre = df_transformed[pre_mask]
        
        if len(df_seg) == 0 or len(df_pre) == 0:
            continue
        
        # ---- 4a. Z-score 驱动分析 ----
        z_drivers = {}
        for feat, p in tb.z_params.items():
            if feat not in df_seg.columns:
                continue
            z_seg = np.abs((df_seg[feat].values - p['median']) / (p['scale'] + 1e-8))
            z_pre = np.abs((df_pre[feat].values - p['median']) / (p['scale'] + 1e-8))
            
            z_drivers[feat] = {
                'seg_mean_z': float(np.mean(z_seg)),
                'seg_max_z': float(np.max(z_seg)),
                'pre_mean_z': float(np.mean(z_pre)),
                'z_increase': float(np.mean(z_seg) - np.mean(z_pre)),
                'z_ratio': float(np.mean(z_seg) / (np.mean(z_pre) + 1e-8)),
            }
        
        # 排序：按z_increase降序
        top_drivers = sorted(z_drivers.items(), 
                            key=lambda x: x[1]['z_increase'], reverse=True)[:5]
        
        # ---- 4b. 形态判断 ----
        # 取残差时序
        resid_seg = res_metrics[
            (res_metrics['time_idx'] >= t_start) & 
            (res_metrics['time_idx'] <= t_end)
        ]['residual'].abs().values
        
        if len(resid_seg) < 3:
            morphology = 'single_spike'
        else:
            # 尖峰 vs 持续：看峰值与均值的比
            peak_to_mean = resid_seg.max() / (resid_seg.mean() + 1e-8)
            # 趋势：拟合斜率
            slope = np.polyfit(range(len(resid_seg)), resid_seg, 1)[0] if len(resid_seg) > 2 else 0
            # 波动：std/mean
            cv = resid_seg.std() / (resid_seg.mean() + 1e-8)
            
            if seg['length'] <= 4 and peak_to_mean > 2.5:
                morphology = 'sharp_spike'
            elif abs(slope) > 0.01 * resid_seg.mean():
                morphology = 'gradual_drift' if slope > 0 else 'gradual_recovery'
            elif cv < 0.3:
                morphology = 'sustained_shift'
            elif cv > 0.8:
                morphology = 'volatility_burst'
            else:
                morphology = 'mixed'
        
        # ---- 4c. 单特征 vs 多特征 ----
        n_significant_drivers = sum(1 for _, d in top_drivers if d['z_increase'] > 1.0)
        
        # ---- 4d. Post vs Comment 主导 ----
        post_z = sum(d['z_increase'] for f, d in top_drivers if '_post' in f)
        comment_z = sum(d['z_increase'] for f, d in top_drivers if '_comment' in f)
        dominant_channel = 'post' if post_z > comment_z * 1.5 else \
                          ('comment' if comment_z > post_z * 1.5 else 'both')
        
        # 汇总
        ts_start = df_seg['timestamp'].iloc[0] if 'timestamp' in df_seg.columns else t_start
        ts_end = df_seg['timestamp'].iloc[-1] if 'timestamp' in df_seg.columns else t_end
        
        result = {
            'segment_id': seg_idx,
            'time_start': str(ts_start),
            'time_end': str(ts_end),
            'duration_h': seg['length'] * 15 / 60,
            'duration_windows': seg['length'],
            'max_ratio_p95': seg['ratio_to_p95'],
            'morphology': morphology,
            'n_drivers': n_significant_drivers,
            'dominant_channel': dominant_channel,
            'top1_feature': top_drivers[0][0] if top_drivers else '?',
            'top1_z_increase': top_drivers[0][1]['z_increase'] if top_drivers else 0,
            'top2_feature': top_drivers[1][0] if len(top_drivers) > 1 else '?',
            'top2_z_increase': top_drivers[1][1]['z_increase'] if len(top_drivers) > 1 else 0,
            'top3_feature': top_drivers[2][0] if len(top_drivers) > 2 else '?',
            'top3_z_increase': top_drivers[2][1]['z_increase'] if len(top_drivers) > 2 else 0,
        }
        
        # 添加关键残差形态数值
        if len(resid_seg) > 2:
            result['peak_to_mean'] = float(resid_seg.max() / (resid_seg.mean() + 1e-8))
            result['resid_cv'] = float(resid_seg.std() / (resid_seg.mean() + 1e-8))
            result['resid_slope'] = float(np.polyfit(range(len(resid_seg)), resid_seg, 1)[0])
        
        results.append(result)
    
    return results


# 运行形态分析
print("=" * 70)
print("  Phase 4: 异常区间形态分析")
print("=" * 70)

all_morphology = {}
for mn in engine_PC.models:
    segs = all_segments[mn]
    if len(segs) == 0:
        continue
    
    morph = analyze_anomaly_morphology(
        df_real_transformed, results_real[mn]['metrics'],
        segs, engine_PC.baselines[mn], tb, mn
    )
    all_morphology[mn] = pd.DataFrame(morph)
    
    print(f"\n{'='*60}")
    print(f"  [{mn}] 异常形态汇总")
    print(f"{'='*60}")
    df_morph = all_morphology[mn]
    print(df_morph[['time_start', 'time_end', 'duration_h', 'morphology', 
                     'n_drivers', 'dominant_channel', 'max_ratio_p95',
                     'top1_feature', 'top1_z_increase']].to_string(index=False))
    
    print(f"\n  形态分布:")
    print(f"  {df_morph['morphology'].value_counts().to_dict()}")
    print(f"  主导通道:")
    print(f"  {df_morph['dominant_channel'].value_counts().to_dict()}")
    print(f"  驱动特征数 (median): {df_morph['n_drivers'].median():.1f}")

In [ ]:
# ============================================================
# Phase 5: 注意力权重 & VSN 权重在异常区间的变化
# ============================================================

from scipy.stats import spearmanr

for mn in engine_PC.models:
    bl = engine_PC.baselines[mn]
    att_base = bl['att_mean']
    vsn_base = bl['vsn_mean']
    
    att_real = results_real[mn]['attention']
    vsn_real = results_real[mn]['vsn']
    metrics_real = results_real[mn]['metrics']
    
    segs = all_segments[mn]
    if len(segs) == 0:
        print(f"[{mn}] 无异常段，跳过")
        continue
    
    print(f"\n{'='*60}")
    print(f"  [{mn}] 注意力 & VSN 异常期偏移分析")
    print(f"{'='*60}")
    
    # 全局时序的KL散度
    n = len(att_real)
    att_kl_series = np.zeros(n)
    vsn_js_series = np.zeros(n)
    vsn_rank_shift_series = np.zeros(n)
    
    baseline_rank = np.argsort(np.argsort(-np.abs(vsn_base.flatten())))
    
    for i in range(n):
        # Attention KL
        p = att_real[i].flatten()
        q = att_base.flatten()
        p = p / (p.sum() + 1e-10) + 1e-10
        q = q / (q.sum() + 1e-10) + 1e-10
        p = p / p.sum()
        q = q / q.sum()
        att_kl_series[i] = np.sum(p * np.log(p / q))
        
        # VSN JS
        v_i = np.abs(vsn_real[i].flatten()) + 1e-10
        v_b = np.abs(vsn_base.flatten()) + 1e-10
        p_v = v_i / v_i.sum()
        q_v = v_b / v_b.sum()
        m_v = 0.5 * (p_v + q_v)
        vsn_js_series[i] = 0.5 * np.sum(p_v * np.log(p_v / m_v)) + \
                           0.5 * np.sum(q_v * np.log(q_v / m_v))
        
        # VSN rank shift
        curr_rank = np.argsort(np.argsort(-np.abs(vsn_real[i].flatten())))
        corr, _ = spearmanr(baseline_rank, curr_rank)
        vsn_rank_shift_series[i] = 1 - corr if not np.isnan(corr) else 1.0
    
    # 可视化
    time_idx = metrics_real['time_idx'].values
    fig, axes = plt.subplots(4, 1, figsize=(18, 14), sharex=True)
    
    # 残差
    axes[0].plot(time_idx, np.abs(metrics_real['residual'].values), 
                 linewidth=0.3, alpha=0.6, color='coral')
    axes[0].axhline(bl['residual_p95'], color='r', ls='--', alpha=0.3)
    axes[0].set_ylabel('|Residual|')
    axes[0].set_title(f'{mn}: 真实数据 TFT 信号全景')
    
    # Attention KL
    axes[1].plot(time_idx, att_kl_series, linewidth=0.3, alpha=0.6, color='purple')
    axes[1].set_ylabel('Attention KL')
    
    # VSN JS
    axes[2].plot(time_idx, vsn_js_series, linewidth=0.3, alpha=0.6, color='green')
    axes[2].set_ylabel('VSN JS-div')
    
    # VSN rank shift
    axes[3].plot(time_idx, vsn_rank_shift_series, linewidth=0.3, alpha=0.6, color='brown')
    axes[3].set_ylabel('VSN Rank Shift')
    axes[3].set_xlabel('time_idx')
    
    # 在所有子图上标注异常段
    for _, seg in segs.iterrows():
        for ax in axes:
            ax.axvspan(seg['start_idx'], seg['end_idx'], 
                      alpha=0.15, color='red', zorder=0)
    
    plt.tight_layout()
    plt.savefig(f'./output/compare_tft_signals_{mn}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # 数值对比：异常段 vs 正常段
    normal_mask_arr = np.ones(n, dtype=bool)
    for _, seg in segs.iterrows():
        seg_m = (time_idx >= seg['start_idx']) & (time_idx <= seg['end_idx'])
        normal_mask_arr &= ~seg_m
    
    anom_mask_arr = ~normal_mask_arr
    
    print(f"\n  信号对比 (正常段 vs 异常段):")
    print(f"  {'Signal':25s} | {'Normal_P50':>10s} | {'Normal_P95':>10s} | {'Anom_P50':>10s} | {'Anom_P95':>10s} | {'Ratio':>8s}")
    print(f"  {'-'*80}")
    for name, arr in [('|Residual|', np.abs(metrics_real['residual'].values)),
                       ('Att_KL', att_kl_series),
                       ('VSN_JS', vsn_js_series),
                       ('VSN_RankShift', vsn_rank_shift_series)]:
        n_p50 = np.median(arr[normal_mask_arr])
        n_p95 = np.quantile(arr[normal_mask_arr], 0.95)
        a_p50 = np.median(arr[anom_mask_arr]) if anom_mask_arr.sum() > 0 else 0
        a_p95 = np.quantile(arr[anom_mask_arr], 0.95) if anom_mask_arr.sum() > 0 else 0
        ratio = a_p50 / (n_p50 + 1e-10)
        print(f"  {name:25s} | {n_p50:10.4f} | {n_p95:10.4f} | {a_p50:10.4f} | {a_p95:10.4f} | {ratio:8.2f}×")

In [ ]:
# ============================================================
# Phase 6: 异常区间 vs 已知 crisis 事件对齐
# ============================================================

print("=" * 70)
print("  Phase 6: 异常段与已知crisis事件的对齐")
print("=" * 70)

# crisis事件时间
crisis_events = []
for _, row in crisis_df.iterrows():
    crisis_events.append({
        'name': row.get('crisis_name', '?'),
        'level': row.get('crisis_level', '?'),
        'start': pd.to_datetime(row['t_crisis_start']),
        'peak': pd.to_datetime(row['t_crisis_peak']),
        'end': pd.to_datetime(row['t_crisis_end']),
        'resp': pd.to_datetime(row['t_official_resp']),
    })

for mn in engine_PC.models:
    morph_df = all_morphology.get(mn)
    if morph_df is None or len(morph_df) == 0:
        continue
    
    print(f"\n  [{mn}]")
    for _, seg in morph_df.iterrows():
        seg_start = pd.to_datetime(seg['time_start'])
        seg_end = pd.to_datetime(seg['time_end'])
        
        # 查找重叠的crisis事件
        matched = []
        for evt in crisis_events:
            # 有重叠
            if seg_start <= evt['end'] and seg_end >= evt['start']:
                overlap_start = max(seg_start, evt['start'])
                overlap_end = min(seg_end, evt['end'])
                overlap_h = (overlap_end - overlap_start).total_seconds() / 3600
                matched.append(f"{evt['name']}(L{evt['level']}, overlap={overlap_h:.1f}h)")
        
        match_str = ', '.join(matched) if matched else '❓ 未匹配已知事件'
        
        print(f"    {seg['time_start']} ~ {seg['time_end']} "
              f"| {seg['duration_h']:.1f}h | {seg['morphology']:18s} "
              f"| {seg['dominant_channel']:7s} | {seg['top1_feature']:25s} "
              f"| → {match_str}")

In [ ]:
# ============================================================
# Phase 7: 反推注入策略优化建议
# ============================================================

print("=" * 70)
print("  Phase 7: 注入策略优化建议（基于真实数据形态）")
print("=" * 70)

# 汇总所有模型的形态
all_morph_combined = pd.concat(all_morphology.values(), ignore_index=True)

if len(all_morph_combined) == 0:
    print("⚠️ 未检测到异常段，可能原因:")
    print("  1. 阈值设太高 → 降低 k_threshold")
    print("  2. TFT 未收敛 → 检查训练loss")
    print("  3. 真实数据在训练期确实平稳")
else:
    # ---- 7a. 形态分布 ----
    print("\n📊 真实异常形态分布:")
    morph_counts = all_morph_combined['morphology'].value_counts()
    for m, c in morph_counts.items():
        pct = c / len(all_morph_combined) * 100
        print(f"  {m:20s}: {c:3d} ({pct:.1f}%)")
    
    # ---- 7b. 持续时间分布 ----
    print(f"\n📊 持续时间统计:")
    dur = all_morph_combined['duration_h']
    print(f"  P25={dur.quantile(0.25):.1f}h, P50={dur.median():.1f}h, "
          f"P75={dur.quantile(0.75):.1f}h, P95={dur.quantile(0.95):.1f}h")
    
    # ---- 7c. 驱动特征频率 ----
    print(f"\n📊 Top驱动特征频率:")
    feat_freq = {}
    for col in ['top1_feature', 'top2_feature', 'top3_feature']:
        for f in all_morph_combined[col]:
            if f != '?':
                feat_freq[f] = feat_freq.get(f, 0) + 1
    for f, c in sorted(feat_freq.items(), key=lambda x: -x[1])[:10]:
        print(f"  {f:30s}: {c}次")
    
    # ---- 7d. 通道主导性 ----
    print(f"\n📊 通道主导性:")
    ch_counts = all_morph_combined['dominant_channel'].value_counts()
    for ch, c in ch_counts.items():
        print(f"  {ch:10s}: {c} ({c/len(all_morph_combined)*100:.1f}%)")
    
    # ---- 7e. 具体建议 ----
    print(f"\n{'='*60}")
    print("  📌 注入策略优化建议")
    print(f"{'='*60}")
    
    # 建议1: 形态匹配
    if morph_counts.get('sustained_shift', 0) > morph_counts.get('sharp_spike', 0):
        print("\n  1️⃣ [形态] 真实异常以「持续偏移」为主，而非尖峰")
        print("     → 增加 mean_changepoint 的注入比例")
        print("     → 降低 positive_spike 的magnitude，增加 duration")
        print(f"     → 建议 duration 范围: {max(4, int(dur.median()*4))}~{max(16, int(dur.quantile(0.75)*4))} 窗口")
    elif morph_counts.get('sharp_spike', 0) > 0:
        print("\n  1️⃣ [形态] 真实异常包含尖峰事件")
        print("     → 保持 positive_spike / event_shock 的注入")
        print("     → 确保 magnitude 足够大（参考 ratio_to_p95）")
    
    if morph_counts.get('volatility_burst', 0) > 0:
        print("\n  2️⃣ [波动] 检测到波动率爆发形态")
        print("     → 增加 volatility_burst 的注入频率")
        print("     → volatility_multiplier 建议: 2.0~4.0")
    
    if morph_counts.get('gradual_drift', 0) > 0:
        print("\n  3️⃣ [趋势] 检测到渐变漂移")
        print("     → 增加 gradual_drift 类注入（当前C_METHOD_POOL可能缺少）")
        print("     → 考虑在 C_METHOD_POOL 中加回 'gradual_drift'")
    
    # 建议2: 特征覆盖
    top_real_feats = [f for f, _ in sorted(feat_freq.items(), key=lambda x: -x[1])[:5]]
    current_inject_feats = set(traffic_features + sentiment_features + 
                               semantic_features + concentration_features)
    uncovered = [f for f in top_real_feats if f not in current_inject_feats]
    
    if uncovered:
        print(f"\n  4️⃣ [特征] 真实异常驱动特征中有未覆盖的:")
        for f in uncovered:
            print(f"     → {f} (出现{feat_freq[f]}次)")
        print("     → 在注入特征池中补充这些特征")
    
    # 建议3: 通道比例
    post_pct = ch_counts.get('post', 0) / len(all_morph_combined)
    comment_pct = ch_counts.get('comment', 0) / len(all_morph_combined)
    both_pct = ch_counts.get('both', 0) / len(all_morph_combined)
    
    print(f"\n  5️⃣ [通道] post={post_pct:.0%}, comment={comment_pct:.0%}, both={both_pct:.0%}")
    if both_pct > 0.3:
        print("     → 真实异常多为 post+comment 协同")
        print("     → 增加 correlated_anomaly 的注入比例")
        print("     → 确保协同注入同时影响 post 和 comment 特征")
    
    # 建议4: 强度校准
    real_ratio = all_morph_combined['max_ratio_p95'].median()
    print(f"\n  6️⃣ [强度] 真实异常的 median ratio_to_P95 = {real_ratio:.2f}×")
    if real_ratio < 2.0:
        print("     → 真实异常强度温和，当前 magnitude_multiplier=2.0 可能偏大")
        print(f"     → 建议 magnitude_multiplier={max(1.0, real_ratio * 0.8):.1f}~{real_ratio * 1.2:.1f}")
    elif real_ratio > 5.0:
        print("     → 真实异常强度很大，当前注入可能不够")
        print(f"     → 建议 magnitude_multiplier={real_ratio * 0.6:.1f}~{real_ratio * 0.8:.1f}")
    else:
        print(f"     → 当前 magnitude_multiplier=2.0 大致匹配")

print(f"\n{'='*70}")
print("  分析完成。根据以上建议调整注入参数后重新生成25场景训练集。")
print(f"{'='*70}")

In [ ]:
# ============================================================
# Phase 8: 总览可视化（干净 vs 真实 PC Score 时序对比）
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(22, 12), sharex='col')

# 左列: 干净数据 | 右列: 真实数据
titles = [('干净数据 (df_normal)', '真实数据 (df_features)')]

for col_idx, (df_data, res_data, title) in enumerate([
    (df_full, results_PC, '干净数据'),
    (df_real_transformed, results_real, '真实数据'),
]):
    # PC targets
    for row_idx, pc_col in enumerate(['comment_pc1', 'post_pc2']):
        if pc_col not in df_data.columns:
            continue
        ax = axes[row_idx, col_idx]
        ax.plot(df_data['time_idx'].values, df_data[pc_col].values,
                linewidth=0.3, alpha=0.6)
        
        p5 = df_data[pc_col].quantile(0.05)
        p95 = df_data[pc_col].quantile(0.95)
        ax.axhline(p95, color='r', ls=':', alpha=0.3)
        ax.axhline(p5, color='b', ls=':', alpha=0.3)
        ax.set_ylabel(pc_col)
        ax.set_title(f'{title} — {pc_col}')
    
    # feature_energy
    ax = axes[2, col_idx]
    ax.plot(df_data['time_idx'].values, df_data['time_idx'].values,
            linewidth=0.3, alpha=0.6, color='green')
    ax.axhline(tb.quiet_energy_p95, color='r', ls='--', alpha=0.3, label='quiet P95')
    ax.set_ylabel('feature_energy')
    ax.set_title(f'{title} — feature_energy')
    ax.legend(fontsize=7)
    ax.set_xlabel('time_idx')

    # 标注异常段（仅真实数据列）
    if col_idx == 1:
        for mn, segs in all_segments.items():
            for _, seg in segs.iterrows():
                for ax in axes[:, col_idx]:
                    ax.axvspan(seg['start_idx'], seg['end_idx'],
                              alpha=0.1, color='red')

plt.suptitle('干净数据 vs 真实数据: PC Score & Feature Energy 对比', fontsize=14)
plt.tight_layout()
plt.savefig('./output/compare_clean_vs_real_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## load

# 引入异常注入器

## 定义注入器

### 定义注入异常方式

### 方法辅助函数

### 从事件池注入

### 额外注入

## 注入流程

## 验证

## 多世界注入

## 可视化

# 诊断

# 分类器构造

## 因果时序特征工程

In [ ]:
# ==================== 单元格: 因果时序特征工程（PC Score 版 v2 精简）====================

def build_causal_features_pc(df, 
                              pc_targets,
                              tb,
                              time_col='time_idx',
                              baseline_stats=None):
    """
    基于 PC Score 的因果特征构造（精简版）
    
    精简依据（真实数据形态分析）：
    - 真实异常 P50=1.2h(5窗口), P75=1.8h(7窗口), P95=3.8h(15窗口)
    - comment_pc1: 267段, 主驱动=neg_ratio_comment, 通道=comment 45%
    - post_pc2: 27段, 主驱动=vis_concentration_post, 通道=post 78%
    - 两模型低相关, 合成energy会稀释信号 → 保留但不过度衍生
    
    精简内容：
    1. 去掉 {pc}_raw / {pc}_above_p95 / {pc}_below_p05（无物理意义/信息量低）
    2. energy rolling 从5尺度减为3尺度 [4,16,32]（去掉k=8冗余, k=96过长）
    3. 去掉 PC raw rolling，改为 PC zscore rolling [8,32]
    4. 去掉 energy_slope（与energy_trend_8冗余且慢）
    5. 去掉 energy_accel（短窗口噪声大，gradual形态不适用）
    6. 去掉 above_p95_density（与above_ratio完全重复）
    7. energy_tstat/mean_ratio 从6个减为1个(tstat_32)
    8. above_2x_density 从4个减为1个(k=16)
    9. PC diff 改为基于 zscore
    
    Parameters
    ----------
    df : DataFrame, 必须包含 time_col 和 pc_targets 列
    pc_targets : list[str], e.g. ['comment_pc1', 'post_pc2']
    tb : TargetBuilder, 用于获取 Z_CLIP 等参数
    baseline_stats : dict or None, 各 PC 的基线统计；None 时从 df 推断
    """
    df = df.sort_values(time_col).copy()
    X = pd.DataFrame(index=df.index)
    
    # ====== 0. 准备 baseline 统计量 ======
    if baseline_stats is None:
        baseline_stats = {}
        for pc in pc_targets:
            if pc in df.columns:
                vals = df[pc].dropna()
                baseline_stats[pc] = {
                    'median': float(vals.median()),
                    'std': float(vals.std()),
                    'mad': float((vals - vals.median()).abs().median()) * 1.4826,
                    'p95': float(vals.quantile(0.95)),
                    'p05': float(vals.quantile(0.05)),
                }
    
    # ====== 1. 合成 energy（跨 PC 的综合异常度）======
    z_clip = tb.Z_CLIP if hasattr(tb, 'Z_CLIP') else 10
    z_scores_all = pd.DataFrame(index=df.index)
    for pc in pc_targets:
        if pc not in df.columns:
            continue
        bs = baseline_stats[pc]
        scale = bs['mad'] if bs['mad'] > 1e-8 else bs['std'] if bs['std'] > 1e-8 else 1.0
        z = (df[pc] - bs['median']).abs() / scale
        z_scores_all[f'{pc}_z'] = z.clip(upper=z_clip)
    
    # 综合 energy = 各 PC z-score 的 L2 范数
    if len(z_scores_all.columns) > 0:
        energy = np.sqrt((z_scores_all ** 2).sum(axis=1))
    else:
        energy = pd.Series(0.0, index=df.index)
    
    X['energy'] = energy
    
    # 各 PC 的绝对 z-score（独立特征，不合成）
    for col in z_scores_all.columns:
        X[col] = z_scores_all[col]
    
    # energy 的 baseline 统计
    energy_median = float(energy.median())
    energy_std = float(energy.std()) if energy.std() > 1e-8 else 1.0
    energy_p95 = float(energy.quantile(0.95))
    ap_threshold = energy_p95
    
    X['energy_zscore'] = (energy - energy_median) / (energy_std + 1e-8)
    
    # ====== 2. 各 PC score 的带符号 z-score ======
    # （去掉 raw、above_p95、below_p05）
    for pc in pc_targets:
        if pc not in df.columns:
            continue
        vals = df[pc]
        bs = baseline_stats[pc]
        scale = bs['mad'] if bs['mad'] > 1e-8 else bs['std'] if bs['std'] > 1e-8 else 1.0
        X[f'{pc}_zscore'] = (vals - bs['median']) / scale
    
    # ====== 3. energy 回溯统计（3尺度：短4/中16/长32）======
    for k in [4, 16, 32]:
        roll = energy.rolling(k, min_periods=1)
        X[f'energy_mean_{k}'] = roll.mean()
        X[f'energy_max_{k}'] = roll.max()
        X[f'energy_std_{k}'] = roll.std().fillna(0)
        X[f'energy_ratio_{k}'] = energy / (X[f'energy_mean_{k}'] + 1e-8)
        above = (energy > ap_threshold).astype(float)
        X[f'above_ratio_{k}'] = above.rolling(k, min_periods=1).mean()
    
    # ====== 4. PC zscore rolling（2尺度×2统计×nPC）======
    # 用 zscore 而非 raw，跨场景可比
    for pc in pc_targets:
        zs_col = f'{pc}_zscore'
        if zs_col not in X.columns:
            continue
        pc_zs = X[zs_col]
        for k in [8, 32]:
            pc_roll = pc_zs.abs().rolling(k, min_periods=1)
            X[f'{pc}_zs_mean_{k}'] = pc_roll.mean()
            X[f'{pc}_zs_max_{k}'] = pc_roll.max()
    
    # ====== 5. 变化率 ======
    X['energy_diff_1'] = energy.diff(1).fillna(0)
    X['energy_diff_4'] = energy.diff(4).fillna(0)
    X['energy_diff_16'] = energy.diff(16).fillna(0)
    X['energy_trend_8'] = energy.rolling(8, min_periods=2).apply(
        lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) > 1 else 0,
        raw=False
    ).fillna(0)
    
    # PC zscore 差分（替代原来基于 raw 的差分）
    for pc in pc_targets:
        zs_col = f'{pc}_zscore'
        if zs_col not in X.columns:
            continue
        pc_zs = X[zs_col]
        X[f'{pc}_zs_diff_1'] = pc_zs.diff(1).fillna(0)
        X[f'{pc}_zs_diff_4'] = pc_zs.diff(4).fillna(0)
    
    # ====== 6. 持续性 ======
    above_thresh = (energy > ap_threshold).astype(int).values
    run_length = np.zeros(len(above_thresh))
    for i in range(1, len(above_thresh)):
        if above_thresh[i] == 1:
            run_length[i] = run_length[i-1] + 1
        else:
            run_length[i] = 0
    X['run_length'] = run_length
    
    below_thresh = (energy <= energy_median * 1.5).astype(int).values
    dist_from_normal = np.zeros(len(below_thresh))
    last_normal = 0
    for i in range(len(below_thresh)):
        if below_thresh[i] == 1:
            last_normal = i
        dist_from_normal[i] = i - last_normal
    X['dist_from_normal'] = dist_from_normal
    
    # ====== 7. 前后对比 ======
    recent = energy.rolling(32, min_periods=1).mean()
    earlier = energy.shift(32).rolling(32, min_periods=1).mean()
    X['energy_shift_ratio'] = (recent / (earlier + 1e-8)).fillna(1)
    
    # ====== 8. 分布偏移（仅保留 tstat_32，CP检测核心特征）======
    recent_mean = energy.rolling(32, min_periods=1).mean()
    earlier_mean = energy.shift(32).rolling(32, min_periods=1).mean()
    recent_std = energy.rolling(32, min_periods=2).std().fillna(1e-8)
    earlier_std = energy.shift(32).rolling(32, min_periods=2).std().fillna(1e-8)
    pooled = np.sqrt((recent_std**2 + earlier_std**2) / 2) + 1e-8
    X['energy_tstat_32'] = ((recent_mean - earlier_mean) / pooled).fillna(0)
    
    # ====== 9. 超阈值浓度（仅保留 above_2x_density_16）======
    above_2x = (energy > energy_median * 2).astype(float)
    X['above_2x_density_16'] = above_2x.rolling(16, min_periods=1).mean()
    
    # ====== 10. PC Score 间的交互特征 ======
    # 区分 CP（双通道同时偏移）vs AP（单通道脉冲）的关键特征
    if len(pc_targets) >= 2:
        for i in range(len(pc_targets)):
            for j in range(i+1, len(pc_targets)):
                pc_i, pc_j = pc_targets[i], pc_targets[j]
                zi_col = f'{pc_i}_z'
                zj_col = f'{pc_j}_z'
                if zi_col in z_scores_all.columns and zj_col in z_scores_all.columns:
                    # 同时异常（两个PC同时高z → CP信号）
                    X[f'{pc_i}_{pc_j}_joint_z'] = z_scores_all[zi_col] * z_scores_all[zj_col]
                    # z差异（单通道偏移 → AP信号）
                    X[f'{pc_i}_{pc_j}_z_diff'] = (z_scores_all[zi_col] - z_scores_all[zj_col]).abs()
    
    return X, baseline_stats

In [ ]:
# # ==================== 单元格: TFT 信号合并（PC Score 版）====================

# def merge_tft_signals_pc(df_all, results_tft, engine):
#     """
#     PC Score 版 TFT 信号合并
    
#     与旧版区别：
#     - 不再要求 df_all 中有 feature_energy
#     - 信号命名统一为 model_{model_name}_{signal_type}
#     """
#     for model_name in results_tft:
#         metrics = results_tft[model_name]['metrics']
#         attention = results_tft[model_name]['attention']
#         vsn = results_tft[model_name]['vsn']
#         baseline = engine.baselines[model_name]
        
#         n = len(metrics)
        
#         # 残差比
#         residual_abs = metrics['residual'].abs().values
#         baseline_p95 = max(abs(baseline['residual_p95']),
#                           abs(baseline.get('residual_p5', 0)),
#                           baseline['residual_std'] * 1.645, 1e-8)
#         sig_residual = np.clip(residual_abs / baseline_p95, 0, 20)
        
#         # 不确定性膨胀比
#         div_vals = metrics['divergence'].values if 'divergence' in metrics.columns else np.zeros(n)
#         baseline_div = max(baseline.get('divergence_mean', 1e-8), 1e-8)
#         sig_div = np.clip(div_vals / baseline_div, 0, 20)
        
#         # 注意力KL散度
#         att_base = baseline['att_mean']
#         sig_att_kl = np.zeros(n)
#         for i in range(n):
#             p = attention[i].flatten()
#             q = att_base.flatten()
#             p = p / (p.sum() + 1e-10) + 1e-10
#             q = q / (q.sum() + 1e-10) + 1e-10
#             p = p / p.sum()
#             q = q / q.sum()
#             sig_att_kl[i] = np.sum(p * np.log(p / q))
        
#         # VSN JS散度
#         vsn_base = baseline['vsn_mean']
#         sig_vsn_js = np.zeros(n)
#         for i in range(n):
#             v_i = np.abs(vsn[i].flatten()) + 1e-10
#             v_b = np.abs(vsn_base.flatten()) + 1e-10
#             p = v_i / v_i.sum()
#             q = v_b / v_b.sum()
#             m = 0.5 * (p + q)
#             sig_vsn_js[i] = 0.5 * np.sum(p * np.log(p/m)) + 0.5 * np.sum(q * np.log(q/m))
        
#         # VSN排序变化
#         from scipy.stats import spearmanr
#         baseline_rank = np.argsort(np.argsort(-np.abs(vsn_base.flatten())))
#         sig_vsn_rank = np.zeros(n)
#         for i in range(n):
#             curr_rank = np.argsort(np.argsort(-np.abs(vsn[i].flatten())))
#             corr, _ = spearmanr(baseline_rank, curr_rank)
#             sig_vsn_rank[i] = 1 - corr if not np.isnan(corr) else 1.0
        
#         # 组装
#         prefix = f'model_{model_name}'
#         tft_df = pd.DataFrame({
#             'time_idx': metrics['time_idx'].values,
#             f'{prefix}_residual_ratio': sig_residual,
#             f'{prefix}_div_ratio': sig_div,
#             f'{prefix}_att_kl': np.clip(sig_att_kl, 0, 20),
#             f'{prefix}_vsn_js': np.clip(sig_vsn_js, 0, 5),
#             f'{prefix}_vsn_rank_shift': np.clip(sig_vsn_rank, 0, 2),
#         })
        
#         df_all = df_all.merge(tft_df, on='time_idx', how='left')
    
#     return df_all

In [ ]:
def merge_tft_signals_pc(df_all, results_tft, engine):
    """
    PC Score 版 TFT 信号合并（修复版：防止 merge 膨胀）
    """
    for model_name in results_tft:
        metrics = results_tft[model_name]['metrics']
        attention = results_tft[model_name]['attention']
        vsn = results_tft[model_name]['vsn']
        baseline = engine.baselines[model_name]
        
        n = len(metrics)
        
        # 残差比
        residual_abs = metrics['residual'].abs().values
        baseline_p95 = max(abs(baseline['residual_p95']),
                          abs(baseline.get('residual_p5', 0)),
                          baseline['residual_std'] * 1.645, 1e-8)
        sig_residual = np.clip(residual_abs / baseline_p95, 0, 20)
        
        # 不确定性膨胀比
        div_vals = metrics['divergence'].values if 'divergence' in metrics.columns else np.zeros(n)
        baseline_div = max(baseline.get('divergence_mean', 1e-8), 1e-8)
        sig_div = np.clip(div_vals / baseline_div, 0, 20)
        
        # 注意力KL散度
        att_base = baseline['att_mean']
        sig_att_kl = np.zeros(n)
        for i in range(n):
            p = attention[i].flatten()
            q = att_base.flatten()
            p = p / (p.sum() + 1e-10) + 1e-10
            q = q / (q.sum() + 1e-10) + 1e-10
            p = p / p.sum()
            q = q / q.sum()
            sig_att_kl[i] = np.sum(p * np.log(p / q))
        
        # VSN JS散度
        vsn_base = baseline['vsn_mean']
        sig_vsn_js = np.zeros(n)
        for i in range(n):
            v_i = np.abs(vsn[i].flatten()) + 1e-10
            v_b = np.abs(vsn_base.flatten()) + 1e-10
            p = v_i / v_i.sum()
            q = v_b / v_b.sum()
            m = 0.5 * (p + q)
            sig_vsn_js[i] = 0.5 * np.sum(p * np.log(p/m)) + 0.5 * np.sum(q * np.log(q/m))
        
        # VSN排序变化
        from scipy.stats import spearmanr
        baseline_rank = np.argsort(np.argsort(-np.abs(vsn_base.flatten())))
        sig_vsn_rank = np.zeros(n)
        for i in range(n):
            curr_rank = np.argsort(np.argsort(-np.abs(vsn[i].flatten())))
            corr, _ = spearmanr(baseline_rank, curr_rank)
            sig_vsn_rank[i] = 1 - corr if not np.isnan(corr) else 1.0
        
        # 组装 TFT 信号 DataFrame
        prefix = f'model_{model_name}'
        tft_df = pd.DataFrame({
            'time_idx': metrics['time_idx'].values,
            f'{prefix}_residual_ratio': sig_residual,
            f'{prefix}_div_ratio': sig_div,
            f'{prefix}_att_kl': np.clip(sig_att_kl, 0, 20),
            f'{prefix}_vsn_js': np.clip(sig_vsn_js, 0, 5),
            f'{prefix}_vsn_rank_shift': np.clip(sig_vsn_rank, 0, 2),
        })
        
        # ★ 修复：对 tft_df 按 time_idx 去重，保留最后一次（避免 rolling 重叠导致的重复）
        tft_df = tft_df.drop_duplicates(subset='time_idx', keep='last')
        
        # ★ 修复：验证 merge 前后行数一致
        n_before = len(df_all)
        df_all = df_all.merge(tft_df, on='time_idx', how='left')
        n_after = len(df_all)
        
        if n_after != n_before:
            print(f"  ⚠️ [{model_name}] merge导致行数变化: {n_before} → {n_after} "
                  f"(膨胀{n_after - n_before}行)")
            # 回退并使用 index-aligned 方式
            # 先删除刚 merge 的列
            new_cols = [c for c in tft_df.columns if c != 'time_idx']
            df_all = df_all.drop(columns=new_cols)
            df_all = df_all.iloc[:n_before]  # 恢复行数
            
            # 使用 map 方式逐列赋值（绝对不会改变行数）
            time_to_idx = dict(zip(tft_df['time_idx'], tft_df.index))
            for col in new_cols:
                val_map = dict(zip(tft_df['time_idx'], tft_df[col]))
                df_all[col] = df_all['time_idx'].map(val_map)
            
            print(f"    → 已切换为 map 赋值，行数保持 {len(df_all)}")
    
    return df_all

In [ ]:

# ============================================================
# Step 1：扩展CP标签从8窗口到32窗口
# ============================================================

def extend_cp_labels(df_all, cp_extend_windows=32):
    """
    将CP标签从onset的8个窗口扩展到32个窗口
    
    原始设计：
      CP onset (8窗口): label='CP', regime='cp_onset'
      CP drift (672窗口): label='N', regime='cp_drift' → 被排除
    
    修改后：
      CP onset+确认 (32窗口): label='CP', regime='cp_onset'
      CP drift剩余 (648窗口): label='N', regime='cp_drift' → 仍排除
    """
    df = df_all.copy()
    
    total_extended = 0
    
    for sid in sorted(df['scenario_id'].unique()):
        sid_mask = df['scenario_id'] == sid
        df_s = df[sid_mask].sort_values('time_idx')
        
        # 找到当前CP标签的窗口
        cp_windows = df_s[df_s['label'] == 'CP']
        
        if len(cp_windows) == 0:
            continue
        
        # 对每个CP事件（可能有多个）
        # 找连续CP段的起始点
        cp_times = cp_windows['time_idx'].values
        cp_events = []
        
        if len(cp_times) > 0:
            event_start = cp_times[0]
            event_end = cp_times[0]
            
            for t in cp_times[1:]:
                if t - event_end <= 2:  # 连续
                    event_end = t
                else:
                    cp_events.append((event_start, event_end))
                    event_start = t
                    event_end = t
            cp_events.append((event_start, event_end))
        
        # 对每个CP事件，向后扩展标签
        for cp_start, cp_end in cp_events:
            current_len = cp_end - cp_start + 1
            extend_to = cp_start + cp_extend_windows - 1
            
            # 只扩展到还没有其他事件标签的窗口
            extend_mask = (
                sid_mask &
                (df['time_idx'] > cp_end) &
                (df['time_idx'] <= extend_to) &
                (df['label'] == 'N')  # 只改N标签的窗口
            )
            
            n_extended = extend_mask.sum()
            df.loc[extend_mask, 'label'] = 'CP'
            df.loc[extend_mask, 'regime'] = 'cp_onset'  # 改regime使其不被排除
            
            total_extended += n_extended
    
    # 汇报
    print(f"[CP扩展] 共扩展 {total_extended} 个窗口的标签从N改为CP")
    
    mask = df['regime'] != 'cp_drift'
    print(f"扩展后标签分布（排除cp_drift）:")
    print(df.loc[mask, 'label'].value_counts())
    
    return df



In [ ]:
# # ==================== 单元格: 构造全场景特征矩阵（PC Score 版 v2）====================

# def build_all_features_pc(df_all, tb, engine_PC, results_PC=None,
#                            pc_targets=None):
#     """
#     PC Score 版特征构造主函数（v2 精简版）
    
#     变更：调用精简后的 build_causal_features_pc
#     """
#     # 自动推断 PC targets
#     if pc_targets is None:
#         pc_targets = []
#         for mn, mc in engine_PC.config.get('tft_models', {}).items():
#             t = mc.get('target')
#             if t and t in df_all.columns:
#                 pc_targets.append(t)
#         if not pc_targets:
#             pc_targets = [c for c in df_all.columns 
#                           if ('_pc' in c and any(c.startswith(p) for p in ['post', 'comment']))]
    
#     print(f"[PC Targets] {pc_targets}")
    
#     # ---- 识别已有 TFT 信号列 ----
#     tft_cols = [c for c in df_all.columns 
#                 if any(kw in c for kw in ['residual_ratio', 'div_ratio', 
#                                            'att_kl', 'vsn_js', 'vsn_rank_shift'])]
#     if tft_cols:
#         print(f"[TFT信号列] {tft_cols}")
    
#     # ---- 计算全局 baseline 统计量（从所有场景的 N 标签数据）----
#     df_normal_all = df_all[df_all['label'] == 'N']
#     baseline_stats = {}
#     for pc in pc_targets:
#         if pc not in df_normal_all.columns:
#             continue
#         vals = df_normal_all[pc].dropna()
#         baseline_stats[pc] = {
#             'median': float(vals.median()),
#             'std': float(vals.std()),
#             'mad': float((vals - vals.median()).abs().median()) * 1.4826,
#             'p95': float(vals.quantile(0.95)),
#             'p05': float(vals.quantile(0.05)),
#         }
    
#     print(f"[Baseline Stats]")
#     for pc, bs in baseline_stats.items():
#         print(f"  {pc}: median={bs['median']:.4f}, mad={bs['mad']:.4f}, "
#               f"p95={bs['p95']:.4f}, p05={bs['p05']:.4f}")
    
#     # ---- 逐场景构造特征 ----
#     all_X = []
#     all_y = []
#     all_meta = []
    
#     for sid in sorted(df_all['scenario_id'].unique()):
#         df_s = df_all[df_all['scenario_id'] == sid].copy()
#         mask = df_s.get('regime', pd.Series('normal', index=df_s.index)) != 'cp_drift'
#         df_s = df_s[mask]
        
#         if len(df_s) == 0:
#             continue
        
#         # 构造 PC 因果特征（精简版）
#         X_s, _ = build_causal_features_pc(
#             df_s,
#             pc_targets=pc_targets,
#             tb=tb,
#             baseline_stats=baseline_stats,
#         )
        
#         # TFT 偏差信号（已在 df_all 中，直接取 + rolling）
#         for col in tft_cols:
#             if col in df_s.columns:
#                 X_s[col] = df_s[col].values
#                 for k in [4, 8]:
#                     X_s[f'{col}_rollmax_{k}'] = df_s[col].rolling(k, min_periods=1).max().values
#                     X_s[f'{col}_rollmean_{k}'] = df_s[col].rolling(k, min_periods=1).mean().values
        
#         all_X.append(X_s)
#         all_y.append(df_s['label'])
#         all_meta.append(df_s[['time_idx', 'scenario_id', 'label', 'regime']])
    
#     X = pd.concat(all_X, ignore_index=True)
#     y = pd.concat(all_y, ignore_index=True)
#     meta = pd.concat(all_meta, ignore_index=True)
    
#     # 清理 NaN / Inf
#     X = X.replace([np.inf, -np.inf], 0).fillna(0)
    
#     print(f"\n[特征矩阵] {X.shape}")
#     print(f"[标签分布]\n{y.value_counts()}")
#     print(f"[特征列数] {len(X.columns)}")
#     print(f"[特征列表]\n  " + "\n  ".join(X.columns.tolist()))
    
#     return X, y, meta, baseline_stats

In [ ]:
# ==================== 修改: 逐场景 TFT 推理 + 信号合并 ====================

def build_all_features_pc(df_all, df_normal_all,tb, engine_PC, results_PC=None,
                           pc_targets=None):
    """
    PC Score 版特征构造主函数（v2.1 修复版）
    
    关键修复：逐场景重新 TFT 推理，而非共享干净数据的推理结果
    """
    import io, contextlib
    
    # 自动推断 PC targets
    if pc_targets is None:
        pc_targets = []
        for mn, mc in engine_PC.config.get('tft_models', {}).items():
            t = mc.get('target')
            if t and t in df_all.columns:
                pc_targets.append(t)
        if not pc_targets:
            pc_targets = [c for c in df_all.columns 
                          if ('_pc' in c and any(c.startswith(p) for p in ['post', 'comment']))]
    
    print(f"[PC Targets] {pc_targets}")
    
    # ---- 计算全局 baseline 统计量（从所有场景的 N 标签数据）----
    # df_normal_all = df_all[df_all['label'] == 'N']
    baseline_stats = {}
    for pc in pc_targets:
        if pc not in df_normal_all.columns:
            continue
        vals = df_normal_all[pc].dropna()
        baseline_stats[pc] = {
            'median': float(vals.median()),
            'std': float(vals.std()),
            'mad': float((vals - vals.median()).abs().median()) * 1.4826,
            'p95': float(vals.quantile(0.95)),
            'p05': float(vals.quantile(0.05)),
        }
    
    print(f"[Baseline Stats]")
    for pc, bs in baseline_stats.items():
        print(f"  {pc}: median={bs['median']:.4f}, mad={bs['mad']:.4f}, "
              f"p95={bs['p95']:.4f}, p05={bs['p05']:.4f}")
    
    # ---- 逐场景构造特征 ----
    all_X = []
    all_y = []
    all_meta = []
    print(f"\n[场景总数] {len(df_all['scenario_id'].unique())}")
    scenarios = sorted(df_all['scenario_id'].unique())
    
    drop_columns=['total_volume_post', 'total_volume_comment', 'gini_post',
       'gini_comment', 'senti_symbol_post', 'senti_symbol_comment',
       'comp_ratio_post', 'comp_ratio_comment', 'retweet_ratio_post',
       'total_short_post', 'total_long_post', 'total_short_comment',
       'total_long_comment', 'neg_ratio_post', 'neg_ratio_comment',
       'semantic_shift_post', 'semantic_shift_comment',
       'vis_abs_redundancy_post', 'vis_concentration_post', 
       'hour', 'dayofweek', 'month', 'day', 'is_weekend', 'is_holiday',
       'time_slot', 'event_code']
    
    # for sid_idx, sid in enumerate(scenarios):
    for sid_idx, sid in enumerate(tqdm(scenarios, desc="处理合成场景中")):
        print(f"\nProcessing Scenario {sid_idx+1}/{len(scenarios)} (SID={sid})...")
        df_s = df_all[df_all['scenario_id'] == sid].copy()
        mask = df_s.get('regime', pd.Series('normal', index=df_s.index)) != 'cp_drift'
        df_s = df_s[mask]
        
        # df_processed_s = processor.transform(df_s,df_official_real).data  # ★ 场景内数据处理（如缺失值填充、时间特征等）
        if len(df_s) == 0:
            continue
        n_before_tft = len(df_s)
        print(f"  排除cp_drift后: {n_before_tft} 行, "
              f"标签分布: {df_s['label'].value_counts().to_dict()}")
        
        # ★ 关键修复：逐场景 TFT 推理
        tft_results_s = {}
        for mn in engine_PC.models:
            with contextlib.redirect_stdout(io.StringIO()), \
                 contextlib.redirect_stderr(io.StringIO()):
                res = engine_PC.analyze_rolling(mn, df_s, baseline_end_idx=None)
            tft_results_s[mn] = res
            # ★ 诊断：检查 metrics 中 time_idx 是否有重复
            metrics_tidx = res['metrics']['time_idx'].values
            n_unique = len(np.unique(metrics_tidx))
            n_total_metrics = len(metrics_tidx)
            if n_unique != n_total_metrics:
                print(f"  ⚠️ [{mn}] metrics time_idx 有重复: "
                      f"total={n_total_metrics}, unique={n_unique}")
        
        # 只保留需要的列（drop 原始特征列）
        cols_to_drop = [c for c in drop_columns if c in df_s.columns]
        df_s_slim = df_s.drop(columns=cols_to_drop)
        
        # 合并 TFT 信号到场景数据
        df_s_merged = merge_tft_signals_pc(df_s_slim, tft_results_s, engine_PC)
        
        # ★ 行数校验
        n_after_merge = len(df_s_merged)
        if n_after_merge != n_before_tft:
            print(f"  ❌ SID={sid} 合并后行数异常: "
                  f"{n_before_tft} → {n_after_merge} (差{n_after_merge - n_before_tft})")
            # 直接 continue 跳过此场景，或尝试修复
            continue
        else:
            print(f"  ✅ SID={sid} 合并后行数正常: {n_after_merge} 行")
        # print(df_s.columns)
        # 识别 TFT 信号列
        tft_cols = [c for c in df_s_merged.columns 
                    if any(kw in c for kw in ['residual_ratio', 'div_ratio', 
                                               'att_kl', 'vsn_js', 'vsn_rank_shift'])]
        # print(f"[TFT信号列] {tft_cols}")
        # 构造 PC 因果特征（精简版）
        X_s, _ = build_causal_features_pc(
            df_s_merged,
            pc_targets=pc_targets,
            tb=tb,
            baseline_stats=baseline_stats,
        )
        # print(f"[PC特征] {X_s.columns.tolist()}")
        # TFT 偏差信号 + rolling
        for col in tft_cols:
            if col in df_s_merged.columns:
                X_s[col] = df_s_merged[col].values
                for k in [4, 8]:
                    X_s[f'{col}_rollmax_{k}'] = df_s_merged[col].rolling(k, min_periods=1).max().values
                    X_s[f'{col}_rollmean_{k}'] = df_s_merged[col].rolling(k, min_periods=1).mean().values
        # print(len(X_s.columns))
        # ★ 最终行数校验
        assert len(X_s) == len(df_s_merged), \
            f"SID={sid} 特征矩阵行数不匹配: X={len(X_s)}, df={len(df_s_merged)}"
        
        all_X.append(X_s)
        all_y.append(df_s['label'])
        all_meta.append(df_s[['time_idx', 'scenario_id', 'label', 'regime']])
        
        # if (sid_idx + 1) % 5 == 0:
        #     print(f"  场景 {sid_idx+1}/{len(scenarios)} 完成 (SID={sid})")
        # break  # ★ 测试阶段先处理一个场景，确认无误后再去掉
    
    X = pd.concat(all_X, ignore_index=True)
    y = pd.concat(all_y, ignore_index=True)
    meta = pd.concat(all_meta, ignore_index=True)
    
    # 清理 NaN / Inf
    X = X.replace([np.inf, -np.inf], 0).fillna(0)
    
    print(f"\n[特征矩阵] {X.shape}")
    print(f"[标签分布]\n{y.value_counts()}")
    print(f"[特征列数] {len(X.columns)}")
    print(f"[特征列表]\n  " + "\n  ".join(X.columns.tolist()))

        # ★ 最终一致性校验
    expected_rows = sum(
        len(df_all[(df_all['scenario_id'] == sid) & 
                   (df_all.get('regime', pd.Series('normal')) != 'cp_drift')])
        for sid in scenarios
    )
    print(f"\n[一致性校验]")
    print(f"  预期总行数 (排除cp_drift): {expected_rows}")
    print(f"  实际总行数: {len(X)}")
    if len(X) != expected_rows:
        print(f"  ⚠️ 差异: {len(X) - expected_rows} 行 "
              f"(可能有场景因merge失败被跳过)")
    else:
        print(f"  ✅ 完全一致")
    
    return X, y, meta, baseline_stats


## LOSO交叉验证

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score, precision_recall_curve,precision_score,recall_score)
import lightgbm as lgb


In [ ]:
def plot_lgbm_curves(evals_results, fold_labels=None, save_path="./figures/lgbm_loss.png"):
    """绘制LightGBM训练/验证loss曲线"""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    n = len(evals_results)
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows), squeeze=False)

    for i, ev in enumerate(evals_results):
        ax = axes[i // cols][i % cols]
        ax.plot(ev['train']['multi_logloss'], label='train', alpha=0.7)
        ax.plot(ev['valid']['multi_logloss'], label='val', alpha=0.7)
        ax.set_xlabel('iteration')
        ax.set_ylabel('multi_logloss')
        ax.set_title(fold_labels[i] if fold_labels else f'Fold {i+1}')
        ax.legend(fontsize=7)

        # 过拟合指标
        t_final = ev['train']['multi_logloss'][-1]
        v_best = min(ev['valid']['multi_logloss'])
        v_final = ev['valid']['multi_logloss'][-1]
        gap = v_final - t_final
        
        ax.text(0.95, 0.95, f'gap={gap:.3f}\nbest_val={v_best:.3f}',
                transform=ax.transAxes, ha='right', va='top', fontsize=7,
                color='red' if gap > 0.3 else 'green')

    for j in range(n, rows * cols):
        axes[j // cols][j % cols].axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ LightGBM loss curves saved to: {save_path}")

In [ ]:
# ==================== 单元格: LOSO交叉验证（PC Score 版）====================

def lgbm_loso_pc(X, y, meta):
    """
    LOSO交叉验证（PC Score 版）
    与旧版区别：layer_cols 从 PC 特征中取
    """
    scenarios = sorted(meta['scenario_id'].unique())
    oof_parts = []
    evals_results = []

    # 传递给分层预警的特征列
    layer_cols = ['energy', 'energy_zscore', 'run_length', 'energy_shift_ratio']
    # 加上 TFT 信号列
    for c in X.columns:
        if 'residual_ratio' in c:
            layer_cols.append(c)
    # 加上 PC 交互特征
    for c in X.columns:
        if '_joint_z' in c or '_z_diff' in c:
            layer_cols.append(c)
    layer_cols = [c for c in layer_cols if c in X.columns]

    label_map = {'N': 0, 'AP': 1, 'CP': 2}
    inv_map = {0: 'N', 1: 'AP', 2: 'CP'}

    for fold, test_sid in enumerate(scenarios):
        # 👈 修改点2: 切分 train 和 val 场景
        train_sids = [s for s in scenarios if s != test_sid]
        test_mask = meta['scenario_id'] == test_sid
        X_te, y_te = X[test_mask], y[test_mask]

        # 从剩余的训练场景中，随机留出20%场景做 early-stopping 的验证集
        n_val = max(1, len(train_sids) // 5)
        rng = np.random.RandomState(42 + fold)
        val_sids = set(rng.choice(train_sids, size=n_val, replace=False))
        tr_sids = [s for s in train_sids if s not in val_sids]
        
        # 构建掩码
        tr_mask = meta['scenario_id'].isin(tr_sids)
        val_mask = meta['scenario_id'].isin(val_sids)
        
        y_tr_enc = y[tr_mask].map(label_map)
        y_val_enc = y[val_mask].map(label_map)

        # 👈 修改点3: 计算样本权重 (仅基于当前纯 training set 的分布)
        counts = y[tr_mask].value_counts()
        n_total = len(y_tr_enc)
        base = n_total / (3 * counts.get('N', 1))
        max_ratio = 2000            
        w = {code: min(n_total / (3 * counts.get(lbl, 1)) / base, max_ratio) * base
             for lbl, code in label_map.items()}

        # 初始化模型 (增加 n_estimators 以备早停)
        model = lgb.LGBMClassifier(
            n_estimators=500,   # 👈 允许跑到2000，靠早停掐断
            max_depth=5, 
            num_leaves=31,
            min_child_samples=50,
            learning_rate=0.05,
            subsample=0.8, 
            colsample_bytree=0.7,
            reg_alpha=1.0, 
            reg_lambda=1.0,
            random_state=42, 
            verbose=-1,
            num_class=3, 
            objective='multiclass',
        )

        # 👈 修改点4: 加入 eval_set 并启用早停
        model.fit(
            X[tr_mask], y_tr_enc, 
            sample_weight=y_tr_enc.map(w).values,
            eval_set=[(X[tr_mask], y_tr_enc), (X[val_mask], y_val_enc)],
            eval_sample_weight=[y_tr_enc.map(w).values, y_val_enc.map(w).values],
            eval_names=['train', 'valid'],
            eval_metric='multi_logloss',
            callbacks=[
                lgb.log_evaluation(0), # 关闭每个迭代的打印，太吵了
                lgb.early_stopping(5,first_metric_only=True,verbose=False)],
        )

        evals_results.append(model.evals_result_)
        proba = model.predict_proba(X_te)

        batch = pd.DataFrame({
            'y_true': y_te.values,
            'y_pred_argmax': [inv_map[p] for p in proba.argmax(axis=1)],
            'prob_N': proba[:, 0], 'prob_AP': proba[:, 1], 'prob_CP': proba[:, 2],
            'anomaly_score': 1 - proba[:, 0],
            'time_idx': meta.loc[test_mask, 'time_idx'].values,
            'scenario_id': test_sid,
        })
        for c in layer_cols:
            if c in X_te.columns:
                batch[c] = X_te[c].values
        oof_parts.append(batch)

        # 👈 修改点5: 打印每个模型的最佳迭代次数和测试集 F1
        best_iter = model.best_iteration_ if hasattr(model, 'best_iteration_') else 500
        f1 = f1_score(y_te != 'N', proba.argmax(axis=1) != 0, zero_division=0)
        print(f"  ✅ Fold {fold+1:02d}/{len(scenarios)} [Test SID: {test_sid}]: "
              f"best_iter={best_iter:4d}, val_loss={model.best_score_['valid']['multi_logloss']:.4f} "
              f"| test_binary_F1={f1:.3f}")

    # 合并所有预测结果
    oof_df = pd.concat(oof_parts, ignore_index=True)
    
    # 👈 修改点6: 统计全局最佳迭代次数
    all_best_iters = []
    for ev in evals_results:
        val_losses = ev['valid']['multi_logloss']
        best_iter = int(np.argmin(val_losses)) + 1
        all_best_iters.append(best_iter)
    mean_best_iter = int(np.mean(all_best_iters))
    
    f1_all = f1_score(oof_df['y_true'] != 'N', oof_df['y_pred_argmax'] != 'N', zero_division=0)
    
    print("\n" + "="*50)
    print(f"🎯 [LOSO 汇总] Total {len(oof_df)} windows")
    print(f"   Binary_F1 (argmax) = {f1_all:.4f}")
    print(f"   Best Iterations    : mean={mean_best_iter}, min={min(all_best_iters)}, max={max(all_best_iters)}")
    print("="*50)
    
    # 👈 修改点7: 调用绘图函数
    plot_lgbm_curves(
        evals_results[::max(1, len(evals_results)//5)], # 挑几个代表性的 fold 画图
        fold_labels=[f'Fold {i+1}' for i in range(0, len(evals_results), max(1, len(evals_results)//5))],
        save_path="./figures/lgbm_loso_loss.png"
    )
    
    return oof_df, mean_best_iter

## 可视化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve
import os

def plot_multiclass_evaluation_V2(oof_df, save_dir="./figures", zoom_window=500):
    """
    绘制三分类综合评估图
    参数:
    - oof_df: 包含真实标签和预测概率的DataFrame
    - save_dir: 图片保存路径
    - zoom_window: 时序截取窗口大小（变点前后各截取的时间步数）
    """
    os.makedirs(save_dir, exist_ok=True)
    
    labels = ['N', 'AP', 'CP']
    
    # ==========================================
    # 1. 混淆矩阵 (Confusion Matrix)
    # ==========================================
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(oof_df['y_true'], oof_df['y_pred_argmax'], labels=labels)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    annot = np.empty_like(cm).astype(str)
    for i in range(3):
        for j in range(3):
            annot[i, j] = f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)"
            
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    plt.title('Confusion Matrix (OOF)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig(f"{save_dir}/confusion_matrix.png", dpi=150)
    plt.close()
    
    # ==========================================
    # 2. ROC & PR 曲线
    # ==========================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    colors = {'N': 'green', 'AP': 'orange', 'CP': 'red'}
    
    for cls in labels:
        y_true_bin = (oof_df['y_true'] == cls).astype(int)
        y_prob = oof_df[f'prob_{cls}']
        
        fpr, tpr, _ = roc_curve(y_true_bin, y_prob)
        roc_auc = auc(fpr, tpr)
        ax1.plot(fpr, tpr, color=colors[cls], lw=2, label=f'{cls} (AUC = {roc_auc:.3f})')
        
        precision, recall, _ = precision_recall_curve(y_true_bin, y_prob)
        pr_auc = auc(recall, precision)
        ax2.plot(recall, precision, color=colors[cls], lw=2, label=f'{cls} (AUC = {pr_auc:.3f})')
        
    ax1.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve (One-vs-Rest)')
    ax1.legend(loc="lower right")
    
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve')
    ax2.legend(loc="lower left")
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/roc_pr_curves.png", dpi=150)
    plt.close()

    # ==========================================
    # 3. 随机抽取一个场景画时序还原图（局部放大版）
    # ==========================================
    cp_scenarios = oof_df[oof_df['y_true'] == 'CP']['scenario_id'].unique()
    if len(cp_scenarios) > 0:
        sample_sid = cp_scenarios[0]
        df_plot_full = oof_df[oof_df['scenario_id'] == sample_sid].sort_values('time_idx')
        
        # 寻找第一个CP变点的发生时间
        first_cp_idx = df_plot_full[df_plot_full['y_true'] == 'CP']['time_idx'].min()
        
        # 计算局部截取区间
        start_idx = max(df_plot_full['time_idx'].min(), first_cp_idx - zoom_window)
        end_idx = min(df_plot_full['time_idx'].max(), first_cp_idx + zoom_window)
        
        # 截取局部数据
        df_plot = df_plot_full[(df_plot_full['time_idx'] >= start_idx) & 
                               (df_plot_full['time_idx'] <= end_idx)]
        
        plt.figure(figsize=(10, 4))
        
        plt.plot(df_plot['time_idx'], df_plot['prob_N'], label='Prob: N', color='green', alpha=0.7)
        plt.plot(df_plot['time_idx'], df_plot['prob_AP'], label='Prob: AP', color='orange', alpha=0.7)
        plt.plot(df_plot['time_idx'], df_plot['prob_CP'], label='Prob: CP', color='red', alpha=0.8, linewidth=2)
        
        # 绘制背景色
        for idx, row in df_plot.iterrows():
            if row['y_true'] == 'AP':
                plt.axvspan(row['time_idx']-0.5, row['time_idx']+0.5, color='orange', alpha=0.2, lw=0)
            elif row['y_true'] == 'CP':
                plt.axvspan(row['time_idx']-0.5, row['time_idx']+0.5, color='red', alpha=0.2, lw=0)
                
        plt.title(f'Prediction Timeline Zoomed-in (Scenario: {sample_sid})')
        plt.xlabel('Time Index')
        plt.ylabel('Predicted Probability')
        
        # 强制设置X轴和Y轴范围，避免由于截取数据导致坐标轴比例失衡
        plt.xlim(start_idx, end_idx)
        plt.ylim(0, 1.05)
        
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{save_dir}/timeline_scenario_{sample_sid}_zoomed.png", dpi=150)
        plt.close()

    print(f"✅ OOF Evaluation plots saved to {save_dir}")

## E_X: 三分类 argmax

##  E_Y: 分层预警（自动阈值，分层独立优化）

### C2定义

In [ ]:
# ============================================================
# 分层预警：模拟 + 优化
# ============================================================

def simulate_layered_alerts(oof_df, params):
    """
    分层预警模拟
    
    Layer1: 统计规则 (OR)  → 黄色（仅标记）
    Layer2: ML确认+持续性   → 橙色 → 标签 AP
    Layer3: CP分型确认      → 红色 → 标签 CP
    """
    labels = np.full(len(oof_df), 'N', dtype=object)

    for sid in oof_df['scenario_id'].unique():
        smask = oof_df['scenario_id'] == sid
        df_s = oof_df[smask].sort_values('time_idx')
        idx = df_s.index.values
        n = len(df_s)

        # --- Layer 1: 统计规则 ---
        l1 = ((df_s['energy'].values > params['l1_energy']) |
              (df_s['energy_zscore'].values > params['l1_zscore']))
        tft_cols = [c for c in df_s.columns if 'residual_ratio' in c]
        for tc in tft_cols:
            l1 |= (df_s[tc].values > params.get('l1_tft', 999))

        # --- Layer 2: ML确认 + L1持续性 ---
        l1_rolling = pd.Series(l1.astype(float)).rolling(
            params['l2_lookback'], min_periods=1).sum().values
        l2 = ((df_s['anomaly_score'].values > params['l2_score']) &
              (l1_rolling >= params['l2_min_l1']))

        # --- Layer 3: CP确认（连续L2 + prob_CP） ---
        consec = np.zeros(n, dtype=int)
        for i in range(n):
            consec[i] = (consec[i-1] + 1) if (i > 0 and l2[i]) else int(l2[i])

        cp_mask = (l2 & (df_s['prob_CP'].values > params['l3_cp_prob'])
                   & (consec >= params['l3_cp_consec']))
        ap_mask = l2 & ~cp_mask

        labels[idx[cp_mask]] = 'CP'
        labels[idx[ap_mask]] = 'AP'

    return labels


def optimize_layered_thresholds(oof_df, tb):
    """
    分层独立优化：
    Layer1 → max recall s.t. FPR<10%
    Layer2 → max binary F1
    Layer3 → max macro F1
    """
    y_true = oof_df['y_true'].values
    is_anom = (y_true != 'N')

    # ===== Layer 1 =====
    best_l1, best_l1_r = {}, -1
    e_grid = [tb.quiet_energy_p95 * m for m in [0.7, 0.8, 0.9, 1.0, 1.1, 1.3]]
    z_grid = [2.5, 3.0, 3.5, 4.0, 5.0]

    for e_th in e_grid:
        for z_th in z_grid:
            l1 = (oof_df['energy'].values > e_th) | (oof_df['energy_zscore'].values > z_th)
            recall = l1[is_anom].mean() if is_anom.any() else 0
            fpr = l1[~is_anom].mean() if (~is_anom).any() else 1
            if fpr <= 0.10 and recall > best_l1_r:
                best_l1_r = recall
                best_l1 = {'l1_energy': e_th, 'l1_zscore': z_th}

    if not best_l1:  # FPR约束过严，退而求F1
        for e_th in e_grid:
            for z_th in z_grid:
                l1 = (oof_df['energy'].values > e_th) | (oof_df['energy_zscore'].values > z_th)
                f = f1_score(is_anom, l1, zero_division=0)
                if f > best_l1_r:
                    best_l1_r = f
                    best_l1 = {'l1_energy': e_th, 'l1_zscore': z_th}

    tft_cols = [c for c in oof_df.columns if 'residual_ratio' in c]
    best_l1['l1_tft'] = 3.0 if tft_cols else 999
    print(f"[Layer1] recall={best_l1_r:.3f}, {best_l1}")

    # ===== Layer 2 =====
    best_l2, best_l2_f1 = {}, -1
    # 固定 L3 为宽松默认值（不影响L2的binary判断）
    l3_default = {'l3_cp_prob': 0.3, 'l3_cp_consec': 8}

    for s in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
        for lb in [2, 3, 4, 6]:
            for ml in [1, 2]:
                p = {**best_l1, 'l2_score': s, 'l2_lookback': lb,
                     'l2_min_l1': ml, **l3_default}
                pred = simulate_layered_alerts(oof_df, p)
                f = f1_score(is_anom, pred != 'N', zero_division=0)
                if f > best_l2_f1:
                    best_l2_f1 = f
                    best_l2 = {'l2_score': s, 'l2_lookback': lb, 'l2_min_l1': ml}

    print(f"[Layer2] binary_F1={best_l2_f1:.3f}, {best_l2}")

    # ===== Layer 3 =====
    best_l3, best_l3_f1 = {}, -1

    for cp_p in [0.15, 0.20, 0.25, 0.30, 0.40]:
        for cp_c in [4, 6, 8, 12]:
            p = {**best_l1, **best_l2, 'l3_cp_prob': cp_p, 'l3_cp_consec': cp_c}
            pred = simulate_layered_alerts(oof_df, p)
            f = f1_score(y_true, pred, labels=['N', 'AP', 'CP'],
                         average='macro', zero_division=0)
            if f > best_l3_f1:
                best_l3_f1 = f
                best_l3 = {'l3_cp_prob': cp_p, 'l3_cp_consec': cp_c}

    print(f"[Layer3] macro_F1={best_l3_f1:.3f}, {best_l3}")

    final = {**best_l1, **best_l2, **best_l3}
    print(f"\n[最优阈值] {final}")
    return final

### 新C2定义

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# ============================================================
# 分层预警：模拟 + 优化 (V8 精准版)
# 针对微弱注入与概率左移进行专门优化，大幅压降误报
# ============================================================

def simulate_layered_alerts_v8(oof_df, params):
    """
    Layer 1: 严格统计初筛 (仅依赖 zscore 和 TFT 残差比)
    Layer 2: 严格 ML 确认 + 持续性
    Layer 3: CP 分型确认
    """
    labels = np.full(len(oof_df), 'N', dtype=object)

    for sid in oof_df['scenario_id'].unique():
        smask = oof_df['scenario_id'] == sid
        df_s = oof_df[smask].sort_values('time_idx')
        idx = df_s.index.values
        n = len(df_s)

        # --- Layer 1: 严格的相对异常初筛 ---
        # 摒弃绝对 energy，只用 zscore 和残差比，门槛提高
        l1 = (df_s['energy_zscore'].values > params['l1_zscore'])
        
        tft_cols = [c for c in df_s.columns if 'residual_ratio' in c]
        for tc in tft_cols:
            l1 |= (df_s[tc].values > params.get('l1_tft', 3.0))

        # --- Layer 2: 高门槛 ML 确认 + 持续性要求 ---
        # 统计过去 lookback 窗口内，L1 报警且 L2 分数也达标的次数
        # 注意：这里要求 ML score 必须非常高
        l2_cond = (df_s['anomaly_score'].values > params['l2_score'])
        combined_signal = (l1 & l2_cond).astype(float)
        
        l1_rolling = pd.Series(combined_signal).rolling(
            params['l2_lookback'], min_periods=1).sum().values
            
        l2 = l1_rolling >= params['l2_min_l1']

        # --- Layer 3: CP 确认（必须持续被判定为 L2 且 CP 概率高） ---
        consec = np.zeros(n, dtype=int)
        for i in range(n):
            consec[i] = (consec[i-1] + 1) if (i > 0 and l2[i]) else int(l2[i])

        cp_mask = (l2 & (df_s['prob_CP'].values > params['l3_cp_prob'])
                   & (consec >= params['l3_cp_consec']))
        ap_mask = l2 & ~cp_mask

        labels[idx[cp_mask]] = 'CP'
        labels[idx[ap_mask]] = 'AP'

    return labels


def optimize_layered_thresholds_v8(oof_df, baseline_p95_zscore=3.0):
    """
    分层独立优化 (V8)
    强行提高各项阈值的下限，牺牲微弱的 Recall 来换取 Precision 的大幅提升
    """
    y_true = oof_df['y_true'].values
    is_anom = (y_true != 'N')

    # ===== Layer 1: 放弃 FPR，直接最大化 F1 (惩罚假阳性) =====
    best_l1, best_l1_f1 = {}, -1
    
    # 网格大幅上调：Z-score 至少 3.5 起步，TFT 至少 3.0 起步
    z_grid = [3.5, 4.0, 4.5, 5.0, 6.0]
    tft_grid = [3.0, 3.5, 4.0, 5.0]
    tft_cols = [c for c in oof_df.columns if 'residual_ratio' in c]

    print("开始搜索 Layer 1 阈值...")
    for z_th in z_grid:
        for tft_th in tft_grid:
            l1 = (oof_df['energy_zscore'].values > z_th)
            for tc in tft_cols:
                l1 |= (oof_df[tc].values > tft_th)
                
            f = f1_score(is_anom, l1, zero_division=0)
            if f > best_l1_f1:
                best_l1_f1 = f
                best_l1 = {'l1_zscore': z_th, 'l1_tft': tft_th}

    # 兜底保障
    if not best_l1:
        best_l1 = {'l1_zscore': 4.0, 'l1_tft': 3.5}
    print(f"[Layer1] F1={best_l1_f1:.3f}, {best_l1}")

    # ===== Layer 2: 极高模型置信度要求 =====
    best_l2, best_l2_f1 = {}, -1
    l3_default = {'l3_cp_prob': 0.5, 'l3_cp_consec': 12}

    # anomaly_score 网格上调：0.5 起步，因为真正的异常大部分都在 0.6 以右
    print("开始搜索 Layer 2 阈值...")
    for s in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85]:
        for lb in [3, 4, 6, 8]:
            for ml in [2, 3]: # 至少需要2次确认，杜绝单点闪烁报警
                p = {**best_l1, 'l2_score': s, 'l2_lookback': lb,
                     'l2_min_l1': ml, **l3_default}
                pred = simulate_layered_alerts_v8(oof_df, p)
                f = f1_score(is_anom, pred != 'N', zero_division=0)
                if f > best_l2_f1:
                    best_l2_f1 = f
                    best_l2 = {'l2_score': s, 'l2_lookback': lb, 'l2_min_l1': ml}

    print(f"[Layer2] binary_F1={best_l2_f1:.3f}, {best_l2}")

    # ===== Layer 3: CP 分型确认 =====
    best_l3, best_l3_f1 = {}, -1
    print("开始搜索 Layer 3 阈值...")
    for cp_p in [0.20, 0.30, 0.40, 0.50]: # CP 在三分类里概率往往不高，这里可以放宽
        for cp_c in [4, 8, 12, 16]:
            p = {**best_l1, **best_l2, 'l3_cp_prob': cp_p, 'l3_cp_consec': cp_c}
            pred = simulate_layered_alerts_v8(oof_df, p)
            f = f1_score(y_true, pred, labels=['N', 'AP', 'CP'],
                         average='macro', zero_division=0)
            if f > best_l3_f1:
                best_l3_f1 = f
                best_l3 = {'l3_cp_prob': cp_p, 'l3_cp_consec': cp_c}

    print(f"[Layer3] macro_F1={best_l3_f1:.3f}, {best_l3}")

    final = {**best_l1, **best_l2, **best_l3}
    print(f"\n[V8 最优阈值] {final}")
    return final

## 定义评估方法

### 统一评估（窗口 + 事件 + 运营）

In [ ]:
def _extract_events(y_true, y_pred, meta):
    """从连续标签段提取事件，计算检出和延迟"""
    rows = []
    for sid in meta['scenario_id'].unique():
        smask = (meta['scenario_id'] == sid).values
        yt, yp = y_true[smask], y_pred[smask]
        tidx = meta.loc[smask, 'time_idx'].values
        order = np.argsort(tidx)
        yt, yp = yt[order], yp[order]

        i = 0
        while i < len(yt):
            if yt[i] != 'N':
                etype, start = yt[i], i
                while i < len(yt) and yt[i] == etype:
                    i += 1
                seg = yp[start:i]
                detected = np.any(seg != 'N')
                type_ok = np.any(seg == etype)
                delay = next((j for j in range(len(seg)) if seg[j] != 'N'), -1)
                rows.append({'scenario': sid, 'type': etype, 'length': i - start,
                             'detected': detected, 'type_correct': type_ok, 'delay': delay})
            else:
                i += 1
    return pd.DataFrame(rows) if rows else pd.DataFrame()


def evaluate_unified(y_true, y_pred, meta, name=""):
    """窗口级 + 事件级 + 运营级 统一评估"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    print(f"\n{'='*60}\n  {name}\n{'='*60}")

    # --- 窗口级 ---
    bin_p = precision_score(y_true != 'N', y_pred != 'N', zero_division=0)
    bin_r = recall_score(y_true != 'N', y_pred != 'N', zero_division=0)
    bin_f1 = f1_score(y_true != 'N', y_pred != 'N', zero_division=0)
    f1_m = f1_score(y_true, y_pred, labels=['N', 'AP', 'CP'], average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=['N', 'AP', 'CP'])
    cp_r = cm[2, 2] / cm[2].sum() if cm[2].sum() > 0 else 0
    cp_p = cm[2, 2] / cm[:, 2].sum() if cm[:, 2].sum() > 0 else 0

    print(f"[窗口] Binary: P={bin_p:.3f} R={bin_r:.3f} F1={bin_f1:.3f}")
    print(f"       Macro-F1={f1_m:.3f}  CP: recall={cp_r:.3f} precision={cp_p:.3f}")
    print(f"       CM:\n{pd.DataFrame(cm, index=['t_N','t_AP','t_CP'], columns=['p_N','p_AP','p_CP'])}")

    # 场景一致性
    fold_f1s = []
    for sid in meta['scenario_id'].unique():
        sm = (meta['scenario_id'] == sid).values
        fold_f1s.append(f1_score(y_true[sm] != 'N', y_pred[sm] != 'N', zero_division=0))
    print(f"       Per-scenario F1: mean={np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")

    # --- 事件级 ---
    events = _extract_events(y_true, y_pred, meta)
    evt_metrics = {}
    if len(events) > 0:
        print(f"\n[事件] 共 {len(events)} 个事件")
        for et in ['AP', 'CP']:
            sub = events[events['type'] == et]
            if len(sub) == 0:
                continue
            det = sub['detected'].mean()
            tc = sub['type_correct'].mean()
            det_sub = sub[sub['detected']]
            avg_d = det_sub['delay'].mean() if len(det_sub) > 0 else -1
            print(f"  {et}: n={len(sub)}, detected={det:.1%}, type_correct={tc:.1%}, "
                  f"avg_delay={avg_d:.1f} win ({avg_d * 15:.0f} min)")
            evt_metrics[f'{et}_detect_rate'] = det
            evt_metrics[f'{et}_type_correct'] = tc
            evt_metrics[f'{et}_avg_delay'] = avg_d

    # --- 运营级 ---
    n_alerts = (y_pred != 'N').sum()
    n_sc = meta['scenario_id'].nunique()
    days_per_sc = len(y_pred) / 96 / n_sc
    daily = n_alerts / n_sc / max(days_per_sc, 1)
    print(f"\n[运营] 总预警={n_alerts}, 日均/场景={daily:.1f}, 预警精准率={bin_p:.3f}")

    return {
        'binary_f1': bin_f1, 'binary_p': bin_p, 'binary_r': bin_r,
        'macro_f1': f1_m, 'cp_recall': cp_r, 'cp_precision': cp_p,
        'daily_alerts': daily, 'scenario_f1_std': np.std(fold_f1s),
        **evt_metrics,
    }

# 运行分类器

In [ ]:
pc_targets_list = [mc.get('target') for mc in config_raw.get('tft_models', {}).values()
                   if mc.get('target')]
print(f"PC Targets: {pc_targets_list}")

### 特征构造

In [ ]:
# if 'results_PC' in dir() and results_PC:
#     print("合并TFT信号...")
#     df_all_25 = merge_tft_signals_pc(df_all, results_PC, engine_PC)
#     print(f"合并后列数: {len(df_all_25.columns)}")
# else:
#     df_all_25 = df_all.copy()
#     print("无TFT结果，仅使用PC Score特征")

In [ ]:
# 标签统一
label_mapping = {'A': 'AP', 'C': 'CP', 'Anomaly': 'AP',
                 'anomaly_point': 'AP', 'changepoint': 'CP'}
df_all['label'] = df_all['label'].replace(label_mapping)

In [ ]:
df_all.columns

In [ ]:
# 扩展 CP 标签
df_all_extended = extend_cp_labels(df_all, cp_extend_windows=32)

In [ ]:
df_all_extended.columns

In [ ]:
df_train.columns

In [ ]:
# 构造特征(纯训练集、纯标签、映射关系、基线统计量)
X_ext, y_ext, meta_ext, baseline_stats = build_all_features_pc(
    df_all_extended, df_train, tb, engine_PC, results_PC,
    pc_targets=pc_targets_list
)

print(f"\nCP样本量: {(y_ext == 'CP').sum()}")
print(f"AP样本量: {(y_ext == 'AP').sum()}")
print(f"N样本量:  {(y_ext == 'N').sum()}")

In [ ]:
X_ext.columns

In [ ]:
for i,j in enumerate(list(X_ext.columns)):
    print(i,j)

## LOSO

In [ ]:
import pandas as pd
import numpy as np

print("="*60)
print(" 🕵️ 信号链路追踪诊断 (SID = 0)")
print("="*60)

# 选定第一个场景
mask_sid = meta_ext['scenario_id'] == 0
X_diag = X_ext[mask_sid].copy()
y_diag = y_ext[mask_sid].copy()

# 要检查的核心特征 (如果你的列名不同，请微调)
check_cols = [
    'comment_pc1_zscore',  # 1. PCA空间还有信号吗？
    'post_pc2_zscore',
    'energy_shift_ratio',  # 2. 相对漂移特征起效了吗？
]
# 加上 TFT 信号
tft_cols = [c for c in X_diag.columns if 'residual_ratio' in c and 'roll' not in c]
check_cols.extend(tft_cols)

check_cols = [c for c in check_cols if c in X_diag.columns]
print(check_cols)
# 分离正常和异常数据


In [ ]:
mask_n = y_diag == 'N'
mask_anom = y_diag != 'N'

print(f"提取出 {mask_n.sum()} 个正常窗口，{mask_anom.sum()} 个异常窗口。\n")

for col in check_cols:
    vals_n = X_diag.loc[mask_n, col].dropna()
    vals_anom = X_diag.loc[mask_anom, col].dropna()
    
    if len(vals_n) == 0 or len(vals_anom) == 0:
        continue
        
    mean_n, std_n = vals_n.mean(), vals_n.std()
    mean_a = vals_anom.mean()
    
    # 计算效应量 Cohen's d
    d = abs(mean_a - mean_n) / (std_n + 1e-8)
    
    # 显著性标记
    if d < 0.2:
        status = "❌ 信号全无 (淹没在噪声中)"
    elif d < 0.5:
        status = "⚠️ 信号微弱"
    else:
        status = "✅ 信号显著"
        
    print(f"🔹 {col:40s}")
    print(f"   正常均值: {mean_n:8.4f} | 异常均值: {mean_a:8.4f}")
    print(f"   Cohen's d: {d:6.3f}  ->  {status}")
    print("-" * 60)

In [ ]:
# 在调用 lgbm_loso_pc 之前加入

# ★ 确认 meta 中不存在 cp_drift
if 'regime' in meta_ext.columns:
    n_drift = (meta_ext['regime'] == 'cp_drift').sum()
    if n_drift > 0:
        print(f"⚠️ meta_ext 中仍有 {n_drift} 行 cp_drift，将移除")
        valid_mask = meta_ext['regime'] != 'cp_drift'
        X_ext = X_ext[valid_mask].reset_index(drop=True)
        y_ext = y_ext[valid_mask].reset_index(drop=True)
        meta_ext = meta_ext[valid_mask].reset_index(drop=True)
    
print(f"[LOSO 前最终校验]")
print(f"  X_ext: {X_ext.shape}")
print(f"  标签分布:\n{y_ext.value_counts()}")
print(f"  预期 CP/N/AP 比例: CP≈{(y_ext=='CP').sum()}, AP≈{(y_ext=='AP').sum()}, N≈{(y_ext=='N').sum()}")

# 合理性检查
n_scenarios = meta_ext['scenario_id'].nunique()
avg_per_scenario = len(X_ext) / n_scenarios
print(f"  场景数: {n_scenarios}, 平均每场景: {avg_per_scenario:.0f} 行")
# 干净数据每场景约 30000+ 行（假设 df_normal 有 ~30k 行）
# 排除 cp_drift 后应该差不多

In [ ]:


# ============================================================
# LOSO（共用一次训练，E_X / E_Y 各自取结果）
# ============================================================
oof_df, mean_best_iter = lgbm_loso_pc(X_ext, y_ext, meta_ext)


In [ ]:
oof_meta = oof_df[['time_idx', 'scenario_id']]


In [ ]:
plot_multiclass_evaluation_V2(oof_df)

### 评估分类器必要性

In [ ]:
# ==================== TFT直接判断 vs LGBM 决策 ====================

print("=" * 70)
print("  TFT 直接判断 vs LGBM LOSO 决策")
print("=" * 70)

# 1. TFT 直接判断
tft_results_summary = {}
for mn in engine_PC.models:
    bl = engine_PC.baselines[mn]
    target = engine_PC.config['tft_models'][mn]['target']
    metrics = results_PC[mn]['metrics']
    
    residual_abs = metrics['residual'].abs()
    
    # 多阈值扫描
    for k in [1.0, 1.5, 2.0, 3.0]:
        threshold = bl['residual_p95'] * k
        y_true_bin = (df_all_25['label'] != 'N').astype(int)
        y_pred_tft = (residual_abs > threshold).astype(int)
        n = min(len(y_true_bin), len(y_pred_tft))
        
        from sklearn.metrics import f1_score, precision_score, recall_score
        f1 = f1_score(y_true_bin[:n], y_pred_tft[:n], zero_division=0)
        prec = precision_score(y_true_bin[:n], y_pred_tft[:n], zero_division=0)
        rec = recall_score(y_true_bin[:n], y_pred_tft[:n], zero_division=0)
        
        tft_results_summary[f'{mn}_k{k}'] = {
            'P': prec, 'R': rec, 'F1': f1, 'threshold': threshold
        }
        
        if k == 1.5:
            print(f"  [{mn}] target={target}, k=1.5x P95:")
            print(f"    P={prec:.3f} R={rec:.3f} F1={f1:.3f}")

# 2. 对比 LGBM (从oof_df)
if 'oof_df' in dir():
    lgbm_f1 = f1_score(
        oof_df['y_true'] != 'N', 
        oof_df['y_pred_argmax'] != 'N', 
        zero_division=0
    )
    lgbm_macro = f1_score(
        oof_df['y_true'], oof_df['y_pred_argmax'],
        labels=['N', 'AP', 'CP'], average='macro', zero_division=0
    )
    print(f"\n  [LGBM LOSO] binary_F1={lgbm_f1:.3f}, macro_F1={lgbm_macro:.3f}")

# 3. 自动决策
best_tft_f1 = max(v['F1'] for v in tft_results_summary.values())
print(f"\n  TFT 最佳 binary_F1 = {best_tft_f1:.3f}")

print("\n" + "=" * 70)
if best_tft_f1 > 0.5:
    print("  📌 TFT 直接判断有一定效果，但建议仍用 LGBM 融合")
    print("     原因：LGBM 能融合多尺度统计特征 + 多TFT模型信号")
else:
    print("  📌 TFT 直接判断效果不足，必须用 LGBM 融合")
    print("     TFT 的价值 = 提供 residual/attention/VSN 变化信号作为 LGBM 特征")

print("\n  ✅ 推荐方案: LGBM LOSO + 分层预警（当前架构）")
print("     TFT 作为特征提供者，LGBM 作为决策器")
print("=" * 70)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# 1. 过滤掉无用的数据进行评估
valid_mask = oof_df['y_true'] != 'N'  # 实际为异常的样本
all_mask = oof_df['y_true'].notna()

# 2. 计算真正衡量能力的 ROC-AUC (二分类：正常 vs 异常)
y_binary_true = (oof_df['y_true'] != 'N').astype(int)
auc_score = roc_auc_score(y_binary_true, oof_df['anomaly_score'])

print("="*60)
print(f"🚀 LightGBM 真实的概率排序能力 (ROC-AUC): {auc_score:.4f}")
print("   * AUC > 0.70 说明具有良好的判别力")
print("   * AUC > 0.80 说明特征工程非常成功")
print("="*60)

# 3. 绘制概率密度分布对比图
plt.figure(figsize=(12, 6))
sns.kdeplot(data=oof_df[oof_df['y_true'] == 'N'], x='anomaly_score', 
            fill=True, label='True Normal (N)', color='#2ca02c', alpha=0.3)
sns.kdeplot(data=oof_df[oof_df['y_true'] == 'AP'], x='anomaly_score', 
            fill=True, label='True Anomaly (AP)', color='#ff7f0e', alpha=0.4)
sns.kdeplot(data=oof_df[oof_df['y_true'] == 'CP'], x='anomaly_score', 
            fill=True, label='True Changepoint (CP)', color='#d62728', alpha=0.4)

plt.axvline(0.5, color='black', linestyle=':', label='Argmax Threshold (0.5)')
plt.title(f'LightGBM Anomaly Score Distribution (Overall AUC = {auc_score:.4f})', fontsize=14, fontweight='bold')
plt.xlabel('Anomaly Score (1 - prob_N)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.xlim(-0.05, 1.05)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/anomaly_score_distribution.png', dpi=150)
plt.show()

### 分类器

In [ ]:
# ==================== 诊断单元格：确认根因 ====================

print("=" * 70)
print("  诊断 1: 注入异常在 PC Score 空间的可见性")
print("=" * 70)

# 从 df_all_extended 中取一个有异常的场景
for sid in sorted(df_all_extended['scenario_id'].unique())[:5]:
    df_s = df_all_extended[df_all_extended['scenario_id'] == sid]
    mask_drift = df_s['regime'] != 'cp_drift'
    df_s = df_s[mask_drift]
    
    for pc in pc_targets_list:
        n_vals = df_s.loc[df_s['label'] == 'N', pc]
        ap_vals = df_s.loc[df_s['label'] == 'AP', pc]
        cp_vals = df_s.loc[df_s['label'] == 'CP', pc]
        
        if len(ap_vals) == 0 and len(cp_vals) == 0:
            continue
        
        # Cohen's d
        for lbl, vals in [('AP', ap_vals), ('CP', cp_vals)]:
            if len(vals) == 0:
                continue
            d = abs(vals.mean() - n_vals.mean()) / (n_vals.std() + 1e-8)
            print(f"  SID={sid} | {pc} | {lbl}: Cohen_d={d:.3f}, "
                  f"N_mean={n_vals.mean():.4f}, {lbl}_mean={vals.mean():.4f}")

print("\n" + "=" * 70)
print("  诊断 2: energy 在异常窗口的分布")
print("=" * 70)

# 用 baseline_stats 计算 energy # 如果不能直接import就在上面的cell里复用

for sid in sorted(df_all_extended['scenario_id'].unique())[:3]:
    df_s = df_all_extended[df_all_extended['scenario_id'] == sid].copy()
    mask_drift = df_s['regime'] != 'cp_drift'
    df_s = df_s[mask_drift]
    
    X_diag, _ = build_causal_features_pc(
        df_s, pc_targets=pc_targets_list, tb=tb, baseline_stats=baseline_stats
    )
    
    energy_n = X_diag.loc[df_s['label'] == 'N', 'energy']
    energy_ap = X_diag.loc[df_s['label'] == 'AP', 'energy']
    energy_cp = X_diag.loc[df_s['label'] == 'CP', 'energy']
    
    print(f"\n  SID={sid}:")
    print(f"    energy_N:  P50={energy_n.median():.3f}, P95={energy_n.quantile(0.95):.3f}")
    if len(energy_ap) > 0:
        print(f"    energy_AP: P50={energy_ap.median():.3f}, P95={energy_ap.quantile(0.95):.3f}, "
              f"Cohen_d={abs(energy_ap.mean()-energy_n.mean())/(energy_n.std()+1e-8):.3f}")
    if len(energy_cp) > 0:
        print(f"    energy_CP: P50={energy_cp.median():.3f}, P95={energy_cp.quantile(0.95):.3f}, "
              f"Cohen_d={abs(energy_cp.mean()-energy_n.mean())/(energy_n.std()+1e-8):.3f}")

print("\n" + "=" * 70)
print("  诊断 3: TFT 信号列是否按场景区分")
print("=" * 70)

tft_cols_check = [c for c in df_all_extended.columns 
                  if any(kw in c for kw in ['residual_ratio', 'att_kl', 'vsn_js'])]
print(f"  TFT 信号列: {tft_cols_check}")

if tft_cols_check:
    # 检查不同场景的 TFT 信号是否相同
    for col in tft_cols_check[:2]:
        vals_by_scenario = []
        for sid in sorted(df_all_extended['scenario_id'].unique())[:3]:
            v = df_all_extended.loc[df_all_extended['scenario_id'] == sid, col].values
            vals_by_scenario.append(v)
        
        # 如果所有场景的值完全相同 → TFT信号没有按场景重推理
        if len(vals_by_scenario) >= 2:
            same = np.array_equal(vals_by_scenario[0][:100], vals_by_scenario[1][:100])
            print(f"  {col}: SID0 vs SID1 前100值相同? {same}")
            if same:
                print(f"    ⚠️ TFT信号未按场景重推理! 所有场景共享同一套值!")
else:
    print("  ⚠️ 没有找到 TFT 信号列! LGBM 完全没有 TFT 特征输入!")

print("\n" + "=" * 70)
print("  诊断 4: LGBM anomaly_score 分布")
print("=" * 70)

print(f"  anomaly_score 分位数 (全体):")
for q in [0.5, 0.75, 0.9, 0.95, 0.99]:
    print(f"    P{q*100:.0f} = {oof_df['anomaly_score'].quantile(q):.4f}")

print(f"\n  anomaly_score 分位数 (真实异常):")
anom_scores = oof_df.loc[oof_df['y_true'] != 'N', 'anomaly_score']
for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
    print(f"    P{q*100:.0f} = {anom_scores.quantile(q):.4f}")

print(f"\n  anomaly_score 分位数 (正常):")
normal_scores = oof_df.loc[oof_df['y_true'] == 'N', 'anomaly_score']
for q in [0.5, 0.75, 0.9, 0.95, 0.99]:
    print(f"    P{q*100:.0f} = {normal_scores.quantile(q):.4f}")

# 重叠度
overlap_thresh = 0.3
n_normal_above = (normal_scores > overlap_thresh).sum()
n_anom_below = (anom_scores < overlap_thresh).sum()
print(f"\n  概率重叠度 (threshold={overlap_thresh}):")
print(f"    正常窗口 score>{overlap_thresh}: {n_normal_above} ({n_normal_above/len(normal_scores)*100:.2f}%)")
print(f"    异常窗口 score<{overlap_thresh}: {n_anom_below} ({n_anom_below/len(anom_scores)*100:.2f}%)")

print("\n" + "=" * 70)
print("  诊断 5: 特征重要性 Top 10")
print("=" * 70)

if 'final_lgbm' in dir():
    imp = pd.Series(
        final_lgbm.booster_.feature_importance(importance_type='gain'),
        index=feature_config['feature_columns']
    ).sort_values(ascending=False)
    print("  Top 10 (gain):")
    for feat, val in imp.head(10).items():
        is_tft = any(kw in feat for kw in ['residual_ratio', 'att_kl', 'vsn_js', 'vsn_rank'])
        tag = " [TFT]" if is_tft else ""
        print(f"    {feat:45s}: {val:.0f}{tag}")
    
    # TFT 特征的总重要性占比
    tft_imp = sum(v for f, v in imp.items() 
                  if any(kw in f for kw in ['residual_ratio', 'att_kl', 'vsn_js', 'vsn_rank']))
    total_imp = imp.sum()
    print(f"\n  TFT 信号特征总重要性占比: {tft_imp/total_imp*100:.1f}%")

In [ ]:
# ==================== 诊断：注入模式合理性验证 ====================

print("=" * 70)
print("  注入异常 vs 真实异常 模式对齐诊断")
print("=" * 70)

# ---- 1. 注入数据的 energy 分布 vs 真实数据 ----
print("\n📌 诊断A: Energy 分布对比（注入 vs 真实）")

# 注入数据（从 df_all_extended 获取）
mask_no_drift = df_all_extended['regime'] != 'cp_drift'
df_inj = df_all_extended[mask_no_drift]

# 计算注入数据的 energy
X_inj_sample, bs_inj = build_causal_features_pc(
    df_inj[df_inj['scenario_id'] == 0].copy(),
    pc_targets=pc_targets_list, tb=tb, baseline_stats=baseline_stats
)
energy_inj_n = X_inj_sample.loc[df_inj[df_inj['scenario_id']==0]['label'].values == 'N', 'energy']
energy_inj_ap = X_inj_sample.loc[df_inj[df_inj['scenario_id']==0]['label'].values == 'AP', 'energy']
energy_inj_cp = X_inj_sample.loc[df_inj[df_inj['scenario_id']==0]['label'].values == 'CP', 'energy']

print(f"  [注入] energy_N:  P50={energy_inj_n.median():.3f}, P95={energy_inj_n.quantile(0.95):.3f}")
if len(energy_inj_ap) > 0:
    print(f"  [注入] energy_AP: P50={energy_inj_ap.median():.3f}, P95={energy_inj_ap.quantile(0.95):.3f}, "
          f"Cohen_d={abs(energy_inj_ap.mean()-energy_inj_n.mean())/(energy_inj_n.std()+1e-8):.3f}")
if len(energy_inj_cp) > 0:
    print(f"  [注入] energy_CP: P50={energy_inj_cp.median():.3f}, P95={energy_inj_cp.quantile(0.95):.3f}, "
          f"Cohen_d={abs(energy_inj_cp.mean()-energy_inj_n.mean())/(energy_inj_n.std()+1e-8):.3f}")

# 真实数据（从 df_real_transformed 获取）
X_real_all, bs_real = build_causal_features_pc(
    df_real_transformed.copy(),
    pc_targets=pc_targets_list, tb=tb, baseline_stats=baseline_stats
)
energy_real = X_real_all['energy']
print(f"\n  [真实] energy: P50={energy_real.median():.3f}, P95={energy_real.quantile(0.95):.3f}, "
      f"P99={energy_real.quantile(0.99):.3f}")

# 对比：真实数据异常段（从 all_segments 获取）
print(f"\n  [真实异常段] energy 统计:")
for mn, segs in all_segments.items():
    if len(segs) == 0:
        continue
    seg_energies = []
    for _, seg in segs.iterrows():
        seg_mask = (df_real_transformed['time_idx'] >= seg['start_idx']) & \
                   (df_real_transformed['time_idx'] <= seg['end_idx'])
        seg_idx = df_real_transformed[seg_mask].index
        if len(seg_idx) > 0 and all(i in X_real_all.index for i in seg_idx):
            seg_energies.extend(X_real_all.loc[seg_idx, 'energy'].values)
    if seg_energies:
        seg_e = np.array(seg_energies)
        print(f"    [{mn}] P50={np.median(seg_e):.3f}, P95={np.quantile(seg_e, 0.95):.3f}, "
              f"ratio_to_normal_P50={np.median(seg_e)/energy_real.median():.2f}×")

# ---- 2. 特征形态对比 ----
print(f"\n\n📌 诊断B: 特征变化率形态对比（注入AP vs 真实异常段）")

# 注入AP的 energy_diff_1 分布
if len(energy_inj_ap) > 0:
    ediff1_ap = X_inj_sample.loc[df_inj[df_inj['scenario_id']==0]['label'].values == 'AP', 'energy_diff_1']
    ediff1_n = X_inj_sample.loc[df_inj[df_inj['scenario_id']==0]['label'].values == 'N', 'energy_diff_1']
    print(f"  [注入AP] energy_diff_1: P50={ediff1_ap.median():.4f}, P95={ediff1_ap.quantile(0.95):.4f}")
    print(f"  [注入N]  energy_diff_1: P50={ediff1_n.median():.4f}, P95={ediff1_n.quantile(0.95):.4f}")

# 真实异常段的 energy_diff_1 分布
real_anom_diff1 = []
real_norm_diff1 = X_real_all['energy_diff_1'].values
for mn, segs in all_segments.items():
    for _, seg in segs.iterrows():
        seg_mask = (df_real_transformed['time_idx'] >= seg['start_idx']) & \
                   (df_real_transformed['time_idx'] <= seg['end_idx'])
        seg_idx = df_real_transformed[seg_mask].index
        if len(seg_idx) > 0 and all(i in X_real_all.index for i in seg_idx):
            real_anom_diff1.extend(X_real_all.loc[seg_idx, 'energy_diff_1'].values)

if real_anom_diff1:
    rad = np.array(real_anom_diff1)
    print(f"\n  [真实异常] energy_diff_1: P50={np.median(rad):.4f}, P95={np.quantile(rad, 0.95):.4f}")
    print(f"  [真实全体] energy_diff_1: P50={np.median(real_norm_diff1):.4f}, P95={np.quantile(real_norm_diff1, 0.95):.4f}")

# ---- 3. TFT 信号场景一致性 ----
print(f"\n\n📌 诊断C: TFT 信号是否按场景区分")

tft_cols_check = [c for c in df_all_extended.columns 
                  if any(kw in c for kw in ['residual_ratio', 'att_kl', 'vsn_js'])]

if tft_cols_check:
    for col in tft_cols_check[:3]:
        vals_by_scenario = {}
        for sid in sorted(df_all_extended['scenario_id'].unique())[:5]:
            v = df_all_extended.loc[df_all_extended['scenario_id'] == sid, col].dropna().values
            if len(v) > 0:
                vals_by_scenario[sid] = v
        
        if len(vals_by_scenario) >= 2:
            sids = list(vals_by_scenario.keys())
            v0, v1 = vals_by_scenario[sids[0]], vals_by_scenario[sids[1]]
            n_comp = min(100, len(v0), len(v1))
            same = np.allclose(v0[:n_comp], v1[:n_comp], atol=1e-6)
            corr = np.corrcoef(v0[:n_comp], v1[:n_comp])[0, 1] if n_comp > 2 else 1.0
            print(f"  {col}:")
            print(f"    SID{sids[0]} vs SID{sids[1]} 前{n_comp}值: same={same}, corr={corr:.4f}")
            if same:
                print(f"    ⚠️ 完全相同！TFT信号未因注入不同而变化")
                print(f"    → 原因: TFT只在干净数据上推理了一次，25场景共享同一残差")
                print(f"    → 影响: LGBM学到的TFT特征权重在部署时不可靠")
            else:
                print(f"    ✅ TFT信号因场景不同而不同")
else:
    print("  ⚠️ 无TFT信号列！请检查 merge_tft_signals_pc 是否执行")

# ---- 4. Baseline Stats 一致性 ----
print(f"\n\n📌 诊断D: Baseline Stats 一致性（训练时 vs 部署时）")

for pc in pc_targets_list:
    bs_train = baseline_stats.get(pc, {})
    # 模拟部署时的 baseline（从冷启动期数据计算）
    warmup_data = df_real_transformed.head(warmup_windows)
    if pc in warmup_data.columns:
        vals_warmup = warmup_data[pc].dropna()
        bs_deploy = {
            'median': float(vals_warmup.median()),
            'mad': float((vals_warmup - vals_warmup.median()).abs().median()) * 1.4826,
            'p95': float(vals_warmup.quantile(0.95)),
        }
        
        median_shift = abs(bs_train['median'] - bs_deploy['median']) / (bs_train['mad'] + 1e-8)
        scale_ratio = bs_deploy['mad'] / (bs_train['mad'] + 1e-8)
        
        print(f"  {pc}:")
        print(f"    训练baseline: median={bs_train['median']:.4f}, mad={bs_train['mad']:.4f}, p95={bs_train['p95']:.4f}")
        print(f"    部署baseline: median={bs_deploy['median']:.4f}, mad={bs_deploy['mad']:.4f}, p95={bs_deploy['p95']:.4f}")
        print(f"    中位数偏移: {median_shift:.2f}σ, 尺度比: {scale_ratio:.2f}×")
        
        if median_shift > 1.0:
            print(f"    ⚠️ 中位数偏移 >1σ，训练时和部署时的'正常'水平不同")
        if abs(scale_ratio - 1.0) > 0.3:
            print(f"    ⚠️ 尺度差异 >30%，z-score 的含义在训练和部署时不一致")

# ---- 5. 注入形态 vs 真实形态 ----
print(f"\n\n📌 诊断E: 注入形态 vs 真实形态分布对比")

# 真实形态（从 Phase 4 获取）
if 'all_morphology' in dir() and all_morphology:
    all_morph = pd.concat(all_morphology.values(), ignore_index=True)
    real_morph_dist = all_morph['morphology'].value_counts(normalize=True)
    print("  真实形态分布:")
    for m, p in real_morph_dist.items():
        print(f"    {m:25s}: {p:.1%}")
    
    # 注入形态（从 configs 推断）
    print("\n  当前注入方法权重:")
    for stage, methods in A_STAGE_METHOD_POOL.items():
        weights = A_STAGE_METHOD_WEIGHTS.get(stage, [])
        for m, w in zip(methods, weights):
            print(f"    [{stage}] {m:25s}: {w:.0%}")
    
    # 对齐建议
    print("\n  形态映射关系:")
    mapping = {
        'event_shock': 'gradual_recovery (衰减部分)',
        'gradual_drift': 'gradual_drift + sustained_shift',
        'volatility_burst': 'volatility_burst',
        'correlated_anomaly': 'mixed (多特征协同)',
    }
    for inject_m, real_m in mapping.items():
        print(f"    {inject_m:25s} → {real_m}")
    
    # 检查关键缺口
    if real_morph_dist.get('gradual_recovery', 0) > 0.4:
        inject_shock_weight = np.mean([
            A_STAGE_METHOD_WEIGHTS[s][A_STAGE_METHOD_POOL[s].index('event_shock')]
            for s in A_STAGE_METHOD_POOL if 'event_shock' in A_STAGE_METHOD_POOL[s]
        ])
        if inject_shock_weight < 0.4:
            print(f"\n  ⚠️ 真实 gradual_recovery={real_morph_dist.get('gradual_recovery',0):.0%}，"
                  f"但 event_shock 平均权重仅 {inject_shock_weight:.0%}")
            print(f"     建议提高 event_shock 权重到 ≥50%")

print(f"\n{'='*70}")
print("  诊断完成")
print(f"{'='*70}")

In [ ]:
# ==================== 可视化：注入 vs 真实 energy 分布对比 ====================

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# ---- 1. Energy 分布对比（注入N/AP/CP vs 真实全体/异常段）----
ax1 = fig.add_subplot(gs[0, 0])

# 注入数据
sid0_mask = (df_all_extended['scenario_id'] == 0) & (df_all_extended['regime'] != 'cp_drift')
df_sid0 = df_all_extended[sid0_mask]
X_sid0, _ = build_causal_features_pc(
    df_sid0.copy(), pc_targets=pc_targets_list, tb=tb, baseline_stats=baseline_stats
)

for lbl, color, alpha in [('N', 'green', 0.3), ('AP', 'orange', 0.5), ('CP', 'red', 0.5)]:
    mask_lbl = df_sid0['label'].values == lbl
    if mask_lbl.sum() > 0:
        ax1.hist(X_sid0.loc[mask_lbl, 'energy'], bins=80, alpha=alpha, 
                 density=True, label=f'注入_{lbl}', color=color)

# 真实数据
X_real_diag, _ = build_causal_features_pc(
    df_real_transformed.copy(), pc_targets=pc_targets_list, tb=tb, baseline_stats=baseline_stats
)
ax1.hist(X_real_diag['energy'], bins=80, alpha=0.3, density=True, 
         label='真实_全体', color='blue', histtype='step', linewidth=2)

ax1.set_xlabel('Energy')
ax1.set_title('Energy 分布对比')
ax1.legend(fontsize=8)

# ---- 2. energy_diff_1 对比 ----
ax2 = fig.add_subplot(gs[0, 1])

for lbl, color in [('N', 'green'), ('AP', 'orange')]:
    mask_lbl = df_sid0['label'].values == lbl
    if mask_lbl.sum() > 0:
        vals = X_sid0.loc[mask_lbl, 'energy_diff_1'].clip(-5, 5)
        ax2.hist(vals, bins=80, alpha=0.4, density=True, label=f'注入_{lbl}', color=color)

real_diff1 = X_real_diag['energy_diff_1'].clip(-5, 5)
ax2.hist(real_diff1, bins=80, alpha=0.3, density=True, 
         label='真实_全体', color='blue', histtype='step', linewidth=2)

ax2.set_xlabel('energy_diff_1')
ax2.set_title('变化率分布对比')
ax2.legend(fontsize=8)

# ---- 3. energy_zscore 对比 ----
ax3 = fig.add_subplot(gs[0, 2])

for lbl, color in [('N', 'green'), ('AP', 'orange'), ('CP', 'red')]:
    mask_lbl = df_sid0['label'].values == lbl
    if mask_lbl.sum() > 0:
        vals = X_sid0.loc[mask_lbl, 'energy_zscore'].clip(-5, 15)
        ax3.hist(vals, bins=80, alpha=0.4, density=True, label=f'注入_{lbl}', color=color)

real_zs = X_real_diag['energy_zscore'].clip(-5, 15)
ax3.hist(real_zs, bins=80, alpha=0.3, density=True, 
         label='真实_全体', color='blue', histtype='step', linewidth=2)

ax3.set_xlabel('energy_zscore')
ax3.set_title('Z-score 分布对比')
ax3.legend(fontsize=8)

# ---- 4. 各PC的 zscore 对比 ----
for i, pc in enumerate(pc_targets_list[:2]):
    ax = fig.add_subplot(gs[1, i])
    zs_col = f'{pc}_zscore'
    
    if zs_col in X_sid0.columns:
        for lbl, color in [('N', 'green'), ('AP', 'orange'), ('CP', 'red')]:
            mask_lbl = df_sid0['label'].values == lbl
            if mask_lbl.sum() > 0:
                vals = X_sid0.loc[mask_lbl, zs_col].clip(-8, 8)
                ax.hist(vals, bins=80, alpha=0.4, density=True, label=f'注入_{lbl}', color=color)
    
    if zs_col in X_real_diag.columns:
        real_vals = X_real_diag[zs_col].clip(-8, 8)
        ax.hist(real_vals, bins=80, alpha=0.3, density=True, 
                label='真实_全体', color='blue', histtype='step', linewidth=2)
    
    ax.set_xlabel(zs_col)
    ax.set_title(f'{pc} Z-score 对比')
    ax.legend(fontsize=8)

# ---- 5. TFT 信号场景一致性 ----
ax5 = fig.add_subplot(gs[1, 2])
tft_check_cols = [c for c in df_all_extended.columns if 'residual_ratio' in c]
if tft_check_cols:
    col = tft_check_cols[0]
    for sid in range(min(5, df_all_extended['scenario_id'].nunique())):
        vals = df_all_extended.loc[df_all_extended['scenario_id'] == sid, col].values[:500]
        ax5.plot(vals, alpha=0.5, linewidth=0.5, label=f'SID={sid}')
    ax5.set_title(f'TFT信号 跨场景一致性\n({col})')
    ax5.legend(fontsize=7)
    ax5.set_xlabel('窗口')
else:
    ax5.text(0.5, 0.5, '无TFT信号列', ha='center', va='center', transform=ax5.transAxes)
    ax5.set_title('TFT信号 跨场景一致性')

plt.suptitle('注入异常 vs 真实数据 模式对齐诊断', fontsize=14, fontweight='bold')
plt.savefig('./output/injection_vs_real_alignment.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 对齐诊断图已保存")

#### ※A1B1C1

In [ ]:

# ============================================================
# E_X: 三分类 argmax
# ============================================================
results_ex = evaluate_unified(
    oof_df['y_true'].values,
    oof_df['y_pred_argmax'].values,
    oof_meta, name="E_X: 三分类 argmax"
)


#### ※A1B1C2 25场景最优

In [ ]:

# ============================================================
# E_Y: 分层预警（自动阈值）
# ============================================================
best_params = optimize_layered_thresholds(oof_df, tb)
y_pred_layered = simulate_layered_alerts(oof_df, best_params)

results_ey = evaluate_unified(
    oof_df['y_true'].values,
    y_pred_layered,
    oof_meta, name="E_Y: 分层预警"
)


### 评估

In [ ]:

# ============================================================
# 对比
# ============================================================
comp = pd.DataFrame([results_ex, results_ey],
                     index=['E_X (argmax)', 'E_Y (layered)'])
print(f"\n{'='*60}\n对比总表\n{'='*60}")
print(comp.T.to_string())

### 保存

In [ ]:

# 3. 特征配置
feature_config = {
    'baseline_median': tb.quiet_energy_median,
    'baseline_std': tb.quiet_energy_mad,
    'ap_threshold': tb.quiet_energy_p95,
    'feature_columns': X_ext.columns.tolist(),
    'tft_signal_cols': [c for c in X_ext.columns if 'model_' in c],
    'label_map': {'N': 0, 'AP': 1, 'CP': 2},
    'inv_label_map': {0: 'N', 1: 'AP', 2: 'CP'},
    'cp_extend_windows': 32,
}


In [ ]:
# ============================================================
# 提取公共模型参数 (保持与 LOSO 验证时完全一致)
# ============================================================

LGBM_PARAMS = {
    'max_depth': 5,
    'num_leaves': 31,
    'min_child_samples': 50,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'random_state': 42,
    'verbose': -1,
    'num_class': 3,
    'objective': 'multiclass'
}


In [ ]:

def train_final_lgbm(X, y, params=LGBM_PARAMS,n_iterations=mean_best_iter):
    """
    用全量数据和确定的参数训练最终分类器。
    注意：全量训练没有验证集，所以不使用 early_stopping。
    """
    label_map = {'N': 0, 'AP': 1, 'CP': 2}
    y_enc = y.map(label_map)
    
    # 严格使用与 lgbm_loso_v2 相同的权重计算逻辑
    counts = y.value_counts()
    n_total = len(y)
    base = n_total / (3 * counts.get('N', 1))
    max_ratio = 80  # 与 LOSO 中保持一致
    weights = {}
    for label, code in label_map.items():
        raw = n_total / (3 * counts.get(label, 1))
        weights[code] = min(raw / base, max_ratio) * base
    sample_w = y_enc.map(weights).values
    
    # 组合参数并指定确定的迭代次数
    final_params = params.copy()
    final_params['n_estimators'] = n_iterations
    
    model = lgb.LGBMClassifier(**final_params)
    
    # 直接使用 100% 数据进行拟合，无需 eval_set
    model.fit(X, y_enc, sample_weight=sample_w)
    
    print(f"[Final LGBM] Trained on {len(X)} samples with {n_iterations} iterations")
    print(f"  N={counts.get('N',0)}, AP={counts.get('AP',0)}, CP={counts.get('CP',0)}")
    
    return model

# ============================================================
# 执行与保存
# ============================================================
import os
import pickle

In [ ]:
CKPT = "./checkpoints/pretrained_planB"
os.makedirs(CKPT, exist_ok=True)

# 训练最终分类器（全量数据）

final_lgbm = train_final_lgbm(X_ext, y_ext, params=LGBM_PARAMS, n_iterations=mean_best_iter)



In [ ]:

with open(f"{CKPT}/lgbm_classifier.pkl", 'wb') as f:
    pickle.dump(final_lgbm, f)
with open(f"{CKPT}/feature_config.pkl", 'wb') as f:
    pickle.dump(feature_config, f)
with open(f"{CKPT}/processor.pkl", 'wb') as f:
    pickle.dump(processor, f)
with open(f"{CKPT}/layered_params.pkl", 'wb') as f:
    pickle.dump(best_params, f)

print(f"✅ 全部保存到 {CKPT}")

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
import numpy as np
import re
import shap

# ============================================================
# 辅助函数：将工程变量名映射为学术论文标准名称
# ============================================================
def format_feature_name(feat):
    # TFT 偏差信号映射
    if feat.startswith('model_'):
        target = 'PC1' if 'pc1' in feat else '能量'
        signal = '残差比' if 'residual_ratio' in feat else \
                 '散度比' if 'div_ratio' in feat else \
                 '注意力KL散度' if 'att_kl' in feat else \
                 'VSN JS散度' if 'vsn_js' in feat else '偏差'
        
        stat_match = re.search(r'roll(max|mean)_(\d+)', feat)
        if stat_match:
            stat = 'Max' if stat_match.group(1) == 'max' else 'Mean'
            k = stat_match.group(2)
            return f"TFT {signal} ({target}, {stat}, k={k})"
        return f"TFT {signal} ({target})"
    
    # 统计特征映射
    if feat == 'max_abs_z': return '最大绝对Z分数'
    if feat == 'run_length': return '连续超阈值窗口数'
    if feat == 'dist_from_normal': return '偏离稳态时间'
    if feat == 'energy_shift_ratio': return '序列前后均值偏移比'
    if feat.startswith('energy_trend_'): return f"特征能量演化趋势 (k={feat.split('_')[-1]})"
    
    # 滑动窗口统计量映射
    match = re.match(r'(energy|above_ratio)_(mean|max|std|ratio)_(\d+)', feat)
    if match:
        base = '特征能量' if match.group(1) == 'energy' else '能量超阈值比例'
        stat = {'mean': '均值', 'max': '最大值', 'std': '标准差', 'ratio': ''}.get(match.group(2), '')
        k = match.group(3)
        return f"{base}{stat} (k={k})"
    
    return feat

# ============================================================
# 1. 特征重要性排序图（split + gain 双视角）
# ============================================================
fig_dir = "./figures"
os.makedirs(fig_dir, exist_ok=True)

importance_types = ['split', 'gain']
fig, axes = plt.subplots(1, 2, figsize=(20, 12))

for ax, imp_type in zip(axes, importance_types):
    raw_feat_imp = pd.Series(
        final_lgbm.booster_.feature_importance(importance_type=imp_type),
        index=feature_config['feature_columns']
    ).sort_values(ascending=True).tail(30)
    
    # 应用名称映射
    raw_feat_imp.index = raw_feat_imp.index.map(format_feature_name)

    colors = []
    for feat in raw_feat_imp.index:
        if 'TFT' in feat:
            colors.append('#e74c3c')   # TFT信号：红色
        elif '能量' in feat or 'Z分数' in feat:
            colors.append('#2980b9')   # 能量类：蓝色
        elif any(kw in feat for kw in ['窗口数', '时间', '偏移比']):
            colors.append('#27ae60')   # 持续性：绿色
        else:
            colors.append('#95a5a6')   # 其他：灰色

    bars = ax.barh(raw_feat_imp.index, raw_feat_imp.values, color=colors, edgecolor='white', height=0.7)
    ax.set_xlabel(f'Importance ({imp_type})', fontsize=12)
    ax.set_title(f'Top 30 Feature Importance ({imp_type})', fontsize=13, fontweight='bold')
    ax.tick_params(axis='y', labelsize=10)

    for bar, val in zip(bars, raw_feat_imp.values):
        ax.text(val + raw_feat_imp.values.max() * 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.0f}', va='center', ha='left', fontsize=8)

    legend_elements = [
        Patch(facecolor='#e74c3c', label='TFT微观偏差信号'),
        Patch(facecolor='#2980b9', label='宏观能量统计特征'),
        Patch(facecolor='#27ae60', label='时序持续性特征')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(f'Global Feature Importance in LightGBM Model\n'
             f'(n_iter={mean_best_iter}, Validation Setup: LOSO)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
imp_path = f"{fig_dir}/final_lgbm_feature_importance_academic.png"
plt.savefig(imp_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ 学术版特征重要性图已保存: {imp_path}")

# ============================================================
# 2. 各类别 Top15 特征（基于 SHAP 均值绝对值）
# ============================================================
explainer = shap.TreeExplainer(final_lgbm)
sample_idx = np.random.choice(len(X_ext), size=min(1000, len(X_ext)), replace=False)
shap_values = explainer.shap_values(X_ext.iloc[sample_idx])

label_names = ['Normal (N)', 'Anomaly Point (AP)', 'Change Point (CP)']
is_list = isinstance(shap_values, list)
fig2, axes2 = plt.subplots(1, 3, figsize=(22, 8))

for cls_idx, cls_name in enumerate(label_names):
    sv_arr = shap_values[cls_idx] if is_list else shap_values[:, :, cls_idx]
    mean_abs = pd.Series(
        np.abs(sv_arr).mean(axis=0),
        index=feature_config['feature_columns']
    ).sort_values(ascending=True).tail(15)
    
    # 应用名称映射
    mean_abs.index = mean_abs.index.map(format_feature_name)

    axes2[cls_idx].barh(mean_abs.index, mean_abs.values,
                        color=['#e74c3c' if 'TFT' in f else '#2980b9' for f in mean_abs.index],
                        edgecolor='white', height=0.7)
    axes2[cls_idx].set_title(f'SHAP Mean |value| — {cls_name}', fontsize=12, fontweight='bold')
    axes2[cls_idx].set_xlabel('Mean |SHAP value|', fontsize=10)
    axes2[cls_idx].tick_params(axis='y', labelsize=9)
    axes2[cls_idx].grid(axis='x', alpha=0.3, linestyle='--')
    axes2[cls_idx].spines['top'].set_visible(False)
    axes2[cls_idx].spines['right'].set_visible(False)

plt.suptitle('SHAP Feature Attribution by Event Class (Top 15)', fontsize=14, fontweight='bold')
plt.tight_layout()
shap_path = f"{fig_dir}/final_lgbm_shap_importance_academic.png"
plt.savefig(shap_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ 学术版SHAP归因图已保存: {shap_path}")

# 正式训练

## 检测

In [ ]:
from tft_full_period_utils import compute_baseline_from_held_out, create_held_out_split

In [ ]:
# ============================================================
# 冷启动期处理补丁
# ============================================================
def apply_warmup_mask(df, time_col='time_idx',
                      warmup_windows=384,
                      label_col='pred_label'):
    df = df.copy()
    min_idx = df[time_col].min()
    warmup_end = min_idx + warmup_windows
    warmup_mask = df[time_col] < warmup_end
    n_warmup = warmup_mask.sum()
    df['is_warmup'] = warmup_mask.astype(int)
    df['pred_label_raw'] = df[label_col]
    df.loc[warmup_mask, label_col] = 'N'
    if 'anomaly_score' in df.columns:
        df.loc[warmup_mask, 'anomaly_score'] = 0.0
    suppressed = (df.loc[warmup_mask, 'pred_label_raw'] != 'N').sum()
    print(f"[Warmup] 冷启动期: time_idx < {warmup_end} ({n_warmup} windows, {n_warmup*15/60:.0f}h)")
    print(f"  抑制了 {suppressed} 个冷启动期内的异常预测")
    return df



In [ ]:
# ==================== 单元格: 在线推理单窗口（PC Score 版）====================

def _infer_single_window_pc(df_context, engine, tb, lgbm_model, 
                             feature_config, layered_params,
                             pc_targets, baseline_stats,
                             verbose=False):
    """
    PC Score 版在线推理单窗口
    替代旧版 _infer_single_window
    """
    import io, contextlib
    
    df = tb.transform(df_context.copy())
    
    # TFT 推理
    tft_results = {}
    for mn in engine.models:
        if verbose:
            tft_results[mn] = engine.analyze_rolling(mn, df, baseline_end_idx=None)
        else:
            with contextlib.redirect_stdout(io.StringIO()), \
                 contextlib.redirect_stderr(io.StringIO()):
                tft_results[mn] = engine.analyze_rolling(mn, df, baseline_end_idx=None)
    
    # 合并 TFT 信号
    df = merge_tft_signals_pc(df, tft_results, engine)
    
    # 构造 PC 因果特征
    X, _ = build_causal_features_pc(
        df,
        pc_targets=pc_targets,
        tb=tb,
        baseline_stats=baseline_stats,
    )
    
    # TFT 信号的 rolling 特征
    tft_cols = feature_config.get('tft_signal_cols', [])
    for col in tft_cols:
        if col in df.columns:
            X[col] = df[col].values
            for k in [4, 8]:
                X[f'{col}_rollmax_{k}'] = df[col].rolling(k, min_periods=1).max().values
                X[f'{col}_rollmean_{k}'] = df[col].rolling(k, min_periods=1).mean().values
    
    # 对齐特征列
    X = X.reindex(columns=feature_config['feature_columns'], fill_value=0)
    
    # LGBM 预测
    inv_map = feature_config['inv_label_map']
    proba = lgbm_model.predict_proba(X)
    df['pred_label_ex'] = [inv_map[int(p)] for p in proba.argmax(axis=1)]
    df['anomaly_score'] = 1 - proba[:, 0]
    df['prob_N'] = proba[:, 0]
    df['prob_AP'] = proba[:, 1]
    df['prob_CP'] = proba[:, 2]
    
    # 分层预警
    if layered_params is not None:
        oof_like = pd.DataFrame({
            'energy': X['energy'].values if 'energy' in X.columns else 0,
            'energy_zscore': X['energy_zscore'].values if 'energy_zscore' in X.columns else 0,
            'anomaly_score': df['anomaly_score'].values,
            'prob_N': df['prob_N'].values,
            'prob_AP': df['prob_AP'].values,
            'prob_CP': df['prob_CP'].values,
            'scenario_id': 0,
            'time_idx': df['time_idx'].values,
        })
        # TFT residual_ratio 列
        for c in df.columns:
            if 'residual_ratio' in c:
                oof_like[c] = df[c].values
        # run_length / energy_shift_ratio（分层V2需要）
        if 'run_length' in X.columns:
            oof_like['run_length'] = X['run_length'].values
        if 'energy_shift_ratio' in X.columns:
            oof_like['energy_shift_ratio'] = X['energy_shift_ratio'].values
        
        df['pred_label_ey'] = simulate_layered_alerts(oof_like, layered_params)
    else:
        df['pred_label_ey'] = df['pred_label_ex']
    
    last = df.iloc[[-1]].copy()
    return last

In [ ]:


# ============================================================
# Step 1: 在线检测（逐窗口推进）
# ============================================================
# def step1_detect_formal(df_features_transformed, engine, tb,
#                         lgbm_model, feature_config,
#                         layered_params=None,
#                         warmup_windows=192,
#                         start_time=None,
#                         is_warm_start=False,
#                         encoder_context=96,       # TFT encoder所需上下文窗口数
#                         save_path="./output/formal_detection.csv"):
#     """
#     在线模式：逐窗口推进，维护滑动上下文buffer
#     start_time: 开始输出结果的时间点，之前数据仅作encoder上下文
#     is_warm_start: True时跳过冷启动抑制
#     """
#     import os
#     os.makedirs(os.path.dirname(save_path), exist_ok=True)

#     df = df_features_transformed.copy()

#     if start_time is not None:
#         start_time = pd.to_datetime(start_time)
#         context_start = start_time - pd.Timedelta(minutes=15 * encoder_context)
#         df = df[df['timestamp'] >= context_start].copy()
#         print(f"[Start] 从 {start_time} 启动, 上下文起点 {context_start}")

#     timestamps = df['timestamp'].values
#     output_rows = []

#     print(f"[Online] 共 {len(df)} 窗口，逐窗口推进...")
#     for i in range(encoder_context, len(df)):
#         # 滑动上下文：取前encoder_context行作为上下文 + 当前行
#         ctx = df.iloc[max(0, i - encoder_context): i + 1].copy()
#         row = _infer_single_window(ctx, engine, tb, lgbm_model, feature_config, layered_params)
#         output_rows.append(row)

#         if (i - encoder_context) % 96 == 0:
#             ts = pd.to_datetime(timestamps[i])
#             print(f"  [{i}/{len(df)}] {ts}")

#     result_df = pd.concat(output_rows, ignore_index=True)

#     # 冷启动抑制
#     print("[Warmup]...")
#     if is_warm_start:
#         print("  热启动：跳过冷启动抑制")
#         result_df['is_warmup'] = 0
#     else:
#         for col in ['pred_label_ex', 'pred_label_ey']:
#             result_df = apply_warmup_mask(result_df, warmup_windows=warmup_windows, label_col=col)

#     # 只保留start_time之后
#     if start_time is not None:
#         result_df = result_df[result_df['timestamp'] >= start_time].copy()
#         print(f"  输出窗口数: {len(result_df)} (从 {start_time} 起)")

#     for col in ['pred_label_ex', 'pred_label_ey']:
#         vc = result_df[col].value_counts()
#         print(f"  {col}: N={vc.get('N',0)}, AP={vc.get('AP',0)}, CP={vc.get('CP',0)}")

#     result_df.to_csv(save_path, index=False)
#     print(f"✅ {save_path}")
#     return result_df



In [ ]:

# def step1_detect_formal(df_features_transformed, engine, tb,
#                         lgbm_model, feature_config,
#                         layered_params=None,
#                         warmup_windows=192,
#                         start_time=None,
#                         is_warm_start=False,
#                         encoder_context=96,
#                         save_path="./output/formal_detection.csv"):
#     import os, sys
#     os.makedirs(os.path.dirname(save_path), exist_ok=True)

#     df = df_features_transformed.copy()

#     if start_time is not None:
#         start_time = pd.to_datetime(start_time)
#         context_start = start_time - pd.Timedelta(minutes=15 * encoder_context)
#         df = df[df['timestamp'] >= context_start].copy()

#     total = len(df)
#     warmup_end_i = encoder_context + warmup_windows  # 绝对行号分界
#     output_rows = []

#     # 阶段标记
#     _last_phase = [None]

#     def _print_phase(phase, i, ts, extra=''):
#         if _last_phase[0] != phase:
#             if _last_phase[0] is not None:
#                 print()  # 换行分隔
#             phase_label = {
#                 'warmup':  '❄️  [冷启动期]',
#                 'normal':  '✅  [正常运行]',
#                 'parallel':'⚡  [并行对比]',
#             }[phase]
#             print(f"{phase_label} 开始 @ {ts}")
#             _last_phase[0] = phase

#     print(f"[Online] 共 {total} 窗口，逐窗口推进...")
#     for i in range(encoder_context, total):
#         ctx = df.iloc[max(0, i - encoder_context): i + 1].copy()
#         row = _infer_single_window(ctx, engine, tb, lgbm_model, feature_config, layered_params, verbose=False)
#         output_rows.append(row)

#         ts = pd.to_datetime(df['timestamp'].values[i])

#         # 判断当前阶段
#         if i < warmup_end_i:
#             phase = 'warmup'
#         else:
#             phase = 'normal'

#         _print_phase(phase, i, ts)

#         # 进度条（每96窗口刷新一行）
#         if (i - encoder_context) % 96 == 0:
#             pct = (i - encoder_context) / max(total - encoder_context, 1) * 100
#             bar_len = 30
#             filled = int(bar_len * pct / 100)
#             bar = '█' * filled + '░' * (bar_len - filled)
#             sys.stdout.write(f"\r  [{bar}] {pct:5.1f}%  {ts}")
#             sys.stdout.flush()

#     print()  # 最终换行

#     result_df = pd.concat(output_rows, ignore_index=True)

#     if is_warm_start:
#         result_df['is_warmup'] = 0
#     else:
#         for col in ['pred_label_ex', 'pred_label_ey']:
#             result_df = apply_warmup_mask(result_df, warmup_windows=warmup_windows, label_col=col)

#     if start_time is not None:
#         result_df = result_df[result_df['timestamp'] >= start_time].copy()

#     print(f"\n📊 检测完成，输出 {len(result_df)} 窗口")
#     for col in ['pred_label_ex', 'pred_label_ey']:
#         vc = result_df[col].value_counts()
#         print(f"  {col}: N={vc.get('N',0)}, AP={vc.get('AP',0)}, CP={vc.get('CP',0)}")

#     result_df.to_csv(save_path, index=False)
#     print(f"✅ {save_path}")
#     return result_df

In [ ]:
import tempfile, copy


In [ ]:

# ============================================================
# AdaptiveDetector（在线状态机）
# ============================================================
class AdaptiveDetector:
    """
    在线模式：step方法每次接收一个新窗口，维护状态机
    scan_and_adapt用于批量回测（逐窗口模拟在线推进）
    """
    DEFAULTS = dict(
        cp_confirm_windows=32,
        cp_onset_lookback=64,
        cp_onset_consecutive=2,
        accumulate_windows=672,
        held_out_hours_per_day=4,
        finetune_epochs=8,
        finetune_lr_scale=0.1,
        switch_agreement=0.95,
        switch_check_window=96,
        encoder_context=96,
    )

    def __init__(self, engine, tb, lgbm_model, feature_config,
                 layered_params, processor, **kwargs):
        self.engine = engine
        self.tb = tb
        self.lgbm = lgbm_model
        self.feat_cfg = feature_config
        self.layered_params = layered_params
        self.processor = processor
        self.cfg = {**self.DEFAULTS, **kwargs}
        self.cp_events = []

        # 在线状态
        self._cp_consec = 0                  # 连续CP计数器
        self._cp_confirmed = False           # 当前CP是否已确认
        self._onset_ts = None                # onset时间戳
        self._accumulate_buffer = []         # 积累期数据buffer（DataFrame列表）
        self._parallel_engine = None         # 微调后的新引擎
        self._parallel_processor = None      # 新引擎对应的processor（如果微调需要）
        self._parallel_buffer = []           # 并行对比buffer
        self._baseline_version = 0

    def step(self, new_row_df, df_context, df_official, pred_col='pred_label_ey'):
        """
        在线单步处理
        new_row_df: 当前窗口的检测结果（step1输出的单行DataFrame）
        df_context: 包含encoder上下文的原始特征DataFrame（用于微调/重推理）
        df_official: 官方运营日历
        返回: 带final_label/confidence/baseline_version的单行DataFrame
        """
        row = new_row_df.copy()
        row['final_label'] = row[pred_col].values[0]
        row['confidence'] = 'high'
        row['baseline_version'] = self._baseline_version

        label = row[pred_col].values[0]

        # ---- CP计数器 ----
        if label == 'CP':
            self._cp_consec += 1
        else:
            self._cp_consec = max(0, self._cp_consec - 1)

        # ---- CP确认 ----
        if (not self._cp_confirmed and
                self._cp_consec >= self.cfg['cp_confirm_windows']):
            self._cp_confirmed = True
            confirmed_ts = pd.to_datetime(row['timestamp'].values[0])
            self._onset_ts = self._estimate_onset_from_buffer(confirmed_ts)
            print(f"[CP CONFIRMED] at {confirmed_ts}, onset_est={self._onset_ts}")
            self.cp_events.append({
                'onset_est': self._onset_ts,
                'confirmed_at': confirmed_ts,
                'switched_at': None,
            })

        # ---- 积累期：onset之后收集数据 ----
        if self._cp_confirmed and self._parallel_engine is None:
            if (self._onset_ts is not None and
                    pd.to_datetime(row['timestamp'].values[0]) >= self._onset_ts):
                self._accumulate_buffer.append(df_context)
                row['confidence'] = 'degraded'

            # 积累够了触发微调
            if len(self._accumulate_buffer) >= self.cfg['accumulate_windows']:
                df_buf = pd.concat(self._accumulate_buffer, ignore_index=True).drop_duplicates('time_idx')
                print(f"[Adaptive] 积累完成({len(df_buf)}窗口)，开始微调...")
                self._parallel_engine = self._finetune(df_buf, df_official)
                self._parallel_buffer = []
                print("[Parallel] 新引擎就绪，开始并行对比")

        # ---- 并行对比阶段 ----
        if self._parallel_engine is not None and self._cp_confirmed:
                # 用新processor transform上下文，再推理
            ctx_transformed = self._parallel_processor.transform(
                df_context, df_official
            )
            row_new = _infer_single_window(
                ctx_transformed.data, self._parallel_engine,
                self.tb, self.lgbm, self.feat_cfg, self.layered_params
            )
            # 用新引擎对当前上下文重推理
            # row_new = _infer_single_window(
            #     df_context, self._parallel_engine,
            #     self.tb, self.lgbm, self.feat_cfg, self.layered_params
            # )
            pred_old = row[pred_col].values[0]
            pred_new = row_new[pred_col].values[0]
            row['pred_old'] = pred_old
            row['pred_new'] = pred_new

            # 融合
            fused_label, confidence = self._fuse_single(pred_old, pred_new)
            row['final_label'] = fused_label
            row['confidence'] = confidence
            self._parallel_buffer.append((pred_old, pred_new))

            # 切换判断
            if len(self._parallel_buffer) >= self.cfg['switch_check_window']:
                agree = np.mean([p == q for p, q in self._parallel_buffer])
                print(f"[Parallel] 一致率: {agree:.3f} ({len(self._parallel_buffer)}窗口)")
                if agree >= self.cfg['switch_agreement']:
                    self.engine = self._parallel_engine
                    self.processor = self._parallel_processor # ← 新增
                    self._baseline_version += 1
                    row['baseline_version'] = self._baseline_version
                    self.cp_events[-1]['switched_at'] = str(row['timestamp'].values[0])
                    print("[SWITCH] 已切换到新baseline")
                else:
                    print(f"[Parallel] 一致率不足，保持旧baseline")
                # 重置状态
                self._reset_cp_state()

        return row

    def _reset_cp_state(self):
        self._cp_confirmed = False
        self._cp_consec = 0
        self._onset_ts = None
        self._accumulate_buffer = []
        self._parallel_engine = None
        self._parallel_buffer = []

    def _estimate_onset_from_buffer(self, confirmed_ts):
        """从已有结果buffer回溯onset"""
        if not self._parallel_buffer and not self._accumulate_buffer:
            return confirmed_ts
        return confirmed_ts  # 无历史prob_CP时退化为确认时刻

    def scan_and_adapt(self, df_detected, df_features, df_official,
                       pred_col='pred_label_ey', start_time=None):
        """
        批量回测模式：逐行模拟在线step，等价于在线部署效果
        df_detected: step1_detect_formal的输出
        df_features: 完整原始特征（用于微调上下文）
        df_official: 官方运营日历
        """
        df = df_detected.sort_values('time_idx').reset_index(drop=True)
        output_rows = []

        enc = self.cfg['encoder_context']
        df_feat_sorted = df_features.sort_values('timestamp').reset_index(drop=True)

        print(f"[ScanAdapt] 逐窗口模拟在线推进，共{len(df)}窗口...")
        for i, idx in enumerate(df.index):
            row = df.iloc[[idx]]
            ts = pd.to_datetime(row['timestamp'].values[0])

            # 构造对应的特征上下文
            feat_i = df_feat_sorted[df_feat_sorted['timestamp'] <= ts].tail(enc + 1).copy()

            result_row = self.step(row, feat_i, df_official, pred_col=pred_col)
            output_rows.append(result_row)

            if i % 96 == 0:
                print(f"  [{i}/{len(df)}] {ts} | cp_consec={self._cp_consec} | ver={self._baseline_version}")

        df_final = pd.concat(output_rows, ignore_index=True).sort_values('time_idx')
        print(f"[ScanAdapt] 完成，baseline切换次数={self._baseline_version}")
        return df_final

    def _estimate_onset(self, df, confirmed_idx):
        idx_pos = df.index.get_loc(confirmed_idx)
        lb = min(idx_pos, self.cfg['cp_onset_lookback'])
        probs = df['prob_CP'].iloc[idx_pos - lb:idx_pos + 1].values

        if len(probs) > 10:
            base = np.median(probs[:len(probs) // 2])
            th = base + 2 * max(np.std(probs[:len(probs) // 2]), 0.02)
        else:
            th = 0.1

        consec = self.cfg['cp_onset_consecutive']
        for i in range(len(probs)):
            if all(probs[j] > th for j in range(i, min(i + consec, len(probs)))):
                onset_idx = idx_pos - lb + i
                return pd.to_datetime(df.iloc[onset_idx]['timestamp'])
        return pd.to_datetime(df.loc[confirmed_idx, 'timestamp'])

    # def _finetune(self, df_buffer, df_official):
    #     import tempfile
    #     print(f"[FINETUNE] {len(df_buffer)} windows")
    #     with tempfile.TemporaryDirectory() as tmp:
    #         self.engine.save(tmp)
    #         engine_new = TFTEngine.load(tmp, config_path='TFT_config.yaml')

    #     df_buf = self.tb.transform(df_buffer.copy())
    #     tft_ds = self.processor.transform(df_buf, df_official)
    #     df_tft = tft_ds.data

    #     mask = create_held_out_split(df_tft, held_out_days=self.cfg['held_out_hours_per_day'] / 24)
    #     held_tidx = set(df_tft.loc[mask, 'time_idx'].values)

    #     for mn in engine_new.models:
    #         engine_new.unfreeze_all(mn)
    #         cfg = engine_new.config['tft_models'][mn]
    #         orig_lr = cfg.get('learning_rate', 0.03)
    #         cfg['learning_rate'] = orig_lr * self.cfg['finetune_lr_scale']
    #         engine_new.build_and_fit(mn, df_tft[~mask], val_ratio=0.2,
    #                                  max_epochs=self.cfg['finetune_epochs'])
    #         cfg['learning_rate'] = orig_lr
    #         res = engine_new.analyze_rolling(mn, df_tft, baseline_end_idx=None)
    #         engine_new.baselines[mn] = (
    #             compute_baseline_from_held_out(res, held_tidx) if held_tidx
    #             else engine_new._build_baseline(
    #                 res['metrics'], res['attention'], res['vsn'],
    #                 int(df_tft['time_idx'].max()))
    #         )
    #         engine_new.freeze_layers(mn)
    #     return engine_new

    def _finetune(self, df_buffer, df_official):
        print(f"[FINETUNE] {len(df_buffer)} windows")

        # 克隆引擎
        with tempfile.TemporaryDirectory() as tmp:
            self.engine.save(tmp)
            engine_new = TFTEngine.load(tmp, config_path='TFT_config.yaml')

        # ── 新增：克隆processor，仅重新fit feature_transformer ──
        import copy
        processor_new = copy.deepcopy(self.processor)
        processor_new.feature_transformer.fit(df_buffer)          # 用CP后数据重拟合分布参数
        # global_min_time 保持不变（time_idx基准不能漂移）
        # event_processor 保持不变（事件类别集合不变）
        # ────────────────────────────────────────────────────────

        # 准备数据（用新processor transform）
        df_buf = self.tb.transform(df_buffer.copy())
        tft_ds = processor_new.transform(df_buf, df_official)      # ← 用processor_new
        df_tft = tft_ds.data

        mask = create_held_out_split(df_tft, held_out_days=self.cfg['held_out_hours_per_day'] / 24)
        held_tidx = set(df_tft.loc[mask, 'time_idx'].values)

        for mn in engine_new.models:
            # engine_new.unfreeze_all(mn)
            engine_new.freeze_layers(mn)
            cfg = engine_new.config['tft_models'][mn]
            orig_lr = cfg.get('learning_rate', 0.03)
            cfg['learning_rate'] = orig_lr * self.cfg['finetune_lr_scale']
            engine_new.build_and_fit(mn, df_tft[~mask], val_ratio=0.2,
                                    max_epochs=self.cfg['finetune_epochs'])
            cfg['learning_rate'] = orig_lr
            res = engine_new.analyze_rolling(mn, df_tft, baseline_end_idx=None)
            engine_new.baselines[mn] = (
                compute_baseline_from_held_out(res, held_tidx) if held_tidx
                else engine_new._build_baseline(
                    res['metrics'], res['attention'], res['vsn'],
                    int(df_tft['time_idx'].max()))
            )
            engine_new.freeze_layers(mn)

        return engine_new, processor_new       

    def _fuse_single(self, pred_old, pred_new):
        if pred_old == pred_new:
            return pred_new, 'high'
        elif pred_old != 'N' and pred_new == 'N':
            return 'N', 'corrected'
        elif pred_old == 'N' and pred_new != 'N':
            return pred_new, 'low'
        else:
            return pred_new, 'medium'

    def _fuse_parallel(self, df_old, df_new, pred_col):
        merged = df_old[['timestamp', 'time_idx', pred_col, 'anomaly_score',
                          'prob_CP', 'feature_energy']].copy()
        merged.rename(columns={pred_col: 'pred_old'}, inplace=True)
        merged['pred_new'] = df_new[pred_col].values

        def fuse_row(r):
            return self._fuse_single(r['pred_old'], r['pred_new'])

        merged[['final_label', 'confidence']] = merged.apply(
            fuse_row, axis=1, result_type='expand')
        return merged

    def save_state(self, path="./checkpoints/adaptive_state.pkl"):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'wb') as f:
            pickle.dump({'cp_events': self.cp_events, 'cfg': self.cfg}, f)

## 检测函数

In [ ]:
def step1_detect_formal(df_features_transformed, engine, tb,
                        lgbm_model, feature_config,
                        layered_params=None,
                        warmup_windows=192,
                        start_time=None,
                        is_warm_start=False,
                        encoder_context=96,
                        save_path="./output/formal_detection.csv",
                        adaptive_detector=None,   # ← 新增，传入detector实例
                        df_features_raw=None,     # ← 新增，用于微调的原始特征
                        df_official=None):        # ← 新增
    import os, sys, io, contextlib
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    df = df_features_transformed.copy()

    if start_time is not None:
        start_time = pd.to_datetime(start_time)
        context_start = start_time - pd.Timedelta(minutes=15 * encoder_context)
        df = df[df['timestamp'] >= context_start].copy()

    total = len(df)
    warmup_end_i = encoder_context + warmup_windows
    output_rows = []
    _last_phase = [None]

    def _print_phase(phase, ts):
        if _last_phase[0] != phase:
            if _last_phase[0] is not None:
                print()
            phase_label = {
                'warmup':  '❄️  [冷启动期]',
                'normal':  '✅  [正常运行]',
                'parallel':'⚡  [并行对比]',
            }[phase]
            print(f"{phase_label} 开始 @ {ts}")
            _last_phase[0] = phase

    print(f"[Online] 共 {total} 窗口，逐窗口推进...")

    for i in range(encoder_context, total):
        ctx = df.iloc[max(0, i - encoder_context): i + 1].copy()
        ts = pd.to_datetime(df['timestamp'].values[i])

        # ── 当前窗口推理（旧引擎）──
        row = _infer_single_window(
            ctx, engine, tb, lgbm_model, feature_config, layered_params, verbose=False
        )
        row['final_label'] = row['pred_label_ey'].values[0]
        row['confidence'] = 'high'
        row['baseline_version'] = 0

        # ── 交给AdaptiveDetector处理CP逻辑 ──
        if adaptive_detector is not None and df_features_raw is not None:
            # 构造原始特征上下文（用于微调）
            df_feat_sorted = df_features_raw.sort_values('timestamp')
            feat_ctx = df_feat_sorted[df_feat_sorted['timestamp'] <= ts].tail(encoder_context + 1).copy()

            row = adaptive_detector.step(
                row, feat_ctx, df_official,
                pred_col='pred_label_ey'
            )
            # 同步引擎（detector可能已切换）
            engine = adaptive_detector.engine

        # ── 阶段显示 ──
        if adaptive_detector is not None and adaptive_detector._parallel_engine is not None:
            phase = 'parallel'
        elif i < warmup_end_i:
            phase = 'warmup'
        else:
            phase = 'normal'
        _print_phase(phase, ts)

        output_rows.append(row)

        if (i - encoder_context) % 96 == 0:
            pct = (i - encoder_context) / max(total - encoder_context, 1) * 100
            bar_len = 30
            filled = int(bar_len * pct / 100)
            bar = '█' * filled + '░' * (bar_len - filled)
            sys.stdout.write(f"\r  [{'█'*filled}{'░'*(bar_len-filled)}] {pct:5.1f}%  {ts}")
            sys.stdout.flush()

    print()

    result_df = pd.concat(output_rows, ignore_index=True)

    # 冷启动抑制
    if is_warm_start:
        result_df['is_warmup'] = 0
    else:
        for col in ['pred_label_ex', 'pred_label_ey']:
            result_df = apply_warmup_mask(result_df, warmup_windows=warmup_windows, label_col=col)

    if start_time is not None:
        result_df = result_df[result_df['timestamp'] >= start_time].copy()

    print(f"\n📊 检测完成，输出 {len(result_df)} 窗口")
    for col in ['pred_label_ex', 'pred_label_ey', 'final_label']:
        if col in result_df.columns:
            vc = result_df[col].value_counts()
            print(f"  {col}: N={vc.get('N',0)}, AP={vc.get('AP',0)}, CP={vc.get('CP',0)}")

    result_df.to_csv(save_path, index=False)
    print(f"✅ {save_path}")
    return result_df

## 归因

In [ ]:
from TFT_Attribution_engine import ContentAttributionEngine

In [ ]:

# ============================================================
# Step 2: 逐窗口归因（文本主题 + 视觉聚类）
# ============================================================
from TFT_Attribution_engine import DashboardGenerator
def step2_attribute_windows(df_detected, df_raw, tb,
                            attribution_engine,
                            pred_col='pred_label_ey',
                            max_windows=50,
                            save_dir="./output/attribution"):
    """
    对每个异常窗口独立进行：
    1. z-score 驱动特征排序
    2. 窗口内文本 → 词云 + 主题聚类
    3. 窗口内图片 → 高频簇提取
    4. 生成窗口级 dashboard
    """
    import os
    os.makedirs(save_dir, exist_ok=True)

    # 筛选异常窗口
    warmup_col = 'is_warmup' if 'is_warmup' in df_detected.columns else None
    mask = df_detected[pred_col] != 'N'
    if warmup_col:
        mask &= df_detected[warmup_col] == 0
    df_anom = df_detected[mask].sort_values('anomaly_score', ascending=False)

    if max_windows and len(df_anom) > max_windows:
        df_anom = df_anom.head(max_windows)

    print(f"归因窗口数: {len(df_anom)}")
    if len(df_anom) == 0:
        return pd.DataFrame()

    z_params = tb.z_params
    results = []

    for i, (idx, row) in enumerate(df_anom.iterrows()):
        ts = pd.to_datetime(row['timestamp'])
        label = row[pred_col]
        window_id = ts.strftime("%Y%m%d_%H%M")

        # --- 1. z-score 驱动特征 ---
        z_scores = {}
        for feat, p in z_params.items():
            if feat in row.index:
                z = abs((row[feat] - p['median']) / (p['scale'] + 1e-8))
                z_scores[feat] = min(z, tb.Z_CLIP)
        top_z = sorted(z_scores.items(), key=lambda x: x[1], reverse=True)[:3]

        # --- 2. 匹配原始数据窗口 ---
        w_start = ts
        w_end = ts + pd.Timedelta(minutes=15)
        raw_mask = ((df_raw['timestamp'] >= w_start) &
                    (df_raw['timestamp'] < w_end))
        df_window = df_raw[raw_mask]

        # --- 3. 文本归因（窗口级词云 + 主题） ---
        text_summary = None
        if len(df_window) > 0:
            text_summary = attribution_engine.analyze_text(
                df_window, window_id,
                title_info=f"{ts} [{label}]"
            )

        # --- 4. 视觉归因（窗口级高频图片） ---
        visual_summary = None
        if len(df_window) > 0 and 'image_urls' in df_window.columns:
            visual_summary = attribution_engine.vis_detector.analyze_window(
                df_window, image_col='image_urls'
            )

        # --- 5. 生成窗口级 Dashboard ---
        dash_dir = os.path.join(save_dir, window_id)
        os.makedirs(dash_dir, exist_ok=True)
        dash_path = os.path.join(dash_dir, "dashboard.png")

        DashboardGenerator.create_report(
            text_summary, visual_summary, dash_path,
            title_info=f"{ts} [{label}] score={row.get('anomaly_score',0):.3f}"
        )

        # --- 6. 记录 ---
        rec = {
            'timestamp': ts, 'pred_label': label,
            'anomaly_score': row.get('anomaly_score', 0),
            'prob_AP': row.get('prob_AP', 0),
            'prob_CP': row.get('prob_CP', 0),
            'feature_energy': row.get('feature_energy', 0),
            'n_raw_posts': len(df_window),
            'dashboard_path': dash_path,
        }
        for j, (feat, zv) in enumerate(top_z):
            rec[f'top{j+1}_feature'] = feat
            rec[f'top{j+1}_zscore'] = round(zv, 3)

        if text_summary and 'discovered_clusters' in text_summary:
            kw_all = []
            for cinfo in text_summary['discovered_clusters'].values():
                kw_all.extend(cinfo.get('keywords', [])[:3])
            rec['top_keywords'] = ', '.join(kw_all[:8])

        if visual_summary:
            rec['vis_concentration'] = visual_summary.get('vis_concentration', 0)
            rec['n_images'] = visual_summary.get('total_images', 0)

        results.append(rec)

        if (i + 1) % 10 == 0:
            print(f"  归因 {i+1}/{len(df_anom)}")

    df_attr = pd.DataFrame(results)
    attr_path = os.path.join(save_dir, "attribution_table.csv")
    df_attr.to_csv(attr_path, index=False, encoding='utf-8-sig')
    print(f"✅ 归因表: {attr_path} ({len(df_attr)} 窗口)")
    return df_attr

# 正式滚动训练

## 测试

In [ ]:
# 1. 执行数据预处理与特征工程（与训练集保持一致的 PCA 变换）
dataset_real = processor.transform(df_features.fillna(0), df_official_real)
df_real_processed = tb.transform(dataset_real.data)

# 2. 如果真实数据中没有 label 和 regime，需初始化默认值以保证代码兼容性
if 'label' not in df_real_processed.columns:
    df_real_processed['label'] = 'N'
if 'regime' not in df_real_processed.columns:
    df_real_processed['regime'] = 'normal'
if 'scenario_id' not in df_real_processed.columns:
    df_real_processed['scenario_id'] = 0

In [ ]:
df_real_processed.columns

In [ ]:
import io, contextlib
import numpy as np
import pandas as pd

def build_real_features_pc(df_real, df_baseline, tb, engine_PC, pc_targets=None):
    """
    用于真实数据的 PC Score 特征构造函数。
    单场景顺序执行，保留去重防笛卡尔积逻辑，输出格式与训练集保持一致。
    
    Parameters
    ----------
    df_real : DataFrame
        经过 processor 和 tb.transform 转换后的真实数据。
    df_baseline : DataFrame
        纯净的正常历史数据，用于计算 Z-score 等统计量的标准化分母。
    """
    # 自动推断 PC targets
    if pc_targets is None:
        pc_targets = []
        for mn, mc in engine_PC.config.get('tft_models', {}).items():
            t = mc.get('target')
            if t and t in df_real.columns:
                pc_targets.append(t)
        if not pc_targets:
            pc_targets = [c for c in df_real.columns 
                          if ('_pc' in c and any(c.startswith(p) for p in ['post', 'comment']))]
    
    # 1. 计算基线统计量（从传入的 df_baseline 获取静态尺度）
    baseline_stats = {}
    for pc in pc_targets:
        if pc not in df_baseline.columns:
            continue
        vals = df_baseline[pc].dropna()
        baseline_stats[pc] = {
            'median': float(vals.median()),
            'std': float(vals.std()),
            'mad': float((vals - vals.median()).abs().median()) * 1.4826,
            'p95': float(vals.quantile(0.95)),
            'p05': float(vals.quantile(0.05)),
        }

    # 2. 时序排序与索引重置
    df_s = df_real.copy()
    if 'time_idx' in df_s.columns:
        df_s = df_s.sort_values('time_idx').reset_index(drop=True)
    elif 'timestamp' in df_s.columns:
        df_s = df_s.sort_values('timestamp').reset_index(drop=True)

    # 3. 执行 TFT 推理
    tft_results_s = {}
    for mn in engine_PC.models:
        with contextlib.redirect_stdout(io.StringIO()), \
             contextlib.redirect_stderr(io.StringIO()):
            res = engine_PC.analyze_rolling(mn, df_s, baseline_end_idx=None)
        
        # 执行去重，防止 merge 笛卡尔积导致行数增加
        if 'metrics' in res and res['metrics'] is not None:
            res['metrics'] = res['metrics'].drop_duplicates(subset=['time_idx'], keep='last')
        tft_results_s[mn] = res

    # 4. 删除高维特征，合并 TFT 信号
    drop_columns = ['total_volume_post', 'total_volume_comment', 'gini_post',
                    'gini_comment', 'senti_symbol_post', 'senti_symbol_comment',
                    'comp_ratio_post', 'comp_ratio_comment', 'retweet_ratio_post',
                    'total_short_post', 'total_long_post', 'total_short_comment',
                    'total_long_comment', 'neg_ratio_post', 'neg_ratio_comment',
                    'semantic_shift_post', 'semantic_shift_comment',
                    'vis_abs_redundancy_post', 'vis_concentration_post', 
                    'hour', 'dayofweek', 'month', 'day', 'is_weekend', 'is_holiday',
                    'time_slot', 'event_code']
    cols_to_drop = [c for c in drop_columns if c in df_s.columns]
    df_s_slim = df_s.drop(columns=cols_to_drop)
    
    df_s_merged = merge_tft_signals_pc(df_s_slim, tft_results_s, engine_PC)

    # 5. 构造因果特征与 TFT 滚动特征
    X_real, _ = build_causal_features_pc(
        df_s_merged, pc_targets=pc_targets, tb=tb, baseline_stats=baseline_stats
    )

    tft_cols = [c for c in df_s_merged.columns 
                if any(kw in c for kw in ['residual_ratio', 'div_ratio', 
                                           'att_kl', 'vsn_js', 'vsn_rank_shift'])]
    
    for col in tft_cols:
        if col in df_s_merged.columns:
            X_real[col] = df_s_merged[col].values
            for k in [4, 8]:
                X_real[f'{col}_rollmax_{k}'] = df_s_merged[col].rolling(k, min_periods=1).max().values
                X_real[f'{col}_rollmean_{k}'] = df_s_merged[col].rolling(k, min_periods=1).mean().values

    # 6. 元数据提取与绝对值特征过滤
    y_real = df_s_merged['label'].copy()
    meta_cols = ['time_idx', 'scenario_id', 'label', 'regime', 'timestamp']
    meta_cols_exist = [c for c in meta_cols if c in df_s_merged.columns]
    meta_real = df_s_merged[meta_cols_exist].copy()

    X_real = X_real.replace([np.inf, -np.inf], 0).fillna(0)
    
    # 删除存在泄露风险的绝对数值聚合特征
    shortcut_cols = [c for c in X_real.columns if ('_mean_' in c or '_max_' in c or 'above_' in c) and 'roll' not in c]
    if shortcut_cols:
        X_real = X_real.drop(columns=shortcut_cols)

    # 7. 最终校验
    print(f"[特征矩阵] X_real: {X_real.shape}")
    assert len(X_real) == len(df_real), f"输入数据行数({len(df_real)})与输出行数({len(X_real)})不一致。"

    return X_real, y_real, meta_real

In [ ]:
# 1. 提取真实数据特征
X_real, y_real, meta_real = build_real_features_pc(
    df_real=df_real_processed, 
    df_baseline=df_train, 
    tb=tb, 
    engine_PC=engine_PC
)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. 拼接特征与时间戳
df_eval = X_real.copy()
df_eval['timestamp'] = pd.to_datetime(meta_real['timestamp'])

# 2. 定义分析时间窗
# 正常期：4月24日 - 4月26日
# 危机期：4月27日 - 4月30日
mask_analysis = (df_eval['timestamp'] >= '2025-04-24') & (df_eval['timestamp'] < '2025-05-05')
df_window = df_eval[mask_analysis].copy()

df_window['period'] = 'Post-Crisis'
df_window.loc[(df_window['timestamp'] >= '2025-04-24') & (df_window['timestamp'] < '2025-04-27'), 'period'] = 'Normal'
df_window.loc[(df_window['timestamp'] >= '2025-04-27') & (df_window['timestamp'] <= '2025-04-30'), 'period'] = 'Crisis'

# 3. 选择待核验的底层核心特征 (排除TFT相关特征)
check_features = [
    'comment_pc1_zscore',             # 绝对波动 (单维)
    'post_pc2_zscore',                # 绝对波动 (单维)
    'energy_zscore',                  # 绝对波动 (多维综合)
    'comment_pc1_zs_diff_1',          # 瞬态突变率 (一阶导)
    'energy_diff_4',                  # 短期突变率
    'energy_shift_ratio',             # 结构性漂移率
    'comment_pc1_post_pc2_joint_z'    # 跨域协同突变
]

# 仅保留 X_real 中实际存在的列
check_features = [f for f in check_features if f in df_window.columns]

# 4. 统计指标量化对比
print("=" * 80)
print(" 真实特征在危机期间的分布变化 (脱离 TFT 模型)")
print("=" * 80)
print(f"{'Feature':<30} | {'Normal P95':<12} | {'Crisis P95':<12} | {'Crisis Max':<12} | {'P95 倍率':<10}")
print("-" * 80)

stats_records = []
for feat in check_features:
    norm_vals = df_window.loc[df_window['period'] == 'Normal', feat].dropna()
    crisis_vals = df_window.loc[df_window['period'] == 'Crisis', feat].dropna()
    
    if len(norm_vals) == 0 or len(crisis_vals) == 0:
        continue
        
    norm_p95 = norm_vals.quantile(0.95)
    crisis_p95 = crisis_vals.quantile(0.95)
    crisis_max = crisis_vals.max()
    
    ratio = crisis_p95 / (norm_p95 + 1e-5) if norm_p95 > 0 else float('inf')
    
    print(f"{feat:<30} | {norm_p95:<12.3f} | {crisis_p95:<12.3f} | {crisis_max:<12.3f} | {ratio:<10.2f}")

# 5. 可视化信号轨迹
fig, axes = plt.subplots(len(check_features), 1, figsize=(12, 2.5 * len(check_features)), sharex=True)
if len(check_features) == 1:
    axes = [axes]

crisis_start = pd.to_datetime('2025-04-27')
crisis_end = pd.to_datetime('2025-04-30 23:59:59')

for ax, feat in zip(axes, check_features):
    ax.plot(df_window['timestamp'], df_window[feat], color='black', linewidth=1)
    
    # 标注危机区间
    ax.axvspan(crisis_start, crisis_end, color='red', alpha=0.2, label='Crisis Period')
    
    # 绘制 Normal 期的 P95 基线作为参考
    norm_p95 = df_window.loc[df_window['period'] == 'Normal', feat].quantile(0.95)
    ax.axhline(norm_p95, color='green', linestyle='--', alpha=0.8, label=f'Normal P95 ({norm_p95:.2f})')
    
    ax.set_ylabel(feat)
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.xlabel('Timestamp')
plt.suptitle('底层特征在 4.27 危机期间的真实时序轨迹', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. 拼接特征与时间戳
df_eval = X_real.copy()
df_eval['timestamp'] = pd.to_datetime(meta_real['timestamp'])

# 2. 定义分析时间窗
# 建议将基线期提前，避开 4.25 的前期发酵波动，以获取更纯净的基准
mask_analysis = (df_eval['timestamp'] >= '2025-04-20') & (df_eval['timestamp'] < '2025-05-05')
df_window = df_eval[mask_analysis].copy()

df_window['period'] = 'Post-Crisis'
df_window.loc[(df_window['timestamp'] >= '2025-04-20') & (df_window['timestamp'] < '2025-04-26'), 'period'] = 'Normal'
df_window.loc[(df_window['timestamp'] >= '2025-04-27') & (df_window['timestamp'] <= '2025-04-30'), 'period'] = 'Crisis'

# 3. 动态获取所有候选特征
# 排除元数据列
exclude_cols = ['timestamp', 'time_idx', 'scenario_id', 'label', 'regime', 'period']
all_features = [c for c in df_window.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df_window[c])]

# 4. 计算所有特征的统计差异
stats_records = []
for feat in all_features:
    norm_vals = df_window.loc[df_window['period'] == 'Normal', feat].dropna()
    crisis_vals = df_window.loc[df_window['period'] == 'Crisis', feat].dropna()
    
    if len(norm_vals) == 0 or len(crisis_vals) == 0:
        continue
        
    norm_p95 = norm_vals.quantile(0.95)
    crisis_p95 = crisis_vals.quantile(0.95)
    crisis_max = crisis_vals.max()
    
    norm_mean, norm_std = norm_vals.mean(), norm_vals.std()
    crisis_mean = crisis_vals.mean()
    
    # 计算倍率与 Cohen's d (效应量)
    ratio = crisis_p95 / (norm_p95 + 1e-5) if norm_p95 > 0 else float('inf')
    cohen_d = abs(crisis_mean - norm_mean) / (norm_std + 1e-8)
    
    stats_records.append({
        'Feature': feat,
        'Normal P95': norm_p95,
        'Crisis P95': crisis_p95,
        'Crisis Max': crisis_max,
        'Ratio (P95)': ratio,
        'Cohen_d': cohen_d
    })

df_stats = pd.DataFrame(stats_records)

# 按 Cohen's d 降序排列，寻找分布差异最大的特征
df_stats = df_stats.sort_values(by='Cohen_d', ascending=False).reset_index(drop=True)

print("=" * 100)
print(" 真实特征在危机期间的分布差异评估 (按 Cohen's d 降序, Top 20)")
print("=" * 100)
print(f"{'Feature':<45} | {'Norm P95':<10} | {'Crisis P95':<10} | {'Ratio':<8} | {'Cohen_d':<8}")
print("-" * 100)
for _, row in df_stats.head(20).iterrows():
    print(f"{row['Feature']:<45} | {row['Normal P95']:<10.3f} | {row['Crisis P95']:<10.3f} | {row['Ratio (P95)']:<8.2f} | {row['Cohen_d']:<8.3f}")

# 5. 可视化 Top 10 差异特征的轨迹
top_n = 10
top_features = df_stats['Feature'].head(top_n).tolist()

fig, axes = plt.subplots(top_n, 1, figsize=(12, 2.5 * top_n), sharex=True)
if top_n == 1:
    axes = [axes]

crisis_start = pd.to_datetime('2025-04-27')
crisis_end = pd.to_datetime('2025-04-30 23:59:59')

for ax, feat in zip(axes, top_features):
    ax.plot(df_window['timestamp'], df_window[feat], color='black', linewidth=1)
    
    # 标注危机区间
    ax.axvspan(crisis_start, crisis_end, color='red', alpha=0.2, label='Crisis Period')
    
    # 绘制 Normal 期的 P95 基线
    norm_p95 = df_window.loc[df_window['period'] == 'Normal', feat].quantile(0.95)
    ax.axhline(norm_p95, color='green', linestyle='--', alpha=0.8, label=f'Normal P95 ({norm_p95:.2f})')
    
    ax.set_ylabel(feat, fontsize=8)
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.xlabel('Timestamp')
plt.suptitle('Top 10 特征在危机期间的时序轨迹 (包含 TFT 信号)', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:

# 2. 获取预训练分类器输出概率
# 注意: model 为 lgb.LGBMClassifier 实例
proba_real = model.predict_proba(X_real)

# 3. 构造预测结果 DataFrame
df_real_preds = pd.DataFrame({
    'timestamp': meta_real['timestamp'].values,
    'time_idx': meta_real['time_idx'].values,
    'prob_N': proba_real[:, 0],
    'prob_AP': proba_real[:, 1],
    'prob_CP': proba_real[:, 2],
    'anomaly_score': 1 - proba_real[:, 0]
})

# 将特定基础特征带入供分层预警使用
for col in ['energy_zscore', 'energy_shift_ratio']:
    if col in X_real.columns:
        df_real_preds[col] = X_real[col].values
        
for col in X_real.columns:
    if 'residual_ratio' in col and 'roll' not in col:
        df_real_preds[col] = X_real[col].values

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def evaluate_synthetic_scenario_signals(X_data, meta_data, target_sid=0, top_n=10):
    """
    评估指定合成场景中，各个特征对 AP 和 CP 的信号显著性。
    """
    # 1. 过滤指定场景数据
    mask_sid = meta_data['scenario_id'] == target_sid
    df_X = X_data[mask_sid].copy()
    df_meta = meta_data[mask_sid].copy()
    
    if len(df_X) == 0:
        print(f"场景 SID={target_sid} 无数据。")
        return
    
    # 将时间轴和标签加入特征表以供分析
    df_X['label'] = df_meta['label'].values
    df_X['time_idx'] = df_meta['time_idx'].values
    
    # 2. 提取所有候选数值特征
    exclude_cols = ['label', 'time_idx', 'scenario_id', 'regime', 'timestamp']
    features = [c for c in df_X.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df_X[c])]
    
    # 3. 计算统计差异指标
    stats_records = []
    for feat in features:
        vals_n = df_X.loc[df_X['label'] == 'N', feat].dropna()
        vals_ap = df_X.loc[df_X['label'] == 'AP', feat].dropna()
        vals_cp = df_X.loc[df_X['label'] == 'CP', feat].dropna()
        
        if len(vals_n) == 0:
            continue
            
        norm_mean, norm_std = vals_n.mean(), vals_n.std()
        norm_p95 = vals_n.quantile(0.95)
        
        # 计算 AP 指标
        if len(vals_ap) > 0:
            ap_p95 = vals_ap.quantile(0.95)
            ap_ratio = ap_p95 / (norm_p95 + 1e-5) if norm_p95 > 0 else float('inf')
            ap_cohen_d = abs(vals_ap.mean() - norm_mean) / (norm_std + 1e-8)
        else:
            ap_ratio, ap_cohen_d = 0.0, 0.0
            
        # 计算 CP 指标
        if len(vals_cp) > 0:
            cp_p95 = vals_cp.quantile(0.95)
            cp_ratio = cp_p95 / (norm_p95 + 1e-5) if norm_p95 > 0 else float('inf')
            cp_cohen_d = abs(vals_cp.mean() - norm_mean) / (norm_std + 1e-8)
        else:
            cp_ratio, cp_cohen_d = 0.0, 0.0
            
        stats_records.append({
            'Feature': feat,
            'Norm_P95': norm_p95,
            'AP_Ratio': ap_ratio,
            'AP_Cohen_d': ap_cohen_d,
            'CP_Ratio': cp_ratio,
            'CP_Cohen_d': cp_cohen_d,
            'Max_Cohen_d': max(ap_cohen_d, cp_cohen_d)
        })
        
    df_stats = pd.DataFrame(stats_records)
    df_stats = df_stats.sort_values(by='Max_Cohen_d', ascending=False).reset_index(drop=True)
    
    # 4. 打印统计表格
    print("=" * 110)
    print(f" 合成场景 SID={target_sid} 特征显著性评估 (按 Max_Cohen_d 降序, Top 20)")
    print("=" * 110)
    print(f"{'Feature':<45} | {'Norm P95':<10} | {'AP Ratio':<10} | {'AP Cohen_d':<12} | {'CP Ratio':<10} | {'CP Cohen_d':<10}")
    print("-" * 110)
    for _, row in df_stats.head(20).iterrows():
        print(f"{row['Feature']:<45} | {row['Norm_P95']:<10.3f} | {row['AP_Ratio']:<10.2f} | {row['AP_Cohen_d']:<12.3f} | {row['CP_Ratio']:<10.2f} | {row['CP_Cohen_d']:<10.3f}")

    # 5. 可视化 Top N 特征时序轨迹
    top_features = df_stats['Feature'].head(top_n).tolist()
    
    fig, axes = plt.subplots(top_n, 1, figsize=(14, 2.5 * top_n), sharex=True)
    if top_n == 1:
        axes = [axes]
        
    for ax, feat in zip(axes, top_features):
        # 绘制主曲线
        ax.plot(df_X['time_idx'], df_X[feat], color='black', linewidth=1)
        
        # 获取各标签的时间索引
        idx_ap = df_X.loc[df_X['label'] == 'AP', 'time_idx']
        idx_cp = df_X.loc[df_X['label'] == 'CP', 'time_idx']
        
        # 标注 AP 和 CP 区域 (使用 fill_between 以适配离散分布)
        if len(idx_ap) > 0:
            ax.fill_between(df_X['time_idx'], ax.get_ylim()[0], ax.get_ylim()[1], 
                            where=df_X['label']=='AP', color='orange', alpha=0.3, label='AP Period')
        if len(idx_cp) > 0:
            ax.fill_between(df_X['time_idx'], ax.get_ylim()[0], ax.get_ylim()[1], 
                            where=df_X['label']=='CP', color='red', alpha=0.3, label='CP Period')
            
        # 绘制基线
        norm_p95 = df_stats.loc[df_stats['Feature'] == feat, 'Norm_P95'].values[0]
        ax.axhline(norm_p95, color='green', linestyle='--', alpha=0.8, label=f'Norm P95 ({norm_p95:.2f})')
        
        ax.set_ylabel(feat, fontsize=8)
        # 避免重复图例
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.xlabel('Time Index')
    plt.suptitle(f'合成场景 SID={target_sid} Top {top_n} 差异特征轨迹', y=1.01, fontsize=14)
    plt.tight_layout()
    plt.show()

# 调用示例：对 X_ext 和 meta_ext 中 SID=0 的场景进行分析
evaluate_synthetic_scenario_signals(X_ext, meta_ext, target_sid=0, top_n=10)

# 滚动训练测试

## 滚动训练

In [ ]:
# ============================================================
# Cell 1: 加载 + 配置 + 预处理 + baseline建立
# ============================================================
import types, io, contextlib, copy, os, pickle
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pytorch_forecasting import TimeSeriesDataSet

# --- 0. 加载预训练系统 ---
CKPT = "./checkpoints/pretrained_planPC"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine_PC = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)

# --- 1. 绑定去重补丁 ---
def analyze_rolling_patched(self, model_name: str, df: pd.DataFrame,
                            baseline_end_idx=None, predict=True):
    model = self.models[model_name]
    train_dataset = self.datasets[model_name]
    df_inf = df.copy()
    for col in self.known_categoricals:
        df_inf[col] = df_inf[col].astype(str)
    try:
        inference_dataset = TimeSeriesDataSet.from_dataset(
            train_dataset, df_inf, predict=False, stop_randomization=True)
    except ValueError as e:
        if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
            return {}
        raise
    if len(inference_dataset) == 0:
        return {}
    dataloader = inference_dataset.to_dataloader(
        train=False, batch_size=512, num_workers=0, pin_memory=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()
    result_buffer = {}
    with torch.no_grad():
        for x, y in dataloader:
            x = {k: v.to(device) for k, v in x.items()}
            targets = y[0].cpu().numpy().flatten()
            raw_out = model(x)
            preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
            p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
            time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
            interp = model.interpret_output(raw_out, reduction="none")
            batch_att = interp['attention'].cpu().numpy()
            batch_vsn = interp['encoder_variables'].cpu().numpy()
            for i in range(len(time_idx)):
                t_id = int(time_idx[i])
                result_buffer[t_id] = {
                    "target_true": targets[i], "pred_p50": p50[i],
                    "residual": targets[i] - p50[i], "divergence": p90[i] - p10[i],
                    "attention": batch_att[i], "vsn": batch_vsn[i]}
    if not result_buffer:
        return {}
    sorted_t = sorted(result_buffer.keys())
    df_s = pd.DataFrame({
        "time_idx": sorted_t,
        "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
        "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
        "residual": [result_buffer[t]["residual"] for t in sorted_t],
        "divergence": [result_buffer[t]["divergence"] for t in sorted_t]})
    full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
    full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
    result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
    if baseline_end_idx is not None:
        bl = self._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
        result["baseline"] = bl
        self.baselines[model_name] = bl
    return result

engine_PC.analyze_rolling = types.MethodType(analyze_rolling_patched, engine_PC)

# --- 2. 参数配置 ---
QUIET_START = pd.to_datetime('2025-02-18 00:00:00')
QUIET_END   = pd.to_datetime('2025-02-25 00:00:00')
ds_0 = list(engine_PC.datasets.values())[0]
MIN_ENCODER = ds_0.max_encoder_length
MIN_PRED    = ds_0.max_prediction_length
MIN_REQUIRED = MIN_ENCODER + MIN_PRED
BUFFER_SIZE = 672
PRE_CONTEXT = MIN_REQUIRED + 1
STEP_SIZE   = 96  # 每96窗口批量推理一次（而非逐窗口）

print(f"[Config] MIN_ENCODER={MIN_ENCODER}, MIN_REQUIRED={MIN_REQUIRED}, "
      f"BUFFER={BUFFER_SIZE}, STEP={STEP_SIZE}")

# --- 3. 数据预处理 ---
mask_quiet = (df_features['timestamp'] >= QUIET_START) & (df_features['timestamp'] < QUIET_END)
df_quiet_raw = df_features[mask_quiet].copy().fillna(0)

processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_quiet_raw)
print(f"[Processor] re-fitted on {len(df_quiet_raw)} quiet windows, "
      f"global_min_time={processor_real.global_min_time}")

dataset_real = processor_real.transform(df_features.fillna(0), df_official_real)
df_real_all = dataset_real.data.copy()
df_real_all.fillna(0, inplace=True)
df_real_all.replace([np.inf, -np.inf], 0, inplace=True)
df_real_all = tb.transform(df_real_all)
if 'label' not in df_real_all.columns:
    df_real_all['label'] = 'N'
df_real_all = df_real_all.sort_values('time_idx').reset_index(drop=True)
print(f"[Data] {df_real_all.shape}, {df_real_all['timestamp'].min()} ~ {df_real_all['timestamp'].max()}")

# --- 4. 用平静期数据建立 baseline（关键修复）---
quiet_mask_all = (df_real_all['timestamp'] >= QUIET_START) & (df_real_all['timestamp'] < QUIET_END)
quiet_start_tidx = df_real_all.loc[quiet_mask_all, 'time_idx'].min()
# 取平静期前PRE_CONTEXT行作为encoder上下文 + 平静期全部
quiet_idx_start = df_real_all[df_real_all['timestamp'] >= QUIET_START].index[0]
bl_start = max(0, quiet_idx_start - PRE_CONTEXT)
bl_end = df_real_all[df_real_all['timestamp'] < QUIET_END].index[-1] + 1
df_for_baseline = df_real_all.iloc[bl_start:bl_end].copy().reset_index(drop=True)

print(f"\n[Baseline] 用 {len(df_for_baseline)} 窗口建立baseline...")
baseline_end_tidx = int(df_for_baseline.loc[
    df_for_baseline['timestamp'] < QUIET_END, 'time_idx'].max())

for mn in engine_PC.models:
    res = engine_PC.analyze_rolling(mn, df_for_baseline, baseline_end_idx=baseline_end_tidx)
    if 'baseline' in res:
        print(f"  [{mn}] baseline built: residual_std={res['baseline']['residual_std']:.4f}, "
              f"n={res['baseline']['n_samples']}")
    else:
        print(f"  ⚠️ [{mn}] baseline 未生成，尝试从全量构建")
        if 'metrics' in res:
            bl = engine_PC._build_baseline(
                res['metrics'], res['attention'], res['vsn'], baseline_end_tidx)
            engine_PC.baselines[mn] = bl
            print(f"    → 手动构建成功: residual_std={bl['residual_std']:.4f}")

print(f"[Baseline] baselines: {list(engine_PC.baselines.keys())}")

In [ ]:
# ============================================================
# Cell 2: 批量滚动推理 + 合并输出 + 可视化
# ============================================================

def extract_tft_features_batch(df_buffer, engine_PC, output_tidx_set=None):
    """
    对缓冲区执行TFT推理，提取model_开头的TFT信号 + rolling。
    
    Parameters
    ----------
    df_buffer : 完整缓冲区数据（含所有TFT需要的列）
    engine_PC : TFT引擎（baselines已建立）
    output_tidx_set : set[int] or None
        只输出这些time_idx的行；None则输出全部推理结果
    
    Returns
    -------
    pd.DataFrame : time_idx, timestamp + model_* 列
    """
    from scipy.stats import spearmanr
    
    tft_results = {}
    for mn in engine_PC.models:
        with contextlib.redirect_stdout(io.StringIO()), \
             contextlib.redirect_stderr(io.StringIO()):
            res = engine_PC.analyze_rolling(mn, df_buffer, baseline_end_idx=None)
        tft_results[mn] = res
    
    if not any('metrics' in v for v in tft_results.values()):
        return pd.DataFrame()
    
    # 用完整df_buffer做merge（merge_tft_signals_pc需要time_idx列）
    df_merged = df_buffer[['time_idx', 'timestamp']].copy()
    df_merged = merge_tft_signals_pc(df_merged, tft_results, engine_PC)
    
    # 识别 model_ 开头的基础列
    base_tft_cols = [c for c in df_merged.columns 
                     if c.startswith('model_') and 'roll' not in c]
    
    # rolling特征
    for col in base_tft_cols:
        for k in [4, 8]:
            df_merged[f'{col}_rollmax_{k}'] = df_merged[col].rolling(k, min_periods=1).max()
            df_merged[f'{col}_rollmean_{k}'] = df_merged[col].rolling(k, min_periods=1).mean()
    
    df_merged = df_merged.replace([np.inf, -np.inf], 0).fillna(0)
    
    # 只保留 time_idx, timestamp + model_* 列
    keep_cols = ['time_idx', 'timestamp'] + [
        c for c in df_merged.columns if c.startswith('model_')]
    df_merged = df_merged[[c for c in keep_cols if c in df_merged.columns]]
    
    # 筛选输出行
    if output_tidx_set is not None:
        df_merged = df_merged[df_merged['time_idx'].isin(output_tidx_set)]
    
    return df_merged


# ============================================================
# 滚动推理主循环（批量模式：每STEP_SIZE窗口推理一次）
# ============================================================
quiet_start_idx = df_real_all[df_real_all['timestamp'] >= QUIET_START].index[0]
pre_start_idx = max(0, quiet_start_idx - PRE_CONTEXT)

# 初始缓冲区：前置上下文 + 平静期
initial_end_idx = min(quiet_start_idx + BUFFER_SIZE, len(df_real_all))
buffer_df = df_real_all.iloc[pre_start_idx:initial_end_idx].copy().reset_index(drop=True)

print(f"[ColdStart] 初始缓冲区: {len(buffer_df)} 窗口, "
      f"{buffer_df['timestamp'].iloc[0]} ~ {buffer_df['timestamp'].iloc[-1]}")

# 冷启动推理：输出平静期开始之后的所有结果
quiet_start_tidx = int(df_real_all.loc[
    df_real_all['timestamp'] >= QUIET_START, 'time_idx'].min())
output_tidx = set(buffer_df.loc[
    buffer_df['time_idx'] >= quiet_start_tidx, 'time_idx'].astype(int).values)

print(f"[ColdStart] 推理中（输出 {len(output_tidx)} 行）...")
df_cold = extract_tft_features_batch(buffer_df, engine_PC, output_tidx_set=output_tidx)
print(f"[ColdStart] 输出: {len(df_cold)} 行")

all_results = [df_cold] if len(df_cold) > 0 else []
processed_tidx = set(df_cold['time_idx'].astype(int).values) if len(df_cold) > 0 else set()

# 批量滚动
remaining_start = initial_end_idx
total_remaining = len(df_real_all) - remaining_start
n_batches = (total_remaining + STEP_SIZE - 1) // STEP_SIZE

print(f"\n[Rolling] {total_remaining} 窗口, 分 {n_batches} 批 (step={STEP_SIZE})")

for batch_i in tqdm(range(n_batches), desc="Rolling TFT Batch"):
    batch_start = remaining_start + batch_i * STEP_SIZE
    batch_end = min(batch_start + STEP_SIZE, len(df_real_all))
    
    if batch_start >= len(df_real_all):
        break
    
    new_rows = df_real_all.iloc[batch_start:batch_end]
    new_tidx_set = set(new_rows['time_idx'].astype(int).values) - processed_tidx
    
    if not new_tidx_set:
        continue
    
    # 追加到缓冲区
    buffer_df = pd.concat([buffer_df, new_rows], ignore_index=True)
    
    # 裁剪缓冲区
    if len(buffer_df) > BUFFER_SIZE:
        buffer_df = buffer_df.iloc[-BUFFER_SIZE:].reset_index(drop=True)
    
    if len(buffer_df) < MIN_REQUIRED:
        continue
    
    # 批量推理，只取新增的time_idx
    df_feat = extract_tft_features_batch(buffer_df, engine_PC, output_tidx_set=new_tidx_set)
    
    if len(df_feat) > 0:
        all_results.append(df_feat)
        processed_tidx.update(df_feat['time_idx'].astype(int).values)
    
    if (batch_i + 1) % 10 == 0:
        ts = new_rows['timestamp'].iloc[-1]
        print(f"  Batch {batch_i+1}/{n_batches} | {ts} | "
              f"buffer={len(buffer_df)} | total_output={len(processed_tidx)}")

# ============================================================
# 合并输出
# ============================================================
if all_results:
    df_result_real = pd.concat(all_results, ignore_index=True)
    df_result_real = df_result_real.drop_duplicates(subset='time_idx', keep='last')
    df_result_real = df_result_real.sort_values('time_idx').reset_index(drop=True)
else:
    df_result_real = pd.DataFrame()

print(f"\n{'='*60}")
print(f"  df_result_real 输出")
print(f"{'='*60}")
print(f"  行数: {len(df_result_real)}")
if len(df_result_real) > 0:
    print(f"  时间: {df_result_real['timestamp'].min()} ~ {df_result_real['timestamp'].max()}")
    tft_cols = [c for c in df_result_real.columns if c.startswith('model_')]
    print(f"  TFT特征: {len(tft_cols)} 列")
    # 按类型分组显示
    base_cols = [c for c in tft_cols if 'roll' not in c]
    roll_cols = [c for c in tft_cols if 'roll' in c]
    print(f"    基础信号 ({len(base_cols)}): {base_cols}")
    print(f"    Rolling  ({len(roll_cols)})")
print(f"{'='*60}")

# 保存
output_path = "./output/df_result_real_tft_features.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_result_real.to_csv(output_path, index=False)
print(f"✅ 已保存到 {output_path}")

## 分析结果

In [ ]:
# ============================================================
# Cell 3: 可视化 — TFT信号时序全景 + 热力图
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if len(df_result_real) == 0:
    print("⚠️ 无数据")
else:
    df_viz = df_result_real.copy()
    df_viz['timestamp'] = pd.to_datetime(df_viz['timestamp'])
    
    # 基础信号（不含rolling）
    base_signals = [c for c in df_viz.columns if c.startswith('model_') and 'roll' not in c]
    
    # --- 时序全景图 ---
    n_sig = len(base_signals)
    fig, axes = plt.subplots(n_sig, 1, figsize=(20, 3 * n_sig), sharex=True)
    if n_sig == 1: axes = [axes]
    
    # crisis区间
    crisis_periods = []
    for _, row in crisis_df.iterrows():
        crisis_periods.append({
            'start': pd.to_datetime(row['t_crisis_start']),
            'end': pd.to_datetime(row['t_crisis_end']),
            'name': row.get('crisis_name', '?')})
    
    for ax, col in zip(axes, base_signals):
        ax.plot(df_viz['timestamp'], df_viz[col], lw=0.5, alpha=0.7, color='steelblue')
        
        # 平静期P95/P99参考线
        quiet_mask = (df_viz['timestamp'] >= QUIET_START) & (df_viz['timestamp'] < QUIET_END)
        q_vals = df_viz.loc[quiet_mask, col].dropna()
        if len(q_vals) > 10:
            p95 = q_vals.quantile(0.95)
            p99 = q_vals.quantile(0.99)
            ax.axhline(p95, color='orange', ls='--', alpha=0.5, label=f'Q_P95={p95:.3f}')
            ax.axhline(p99, color='red', ls=':', alpha=0.5, label=f'Q_P99={p99:.3f}')
        
        for cp in crisis_periods:
            ax.axvspan(cp['start'], cp['end'], alpha=0.12, color='red')
        ax.axvspan(QUIET_START, QUIET_END, alpha=0.08, color='green', label='Quiet')
        
        short_name = col.replace('model_model_', 'M_').replace('model_', 'M_')
        ax.set_ylabel(short_name, fontsize=8)
        ax.legend(loc='upper right', fontsize=7)
        ax.grid(alpha=0.2)
    
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.xticks(rotation=45)
    plt.suptitle('TFT基础信号时序全景（绿=平静期，红=crisis区间）', fontsize=13)
    plt.tight_layout()
    plt.savefig('./output/tft_signals_panorama.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- 热力图 ---
    roll_cols = [c for c in df_viz.columns if c.startswith('model_') and 'rollmax_8' in c]
    if roll_cols:
        df_heat = df_viz[['timestamp'] + roll_cols].set_index('timestamp').copy()
        quiet_mask_h = (df_heat.index >= QUIET_START) & (df_heat.index < QUIET_END)
        for col in roll_cols:
            qv = df_heat.loc[quiet_mask_h, col]
            med = qv.median()
            mad = (qv - med).abs().median() * 1.4826
            scale = mad if mad > 1e-8 else qv.std() if qv.std() > 1e-8 else 1.0
            df_heat[col] = (df_heat[col] - med) / scale
        df_heat = df_heat.clip(-5, 10)
        
        fig, ax = plt.subplots(figsize=(20, max(4, len(roll_cols) * 0.6)))
        short_names = [c.replace('model_model_', '').replace('_rollmax_8', '') for c in roll_cols]
        im = ax.imshow(df_heat[roll_cols].T.values, aspect='auto', cmap='RdYlBu_r',
                       vmin=-2, vmax=6, interpolation='nearest')
        ax.set_yticks(range(len(short_names)))
        ax.set_yticklabels(short_names, fontsize=8)
        n_ticks = min(20, len(df_heat))
        tick_pos = np.linspace(0, len(df_heat)-1, n_ticks, dtype=int)
        ax.set_xticks(tick_pos)
        ax.set_xticklabels([df_heat.index[i].strftime('%m-%d') for i in tick_pos], rotation=45, fontsize=8)
        plt.colorbar(im, ax=ax, label='Z-score (平静期基准)')
        ax.set_title('TFT RollMax8 热力图（红=异常偏高）', fontsize=13)
        plt.tight_layout()
        plt.savefig('./output/tft_heatmap_rollmax8.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    print("✅ 可视化完成")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. 加载数据
# ============================================================
df = pd.read_csv(r"c:\tongji\0 code\03_core_model_v3\output\df_result_real_tft_features.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"数据量: {len(df)} 行, 时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")

# 加载crisis事件池
crisis_df = pd.read_csv(r"C:\tongji\0 code\00_data\01_EDA\0_crisis_event\crisis_event_pool.csv")
print(f"\nCrisis事件数: {len(crisis_df)}")
print(crisis_df[['t_crisis_start', 't_crisis_peak', 't_crisis_end', 'crisis_level']].to_string())

# ============================================================
# 2. 标记crisis窗口
# ============================================================
df['is_crisis'] = False
for _, row in crisis_df.iterrows():
    t_start = pd.to_datetime(row['t_crisis_start'])
    t_end = pd.to_datetime(row['t_crisis_end'])
    mask = (df['timestamp'] >= t_start) & (df['timestamp'] <= t_end)
    df.loc[mask, 'is_crisis'] = True

# 只保留数据覆盖范围内的crisis
data_start = df['timestamp'].min()
data_end = df['timestamp'].max()
crisis_in_range = crisis_df[
    (pd.to_datetime(crisis_df['t_crisis_start']) >= data_start) &
    (pd.to_datetime(crisis_df['t_crisis_end']) <= data_end)
].copy()
print(f"\n数据范围内的crisis事件: {len(crisis_in_range)}")

n_crisis_windows = df['is_crisis'].sum()
n_normal_windows = (~df['is_crisis']).sum()
print(f"Crisis窗口: {n_crisis_windows}, Normal窗口: {n_normal_windows}")

# ============================================================
# 3. 基础信号区分力诊断
# ============================================================
base_signals = [c for c in df.columns if c.startswith('model_') and 'roll' not in c]
all_signals = [c for c in df.columns if c.startswith('model_')]

print("\n" + "="*80)
print("基础信号区分力 (Cohen's d + KS统计量)")
print("="*80)

from scipy.stats import ks_2samp

results = []
for col in all_signals:
    normal_vals = df.loc[~df['is_crisis'], col].dropna()
    crisis_vals = df.loc[df['is_crisis'], col].dropna()
    
    if len(crisis_vals) == 0:
        continue
    
    # Cohen's d
    pooled_std = np.sqrt((normal_vals.std()**2 + crisis_vals.std()**2) / 2)
    d = (crisis_vals.mean() - normal_vals.mean()) / (pooled_std + 1e-10)
    
    # KS test
    ks_stat, ks_p = ks_2samp(normal_vals, crisis_vals)
    
    # 分位数比较
    normal_p95 = normal_vals.quantile(0.95)
    normal_p99 = normal_vals.quantile(0.99)
    crisis_above_p95 = (crisis_vals > normal_p95).mean()
    crisis_above_p99 = (crisis_vals > normal_p99).mean()
    
    results.append({
        'signal': col,
        'normal_mean': normal_vals.mean(),
        'crisis_mean': crisis_vals.mean(),
        'cohen_d': d,
        'ks_stat': ks_stat,
        'ks_p': ks_p,
        'normal_p95': normal_p95,
        'normal_p99': normal_p99,
        'crisis_above_p95': crisis_above_p95,
        'crisis_above_p99': crisis_above_p99,
    })

diag_df = pd.DataFrame(results).sort_values('cohen_d', ascending=False)
print(diag_df[['signal', 'cohen_d', 'ks_stat', 'crisis_above_p95', 'crisis_above_p99']].head(20).to_string(index=False))

# ============================================================
# 4. 逐事件分析：每个crisis期间的信号响应
# ============================================================
print("\n" + "="*80)
print("逐事件信号响应分析")
print("="*80)

# 选取top信号
top_signals = diag_df.head(10)['signal'].tolist()

for idx, row in crisis_in_range.iterrows():
    t_start = pd.to_datetime(row['t_crisis_start'])
    t_end = pd.to_datetime(row['t_crisis_end'])
    level = row.get('crisis_level', '?')
    
    mask = (df['timestamp'] >= t_start) & (df['timestamp'] <= t_end)
    df_event = df[mask]
    
    if len(df_event) == 0:
        continue
    
    print(f"\n--- Event {idx}: Level={level}, {t_start} ~ {t_end}, windows={len(df_event)} ---")
    
    for sig in top_signals[:5]:
        vals = df_event[sig]
        normal_p95 = df.loc[~df['is_crisis'], sig].quantile(0.95)
        normal_p99 = df.loc[~df['is_crisis'], sig].quantile(0.99)
        
        max_val = vals.max()
        above_p95_count = (vals > normal_p95).sum()
        above_p99_count = (vals > normal_p99).sum()
        
        print(f"  {sig:55s}: max={max_val:.4f}, >P95={above_p95_count}/{len(df_event)}, >P99={above_p99_count}/{len(df_event)}")

# ============================================================
# 5. 正常期误报率分析（不同阈值）
# ============================================================
print("\n" + "="*80)
print("误报率分析（正常期）")
print("="*80)

normal_df = df[~df['is_crisis']].copy()

for sig in top_signals[:5]:
    vals = normal_df[sig]
    for pct in [95, 97.5, 99, 99.5]:
        thresh = vals.quantile(pct / 100)
        fpr = (vals > thresh).mean()
        # 每天误报几次？(每15分钟一个窗口, 96个/天)
        false_alarms_per_day = fpr * 96
        print(f"  {sig:55s} P{pct}: thresh={thresh:.4f}, FPR={fpr:.4f}, ~{false_alarms_per_day:.1f} alarms/day")

# ============================================================
# 6. 组合信号分析
# ============================================================
print("\n" + "="*80)
print("组合信号策略")
print("="*80)

# 策略1: 跨模型 residual_ratio 取max
res_cols = [c for c in base_signals if 'residual_ratio' in c]
df['max_residual_ratio'] = df[res_cols].max(axis=1)

# 策略2: rollmax_8 的 residual_ratio 取max  
res_rollmax8_cols = [c for c in all_signals if 'residual_ratio_rollmax_8' in c]
df['max_residual_rollmax8'] = df[res_rollmax8_cols].max(axis=1)

# 策略3: 所有信号的 rollmax_8 取max
all_rollmax8_cols = [c for c in all_signals if 'rollmax_8' in c]
df['max_all_rollmax8'] = df[all_rollmax8_cols].max(axis=1)

# 策略4: 加权组合（residual为主，att_kl辅助）
att_kl_rollmax8 = [c for c in all_signals if 'att_kl_rollmax_8' in c]
if att_kl_rollmax8:
    df['combined_score'] = 0.5 * df['max_residual_rollmax8'] + 0.3 * df[att_kl_rollmax8].max(axis=1)
    vsn_js_rollmax8 = [c for c in all_signals if 'vsn_js_rollmax_8' in c]
    if vsn_js_rollmax8:
        df['combined_score'] += 0.2 * df[vsn_js_rollmax8].max(axis=1)

combo_signals = ['max_residual_ratio', 'max_residual_rollmax8', 'max_all_rollmax8']
if 'combined_score' in df.columns:
    combo_signals.append('combined_score')

for sig in combo_signals:
    normal_vals = df.loc[~df['is_crisis'], sig]
    crisis_vals = df.loc[df['is_crisis'], sig]
    
    pooled_std = np.sqrt((normal_vals.std()**2 + crisis_vals.std()**2) / 2)
    d = (crisis_vals.mean() - normal_vals.mean()) / (pooled_std + 1e-10)
    
    # 事件级检出率
    for pct in [95, 99, 99.5]:
        thresh = normal_vals.quantile(pct / 100)
        
        n_detected = 0
        n_total = 0
        for _, erow in crisis_in_range.iterrows():
            t_s = pd.to_datetime(erow['t_crisis_start'])
            t_e = pd.to_datetime(erow['t_crisis_end'])
            emask = (df['timestamp'] >= t_s) & (df['timestamp'] <= t_e)
            if emask.sum() == 0:
                continue
            n_total += 1
            if (df.loc[emask, sig] > thresh).any():
                n_detected += 1
        
        fpr = (normal_vals > thresh).mean()
        fa_day = fpr * 96
        print(f"  {sig:30s} @P{pct}: d={d:.3f}, event_recall={n_detected}/{n_total}, FPR={fpr:.4f}, ~{fa_day:.1f} FA/day")

print("\n✅ 分析完成")

In [ ]:
crisis_df.info()

# 分类器2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# 1. 读取数据与定义时间窗
# =====================================================================
df = pd.read_csv("df_result_real_tft_features.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# 设定前 7 天为冷启动平静期（基准期）
start_time = pd.to_datetime('2025-02-18 00:00:00')
warmup_end = start_time + pd.Timedelta(days=7)
df_base = df[df['timestamp'] < warmup_end].copy()

# =====================================================================
# 2. 特征筛选与多维对齐分类
# =====================================================================
# AP 候选特征：扩展至全部 8 个核心滚动特征 (包含残差、VSN漂移、注意力KL、VSN散度)
ap_roll_features = [
    'model_model_comment_pc1_residual_ratio_rollmax_8',
    'model_model_post_pc2_residual_ratio_rollmax_8',
    'model_model_comment_pc1_vsn_rank_shift_rollmax_8',
    'model_model_post_pc2_vsn_rank_shift_rollmax_8',
    'model_model_comment_pc1_att_kl_rollmax_8',
    'model_model_post_pc2_att_kl_rollmax_8',
    'model_model_comment_pc1_vsn_js_rollmax_8',
    'model_model_post_pc2_vsn_js_rollmax_8'
]

# CP 候选特征：扩展至 6 个核心静态特征
cp_static_features = [
    'model_model_comment_pc1_vsn_rank_shift',
    'model_model_post_pc2_vsn_rank_shift',
    'model_model_comment_pc1_att_kl',
    'model_model_post_pc2_att_kl'
]

# 解决量纲不一致：利用冷启动期数据，将所有 CP 静态特征标准化为 Z-score
for feat in cp_static_features:
    base_mean = df_base[feat].mean()
    base_std = df_base[feat].std() + 1e-8
    df[f'{feat}_z'] = (df[feat] - base_mean) / base_std

z_cols = [f'{feat}_z' for feat in cp_static_features]

# 取标准化后的多维 Z-score 最大值，作为 CP 的基础监控信号 (反映全维度的极值突变)
df['static_signal_z_max'] = df[z_cols].max(axis=1)



# # =====================================================================
# # 3. 构造衍生特征 (持续时长、密度与偏差比例)
# # =====================================================================
# # 计算 CP 指标 (96步 = 24小时)
df['cp_density'] = df['static_signal_z_max'].rolling(window=288, min_periods=96).mean()
df['cp_volatility'] = df['static_signal_z_max'].rolling(window=288, min_periods=96).std()

# # 同步更新 df_base 以便计算阈值
df_base = df[df['timestamp'] < warmup_end].copy()


# =====================================================================
# 3. 构造衍生特征 (正确的长程密度机制)
# =====================================================================
# 首先定义一个用于 CP 的单步触发水位（这里取 K=3.0）
CP_BASE_THRESH = df_base['static_signal_z_max'].mean() + 3.0 * df_base['static_signal_z_max'].std()

# 标记每一步是否达到了中高水位 (0 或 1)
df['is_high_watermark'] = (df['static_signal_z_max'] > CP_BASE_THRESH).astype(int)

CP_WINDOW = 288
MIN_PERIODS = 96

# 真正的 CP 密度：过去 96 步中，有多少步处于高水位？ (转化为占比 0.0 ~ 1.0)
df['cp_density'] = df['is_high_watermark'].rolling(window=CP_WINDOW, min_periods=MIN_PERIODS).mean()

# 波动率不变
df['cp_volatility'] = df['static_signal_z_max'].rolling(window=CP_WINDOW, min_periods=MIN_PERIODS).std()

# 同步更新 df_base 
df_base = df[df['timestamp'] < warmup_end].copy()

# =====================================================================
# 4. 基于冷启动平静期计算阈值
# =====================================================================
# AP 阈值字典：提高严苛度，减少长尾日常误报 (K 从 4.0 提至 5.0)
ap_thresholds = {}
for feat in ap_roll_features:
    ap_thresholds[feat] = df_base[feat].mean() + 5.0 * df_base[feat].std()

# CP 密度阈值：要求极高密度 (过去24小时内至少 85% 的时间处于异常高水位)
THRESH_CP_DENSITY = 0.95 
THRESH_CP_VOL = df_base['static_signal_z_max'].std() * 1.5 

print("="*60)
print(f" 冷启动期 ({start_time.date()} 至 {warmup_end.date()}) 阈值校准完成")
print(f" [CP 单步判定水位] > {CP_BASE_THRESH:.3f}")
print(f" [CP 时间密度占比要求] > {THRESH_CP_DENSITY:.2%}")
print(f" [CP 偏差比例上限] < {THRESH_CP_VOL:.3f}")
print("="*60)

# =====================================================================
# 5. 执行在线分类判定
# =====================================================================
df['pred_label'] = 'N'

# 条件1：判别 CP (长窗口时间内，极高比例处于异常状态，且波动稳定)
mask_cp = (df['cp_density'] > THRESH_CP_DENSITY) & (df['cp_volatility'] < THRESH_CP_VOL)
df.loc[mask_cp, 'pred_label'] = 'CP'

# 条件2：判别 AP (多维极值报警)
df['ap_trigger_count'] = 0
for feat in ap_roll_features:
    df['ap_trigger_count'] += (df[feat] > ap_thresholds[feat]).astype(int)

# 要求 8 个特征中至少有 3 个同时触发，大幅降低日常噪音干扰
mask_ap = (df['ap_trigger_count'] >= 4) & (~mask_cp)
df.loc[mask_ap, 'pred_label'] = 'AP'

# 屏蔽冷启动期
df.loc[df['timestamp'] < warmup_end, 'pred_label'] = 'N'

# =====================================================================
# 6. 事件合并与统计输出
# =====================================================================
df['event_change'] = (df['pred_label'] != df['pred_label'].shift(1)).cumsum()
events = df[df['pred_label'] != 'N'].groupby(['pred_label', 'event_change']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    duration_windows=('timestamp', 'count'),
    peak_z_max=('static_signal_z_max', 'max')
).reset_index().sort_values('start_time')

print(f"检测到 {len(events[events['pred_label']=='AP'])} 个 AP 事件片段")
print(f"检测到 {len(events[events['pred_label']=='CP'])} 个 CP 事件片段")
# for _, row in events.iterrows():
#     if row['duration_windows'] >= 4: 
#         print(f"[{row['pred_label']}] {row['start_time']} -> {row['end_time']} | "
#               f"持续: {row['duration_windows']} | Z-score 峰值: {row['peak_z_max']:.2f}")

# =====================================================================
# 6.5 真实危机区间 (crisis_df) 命中率统计
# =====================================================================
if 'crisis_df' in locals() or 'crisis_df' in globals():
    print("\n" + "="*60)
    print(" 报警命中率与误报统计 (对比 crisis_df 事件池)")
    print("="*60)
    
    # 确保 crisis 时间列格式正确
    crisis_df['t_crisis_start'] = pd.to_datetime(crisis_df['t_crisis_start'])
    crisis_df['t_crisis_end'] = pd.to_datetime(crisis_df['t_crisis_end'])
    
    # 判定每个事件片段是否与真实的 crisis 区间有交集
    events['in_crisis'] = False
    for idx, event in events.iterrows():
        e_start = event['start_time']
        e_end = event['end_time']
        
        # 区间重叠条件：报警开始时间 <= 危机结束时间 且 报警结束时间 >= 危机开始时间
        overlap = ((e_start <= crisis_df['t_crisis_end']) & (e_end >= crisis_df['t_crisis_start'])).any()
        events.at[idx, 'in_crisis'] = overlap

    # 汇总统计 (假设每天有 96 个时间步)
    WINDOWS_PER_DAY = 96.0
    
    for label in ['AP', 'CP']:
        sub_events = events[events['pred_label'] == label]
        if len(sub_events) == 0:
            continue
            
        in_events = sub_events[sub_events['in_crisis']]
        out_events = sub_events[~sub_events['in_crisis']]
        
        in_count = len(in_events)
        out_count = len(out_events)
        
        in_days = in_events['duration_windows'].sum() / WINDOWS_PER_DAY
        out_days = out_events['duration_windows'].sum() / WINDOWS_PER_DAY
        
        print(f"[{label} 报警统计]")
        print(f"  落在 Crisis 范围内: {in_count:<3} 次 | 累计天数: {in_days:.2f} 天")
        print(f"  落在 Crisis 范围外: {out_count:<3} 次 | 累计天数: {out_days:.2f} 天\n")
else:
    print("\n[提示] 环境中未找到 crisis_df 变量，跳过命中率统计。")
# =====================================================================
# 7. 可视化
# =====================================================================
plt.figure(figsize=(15, 5))
# 此时绘制的是标准化的多维联合信号
plt.plot(df['timestamp'], df['static_signal_z_max'], label='Static Signal (Max Z-score)', color='gray', alpha=0.4, lw=1)
plt.plot(df['timestamp'], df['cp_density'], label='CP Density (96-roll mean Z-score)', color='blue', lw=1.5)

try:
    for _, row in crisis_df.iterrows():
        start_t = pd.to_datetime(row['t_crisis_start'])
        end_t = pd.to_datetime(row['t_crisis_end'])
        plt.axvspan(start_t, end_t, color='red', alpha=0.15)
except NameError:
    pass

ap_points = df[df['pred_label'] == 'AP']
cp_points = df[df['pred_label'] == 'CP']

plt.scatter(ap_points['timestamp'], ap_points['static_signal_z_max'], color='orange', s=12, label='AP Detected', zorder=5)
plt.scatter(cp_points['timestamp'], cp_points['cp_density'], color='red', s=12, label='CP Detected (Density)', zorder=5)

plt.axhline(THRESH_CP_DENSITY, color='blue', ls='--', alpha=0.8, label=f'CP Density Threshold ({THRESH_CP_DENSITY:.2f})')
plt.axvspan(start_time, warmup_end, color='green', alpha=0.1, label='Warm-up Period')

plt.title('AP/CP Detection based on Multi-Dimensional Z-score Deviation')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
crisis_df.info()

# 分类器带漂移

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# 1. 读取数据与定义时间窗
# =====================================================================
df = pd.read_csv("df_result_real_tft_features.csv")

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)



# =====================================================================
# 2. 特征筛选与多维对齐分类
# =====================================================================
ap_roll_features = [
    'model_model_comment_pc1_residual_ratio_rollmax_8',
    'model_model_post_pc2_residual_ratio_rollmax_8',
    'model_model_comment_pc1_vsn_rank_shift_rollmax_8',
    'model_model_post_pc2_vsn_rank_shift_rollmax_8',
    'model_model_comment_pc1_att_kl_rollmax_8',
    'model_model_post_pc2_att_kl_rollmax_8',
    'model_model_comment_pc1_vsn_js_rollmax_8',
    'model_model_post_pc2_vsn_js_rollmax_8'
]

cp_static_features = [
    'model_model_comment_pc1_vsn_rank_shift',
    'model_model_post_pc2_vsn_rank_shift',
    'model_model_comment_pc1_att_kl',
    'model_model_post_pc2_att_kl'
]

for feat in cp_static_features:
    base_mean = df_base[feat].mean()
    base_std = df_base[feat].std() + 1e-8
    df[f'{feat}_z'] = (df[feat] - base_mean) / base_std

z_cols = [f'{feat}_z' for feat in cp_static_features]
df['static_signal_z_max'] = df[z_cols].max(axis=1)

start_time = pd.to_datetime('2025-02-18 00:00:00')


warmup_end = start_time + pd.Timedelta(days=7)
df_base = df[df['timestamp'] < warmup_end].copy()
# =====================================================================
# 3. 构造衍生特征 (突变密度 + 缓变趋势)
# =====================================================================
# --- 轨道 1：短程突变特征 ---
CP_BASE_THRESH = df_base['static_signal_z_max'].mean() + 3.0 * df_base['static_signal_z_max'].std()
df['is_high_watermark'] = (df['static_signal_z_max'] > CP_BASE_THRESH).astype(int)

CP_WINDOW = 288
MIN_PERIODS = 96
df['cp_density'] = df['is_high_watermark'].rolling(window=CP_WINDOW, min_periods=MIN_PERIODS).mean()
df['cp_volatility'] = df['static_signal_z_max'].rolling(window=CP_WINDOW, min_periods=MIN_PERIODS).std()

# --- 轨道 2：长程缓变特征 (新增) ---
# CP_TREND_WINDOW = 1344  # 14天的长程均值
CP_TREND_WINDOW = 2688  # 14天的长程均值

CP_TREND_K = 1.5        # 缓变触发系数较低
CP_TREND_THRESH = df_base['static_signal_z_max'].mean() + CP_TREND_K * df_base['static_signal_z_max'].std()

# 计算长程移动平均水位
df['cp_trend_mean'] = df['static_signal_z_max'].rolling(window=CP_TREND_WINDOW, min_periods=CP_WINDOW).mean()

# =====================================================================
# 4. 基于冷启动平静期计算阈值
# =====================================================================
ap_thresholds = {}
for feat in ap_roll_features:
    ap_thresholds[feat] = df_base[feat].mean() + 5.0 * df_base[feat].std()

THRESH_CP_DENSITY = 0.95 
THRESH_CP_VOL = df_base['static_signal_z_max'].std() * 1.5 

print("="*60)
print(f" 冷启动期 ({start_time.date()} 至 {warmup_end.date()}) 阈值校准完成")
print(f" [CP 突变单步水位] > {CP_BASE_THRESH:.3f}")
print(f" [CP 突变密度要求] > {THRESH_CP_DENSITY:.2%}")
print(f" [CP 缓变趋势水位] > {CP_TREND_THRESH:.3f} (14天均值)")
print("="*60)

# =====================================================================
# 5. 执行双轨分类判定
# =====================================================================
df['pred_label'] = 'N'
df['cp_type'] = 'N' # 细分漂移类型用于观察

# 条件 1：阶跃突变 (原有)
mask_cp_sudden = (df['cp_density'] > THRESH_CP_DENSITY) & (df['cp_volatility'] < THRESH_CP_VOL)

# 条件 2：缓变漂移 (新增)
mask_cp_incremental = (df['cp_trend_mean'] > CP_TREND_THRESH)

# 任意一轨触发即视为 CP
mask_cp = mask_cp_sudden | mask_cp_incremental
df.loc[mask_cp, 'pred_label'] = 'CP'

# 记录具体是哪种漂移触发的
df.loc[mask_cp_sudden, 'cp_type'] = 'Sudden'
df.loc[mask_cp_incremental & ~mask_cp_sudden, 'cp_type'] = 'Incremental'

# 条件 3：判别 AP 
df['ap_trigger_count'] = 0
for feat in ap_roll_features:
    df['ap_trigger_count'] += (df[feat] > ap_thresholds[feat]).astype(int)
mask_ap = (df['ap_trigger_count'] >= 4) & (~mask_cp)
df.loc[mask_ap, 'pred_label'] = 'AP'

df.loc[df['timestamp'] < warmup_end, 'pred_label'] = 'N'

# =====================================================================
# 6. 事件统计输出
# =====================================================================
print(f"检测到 AP 窗口数: {len(df[df['pred_label']=='AP'])}")
print(f"检测到 CP 窗口数: {len(df[df['pred_label']=='CP'])} "
      f"(其中突变型: {len(df[df['cp_type']=='Sudden'])}, 缓变型: {len(df[df['cp_type']=='Incremental'])})")

# =====================================================================
# 7. 可视化 (增加长程趋势图层)
# =====================================================================
plt.figure(figsize=(15, 6))

# 基础信号层
plt.plot(df['timestamp'], df['static_signal_z_max'], label='Static Z-max (Raw)', color='gray', alpha=0.3, lw=1)

# 短程突变密度层
plt.plot(df['timestamp'], df['cp_density'] * 10, label='CP Density x10 (Sudden)', color='blue', alpha=0.6, lw=1)
plt.axhline(THRESH_CP_DENSITY * 10, color='blue', ls='--', alpha=0.5, label='Density Threshold')

# --- 新增：长程趋势层 ---
plt.plot(df['timestamp'], df['cp_trend_mean'], label='CP Trend Mean (14-day Incremental)', color='purple', lw=2.5)
plt.axhline(CP_TREND_THRESH, color='purple', ls=':', lw=2, alpha=0.8, label=f'Trend Threshold ({CP_TREND_THRESH:.2f})')

try:
    for _, row in crisis_df.iterrows():
        start_t = pd.to_datetime(row['t_crisis_start'])
        end_t = pd.to_datetime(row['t_crisis_end'])
        plt.axvspan(start_t, end_t, color='red', alpha=0.1)
except NameError:
    pass

# 散点打标
ap_points = df[df['pred_label'] == 'AP']
cp_sudden = df[df['cp_type'] == 'Sudden']
cp_incremental = df[df['cp_type'] == 'Incremental']

plt.scatter(ap_points['timestamp'], ap_points['static_signal_z_max'], color='orange', s=10, label='AP Detected', zorder=5)
plt.scatter(cp_sudden['timestamp'], cp_sudden['static_signal_z_max'], color='red', marker='^', s=30, label='CP (Sudden)', zorder=6)
plt.scatter(cp_incremental['timestamp'], cp_incremental['cp_trend_mean'], color='magenta', marker='s', s=20, label='CP (Incremental)', zorder=6)

plt.axvspan(start_time, warmup_end, color='green', alpha=0.1, label='Warm-up Period')

plt.title('Dual-Track AP/CP Detection: Sudden vs. Incremental Drift')
plt.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0), ncol=2)
plt.tight_layout()
plt.savefig('./output/dual_track_ap_cp_detection_4.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df[mask_cp_incremental]

# 正式运行

In [ ]:
# ============================================================
# 0. 加载预训练系统
# ============================================================
CKPT = "./checkpoints/pretrained_planB"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/lgbm_classifier.pkl", 'rb') as f:
    lgbm_model = pickle.load(f)
with open(f"{CKPT}/feature_config.pkl", 'rb') as f:
    feature_config = pickle.load(f)
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)
with open(f"{CKPT}/layered_params.pkl", 'rb') as f:
    best_params = pickle.load(f)


In [ ]:
df_transformed.fillna(0, inplace=True)
df_transformed.replace([np.inf, -np.inf], 0, inplace=True)

## 检测器

In [ ]:
"""
自适应滚动检测器：
  滚动TFT推理 → 分类器2判定 → CP确认 → 回溯288窗口 → 积累微调 → 分层切换
  
  状态机：NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
"""

import os, copy, pickle, tempfile, io, contextlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import spearmanr


# =====================================================================
# 分类器2: 纯统计阈值 AP/CP 判定
# =====================================================================
class StatisticalClassifier2:
    """
    基于冷启动期校准的多维Z-score分类器。
    输入: 滚动TFT推理输出的 model_* 信号列
    输出: 每个窗口的 pred_label (N/AP/CP) + 投票归因信息
    """
    
    # 8个AP候选特征
    AP_ROLL_FEATURES = [
        'model_model_comment_pc1_residual_ratio_rollmax_8',
        'model_model_post_pc2_residual_ratio_rollmax_8',
        'model_model_comment_pc1_vsn_rank_shift_rollmax_8',
        'model_model_post_pc2_vsn_rank_shift_rollmax_8',
        'model_model_comment_pc1_att_kl_rollmax_8',
        'model_model_post_pc2_att_kl_rollmax_8',
        'model_model_comment_pc1_vsn_js_rollmax_8',
        'model_model_post_pc2_vsn_js_rollmax_8',
    ]
    
    # 4个CP静态特征
    CP_STATIC_FEATURES = [
        'model_model_comment_pc1_vsn_rank_shift',
        'model_model_post_pc2_vsn_rank_shift',
        'model_model_comment_pc1_att_kl',
        'model_model_post_pc2_att_kl',
    ]
    
    def __init__(self, ap_k=5.0, cp_base_k=3.0, cp_density_thresh=0.95,
                 cp_window=288, cp_min_periods=96, ap_min_triggers=4):
        self.ap_k = ap_k
        self.cp_base_k = cp_base_k
        self.cp_density_thresh = cp_density_thresh
        self.cp_window = cp_window
        self.cp_min_periods = cp_min_periods
        self.ap_min_triggers = ap_min_triggers
        
        # 由 calibrate() 填充
        self.ap_thresholds = {}
        self.cp_base_thresh = None
        self.cp_vol_thresh = None
        self._z_params = {}  # CP静态特征的Z标准化参数
        self._calibrated = False

        # 动态阈值放宽因子（由外部控制）
        self._ap_relax_factor = 1.0
    
    def calibrate(self, df_base):
        """用冷启动期数据校准所有阈值"""
        # AP阈值
        for feat in self.AP_ROLL_FEATURES:
            if feat in df_base.columns:
                self.ap_thresholds[feat] = (
                    df_base[feat].mean() + self.ap_k * df_base[feat].std()
                )
        
        # CP静态特征Z标准化参数
        for feat in self.CP_STATIC_FEATURES:
            if feat in df_base.columns:
                self._z_params[feat] = {
                    'mean': df_base[feat].mean(),
                    'std': df_base[feat].std() + 1e-8,
                }
        
        # CP单步水位阈值
        z_max_vals = self._compute_z_max(df_base)
        self.cp_base_thresh = z_max_vals.mean() + self.cp_base_k * z_max_vals.std()
        self.cp_vol_thresh = z_max_vals.std() * 1.5
        
        self._calibrated = True
        print(f"[Classifier2] 校准完成: AP阈值={len(self.ap_thresholds)}个, "
              f"CP水位={self.cp_base_thresh:.3f}, CP波动上限={self.cp_vol_thresh:.3f}")
        
    def set_ap_relax_factor(self, factor):
        """设置AP阈值放宽因子"""
        self._ap_relax_factor = factor
    
    def _compute_z_max(self, df):
        """计算各行的多维Z-score最大值"""
        z_vals = []
        for feat, params in self._z_params.items():
            if feat in df.columns:
                z = (df[feat] - params['mean']) / params['std']
                z_vals.append(z)
        if z_vals:
            return pd.concat(z_vals, axis=1).max(axis=1)
        return pd.Series(0.0, index=df.index)
    
    def predict(self, df):
        """
        对DataFrame逐行判定，返回带 pred_label + 归因信息的DataFrame。
        需要先调用 calibrate()。
        
        Returns: df 加上列:
          pred_label, ap_trigger_count, cp_density, cp_volatility,
          dominant_model, dominant_signal, trigger_features
        """
        assert self._calibrated, "必须先调用 calibrate()"
        
        df = df.copy()
        
        # --- CP辅助信号 ---
        z_max = self._compute_z_max(df)
        df['static_signal_z_max'] = z_max
        is_high = (z_max > self.cp_base_thresh).astype(float)
        df['cp_density'] = is_high.rolling(
            self.cp_window, min_periods=self.cp_min_periods).mean()
        df['cp_volatility'] = z_max.rolling(
            self.cp_window, min_periods=self.cp_min_periods).std()
        
        # --- AP触发计数 ---
        trigger_matrix = pd.DataFrame(index=df.index)
        for feat in self.AP_ROLL_FEATURES:
            if feat in df.columns and feat in self.ap_thresholds:
                effective_thresh = self.ap_thresholds[feat] * self._ap_relax_factor
                trigger_matrix[feat] = (df[feat] > effective_thresh).astype(int)
        df['ap_trigger_count'] = trigger_matrix.sum(axis=1)

        # --- 判定 ---
        df['pred_label'] = 'N'
        
        # CP: 高密度 + 低波动
        mask_cp = (
            (df['cp_density'] > self.cp_density_thresh) &
            (df['cp_volatility'] < self.cp_vol_thresh)
        )
        df.loc[mask_cp, 'pred_label'] = 'CP'
        
        # AP: 多维同时触发 且不是CP
        mask_ap = (df['ap_trigger_count'] >= self.ap_min_triggers) & (~mask_cp)
        df.loc[mask_ap, 'pred_label'] = 'AP'
        
        # --- 投票归因 ---
        df['dominant_model'] = 'N/A'
        df['dominant_signal'] = 'N/A'
        df['trigger_features'] = ''
        
        anom_mask = df['pred_label'] != 'N'
        if anom_mask.any():
            # for idx in df[anom_mask].index:
            #     triggers = []
            #     for feat in trigger_matrix.columns:
            #         if trigger_matrix.loc[idx, feat] == 1:
            #             triggers.append(feat)
                
            #     if not triggers:
            #         continue
            for idx in df[anom_mask].index:
                triggers = [f for f in trigger_matrix.columns 
                           if trigger_matrix.loc[idx, f] == 1]
                if not triggers:
                    continue


                # 子模型投票
                comment_v = sum(1 for f in triggers if 'comment_pc1' in f)
                post_v = sum(1 for f in triggers if 'post_pc2' in f)
                dom_model = 'comment_pc1' if comment_v >= post_v else 'post_pc2'
                
                # 信号类型投票
                signal_votes = {}
                for sig_type in ['residual_ratio', 'vsn_rank_shift', 'att_kl', 'vsn_js']:
                    signal_votes[sig_type] = sum(1 for f in triggers if sig_type in f)
                dom_signal = max(signal_votes, key=signal_votes.get)
                
                df.loc[idx, 'dominant_model'] = dom_model
                df.loc[idx, 'dominant_signal'] = dom_signal
                df.loc[idx, 'trigger_features'] = '|'.join(triggers)
        
        return df


# =====================================================================
# 自适应滚动检测器
# =====================================================================
class AdaptiveRollingDetector:
    """
    状态机:
      NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
    
    每个STEP_SIZE窗口执行一次批量TFT推理 + 分类器2判定，
    检测到CP后回溯288窗口，积累到576窗口后微调TFT，分层切换。
    """
    
    # 状态常量
    NORMAL = 'NORMAL'
    CP_TENTATIVE = 'CP_TENTATIVE'
    ACCUMULATING = 'ACCUMULATING'
    SWITCHING = 'SWITCHING'
    
    def __init__(self, engine, tb, processor, classifier2,
                 # 滚动参数
                 buffer_size=672, step_size=96, min_required=None,
                 # CP确认
                 cp_confirm_windows=32,
                 # 回溯与积累
                 retrace_windows=288,
                 accumulate_min=576,
                 # 微调
                 finetune_epochs=50, 
                 finetune_lr_scale=0.1,
                 finetune_batch_size = 32,
                 held_out_days=2,
                 finetune_grad_clip=0.5,
                 held_out_min_windows=96,
                 # 切换 # P3修复：切换（完整并行验证）
                 switch_parallel_windows=96,
                 switch_parallel_extend=192,
                 switch_agree_high=0.90, 
                 switch_agree_low=0.70,
                 # 冷却
                 cooldown_windows=384,
                 # AP阈值保护
                 ap_warmup_windows=48, 
                 ap_warmup_relax=1.2,
                 inference_batch_size=256):
        
        self.engine = engine
        self.tb = tb
        self.processor = processor
        self.classifier2 = classifier2
        
        # 滚动参数
        self.buffer_size = buffer_size
        self.step_size = step_size
        ds_0 = list(engine.datasets.values())[0]
        self.min_encoder = ds_0.max_encoder_length
        self.min_pred = ds_0.max_prediction_length
        self.min_required = min_required or (self.min_encoder + self.min_pred + 1)
        
        # CP确认
        self.cp_confirm_windows = cp_confirm_windows
        
        # 回溯与积累
        self.retrace_windows = retrace_windows
        self.accumulate_min = accumulate_min
        
        # 微调
        self.finetune_epochs = finetune_epochs
        self.finetune_lr_scale = finetune_lr_scale
        self.finetune_batch_size = finetune_batch_size
        self.finetune_grad_clip = finetune_grad_clip
        self.held_out_days = held_out_days
        self.held_out_min_windows = held_out_min_windows
        
        # P3修复：切换
        # self.switch_parallel_windows = switch_parallel_windows
        # self.switch_parallel_extend = switch_parallel_extend
        # self.switch_agree_high = switch_agree_high
        # self.switch_agree_low = switch_agree_low
        # 绑定新的切换参数
        self.switch_parallel_windows = switch_parallel_windows
        self.switch_parallel_extend = switch_parallel_extend
        self.switch_recovery_threshold = switch_recovery_threshold
        self.switch_stability_threshold = switch_stability_threshold
        # P7修复：冷却与保护
        self.cooldown_windows = cooldown_windows
        self.ap_warmup_windows = ap_warmup_windows
        self.ap_warmup_relax = ap_warmup_relax
        
        # 推理配置
        self.inference_batch_size = inference_batch_size
        
        # --- 状态变量 ---
        self.state = self.NORMAL
        self.baseline_version = 0
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self.cooldown_until_tidx = -1
        
        # P3修复：并行切换状态
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False
        
        # P7修复：AP warmup状态
        self._ap_warmup_remaining = 0
        
        # 历史记录
        self.cp_events = []
        self.switch_history = []
        
    # ================================================================
    # P8修复：状态持久化与恢复
    # ================================================================
    def save_state(self, path="./checkpoints/adaptive_state"):
        """持久化当前完整状态"""
        os.makedirs(path, exist_ok=True)
        
        state_dict = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_confirmed_tidx': self.cp_confirmed_tidx,
            'retrace_start_tidx': self.retrace_start_tidx,
            'accumulate_buffer_tidx': self.accumulate_buffer_tidx,
            'cooldown_until_tidx': self.cooldown_until_tidx,
            'ap_warmup_remaining': self._ap_warmup_remaining,
            'parallel_count': self._parallel_count,
            'switch_extended': self._switch_extended,
            'cp_events': self.cp_events,
            'switch_history': self.switch_history,
        }
        
        with open(os.path.join(path, 'state.json'), 'w') as f:
            json.dump(state_dict, f, indent=2, default=str)
        
        # 保存当前版本组件
        version_path = os.path.join(path, f'v{self.baseline_version}')
        os.makedirs(version_path, exist_ok=True)
        
        self.engine.save(os.path.join(version_path, 'engine'))
        with open(os.path.join(version_path, 'processor.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(version_path, 'classifier2.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        # 如果正在切换，保存待切换组件
        if self.state == self.SWITCHING and self._new_engine is not None:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            os.makedirs(new_path, exist_ok=True)
            self._new_engine.save(os.path.join(new_path, 'engine'))
            with open(os.path.join(new_path, 'processor.pkl'), 'wb') as f:
                pickle.dump(self._new_processor, f)
            with open(os.path.join(new_path, 'classifier2.pkl'), 'wb') as f:
                pickle.dump(self._new_classifier2, f)
        
        print(f"✅ 状态已保存到 {path}")
    
    def load_state(self, path="./checkpoints/adaptive_state"):
        """从持久化文件恢复状态"""
        state_file = os.path.join(path, 'state.json')
        if not os.path.exists(state_file):
            print(f"⚠️ 未找到状态文件: {state_file}")
            return False
        
        with open(state_file, 'r') as f:
            state_dict = json.load(f)
        
        # 恢复状态变量
        self.state = state_dict['state']
        self.baseline_version = state_dict['baseline_version']
        self.cp_consec = state_dict['cp_consec']
        self.cp_confirmed_tidx = state_dict.get('cp_confirmed_tidx')
        self.retrace_start_tidx = state_dict.get('retrace_start_tidx')
        self.accumulate_buffer_tidx = state_dict.get('accumulate_buffer_tidx', [])
        self.cooldown_until_tidx = state_dict.get('cooldown_until_tidx', -1)
        self._ap_warmup_remaining = state_dict.get('ap_warmup_remaining', 0)
        self._parallel_count = state_dict.get('parallel_count', 0)
        self._switch_extended = state_dict.get('switch_extended', False)
        self.cp_events = state_dict.get('cp_events', [])
        self.switch_history = state_dict.get('switch_history', [])
        
        # 恢复组件
        version_path = os.path.join(path, f'v{self.baseline_version}')
        if os.path.exists(version_path):
            from TFT_tft_engine import TFTEngine
            self.engine = TFTEngine.load(
                os.path.join(version_path, 'engine'), 
                config_path='TFT_config.yaml')
            with open(os.path.join(version_path, 'processor.pkl'), 'rb') as f:
                self.processor = pickle.load(f)
            with open(os.path.join(version_path, 'classifier2.pkl'), 'rb') as f:
                self.classifier2 = pickle.load(f)
        
        # 恢复待切换组件
        if self.state == self.SWITCHING:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            if os.path.exists(new_path):
                self._new_engine = TFTEngine.load(
                    os.path.join(new_path, 'engine'),
                    config_path='TFT_config.yaml')
                with open(os.path.join(new_path, 'processor.pkl'), 'rb') as f:
                    self._new_processor = pickle.load(f)
                with open(os.path.join(new_path, 'classifier2.pkl'), 'rb') as f:
                    self._new_classifier2 = pickle.load(f)
        
        print(f"✅ 状态已恢复: state={self.state}, version={self.baseline_version}")
        return True
       
    # ================================================================
    # 主入口: run()
    # ================================================================
    def run(self, df_real_all, df_features_raw, df_official,
            quiet_start=None, quiet_end=None, output_path=None, resume=False):
        """
        完整的自适应滚动检测流程。
        
        Parameters
        ----------
        df_real_all : 经processor+tb变换后的完整数据（含time_idx, timestamp, PCA列等）
        df_features_raw : 原始特征数据（用于refit processor）
        df_official : 官方运营日历
        quiet_start, quiet_end : 平静期时间范围（用于初始calibrate）
        output_path : 保存路径
        resume : bool
            是否从持久化状态恢复继续运行
        Returns
        -------
        pd.DataFrame : 每个窗口的检测结果
        """
        df = df_real_all.sort_values('time_idx').reset_index(drop=True)
        
        # P8修复：尝试恢复
        if resume and os.path.exists("./checkpoints/adaptive_state/state.json"):
            if self.load_state("./checkpoints/adaptive_state"):
                print("[Resume] 从保存点继续...")

        # --- 0. 冷启动校准 ---
        if quiet_start and quiet_end:
            quiet_mask = (df['timestamp'] >= quiet_start) & (df['timestamp'] < quiet_end)
            quiet_start_idx = df[df['timestamp'] >= quiet_start].index[0]
        else:
            # 默认前7天
            quiet_end_ts = df['timestamp'].min() + pd.Timedelta(days=7)
            quiet_mask = df['timestamp'] < quiet_end_ts
            quiet_start_idx = 0
        
        # 建立TFT baseline
        pre_ctx = max(0, quiet_start_idx - self.min_required)
        bl_end = df[quiet_mask].index[-1] + 1
        df_for_bl = df.iloc[pre_ctx:bl_end].copy().reset_index(drop=True)
        bl_end_tidx = int(df.loc[quiet_mask, 'time_idx'].max())
        
        print(f"[Init] 建立TFT baseline ({len(df_for_bl)} 窗口)...")
        self._patch_engine_batch_size(self.engine)

        for mn in self.engine.models:
            res = self.engine.analyze_rolling(mn, df_for_bl, baseline_end_idx=bl_end_tidx)
            if 'baseline' not in res and 'metrics' in res:
                bl = self.engine._build_baseline(
                    res['metrics'], res['attention'], res['vsn'], bl_end_tidx)
                self.engine.baselines[mn] = bl
            print(f"  [{mn}] baseline: σ={self.engine.baselines[mn]['residual_std']:.4f}")
        
        # 校准分类器2
        # 先对平静期做一次TFT推理获取model_*列
        df_quiet_for_cal = self._extract_tft_batch(df_for_bl, engine=self.engine)
        if len(df_quiet_for_cal) > 0:
            # 只取平静期时间范围内的行
            cal_mask = df_quiet_for_cal['time_idx'] <= bl_end_tidx
            self.classifier2.calibrate(df_quiet_for_cal[cal_mask])
        else:
            print("  ⚠️ 平静期TFT推理无输出, 用原始数据校准")
            self.classifier2.calibrate(df[quiet_mask])
        
        # --- 1. 初始化缓冲区 ---
        initial_end = min(quiet_start_idx + self.buffer_size, len(df))
        buffer_df = df.iloc[pre_ctx:initial_end].copy().reset_index(drop=True)
        
        quiet_start_tidx = int(df.loc[quiet_mask, 'time_idx'].min())
        output_tidx = set(
            buffer_df.loc[buffer_df['time_idx'] >= quiet_start_tidx, 'time_idx']
            .astype(int).values)
        
        print(f"\n[ColdStart] 缓冲区: {len(buffer_df)} 窗口, 推理中...")
        df_cold = self._extract_tft_batch(buffer_df, engine=self.engine,
                                          output_tidx_set=output_tidx)
        
        all_tft_results = [df_cold] if len(df_cold) > 0 else []
        processed_tidx = set(df_cold['time_idx'].astype(int).values) if len(df_cold) > 0 else set()
        
        # --- 2. 滚动主循环 ---
        remaining_start = initial_end
        total_remaining = len(df) - remaining_start
        n_batches = (total_remaining + self.step_size - 1) // self.step_size
        
        print(f"\n[Rolling] {total_remaining} 窗口, {n_batches} 批 (step={self.step_size})")
        
        for batch_i in tqdm(range(n_batches), desc="自适应滚动检测"):
            batch_start = remaining_start + batch_i * self.step_size
            batch_end = min(batch_start + self.step_size, len(df))
            
            if batch_start >= len(df):
                break
            
            new_rows = df.iloc[batch_start:batch_end]
            new_tidx = set(new_rows['time_idx'].astype(int).values) - processed_tidx
            
            if not new_tidx:
                continue
            
            # 更新缓冲区
            buffer_df = pd.concat([buffer_df, new_rows], ignore_index=True)
            if len(buffer_df) > self.buffer_size:
                buffer_df = buffer_df.iloc[-self.buffer_size:].reset_index(drop=True)
            
            if len(buffer_df) < self.min_required:
                continue
            # P7修复：AP warmup阈值放宽
            if self._ap_warmup_remaining > 0:
                self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
                self._ap_warmup_remaining -= len(new_tidx)
                if self._ap_warmup_remaining <= 0:
                    self.classifier2.set_ap_relax_factor(1.0)
                    print(f"\n  [AP Warmup] 结束")
            # P3修复：并行验证时需要两个引擎都推理
            if self.state == self.SWITCHING and self._new_engine is not None:
                df_feat_old = self._extract_tft_batch(
                    buffer_df, engine=self.engine, output_tidx_set=new_tidx)
                df_feat_new = self._extract_tft_batch(
                    buffer_df, engine=self._new_engine, output_tidx_set=new_tidx)
                
                # 主输出用旧引擎（保守）
                df_feat = df_feat_old
                
                # 收集预测结果用于对比
                if len(df_feat_old) > 0 and len(df_feat_new) > 0:
                    pred_old = self.classifier2.predict(df_feat_old)['pred_label'].values
                    pred_new = self._new_classifier2.predict(df_feat_new)['pred_label'].values
                    
                    self._parallel_preds_old.extend(pred_old.tolist())
                    self._parallel_preds_new.extend(pred_new.tolist())
                    self._parallel_count += len(pred_old)
            else:
                active_engine = self.engine
                df_feat = self._extract_tft_batch(
                    buffer_df, engine=active_engine, output_tidx_set=new_tidx)   
            # 选择当前使用的引擎
            # active_engine = self._new_engine if (
            #     self.state == self.SWITCHING and self._new_engine) else self.engine
            
            # 批量TFT推理
            # df_feat = self._extract_tft_batch(
            #     buffer_df, engine=active_engine, output_tidx_set=new_tidx)
            
            if len(df_feat) == 0:
                continue
            
            all_tft_results.append(df_feat)
            processed_tidx.update(df_feat['time_idx'].astype(int).values)
            
            # 分类器2判定
            df_classified = self.classifier2.predict(df_feat)
            
            # 状态机处理
            current_tidx = int(new_rows['time_idx'].iloc[-1])
            self._process_batch(
                df_classified, current_tidx, df, df_features_raw, df_official)
            
            # 日志
            if (batch_i + 1) % 20 == 0:
                ts = new_rows['timestamp'].iloc[-1]
                n_ap = (df_classified['pred_label'] == 'AP').sum()
                n_cp = (df_classified['pred_label'] == 'CP').sum()
                print(f"  Batch {batch_i+1}/{n_batches} | {ts} | "
                      f"state={self.state} | v={self.baseline_version} | "
                      f"AP={n_ap} CP={n_cp} warmup={self._ap_warmup_remaining}")
        
        # --- 3. 合并所有输出 ---
        if all_tft_results:
            df_all_tft = pd.concat(all_tft_results, ignore_index=True)
            df_all_tft = df_all_tft.drop_duplicates(subset='time_idx', keep='last')
            df_all_tft = df_all_tft.sort_values('time_idx').reset_index(drop=True)
            df_final = self.classifier2.predict(df_all_tft)
            df_final['baseline_version'] = self.baseline_version
            df_final['state'] = self.state
        else:
            df_final = pd.DataFrame()
        
        # 对全量结果执行分类器2（利用完整rolling窗口）
        # print(f"\n[Final] 对 {len(df_all_tft)} 窗口执行最终分类...")
        # if len(df_all_tft) > 0:
        #     df_final = self.classifier2.predict(df_all_tft)
        #     # 附加元信息
        #     df_final['baseline_version'] = self.baseline_version
        #     df_final['state'] = self.state
        # else:
        #     df_final = pd.DataFrame()
        
        # 保存
        if output_path and len(df_final) > 0:
            os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
            df_final.to_csv(output_path, index=False)
            print(f"✅ 保存到 {output_path}")
        
        self._print_summary(df_final)
        self.save_state()

        return df_final
    
    # ================================================================
    # 状态机处理逻辑
    # ================================================================
    def _process_batch(self, df_classified, current_tidx,
                       df_full, df_features_raw, df_official):
        """每批次的状态机更新"""
        
        labels = df_classified['pred_label'].values
        
        # --- NORMAL: 检测CP连续性 ---
        if self.state == self.NORMAL:
            # 冷却期检查
            if current_tidx < self.cooldown_until_tidx:
                return
            
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
                if self.cp_consec >= 1:
                    self.state = self.CP_TENTATIVE
                    print(f"\n  [→ CP_TENTATIVE] consec={self.cp_consec} @ tidx={current_tidx}")
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
        
        # --- CP_TENTATIVE: 等待确认 ---
        elif self.state == self.CP_TENTATIVE:
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
            
            # 确认
            if self.cp_consec >= self.cp_confirm_windows:
                self.cp_confirmed_tidx = current_tidx
                self.retrace_start_tidx = current_tidx - self.retrace_windows
                self.accumulate_buffer_tidx = []
                self.state = self.ACCUMULATING
                
                print(f"\n  [→ ACCUMULATING] confirmed @ tidx={current_tidx}, "
                      f"retrace={self.retrace_windows} to tidx={self.retrace_start_tidx}")
                
                self.cp_events.append({
                    'confirmed_tidx': current_tidx,
                    'retrace_tidx': self.retrace_start_tidx,
                    'retrace_windows': self.retrace_windows,
                    'switched_tidx': None,
                    'switch_type': None,
                    'baseline_version': self.baseline_version,
                })
            
            # 回退到NORMAL
            if self.cp_consec <= 0:
                self.state = self.NORMAL
                self.cp_consec = 0
                print(f"\n  [→ NORMAL] CP疑似解除 @ tidx={current_tidx}")
        
        # --- ACCUMULATING: 积累数据并触发微调 ---
        elif self.state == self.ACCUMULATING:
            # 记录当前批次的tidx范围
            batch_tidx = df_classified['time_idx'].astype(int).tolist()
            self.accumulate_buffer_tidx.extend(batch_tidx)
            
            # 检查积累量（从retrace_start开始算起到当前）
            # accumulated = current_tidx - self.retrace_start_tidx
            accumulated = len(set(self.accumulate_buffer_tidx))
            
            print(f"  [ACCUMULATING] 已积累 {accumulated}/{self.accumulate_min} 窗口", end='\r')
            
            if accumulated >= self.accumulate_min:
                print(f"\n  [微调触发] 积累 {accumulated} 窗口, 开始微调...")
                
                # 提取微调数据范围
                retrace_mask = (
                    (df_full['time_idx'] >= self.retrace_start_tidx) &
                    (df_full['time_idx'] <= current_tidx)
                )
                df_retrace = df_full[retrace_mask].copy()
                
                # 对应的原始特征范围（用于refit processor）
                ts_start = df_retrace['timestamp'].min()
                ts_end = df_retrace['timestamp'].max()
                raw_mask = (
                    (df_features_raw['timestamp'] >= ts_start) &
                    (df_features_raw['timestamp'] <= ts_end)
                )
                df_raw_retrace = df_features_raw[raw_mask].copy()
                
                # # 执行微调
                # self._finetune(df_retrace, df_raw_retrace, df_official)
                
                # # 进入并行切换
                # self.state = self.SWITCHING
                # self._parallel_preds_old = []
                # self._parallel_preds_new = []
                # print(f"  [→ SWITCHING] 开始并行验证 ({self.switch_parallel_windows} 窗口)")
                # 执行微调
                success = self._finetune(df_retrace, df_raw_retrace, df_official)
                
                if success:
                    # 进入并行切换
                    self.state = self.SWITCHING
                    self._parallel_preds_old = []
                    self._parallel_preds_new = []
                    self._parallel_count = 0
                    self._switch_extended = False
                    print(f"  [→ SWITCHING] 开始并行验证 ({self.switch_parallel_windows} 窗口)")
                else:
                    print(f"  [微调失败] 回退到NORMAL")
                    self.state = self.NORMAL
                    self._reset_partial_state()
        
        # --- P3修复：SWITCHING 并行验证 ---
        elif self.state == self.SWITCHING:
            self._check_parallel_switch(current_tidx)

        # --- SWITCHING: 并行验证 ---
        # elif self.state == self.SWITCHING:
            # 两个引擎的预测已在主循环中分别执行
            # 这里比较最近一批的pred_label一致性
            # self._parallel_preds_old.extend(labels.tolist())
            
            # 简化处理：用旧引擎的预测作为对照
            # （新引擎的推理在主循环中通过active_engine切换实现）
            # n_compared = len(self._parallel_preds_old)
            
            # if n_compared >= self.switch_parallel_windows:
            #     self._execute_switch(current_tidx)
    # ================================================================
    # P3修复：完整的并行切换逻辑
    # ================================================================
    # def _check_parallel_switch(self, current_tidx):
    #     """完整的并行验证与分层切换判定"""
        
    #     required_windows = (self.switch_parallel_extend 
    #                        if self._switch_extended 
    #                        else self.switch_parallel_windows)
        
    #     if self._parallel_count < required_windows:
    #         return
        
    #     # 计算一致率
    #     pred_old = np.array(self._parallel_preds_old[-required_windows:])
    #     pred_new = np.array(self._parallel_preds_new[-required_windows:])
        
    #     if len(pred_old) == 0 or len(pred_new) == 0:
    #         return
        
    #     # 使用Cohen's Kappa
    #     try:
    #         kappa = cohen_kappa_score(pred_old, pred_new)
    #     except:
    #         kappa = (pred_old == pred_new).mean()
        
    #     agreement = (pred_old == pred_new).mean()
        
    #     print(f"\n  [并行验证] windows={self._parallel_count}, "
    #           f"agreement={agreement:.3f}, kappa={kappa:.3f}")
        
    #     # 分层判定
    #     if agreement >= self.switch_agree_high:
    #         # 高一致性：完整切换
    #         self._execute_full_switch(current_tidx, agreement, kappa)
        
    #     elif agreement < self.switch_agree_low:
    #         # 低一致性：部分切换（仅baseline+阈值）
    #         self._execute_partial_switch(current_tidx, agreement, kappa)
        
    #     else:
    #         # 中等一致性：延长验证
    #         if not self._switch_extended:
    #             self._switch_extended = True
    #             print(f"  [延长验证] 一致率={agreement:.3f}，"
    #                   f"延长到 {self.switch_parallel_extend} 窗口")
    #         else:
    #             # 已延长过，强制决定
    #             if agreement >= 0.80:
    #                 self._execute_full_switch(current_tidx, agreement, kappa)
    #             else:
    #                 self._execute_partial_switch(current_tidx, agreement, kappa)
    def _check_parallel_switch(self, current_tidx):
        """基于模型稳定性的并行验证与切换判定"""
        
        required_windows = (self.switch_parallel_extend 
                            if self._switch_extended 
                            else self.switch_parallel_windows)
        
        if self._parallel_count < required_windows:
            return
        
        pred_old = np.array(self._parallel_preds_old[-required_windows:])
        pred_new = np.array(self._parallel_preds_new[-required_windows:])
        
        if len(pred_old) == 0 or len(pred_new) == 0:
            return
        
        # 计算正常标签(N)的占比
        old_n_ratio = (pred_old == 'N').mean()
        new_n_ratio = (pred_new == 'N').mean()
        
        print(f"\n  [并行验证] windows={self._parallel_count}, "
              f"旧模型N占比={old_n_ratio:.3f}, 新模型N占比={new_n_ratio:.3f}")
        
        # 判定逻辑：使用类属性定义的阈值
        if old_n_ratio >= self.switch_recovery_threshold:
            print("  [放弃切换] 旧模型已恢复正常，微调冗余")
            self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
            
        elif new_n_ratio >= self.switch_stability_threshold:
            print("  [触发切换] 新模型达到稳态")
            self._execute_full_switch(current_tidx, agreement=0.0, kappa=0.0)
            
        else:
            if not self._switch_extended:
                self._switch_extended = True
                print(f"  [延长验证] 新模型N占比={new_n_ratio:.3f}，未达稳态，"
                      f"延长到 {self.switch_parallel_extend} 窗口")
            else:
                print(f"  [拒绝切换] 延长验证后新模型N占比={new_n_ratio:.3f}，微调未能适应分布")
                self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)

    def _execute_full_switch(self, current_tidx, agreement, kappa):
        """完整切换：模型权重+baseline+阈值"""
        print(f"\n  [完整切换] agreement={agreement:.3f}, kappa={kappa:.3f}")
        
        self.engine = self._new_engine
        self.processor = self._new_processor
        self.classifier2 = self._new_classifier2
        
        self._finalize_switch(current_tidx, 'full', agreement, kappa)
    
    # def _execute_partial_switch(self, current_tidx, agreement, kappa):
    #     """部分切换：仅baseline+阈值，保留旧模型权重"""
    #     print(f"\n  [部分切换] agreement={agreement:.3f}，保留旧模型权重")
        
    #     # 只更新processor和分类器阈值
    #     self.processor = self._new_processor
        
    #     # 更新分类器阈值
    #     self.classifier2.ap_thresholds = copy.deepcopy(self._new_classifier2.ap_thresholds)
    #     self.classifier2.cp_base_thresh = self._new_classifier2.cp_base_thresh
    #     self.classifier2.cp_vol_thresh = self._new_classifier2.cp_vol_thresh
    #     self.classifier2._z_params = copy.deepcopy(self._new_classifier2._z_params)
        
    #     # 更新baseline但保留旧模型
    #     for mn in self._new_engine.baselines:
    #         self.engine.baselines[mn] = self._new_engine.baselines[mn]
        
    #     self._finalize_switch(current_tidx, 'partial', agreement, kappa)
    
    def _finalize_switch(self, current_tidx, switch_type, agreement, kappa):
        """切换完成的收尾工作"""
        
        self.baseline_version += 1
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        
        # P7修复：启动AP warmup
        self._ap_warmup_remaining = self.ap_warmup_windows
        self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
        
        # 记录历史
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
            self.cp_events[-1]['switch_type'] = switch_type
        
        self.switch_history.append({
            'tidx': current_tidx,
            'type': switch_type,
            'agreement': agreement,
            'kappa': kappa,
            'new_version': self.baseline_version,
        })
        
        # 重置状态
        self.state = self.NORMAL
        self._reset_partial_state()
        
        print(f"  [切换完成] type={switch_type}, version={self.baseline_version}, "
              f"冷却至 tidx={self.cooldown_until_tidx}, "
              f"AP warmup={self._ap_warmup_remaining}窗口")
    
    def _reset_partial_state(self):
        """重置切换相关状态"""
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False

    # ================================================================
    # 微调
    # ================================================================
    def _finetune(self, df_retrace, df_raw_retrace, df_official):
        """
        微调三件套: processor refit → TFT微调 → 分类器2阈值更新
        """
        from TFT_tft_engine import TFTEngine
        from tft_full_period_utils import compute_baseline_from_held_out, create_held_out_split
        
        # (a) Processor refit
        print("  [微调 1/3] Processor refit...")
        processor_new = copy.deepcopy(self.processor)
        processor_new.feature_transformer.fit(df_raw_retrace.fillna(0))
        
        # 用新processor变换数据
        dataset_new = processor_new.transform(df_raw_retrace.fillna(0), df_official)
        df_tft = dataset_new.data.copy()

        df_tft.fillna(0, inplace=True)
        df_tft.replace([np.inf, -np.inf], 0, inplace=True)
        df_tft = self.tb.transform(df_tft)
        if 'label' not in df_tft.columns:
            df_tft['label'] = 'N'
        
        print(f"  [微调数据] {len(df_tft)} 窗口")

        
        # (b) TFT微调
        print("  [微调 2/3] TFT模型微调...")
        with tempfile.TemporaryDirectory() as tmp:
            self.engine.save(tmp)
            engine_new = TFTEngine.load(tmp, config_path='TFT_config.yaml')
        
        self._patch_engine_batch_size(engine_new)
        self._patch_engine_early_stopping(engine_new)
        
        # Held-out切分
        held_out_windows = max(
            self.held_out_min_windows,
            int(len(df_tft) * 0.15)  # 最多15%
        )
        held_out_windows = min(held_out_windows, len(df_tft) // 3)

        split_idx = len(df_tft) - held_out_windows
        df_train_ft = df_tft.iloc[:split_idx].copy()
        df_held = df_tft.iloc[split_idx:].copy()
        held_out_tidx = set(df_held['time_idx'].values)
        # held_out_mask, held_mask, split_info = create_held_out_split(
        #     df_tft, time_col='timestamp',
        #     block_days=self.held_out_days,
        #     min_block_windows=96)
        # df_train_ft = df_tft[~held_mask].copy()
        # held_out_tidx = set(df_tft.loc[held_mask, 'time_idx'].values)
        print(f"  [数据切分] 训练={len(df_train_ft)}, held_out={len(df_held)}")
        
        if len(df_train_ft) < self.min_required * 2:
            print(f"  ⚠️ 训练数据不足")
            return False
        
        for mn in engine_new.models:
            # 解冻
            for p in engine_new.models[mn].parameters():
                p.requires_grad = True
            
            cfg = engine_new.config['tft_models'][mn]
            orig_lr = cfg.get('learning_rate', 0.03)
            cfg['learning_rate'] = orig_lr * self.finetune_lr_scale
            # 应用梯度裁剪
            if 'gradient_clip_val' not in cfg:
                cfg['gradient_clip_val'] = self.finetune_grad_clip

            print(f"    [{mn}] 微调 (lr={cfg['learning_rate']:.5f}, "
                  f"epochs={self.finetune_epochs}, data={len(df_train_ft)})...")
            
            with contextlib.redirect_stdout(io.StringIO()):
                engine_new.build_and_fit(
                    mn, df_train_ft, 
                    max_epochs=self.finetune_epochs)
            
            cfg['learning_rate'] = orig_lr
            
            # 重建baseline
            res = engine_new.analyze_rolling(mn, df_tft, baseline_end_idx=None)
            if 'metrics' not in res:
                print(f"    ⚠️ [{mn}] 推理失败")
                continue
            if held_out_tidx:
                bl = compute_baseline_from_held_out(res, held_out_tidx)
            else:
                bl = engine_new._build_baseline(
                    res['metrics'], res['attention'], res['vsn'],
                    int(df_tft['time_idx'].max()))
            engine_new.baselines[mn] = bl
            # engine_new.freeze_layers(mn)
            
            print(f"    [{mn}] 新baseline: σ={bl['residual_std']:.4f}")
        
        # (c) 分类器2阈值更新
        print("  [微调 3/3] 分类器2阈值更新...")
        # 用新引擎推理held-out数据，获取model_*列
        classifier2_new = copy.deepcopy(self.classifier2)

        # df_held = df_tft[held_mask].copy()
        if len(df_held) > self.min_required:
            df_held_tft = self._extract_tft_batch(df_held, engine=engine_new)
            if len(df_held_tft) > 10:
                # classifier2_new = copy.deepcopy(self.classifier2)
                # classifier2_new.calibrate(df_held_tft)
                classifier2_new.calibrate(df_held_tft)
                print(f"    新阈值校准完成")
            else:
                classifier2_new = copy.deepcopy(self.classifier2)
        else:
            classifier2_new = copy.deepcopy(self.classifier2)
        
        # 暂存新组件
        self._new_engine = engine_new
        self._new_processor = processor_new
        self._new_classifier2 = classifier2_new
        
        print("  [微调完成] 新组件已就绪，等待并行验证")
        return True

    # ================================================================
    # TFT批量推理（可配置batch_size）
    # ================================================================
    def _patch_engine_batch_size(self, engine):
        """为引擎应用推理batch_size补丁"""
        
        inference_bs = self.inference_batch_size
        
        def patched_analyze(model_name, df, baseline_end_idx=None, predict=True):
            model = engine.models[model_name]
            train_dataset = engine.datasets[model_name]
            df_inf = df.copy()
            for col in engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)
            
            try:
                inference_dataset = TimeSeriesDataSet.from_dataset(
                    train_dataset, df_inf, predict=False, stop_randomization=True)
            except ValueError as e:
                if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
                    return {}
                raise
            
            if len(inference_dataset) == 0:
                return {}
            
            dataloader = inference_dataset.to_dataloader(
                train=False, 
                batch_size=inference_bs,
                num_workers=0, 
                pin_memory=True)
            
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model.to(device)
            model.eval()
            
            result_buffer = {}
            with torch.no_grad():
                for x, y in dataloader:
                    x = {k: v.to(device) for k, v in x.items()}
                    targets = y[0].cpu().numpy().flatten()
                    raw_out = model(x)
                    preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
                    p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
                    time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
                    interp = model.interpret_output(raw_out, reduction="none")
                    batch_att = interp['attention'].cpu().numpy()
                    batch_vsn = interp['encoder_variables'].cpu().numpy()
                    
                    for i in range(len(time_idx)):
                        t_id = int(time_idx[i])
                        result_buffer[t_id] = {
                            "target_true": targets[i],
                            "pred_p50": p50[i],
                            "residual": targets[i] - p50[i],
                            "divergence": p90[i] - p10[i],
                            "attention": batch_att[i],
                            "vsn": batch_vsn[i]
                        }
            
            if not result_buffer:
                return {}
            
            sorted_t = sorted(result_buffer.keys())
            df_s = pd.DataFrame({
                "time_idx": sorted_t,
                "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
                "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
                "residual": [result_buffer[t]["residual"] for t in sorted_t],
                "divergence": [result_buffer[t]["divergence"] for t in sorted_t]
            })
            full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
            full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
            
            result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
            
            if baseline_end_idx is not None:
                bl = engine._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
                result["baseline"] = bl
                engine.baselines[model_name] = bl
            
            return result
        
        engine.analyze_rolling = patched_analyze
        
    def _patch_engine_early_stopping(self, engine):
        """为引擎应用移除早停机制，并修复 batch_size 硬编码问题的补丁"""
        import types
        import torch
        import pandas as pd
        import lightning.pytorch as pl
        from lightning.pytorch.loggers import CSVLogger
        from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
        from pytorch_forecasting.metrics import QuantileLoss
        from pytorch_forecasting.data import GroupNormalizer
        from pytorch_forecasting.data.encoders import NaNLabelEncoder

        # 获取外部检测器配置的微调批次大小
        ft_batch_size = self.finetune_batch_size

        def patched_build_and_fit(self_engine, model_name: str, df: pd.DataFrame, 
                                  val_ratio: float = 0.2, max_epochs: int = -1,
                                  quiet_end_idx: int = None):
            
            cfg = self_engine.config['tft_models'][model_name]
            pl.seed_everything(cfg.get('seed', 42))

            df_inf = df.copy()
            for col in self_engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)

            if quiet_end_idx is not None:
                time_col = self_engine.global_cfg['time_column']
                df_for_training = df_inf[df_inf[time_col] <= quiet_end_idx].copy()
            else:
                df_for_training = df_inf

            time_col = self_engine.global_cfg['time_column']
            time_steps = df_for_training[time_col].sort_values().unique()
            split_idx = int(len(time_steps) * (1 - val_ratio))
            cutoff_time = time_steps[split_idx]
            
            train_df = df_for_training[df_for_training[time_col] <= cutoff_time]
            max_lookback = cfg.get('max_encoder_length', 48)
            val_start_time = time_steps[max(0, split_idx - max_lookback)]
            val_df = df_for_training[df_for_training[time_col] >= val_start_time]

            categorical_encoders = {
                name: NaNLabelEncoder(add_nan=True) 
                for name in self_engine.known_categoricals
            }
            for g_id in self_engine.global_cfg['group_ids']:
                categorical_encoders[g_id] = NaNLabelEncoder(add_nan=True)

            training = TimeSeriesDataSet(
                train_df,
                time_idx=self_engine.global_cfg['time_column'],
                target=cfg['target'],
                group_ids=self_engine.global_cfg['group_ids'],
                min_encoder_length=cfg.get('min_encoder_length'),
                max_encoder_length=cfg.get('max_encoder_length'),
                max_prediction_length=cfg.get('max_prediction_length', 1),
                time_varying_unknown_reals=self_engine.unknown_reals,
                time_varying_known_reals=self_engine.known_reals,
                time_varying_known_categoricals=self_engine.known_categoricals,
                categorical_encoders=categorical_encoders,
                target_normalizer=GroupNormalizer(groups=self_engine.global_cfg['group_ids'], transformation=None),
                add_relative_time_idx=True,
                add_target_scales=True,
                add_encoder_length=True,
                allow_missing_timesteps=True
            )

            validation = TimeSeriesDataSet.from_dataset(training, val_df, predict=False, stop_randomization=True)

            # 核心修改：使用 ft_batch_size 替代写死的 512
            train_dataloader = training.to_dataloader(train=True, batch_size=ft_batch_size, num_workers=0, pin_memory=True)
            val_dataloader = validation.to_dataloader(train=False, batch_size=ft_batch_size, num_workers=0, pin_memory=True)

            model = TemporalFusionTransformer.from_dataset(
                training,
                learning_rate=cfg.get('learning_rate', 0.03),
                hidden_size=cfg.get('hidden_size', 16),
                attention_head_size=4,
                dropout=0.1,
                hidden_continuous_size=8,
                output_size=len(self_engine.quantiles),
                loss=QuantileLoss(quantiles=self_engine.quantiles),
                reduce_on_plateau_patience=4
            )

            logger = CSVLogger("lightning_logs", name=model_name, flush_logs_every_n_steps=10)
            self_engine.log_dirs[model_name] = logger.log_dir

            trainer_kwargs = {
                "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
                "devices": 1,
                "enable_model_summary": False,
                "enable_checkpointing": True,
                "logger": logger,
                "enable_progress_bar": True
            }

            if max_epochs == -1:
                trainer = pl.Trainer(**trainer_kwargs)
            elif max_epochs != 1:
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)
            elif max_epochs == 1:
                trainer_kwargs.update({"limit_train_batches": 5, "limit_val_batches": 5})
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)

            trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

            self_engine.plot_training_history(model_name)
            
            self_engine.datasets[model_name] = training
            self_engine.models[model_name] = model

        engine.build_and_fit = types.MethodType(patched_build_and_fit, engine) 
    # ================================================================
    # 切换
    # ================================================================
    def _execute_switch(self, current_tidx):
        """执行分层切换判定"""
        
        # 由于并行推理在简化实现中无法精确对比，这里直接执行切换
        # 完整实现中应同时用新旧引擎推理同一批数据并比较一致率
        
        # 立即切换: processor + baseline + 分类器阈值
        print(f"\n  [SWITCH] 执行切换 @ tidx={current_tidx}")
        
        self.processor = self._new_processor
        self.engine = self._new_engine
        self.classifier2 = self._new_classifier2
        
        self.baseline_version += 1
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        self._switch_warmup_count = self.ap_warmup_windows
        
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
        
        # 重置状态
        self.state = self.NORMAL
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        
        print(f"  [SWITCH] 完成! baseline_version={self.baseline_version}, "
              f"冷却至 tidx={self.cooldown_until_tidx}")
    
    # ================================================================
    # TFT批量推理（复用滚动训练Cell 2的逻辑）
    # ================================================================
    def _extract_tft_batch(self, df_buffer, engine=None, output_tidx_set=None):
        """
        对缓冲区执行TFT推理，提取model_*信号 + rolling。
        直接复用notebook Cell 2 的 extract_tft_features_batch 逻辑。
        """
        if engine is None:
            engine = self.engine
        
        # --- TFT推理 ---
        tft_results = {}
        for mn in engine.models:
            with contextlib.redirect_stdout(io.StringIO()), \
                 contextlib.redirect_stderr(io.StringIO()):
                res = engine.analyze_rolling(mn, df_buffer, baseline_end_idx=None)
            tft_results[mn] = res
        
        if not any('metrics' in v for v in tft_results.values()):
            return pd.DataFrame()
        
        # --- 合并TFT信号 ---
        df_merged = df_buffer[['time_idx', 'timestamp']].copy()
        
        for model_name in tft_results:
            res = tft_results[model_name]
            if 'metrics' not in res:
                continue
            
            metrics = res['metrics']
            attention = res['attention']
            vsn = res['vsn']
            baseline = engine.baselines.get(model_name, {})
            
            if not baseline:
                continue
            
            n = len(metrics)
            
            # 残差比
            residual_abs = metrics['residual'].abs().values
            baseline_p95 = max(
                abs(baseline.get('residual_p95', 0)),
                abs(baseline.get('residual_p5', 0)),
                baseline.get('residual_std', 1e-8) * 1.645, 1e-8)
            sig_residual = np.clip(residual_abs / baseline_p95, 0, 20)
            
            # 注意力KL散度
            att_base = baseline.get('att_mean')
            sig_att_kl = np.zeros(n)
            if att_base is not None:
                for i in range(n):
                    p = attention[i].flatten()
                    q = att_base.flatten()
                    p = p / (p.sum() + 1e-10) + 1e-10
                    q = q / (q.sum() + 1e-10) + 1e-10
                    p, q = p / p.sum(), q / q.sum()
                    sig_att_kl[i] = np.sum(p * np.log(p / q))
            
            # VSN JS散度
            vsn_base = baseline.get('vsn_mean')
            sig_vsn_js = np.zeros(n)
            if vsn_base is not None:
                for i in range(n):
                    v_i = np.abs(vsn[i].flatten()) + 1e-10
                    v_b = np.abs(vsn_base.flatten()) + 1e-10
                    p_v, q_v = v_i / v_i.sum(), v_b / v_b.sum()
                    m_v = 0.5 * (p_v + q_v)
                    sig_vsn_js[i] = (
                        0.5 * np.sum(p_v * np.log(p_v / m_v)) +
                        0.5 * np.sum(q_v * np.log(q_v / m_v)))
            
            # VSN排序变化
            sig_vsn_rank = np.zeros(n)
            if vsn_base is not None:
                baseline_rank = np.argsort(np.argsort(-np.abs(vsn_base.flatten())))
                for i in range(n):
                    curr_rank = np.argsort(np.argsort(-np.abs(vsn[i].flatten())))
                    corr, _ = spearmanr(baseline_rank, curr_rank)
                    sig_vsn_rank[i] = 1 - corr if not np.isnan(corr) else 1.0
            
            # 组装
            prefix = f'model_{model_name}'
            tft_df = pd.DataFrame({
                'time_idx': metrics['time_idx'].values,
                f'{prefix}_residual_ratio': sig_residual,
                f'{prefix}_att_kl': np.clip(sig_att_kl, 0, 20),
                f'{prefix}_vsn_js': np.clip(sig_vsn_js, 0, 5),
                f'{prefix}_vsn_rank_shift': np.clip(sig_vsn_rank, 0, 2),
            }).drop_duplicates(subset='time_idx', keep='last')
            
            # merge (防膨胀)
            n_before = len(df_merged)
            df_merged = df_merged.merge(tft_df, on='time_idx', how='left')
            if len(df_merged) != n_before:
                # 回退, 使用map方式
                new_cols = [c for c in tft_df.columns if c != 'time_idx']
                df_merged = df_merged.drop(columns=new_cols).iloc[:n_before]
                for col in new_cols:
                    val_map = dict(zip(tft_df['time_idx'], tft_df[col]))
                    df_merged[col] = df_merged['time_idx'].map(val_map)
        
        # --- Rolling特征 ---
        base_tft_cols = [c for c in df_merged.columns
                         if c.startswith('model_') and 'roll' not in c]
        for col in base_tft_cols:
            for k in [4, 8]:
                df_merged[f'{col}_rollmax_{k}'] = (
                    df_merged[col].rolling(k, min_periods=1).max())
                df_merged[f'{col}_rollmean_{k}'] = (
                    df_merged[col].rolling(k, min_periods=1).mean())
        
        df_merged = df_merged.replace([np.inf, -np.inf], 0).fillna(0)
        
        # 只保留model_*列 + time_idx + timestamp
        keep_cols = ['time_idx', 'timestamp'] + [
            c for c in df_merged.columns if c.startswith('model_')]
        df_merged = df_merged[[c for c in keep_cols if c in df_merged.columns]]
        
        # 筛选输出行
        if output_tidx_set is not None:
            df_merged = df_merged[df_merged['time_idx'].isin(output_tidx_set)]
        
        return df_merged
    
    # ================================================================
    # 工具
    # ================================================================
    def _print_summary(self, df_final):
        """打印运行总结"""
        print(f"\n{'='*60}")
        print(f"  自适应滚动检测 — 运行总结")
        print(f"{'='*60}")
        if len(df_final) > 0:
            vc = df_final['pred_label'].value_counts()
            print(f"  总窗口: {len(df_final)}")
            print(f"  N={vc.get('N',0)}, AP={vc.get('AP',0)}, CP={vc.get('CP',0)}")
        print(f"  最终baseline版本: {self.baseline_version}")
        print(f"  CP事件: {len(self.cp_events)}")
        for i, evt in enumerate(self.cp_events):
            print(f"    #{i}: confirmed@{evt['confirmed_tidx']}, "
                  f"retrace@{evt['retrace_tidx']}, "
                  f"switched@{evt.get('switched_tidx', '未切换')}")
        print(f"{'='*60}")
    
    def save_state(self, path="./checkpoints/adaptive_state"):
        """持久化当前状态"""
        os.makedirs(path, exist_ok=True)
        state = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_events': self.cp_events,
            'cooldown_until_tidx': self.cooldown_until_tidx,
        }
        with open(os.path.join(path, 'state.json'), 'w') as f:
            import json
            json.dump(state, f, indent=2, default=str)
        
        self.engine.save(os.path.join(path, f'engine_v{self.baseline_version}'))
        with open(os.path.join(path, f'processor_v{self.baseline_version}.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(path, f'classifier2_v{self.baseline_version}.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        print(f"✅ 状态保存到 {path}")

## 窗口检测

In [ ]:
# ============================================================
# Cell: 自适应滚动检测 — 完整运行
# ============================================================
import types, io, contextlib, copy, os, pickle
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pytorch_forecasting import TimeSeriesDataSet
from TFT_tft_engine import TFTEngine
from target_builder import TargetBuilder
# from adaptive_rolling_detector import StatisticalClassifier2, AdaptiveRollingDetector

# =====================================================================
# 0. 加载预训练系统
# =====================================================================
CKPT = "./checkpoints/pretrained_planPC"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine_PC = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)

# 绑定去重补丁（与Cell 1相同）
def analyze_rolling_patched(self, model_name, df, baseline_end_idx=None, predict=True):
    model = self.models[model_name]
    train_dataset = self.datasets[model_name]
    df_inf = df.copy()
    for col in self.known_categoricals:
        df_inf[col] = df_inf[col].astype(str)
    try:
        inference_dataset = TimeSeriesDataSet.from_dataset(
            train_dataset, df_inf, predict=False, stop_randomization=True)
    except ValueError as e:
        if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
            return {}
        raise
    if len(inference_dataset) == 0:
        return {}
    dataloader = inference_dataset.to_dataloader(
        train=False, batch_size=512, num_workers=0, pin_memory=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()
    result_buffer = {}
    with torch.no_grad():
        for x, y in dataloader:
            x = {k: v.to(device) for k, v in x.items()}
            targets = y[0].cpu().numpy().flatten()
            raw_out = model(x)
            preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
            p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
            time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
            interp = model.interpret_output(raw_out, reduction="none")
            batch_att = interp['attention'].cpu().numpy()
            batch_vsn = interp['encoder_variables'].cpu().numpy()
            for i in range(len(time_idx)):
                t_id = int(time_idx[i])
                result_buffer[t_id] = {
                    "target_true": targets[i], "pred_p50": p50[i],
                    "residual": targets[i] - p50[i], "divergence": p90[i] - p10[i],
                    "attention": batch_att[i], "vsn": batch_vsn[i]}
    if not result_buffer:
        return {}
    sorted_t = sorted(result_buffer.keys())
    df_s = pd.DataFrame({
        "time_idx": sorted_t,
        "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
        "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
        "residual": [result_buffer[t]["residual"] for t in sorted_t],
        "divergence": [result_buffer[t]["divergence"] for t in sorted_t]})
    full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
    full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
    result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
    if baseline_end_idx is not None:
        bl = self._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
        result["baseline"] = bl
        self.baselines[model_name] = bl
    return result

engine_PC.analyze_rolling = types.MethodType(analyze_rolling_patched, engine_PC)

# =====================================================================
# 1. 数据预处理（与滚动训练Cell 1一致）
# =====================================================================
QUIET_START = pd.to_datetime('2025-02-18 00:00:00')
QUIET_END   = pd.to_datetime('2025-02-25 00:00:00')

# 用平静期refit processor
mask_quiet = (df_features['timestamp'] >= QUIET_START) & (df_features['timestamp'] < QUIET_END)
df_quiet_raw = df_features[mask_quiet].copy().fillna(0)

processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_quiet_raw)
print(f"[Processor] re-fitted, global_min_time={processor_real.global_min_time}")

# 变换全量数据
dataset_real = processor_real.transform(df_features.fillna(0), df_official_real)
df_real_all = dataset_real.data.copy()
df_real_all.fillna(0, inplace=True)
df_real_all.replace([np.inf, -np.inf], 0, inplace=True)
df_real_all = tb.transform(df_real_all)
if 'label' not in df_real_all.columns:
    df_real_all['label'] = 'N'
df_real_all = df_real_all.sort_values('time_idx').reset_index(drop=True)
print(f"[Data] {df_real_all.shape}, {df_real_all['timestamp'].min()} ~ {df_real_all['timestamp'].max()}")

# =====================================================================
# 2. 构建分类器2 & 自适应检测器
# =====================================================================
classifier2 = StatisticalClassifier2(
    ap_k=5.0, cp_base_k=3.0, cp_density_thresh=0.95,
    cp_window=288, cp_min_periods=96, ap_min_triggers=4
)

detector = AdaptiveRollingDetector(
    engine=engine_PC,
    tb=tb,
    processor=processor_real,
    classifier2=classifier2,
    # 滚动
    buffer_size=672, step_size=96,min_required=None,
    # CP确认
    # cp_confirm_windows=32,
    cp_confirm_windows=1,

    # 回溯与积累
    retrace_windows=288, accumulate_min=576,
    # 微调
    finetune_epochs=100, finetune_lr_scale=0.005, finetune_batch_size=16, held_out_days=1,
    # 切换
    switch_parallel_windows=96,
    switch_agree_high=0.90, switch_agree_low=0.70,
    # 冷却
    cooldown_windows=384,ap_warmup_windows=96, ap_warmup_relax=1.2,
    inference_batch_size=256
)

# =====================================================================
# 3. 运行
# =====================================================================
df_result = detector.run(
    df_real_all=df_real_all,
    df_features_raw=df_features.fillna(0),
    df_official=df_official_real,
    quiet_start=QUIET_START,
    quiet_end=QUIET_END,
    output_path="./output/adaptive_detection_result.csv"
)

# 保存状态
detector.save_state("./checkpoints/adaptive_state")

print(f"\n🎯 检测完成, 结果: {len(df_result)} 窗口")

## 检测V2

In [ ]:
"""
自适应滚动检测器：
  滚动TFT推理 → 分类器2判定 → CP确认 → 回溯288窗口 → 积累微调 → 分层切换
  
  状态机：NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
"""

import os, copy, pickle, tempfile, io, contextlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import spearmanr

import logging

# 关闭 PyTorch Lightning 的硬件与环境提示
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

# 关闭 数据处理器与 TargetBuilder 的常规INFO日志
logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
logging.getLogger("TargetBuilder").setLevel(logging.WARNING)
# 如果有其他自定义 logger，也可以一并设置为 WARNING
# =====================================================================
# 分类器2: 纯统计阈值 AP/CP 判定
# =====================================================================
class StatisticalClassifier3:
    """
    基于冷启动期校准的多维Z-score分类器。
    输入: 滚动TFT推理输出的 model_* 信号列
    输出: 每个窗口的 pred_label (N/AP/CP) + 投票归因信息
    """
    
    # 8个AP候选特征
    AP_ROLL_FEATURES = [
        'model_model_comment_pc1_residual_ratio_rollmax_8',
        'model_model_post_pc2_residual_ratio_rollmax_8',
        'model_model_comment_pc1_vsn_rank_shift_rollmax_8',
        'model_model_post_pc2_vsn_rank_shift_rollmax_8',
        'model_model_comment_pc1_att_kl_rollmax_8',
        'model_model_post_pc2_att_kl_rollmax_8',
        'model_model_comment_pc1_vsn_js_rollmax_8',
        'model_model_post_pc2_vsn_js_rollmax_8',
    ]
    
    # 4个CP静态特征
    CP_STATIC_FEATURES = [
        'model_model_comment_pc1_vsn_rank_shift',
        'model_model_post_pc2_vsn_rank_shift',
        'model_model_comment_pc1_att_kl',
        'model_model_post_pc2_att_kl',
    ]
    
    def __init__(self, ap_k=5.0, cp_base_k=3.0, cp_density_thresh=0.95,
                 cp_window=288, cp_min_periods=96, ap_min_triggers=4):
        self.ap_k = ap_k
        self.cp_base_k = cp_base_k
        self.cp_density_thresh = cp_density_thresh
        self.cp_window = cp_window
        self.cp_min_periods = cp_min_periods
        self.ap_min_triggers = ap_min_triggers
        
        # 由 calibrate() 填充
        self.ap_thresholds = {}
        self.cp_base_thresh = None
        self.cp_vol_thresh = None
        self._z_params = {}  # CP静态特征的Z标准化参数
        self._calibrated = False

        # 动态阈值放宽因子（由外部控制）
        self._ap_relax_factor = 1.0
    
    def calibrate(self, df_base):
        """用冷启动期数据校准所有阈值"""
        # AP阈值
        for feat in self.AP_ROLL_FEATURES:
            if feat in df_base.columns:
                self.ap_thresholds[feat] = (
                    df_base[feat].mean() + self.ap_k * df_base[feat].std()
                )
        
        # CP静态特征Z标准化参数
        for feat in self.CP_STATIC_FEATURES:
            if feat in df_base.columns:
                self._z_params[feat] = {
                    'mean': df_base[feat].mean(),
                    'std': df_base[feat].std() + 1e-8,
                }
        
        # CP单步水位阈值
        z_max_vals = self._compute_z_max(df_base)
        self.cp_base_thresh = z_max_vals.mean() + self.cp_base_k * z_max_vals.std()
        self.cp_vol_thresh = z_max_vals.std() * 1.5
        
        self._calibrated = True
        print(f"[Classifier2] 校准完成: AP阈值={len(self.ap_thresholds)}个, "
              f"CP水位={self.cp_base_thresh:.3f}, CP波动上限={self.cp_vol_thresh:.3f}")
        
    def set_ap_relax_factor(self, factor):
        """设置AP阈值放宽因子"""
        self._ap_relax_factor = factor
    
    def _compute_z_max(self, df):
        """计算各行的多维Z-score最大值"""
        z_vals = []
        for feat, params in self._z_params.items():
            if feat in df.columns:
                z = (df[feat] - params['mean']) / params['std']
                z_vals.append(z)
        if z_vals:
            return pd.concat(z_vals, axis=1).max(axis=1)
        return pd.Series(0.0, index=df.index)
    
    def predict(self, df):
        """
        对DataFrame逐行判定，返回带 pred_label + 归因信息的DataFrame。
        需要先调用 calibrate()。
        
        Returns: df 加上列:
          pred_label, ap_trigger_count, cp_density, cp_volatility,
          dominant_model, dominant_signal, trigger_features
        """
        assert self._calibrated, "必须先调用 calibrate()"
        
        df = df.copy()
        
        # --- CP辅助信号 ---
        z_max = self._compute_z_max(df)
        df['static_signal_z_max'] = z_max
        is_high = (z_max > self.cp_base_thresh).astype(float)
        df['cp_density'] = is_high.rolling(
            self.cp_window, min_periods=self.cp_min_periods).mean()
        df['cp_volatility'] = z_max.rolling(
            self.cp_window, min_periods=self.cp_min_periods).std()
        
        # --- AP触发计数 ---
        trigger_matrix = pd.DataFrame(index=df.index)
        for feat in self.AP_ROLL_FEATURES:
            if feat in df.columns and feat in self.ap_thresholds:
                effective_thresh = self.ap_thresholds[feat] * self._ap_relax_factor
                trigger_matrix[feat] = (df[feat] > effective_thresh).astype(int)
        df['ap_trigger_count'] = trigger_matrix.sum(axis=1)

        # --- 判定 ---
        df['pred_label'] = 'N'
        
        # CP: 高密度 + 低波动
        mask_cp = (
            (df['cp_density'] > self.cp_density_thresh) &
            (df['cp_volatility'] < self.cp_vol_thresh)
        )
        df.loc[mask_cp, 'pred_label'] = 'CP'
        
        # AP: 多维同时触发 且不是CP
        mask_ap = (df['ap_trigger_count'] >= self.ap_min_triggers) & (~mask_cp)
        df.loc[mask_ap, 'pred_label'] = 'AP'
        
        # --- 投票归因 ---
        df['dominant_model'] = 'N/A'
        df['dominant_signal'] = 'N/A'
        df['trigger_features'] = ''
        
        anom_mask = df['pred_label'] != 'N'
        if anom_mask.any():
            # for idx in df[anom_mask].index:
            #     triggers = []
            #     for feat in trigger_matrix.columns:
            #         if trigger_matrix.loc[idx, feat] == 1:
            #             triggers.append(feat)
                
            #     if not triggers:
            #         continue
            for idx in df[anom_mask].index:
                triggers = [f for f in trigger_matrix.columns 
                           if trigger_matrix.loc[idx, f] == 1]
                if not triggers:
                    continue


                # 子模型投票
                comment_v = sum(1 for f in triggers if 'comment_pc1' in f)
                post_v = sum(1 for f in triggers if 'post_pc2' in f)
                dom_model = 'comment_pc1' if comment_v >= post_v else 'post_pc2'
                
                # 信号类型投票
                signal_votes = {}
                for sig_type in ['residual_ratio', 'vsn_rank_shift', 'att_kl', 'vsn_js']:
                    signal_votes[sig_type] = sum(1 for f in triggers if sig_type in f)
                dom_signal = max(signal_votes, key=signal_votes.get)
                
                df.loc[idx, 'dominant_model'] = dom_model
                df.loc[idx, 'dominant_signal'] = dom_signal
                df.loc[idx, 'trigger_features'] = '|'.join(triggers)
        
        return df



### 检测

In [ ]:

# =====================================================================
# 自适应滚动检测器
# =====================================================================
class AdaptiveRollingDetector2:
    """
    状态机:
      NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
    
    每个STEP_SIZE窗口执行一次批量TFT推理 + 分类器2判定，
    检测到CP后回溯288窗口，积累到576窗口后微调TFT，分层切换。
    """
    
    # 状态常量
    NORMAL = 'NORMAL'
    CP_TENTATIVE = 'CP_TENTATIVE'
    ACCUMULATING = 'ACCUMULATING'
    SWITCHING = 'SWITCHING'
    
    def __init__(self, engine, tb, processor, classifier2,
                 # 滚动参数
                 buffer_size=672, step_size=96, min_required=None,
                 # CP确认
                 cp_confirm_windows=32,
                 # 回溯与积累
                 retrace_windows=288,
                 accumulate_min=576,
                 # 微调
                 finetune_epochs=50, 
                 finetune_lr_scale=0.1,
                 finetune_batch_size = 32,
                 held_out_days=2,
                 finetune_grad_clip=0.5,
                 held_out_min_windows=96,
                 # 切换 # P3修复：切换（完整并行验证）
                #  switch_parallel_windows=96,
                #  switch_parallel_extend=192,
                #  switch_agree_high=0.90, 
                #  switch_agree_low=0.70,
                # 切换参数（修改此部分）
                 switch_parallel_windows=96,
                 switch_parallel_extend=192,
                 switch_recovery_threshold=0.95,  # 新增：旧模型恢复正常的N标签比例阈值
                 switch_stability_threshold=0.85,
                 # 冷却
                 cooldown_windows=384,
                 # AP阈值保护
                 ap_warmup_windows=48, 
                 ap_warmup_relax=1.2,
                 inference_batch_size=256):
        
        self.engine = engine
        self.tb = tb
        self.processor = processor
        self.classifier2 = classifier2
        
        # 滚动参数
        self.buffer_size = buffer_size
        self.step_size = step_size
        ds_0 = list(engine.datasets.values())[0]
        self.min_encoder = ds_0.max_encoder_length
        self.min_pred = ds_0.max_prediction_length
        self.min_required = min_required or (self.min_encoder + self.min_pred + 1)
        
        # CP确认
        self.cp_confirm_windows = cp_confirm_windows
        
        # 回溯与积累
        self.retrace_windows = retrace_windows
        self.accumulate_min = accumulate_min
        
        # 微调
        self.finetune_epochs = finetune_epochs
        self.finetune_lr_scale = finetune_lr_scale
        self.finetune_batch_size = finetune_batch_size
        self.finetune_grad_clip = finetune_grad_clip
        self.held_out_days = held_out_days
        self.held_out_min_windows = held_out_min_windows
        
        # P3修复：切换
        self.switch_parallel_windows = switch_parallel_windows
        self.switch_parallel_extend = switch_parallel_extend
        # self.switch_agree_high = switch_agree_high
        # self.switch_agree_low = switch_agree_low
        self.switch_recovery_threshold =  switch_recovery_threshold  # 新增：旧模型恢复正常的N标签比例阈值
        self.switch_stability_threshold = switch_stability_threshold
        
        # P7修复：冷却与保护
        self.cooldown_windows = cooldown_windows
        self.ap_warmup_windows = ap_warmup_windows
        self.ap_warmup_relax = ap_warmup_relax
        
        # 推理配置
        self.inference_batch_size = inference_batch_size
        
        # --- 状态变量 ---
        self.state = self.NORMAL
        self.baseline_version = 0
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self.cooldown_until_tidx = -1
        
        # P3修复：并行切换状态
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False
        
        # P7修复：AP warmup状态
        self._ap_warmup_remaining = 0
        
        # 历史记录
        self.cp_events = []
        self.switch_history = []
        
    # ================================================================
    # P8修复：状态持久化与恢复
    # ================================================================
    def save_state(self, path="./checkpoints/adaptive_state"):
        """持久化当前完整状态"""
        os.makedirs(path, exist_ok=True)
        
        state_dict = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_confirmed_tidx': self.cp_confirmed_tidx,
            'retrace_start_tidx': self.retrace_start_tidx,
            'accumulate_buffer_tidx': self.accumulate_buffer_tidx,
            'cooldown_until_tidx': self.cooldown_until_tidx,
            'ap_warmup_remaining': self._ap_warmup_remaining,
            'parallel_count': self._parallel_count,
            'switch_extended': self._switch_extended,
            'cp_events': self.cp_events,
            'switch_history': self.switch_history,
        }
        
        with open(os.path.join(path, 'state.json'), 'w') as f:
            json.dump(state_dict, f, indent=2, default=str)
        
        # 保存当前版本组件
        version_path = os.path.join(path, f'v{self.baseline_version}')
        os.makedirs(version_path, exist_ok=True)
        
        self.engine.save(os.path.join(version_path, 'engine'))
        with open(os.path.join(version_path, 'processor.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(version_path, 'classifier2.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        # 如果正在切换，保存待切换组件
        if self.state == self.SWITCHING and self._new_engine is not None:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            os.makedirs(new_path, exist_ok=True)
            self._new_engine.save(os.path.join(new_path, 'engine'))
            with open(os.path.join(new_path, 'processor.pkl'), 'wb') as f:
                pickle.dump(self._new_processor, f)
            with open(os.path.join(new_path, 'classifier2.pkl'), 'wb') as f:
                pickle.dump(self._new_classifier2, f)
        
        print(f"✅ 状态已保存到 {path}")
    
    def load_state(self, path="./checkpoints/adaptive_state"):
        """从持久化文件恢复状态"""
        state_file = os.path.join(path, 'state.json')
        if not os.path.exists(state_file):
            print(f"⚠️ 未找到状态文件: {state_file}")
            return False
        
        with open(state_file, 'r') as f:
            state_dict = json.load(f)
        
        # 恢复状态变量
        self.state = state_dict['state']
        self.baseline_version = state_dict['baseline_version']
        self.cp_consec = state_dict['cp_consec']
        self.cp_confirmed_tidx = state_dict.get('cp_confirmed_tidx')
        self.retrace_start_tidx = state_dict.get('retrace_start_tidx')
        self.accumulate_buffer_tidx = state_dict.get('accumulate_buffer_tidx', [])
        self.cooldown_until_tidx = state_dict.get('cooldown_until_tidx', -1)
        self._ap_warmup_remaining = state_dict.get('ap_warmup_remaining', 0)
        self._parallel_count = state_dict.get('parallel_count', 0)
        self._switch_extended = state_dict.get('switch_extended', False)
        self.cp_events = state_dict.get('cp_events', [])
        self.switch_history = state_dict.get('switch_history', [])
        
        # 恢复组件
        version_path = os.path.join(path, f'v{self.baseline_version}')
        if os.path.exists(version_path):
            from TFT_tft_engine import TFTEngine
            self.engine = TFTEngine.load(
                os.path.join(version_path, 'engine'), 
                config_path='TFT_config.yaml')
            with open(os.path.join(version_path, 'processor.pkl'), 'rb') as f:
                self.processor = pickle.load(f)
            with open(os.path.join(version_path, 'classifier2.pkl'), 'rb') as f:
                self.classifier2 = pickle.load(f)
        
        # 恢复待切换组件
        if self.state == self.SWITCHING:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            if os.path.exists(new_path):
                self._new_engine = TFTEngine.load(
                    os.path.join(new_path, 'engine'),
                    config_path='TFT_config.yaml')
                with open(os.path.join(new_path, 'processor.pkl'), 'rb') as f:
                    self._new_processor = pickle.load(f)
                with open(os.path.join(new_path, 'classifier2.pkl'), 'rb') as f:
                    self._new_classifier2 = pickle.load(f)
        
        print(f"✅ 状态已恢复: state={self.state}, version={self.baseline_version}")
        return True
       
    # ================================================================
    # 主入口: run()
    # ================================================================
    def run(self, df_real_all, df_features_raw, df_official,
            quiet_start=None, quiet_end=None, output_path=None, resume=False):
        """
        完整的自适应滚动检测流程。
        
        Parameters
        ----------
        df_real_all : 经processor+tb变换后的完整数据（含time_idx, timestamp, PCA列等）
        df_features_raw : 原始特征数据（用于refit processor）
        df_official : 官方运营日历
        quiet_start, quiet_end : 平静期时间范围（用于初始calibrate）
        output_path : 保存路径
        resume : bool
            是否从持久化状态恢复继续运行
        Returns
        -------
        pd.DataFrame : 每个窗口的检测结果
        """
        df = df_real_all.sort_values('time_idx').reset_index(drop=True)
        
        # P8修复：尝试恢复
        if resume and os.path.exists("./checkpoints/adaptive_state/state.json"):
            if self.load_state("./checkpoints/adaptive_state"):
                print("[Resume] 从保存点继续...")

        # --- 0. 冷启动校准 ---
        if quiet_start and quiet_end:
            quiet_mask = (df['timestamp'] >= quiet_start) & (df['timestamp'] < quiet_end)
            quiet_start_idx = df[df['timestamp'] >= quiet_start].index[0]
        else:
            # 默认前7天
            quiet_end_ts = df['timestamp'].min() + pd.Timedelta(days=7)
            quiet_mask = df['timestamp'] < quiet_end_ts
            quiet_start_idx = 0
        
        # 建立TFT baseline
        pre_ctx = max(0, quiet_start_idx - self.min_required)
        bl_end = df[quiet_mask].index[-1] + 1
        df_for_bl = df.iloc[pre_ctx:bl_end].copy().reset_index(drop=True)
        bl_end_tidx = int(df.loc[quiet_mask, 'time_idx'].max())
        
        print(f"[Init] 建立TFT baseline ({len(df_for_bl)} 窗口)...")
        self._patch_engine_batch_size(self.engine)

        for mn in self.engine.models:
            res = self.engine.analyze_rolling(mn, df_for_bl, baseline_end_idx=bl_end_tidx)
            if 'baseline' not in res and 'metrics' in res:
                bl = self.engine._build_baseline(
                    res['metrics'], res['attention'], res['vsn'], bl_end_tidx)
                self.engine.baselines[mn] = bl
            print(f"  [{mn}] baseline: σ={self.engine.baselines[mn]['residual_std']:.4f}")
        
        # 校准分类器2
        # 先对平静期做一次TFT推理获取model_*列
        df_quiet_for_cal = self._extract_tft_batch(df_for_bl, engine=self.engine)
        if len(df_quiet_for_cal) > 0:
            # 只取平静期时间范围内的行
            cal_mask = df_quiet_for_cal['time_idx'] <= bl_end_tidx
            self.classifier2.calibrate(df_quiet_for_cal[cal_mask])
        else:
            print("  ⚠️ 平静期TFT推理无输出, 用原始数据校准")
            self.classifier2.calibrate(df[quiet_mask])
        
        # --- 1. 初始化缓冲区 ---
        initial_end = min(quiet_start_idx + self.buffer_size, len(df))
        buffer_df = df.iloc[pre_ctx:initial_end].copy().reset_index(drop=True)
        
        quiet_start_tidx = int(df.loc[quiet_mask, 'time_idx'].min())
        output_tidx = set(
            buffer_df.loc[buffer_df['time_idx'] >= quiet_start_tidx, 'time_idx']
            .astype(int).values)
        
        print(f"\n[ColdStart] 缓冲区: {len(buffer_df)} 窗口, 推理中...")
        df_cold = self._extract_tft_batch(buffer_df, engine=self.engine,
                                          output_tidx_set=output_tidx)
        
        all_tft_results = [df_cold] if len(df_cold) > 0 else []
        processed_tidx = set(df_cold['time_idx'].astype(int).values) if len(df_cold) > 0 else set()
        
        # --- 2. 滚动主循环 ---
        remaining_start = initial_end
        total_remaining = len(df) - remaining_start
        n_batches = (total_remaining + self.step_size - 1) // self.step_size
        
        print(f"\n[Rolling] {total_remaining} 窗口, {n_batches} 批 (step={self.step_size})")
        
        for batch_i in tqdm(range(n_batches), desc="自适应滚动检测"):
            batch_start = remaining_start + batch_i * self.step_size
            batch_end = min(batch_start + self.step_size, len(df))
            
            if batch_start >= len(df):
                break
            
            new_rows = df.iloc[batch_start:batch_end]
            new_tidx = set(new_rows['time_idx'].astype(int).values) - processed_tidx
            
            if not new_tidx:
                continue
            
            # 更新缓冲区
            buffer_df = pd.concat([buffer_df, new_rows], ignore_index=True)
            if len(buffer_df) > self.buffer_size:
                buffer_df = buffer_df.iloc[-self.buffer_size:].reset_index(drop=True)
            
            if len(buffer_df) < self.min_required:
                continue
            # P7修复：AP warmup阈值放宽
            if self._ap_warmup_remaining > 0:
                self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
                self._ap_warmup_remaining -= len(new_tidx)
                if self._ap_warmup_remaining <= 0:
                    self.classifier2.set_ap_relax_factor(1.0)
                    print(f"\n  [AP Warmup] 结束")
            # P3修复：并行验证时需要两个引擎都推理
            if self.state == self.SWITCHING and self._new_engine is not None:
                df_feat_old = self._extract_tft_batch(
                    buffer_df, engine=self.engine, output_tidx_set=new_tidx)
                df_feat_new = self._extract_tft_batch(
                    buffer_df, engine=self._new_engine, output_tidx_set=new_tidx)
                
                # 主输出用旧引擎（保守）
                df_feat = df_feat_old
                
                # 收集预测结果用于对比
                if len(df_feat_old) > 0 and len(df_feat_new) > 0:
                    pred_old = self.classifier2.predict(df_feat_old)['pred_label'].values
                    pred_new = self._new_classifier2.predict(df_feat_new)['pred_label'].values
                    
                    self._parallel_preds_old.extend(pred_old.tolist())
                    self._parallel_preds_new.extend(pred_new.tolist())
                    self._parallel_count += len(pred_old)
            else:
                active_engine = self.engine
                df_feat = self._extract_tft_batch(
                    buffer_df, engine=active_engine, output_tidx_set=new_tidx)   
            # 选择当前使用的引擎
            # active_engine = self._new_engine if (
            #     self.state == self.SWITCHING and self._new_engine) else self.engine
            
            # 批量TFT推理
            # df_feat = self._extract_tft_batch(
            #     buffer_df, engine=active_engine, output_tidx_set=new_tidx)
            
            if len(df_feat) == 0:
                continue
            
            all_tft_results.append(df_feat)
            processed_tidx.update(df_feat['time_idx'].astype(int).values)
            
            # 分类器2判定
            df_classified = self.classifier2.predict(df_feat)
            
            # 状态机处理
            current_tidx = int(new_rows['time_idx'].iloc[-1])
            self._process_batch(
                df_classified, current_tidx, df, df_features_raw, df_official)
            
            # 日志
            if (batch_i + 1) % 20 == 0:
                ts = new_rows['timestamp'].iloc[-1]
                n_ap = (df_classified['pred_label'] == 'AP').sum()
                n_cp = (df_classified['pred_label'] == 'CP').sum()
                print(f"  Batch {batch_i+1}/{n_batches} | {ts} | "
                      f"state={self.state} | v={self.baseline_version} | "
                      f"AP={n_ap} CP={n_cp} warmup={self._ap_warmup_remaining}")
        
        # --- 3. 合并所有输出 ---
        if all_tft_results:
            df_all_tft = pd.concat(all_tft_results, ignore_index=True)
            df_all_tft = df_all_tft.drop_duplicates(subset='time_idx', keep='last')
            df_all_tft = df_all_tft.sort_values('time_idx').reset_index(drop=True)
            df_final = self.classifier2.predict(df_all_tft)
            df_final['baseline_version'] = self.baseline_version
            df_final['state'] = self.state
        else:
            df_final = pd.DataFrame()
        
        # 对全量结果执行分类器2（利用完整rolling窗口）
        # print(f"\n[Final] 对 {len(df_all_tft)} 窗口执行最终分类...")
        # if len(df_all_tft) > 0:
        #     df_final = self.classifier2.predict(df_all_tft)
        #     # 附加元信息
        #     df_final['baseline_version'] = self.baseline_version
        #     df_final['state'] = self.state
        # else:
        #     df_final = pd.DataFrame()
        
        # 保存
        if output_path and len(df_final) > 0:
            os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
            df_final.to_csv(output_path, index=False)
            print(f"✅ 保存到 {output_path}")
        
        self._print_summary(df_final)
        self.save_state()

        return df_final
    
    # ================================================================
    # 状态机处理逻辑
    # ================================================================
    def _process_batch(self, df_classified, current_tidx,
                       df_full, df_features_raw, df_official):
        """每批次的状态机更新"""
        
        labels = df_classified['pred_label'].values
        
        # --- NORMAL: 检测CP连续性 ---
        if self.state == self.NORMAL:
            # 冷却期检查
            if current_tidx < self.cooldown_until_tidx:
                return
            
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
                if self.cp_consec >= 1:
                    self.state = self.CP_TENTATIVE
                    print(f"\n  [→ CP_TENTATIVE] consec={self.cp_consec} @ tidx={current_tidx}")
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
        
        # --- CP_TENTATIVE: 等待确认 ---
        elif self.state == self.CP_TENTATIVE:
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
            
            # 确认
            if self.cp_consec >= self.cp_confirm_windows:
                self.cp_confirmed_tidx = current_tidx
                self.retrace_start_tidx = current_tidx - self.retrace_windows
                self.accumulate_buffer_tidx = []
                self.state = self.ACCUMULATING
                
                print(f"\n  [→ ACCUMULATING] confirmed @ tidx={current_tidx}, "
                      f"retrace={self.retrace_windows} to tidx={self.retrace_start_tidx}")
                
                self.cp_events.append({
                    'confirmed_tidx': current_tidx,
                    'retrace_tidx': self.retrace_start_tidx,
                    'retrace_windows': self.retrace_windows,
                    'switched_tidx': None,
                    'switch_type': None,
                    'baseline_version': self.baseline_version,
                })
            
            # 回退到NORMAL
            if self.cp_consec <= 0:
                self.state = self.NORMAL
                self.cp_consec = 0
                print(f"\n  [→ NORMAL] CP疑似解除 @ tidx={current_tidx}")
        
        # --- ACCUMULATING: 积累数据并触发微调 ---
        elif self.state == self.ACCUMULATING:
            # 记录当前批次的tidx范围
            batch_tidx = df_classified['time_idx'].astype(int).tolist()
            self.accumulate_buffer_tidx.extend(batch_tidx)
            
            # 检查积累量（从retrace_start开始算起到当前）
            # accumulated = current_tidx - self.retrace_start_tidx
            accumulated = len(set(self.accumulate_buffer_tidx))
            
            print(f"  [ACCUMULATING] 已积累 {accumulated}/{self.accumulate_min} 窗口", end='\r')
            
            if accumulated >= self.accumulate_min:
                print(f"\n  [微调触发] 积累 {accumulated} 窗口, 开始微调...")
                
                # 提取微调数据范围
                retrace_mask = (
                    (df_full['time_idx'] >= self.retrace_start_tidx) &
                    (df_full['time_idx'] <= current_tidx)
                )
                df_retrace = df_full[retrace_mask].copy()
                
                # 对应的原始特征范围（用于refit processor）
                ts_start = df_retrace['timestamp'].min()
                ts_end = df_retrace['timestamp'].max()
                raw_mask = (
                    (df_features_raw['timestamp'] >= ts_start) &
                    (df_features_raw['timestamp'] <= ts_end)
                )
                df_raw_retrace = df_features_raw[raw_mask].copy()
                
                # # 执行微调
                # self._finetune(df_retrace, df_raw_retrace, df_official)
                
                # # 进入并行切换
                # self.state = self.SWITCHING
                # self._parallel_preds_old = []
                # self._parallel_preds_new = []
                # print(f"  [→ SWITCHING] 开始并行验证 ({self.switch_parallel_windows} 窗口)")
                # 执行微调
                success = self._finetune(df_retrace, df_raw_retrace, df_official)
                
                if success:
                    # 进入并行切换
                    self.state = self.SWITCHING
                    self._parallel_preds_old = []
                    self._parallel_preds_new = []
                    self._parallel_count = 0
                    self._switch_extended = False
                    print(f"  [→ SWITCHING] 开始并行验证 ({self.switch_parallel_windows} 窗口)")
                else:
                    print(f"  [微调失败] 回退到NORMAL")
                    self.state = self.NORMAL
                    self._reset_partial_state()
        
        # --- P3修复：SWITCHING 并行验证 ---
        elif self.state == self.SWITCHING:
            self._check_parallel_switch(current_tidx)

        # --- SWITCHING: 并行验证 ---
        # elif self.state == self.SWITCHING:
            # 两个引擎的预测已在主循环中分别执行
            # 这里比较最近一批的pred_label一致性
            # self._parallel_preds_old.extend(labels.tolist())
            
            # 简化处理：用旧引擎的预测作为对照
            # （新引擎的推理在主循环中通过active_engine切换实现）
            # n_compared = len(self._parallel_preds_old)
            
            # if n_compared >= self.switch_parallel_windows:
            #     self._execute_switch(current_tidx)
    # ================================================================
    # P3修复：完整的并行切换逻辑
    # ================================================================
    def _check_parallel_switch(self, current_tidx):
        """基于模型稳定性的并行验证与切换判定"""
        
        required_windows = (self.switch_parallel_extend 
                            if self._switch_extended 
                            else self.switch_parallel_windows)
        
        if self._parallel_count < required_windows:
            return
        
        pred_old = np.array(self._parallel_preds_old[-required_windows:])
        pred_new = np.array(self._parallel_preds_new[-required_windows:])
        
        if len(pred_old) == 0 or len(pred_new) == 0:
            return
        
        # 计算正常标签(N)的占比
        old_n_ratio = (pred_old == 'N').mean()
        new_n_ratio = (pred_new == 'N').mean()
        
        print(f"\n  [并行验证] windows={self._parallel_count}, "
              f"旧模型N占比={old_n_ratio:.3f}, 新模型N占比={new_n_ratio:.3f}")
        
        # 判定逻辑：增加相对优劣对比
        if old_n_ratio >= self.switch_recovery_threshold:
            print("  [放弃切换] 旧模型已恢复正常，微调冗余")
            self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
            
        # 核心修改：新模型稳定性必须达标，且表现不能劣于旧模型
        elif new_n_ratio >= self.switch_stability_threshold and new_n_ratio >= old_n_ratio:
            print("  [触发切换] 新模型达到稳态且优于旧模型")
            self._execute_full_switch(current_tidx, agreement=0.0, kappa=0.0)
            
        else:
            if not self._switch_extended:
                self._switch_extended = True
                print(f"  [延长验证] 新模型N占比={new_n_ratio:.3f}，未满足切换优势条件，"
                      f"延长到 {self.switch_parallel_extend} 窗口")
            else:
                print(f"  [拒绝切换] 延长验证后新模型表现仍未占优，微调过拟合或未能适应分布")
                self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
    
    def _execute_full_switch(self, current_tidx, agreement, kappa):
        """完整切换：模型权重+baseline+阈值"""
        print(f"\n  [完整切换] agreement={agreement:.3f}, kappa={kappa:.3f}")
        
        self.engine = self._new_engine
        self.processor = self._new_processor
        self.classifier2 = self._new_classifier2
        
        self._finalize_switch(current_tidx, 'full', agreement, kappa)
    
    def _execute_partial_switch(self, current_tidx, agreement, kappa):
        """部分切换：仅baseline+阈值，保留旧模型权重"""
        print(f"\n  [部分切换] agreement={agreement:.3f}，保留旧模型权重")
        
        # 只更新processor和分类器阈值
        self.processor = self._new_processor
        
        # 更新分类器阈值
        self.classifier2.ap_thresholds = copy.deepcopy(self._new_classifier2.ap_thresholds)
        self.classifier2.cp_base_thresh = self._new_classifier2.cp_base_thresh
        self.classifier2.cp_vol_thresh = self._new_classifier2.cp_vol_thresh
        self.classifier2._z_params = copy.deepcopy(self._new_classifier2._z_params)
        
        # 更新baseline但保留旧模型
        for mn in self._new_engine.baselines:
            self.engine.baselines[mn] = self._new_engine.baselines[mn]
        
        self._finalize_switch(current_tidx, 'partial', agreement, kappa)
    
    def _finalize_switch(self, current_tidx, switch_type, agreement, kappa):
        """切换完成的收尾工作"""
        
        self.baseline_version += 1
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        
        # P7修复：启动AP warmup
        self._ap_warmup_remaining = self.ap_warmup_windows
        self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
        
        # 记录历史
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
            self.cp_events[-1]['switch_type'] = switch_type
        
        self.switch_history.append({
            'tidx': current_tidx,
            'type': switch_type,
            'agreement': agreement,
            'kappa': kappa,
            'new_version': self.baseline_version,
        })
        
        # 重置状态
        self.state = self.NORMAL
        self._reset_partial_state()
        
        print(f"  [切换完成] type={switch_type}, version={self.baseline_version}, "
              f"冷却至 tidx={self.cooldown_until_tidx}, "
              f"AP warmup={self._ap_warmup_remaining}窗口")
    
    def _reset_partial_state(self):
        """重置切换相关状态"""
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False

    # ================================================================
    # 微调
    # ================================================================
    def _finetune(self, df_retrace, df_raw_retrace, df_official):
        """
        微调三件套: processor refit → TFT微调 → 分类器2阈值更新
        """
        from TFT_tft_engine import TFTEngine
        from tft_full_period_utils import compute_baseline_from_held_out, create_held_out_split
        
        # (a) Processor refit
        print("  [微调 1/3] Processor refit...")
        processor_new = copy.deepcopy(self.processor)
        processor_new.feature_transformer.fit(df_raw_retrace.fillna(0))
        
        # 用新processor变换数据
        dataset_new = processor_new.transform(df_raw_retrace.fillna(0), df_official)
        df_tft = dataset_new.data.copy()

        df_tft.fillna(0, inplace=True)
        df_tft.replace([np.inf, -np.inf], 0, inplace=True)
        df_tft = self.tb.transform(df_tft)
        if 'label' not in df_tft.columns:
            df_tft['label'] = 'N'
        
        print(f"  [微调数据] {len(df_tft)} 窗口")

        
        # (b) TFT微调
        print("  [微调 2/3] TFT模型微调...")
        with tempfile.TemporaryDirectory() as tmp:
            self.engine.save(tmp)
            engine_new = TFTEngine.load(tmp, config_path='TFT_config.yaml')
        
        self._patch_engine_batch_size(engine_new)
        self._patch_engine_early_stopping(engine_new)
        
        # Held-out切分
        held_out_windows = max(
            self.held_out_min_windows,
            int(len(df_tft) * 0.15)  # 最多15%
        )
        held_out_windows = min(held_out_windows, len(df_tft) // 3)

        split_idx = len(df_tft) - held_out_windows
        df_train_ft = df_tft.iloc[:split_idx].copy()
        df_held = df_tft.iloc[split_idx:].copy()
        held_out_tidx = set(df_held['time_idx'].values)
        # held_out_mask, held_mask, split_info = create_held_out_split(
        #     df_tft, time_col='timestamp',
        #     block_days=self.held_out_days,
        #     min_block_windows=96)
        # df_train_ft = df_tft[~held_mask].copy()
        # held_out_tidx = set(df_tft.loc[held_mask, 'time_idx'].values)
        print(f"  [数据切分] 训练={len(df_train_ft)}, held_out={len(df_held)}")
        
        if len(df_train_ft) < self.min_required * 2:
            print(f"  ⚠️ 训练数据不足")
            return False
        
        for mn in engine_new.models:
            # 解冻
            for p in engine_new.models[mn].parameters():
                p.requires_grad = True
            
            cfg = engine_new.config['tft_models'][mn]
            orig_lr = cfg.get('learning_rate', 0.03)
            cfg['learning_rate'] = orig_lr * self.finetune_lr_scale
            # 应用梯度裁剪
            if 'gradient_clip_val' not in cfg:
                cfg['gradient_clip_val'] = self.finetune_grad_clip

            print(f"    [{mn}] 微调 (lr={cfg['learning_rate']:.5f}, "
                  f"epochs={self.finetune_epochs}, data={len(df_train_ft)})...")
            
            with contextlib.redirect_stdout(io.StringIO()):
                engine_new.build_and_fit(
                    mn, df_train_ft, 
                    max_epochs=self.finetune_epochs)
            
            cfg['learning_rate'] = orig_lr
            
            # 重建baseline
            res = engine_new.analyze_rolling(mn, df_tft, baseline_end_idx=None)
            if 'metrics' not in res:
                print(f"    ⚠️ [{mn}] 推理失败")
                continue
            if held_out_tidx:
                bl = compute_baseline_from_held_out(res, held_out_tidx)
            else:
                bl = engine_new._build_baseline(
                    res['metrics'], res['attention'], res['vsn'],
                    int(df_tft['time_idx'].max()))
            engine_new.baselines[mn] = bl
            # engine_new.freeze_layers(mn)
            
            print(f"    [{mn}] 新baseline: σ={bl['residual_std']:.4f}")
        
        # (c) 分类器2阈值更新
        print("  [微调 3/3] 分类器2阈值更新...")
        # 用新引擎推理held-out数据，获取model_*列
        classifier2_new = copy.deepcopy(self.classifier2)

        # df_held = df_tft[held_mask].copy()
        if len(df_held) > self.min_required:
            df_held_tft = self._extract_tft_batch(df_held, engine=engine_new)
            if len(df_held_tft) > 10:
                # classifier2_new = copy.deepcopy(self.classifier2)
                # classifier2_new.calibrate(df_held_tft)
                classifier2_new.calibrate(df_held_tft)
                print(f"    新阈值校准完成")
            else:
                classifier2_new = copy.deepcopy(self.classifier2)
        else:
            classifier2_new = copy.deepcopy(self.classifier2)
        
        # 暂存新组件
        self._new_engine = engine_new
        self._new_processor = processor_new
        self._new_classifier2 = classifier2_new
        
        print("  [微调完成] 新组件已就绪，等待并行验证")
        return True

    # ================================================================
    # TFT批量推理（可配置batch_size）
    # ================================================================
    def _patch_engine_batch_size(self, engine):
        """为引擎应用推理batch_size补丁"""
        
        inference_bs = self.inference_batch_size
        
        def patched_analyze(model_name, df, baseline_end_idx=None, predict=True):
            model = engine.models[model_name]
            train_dataset = engine.datasets[model_name]
            df_inf = df.copy()
            for col in engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)
            
            try:
                inference_dataset = TimeSeriesDataSet.from_dataset(
                    train_dataset, df_inf, predict=False, stop_randomization=True)
            except ValueError as e:
                if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
                    return {}
                raise
            
            if len(inference_dataset) == 0:
                return {}
            
            dataloader = inference_dataset.to_dataloader(
                train=False, 
                batch_size=inference_bs,
                num_workers=0, 
                pin_memory=True)
            
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model.to(device)
            model.eval()
            
            result_buffer = {}
            with torch.no_grad():
                for x, y in dataloader:
                    x = {k: v.to(device) for k, v in x.items()}
                    targets = y[0].cpu().numpy().flatten()
                    raw_out = model(x)
                    preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
                    p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
                    time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
                    interp = model.interpret_output(raw_out, reduction="none")
                    batch_att = interp['attention'].cpu().numpy()
                    batch_vsn = interp['encoder_variables'].cpu().numpy()
                    
                    for i in range(len(time_idx)):
                        t_id = int(time_idx[i])
                        result_buffer[t_id] = {
                            "target_true": targets[i],
                            "pred_p50": p50[i],
                            "residual": targets[i] - p50[i],
                            "divergence": p90[i] - p10[i],
                            "attention": batch_att[i],
                            "vsn": batch_vsn[i]
                        }
            
            if not result_buffer:
                return {}
            
            sorted_t = sorted(result_buffer.keys())
            df_s = pd.DataFrame({
                "time_idx": sorted_t,
                "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
                "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
                "residual": [result_buffer[t]["residual"] for t in sorted_t],
                "divergence": [result_buffer[t]["divergence"] for t in sorted_t]
            })
            full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
            full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
            
            result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
            
            if baseline_end_idx is not None:
                bl = engine._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
                result["baseline"] = bl
                engine.baselines[model_name] = bl
            
            return result
        
        engine.analyze_rolling = patched_analyze
        
    def _patch_engine_early_stopping(self, engine):
        """为引擎应用具有高容忍度的早停机制及最佳权重回滚补丁"""
        import types
        import torch
        import pandas as pd
        import lightning.pytorch as pl
        from lightning.pytorch.loggers import CSVLogger
        from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
        from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
        from pytorch_forecasting.metrics import QuantileLoss
        from pytorch_forecasting.data import GroupNormalizer
        from pytorch_forecasting.data.encoders import NaNLabelEncoder

        ft_batch_size = self.finetune_batch_size

        def patched_build_and_fit(self_engine, model_name: str, df: pd.DataFrame, 
                                  val_ratio: float = 0.2, max_epochs: int = -1,
                                  quiet_end_idx: int = None):
            
            cfg = self_engine.config['tft_models'][model_name]
            pl.seed_everything(cfg.get('seed', 42))

            df_inf = df.copy()
            for col in self_engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)

            if quiet_end_idx is not None:
                time_col = self_engine.global_cfg['time_column']
                df_for_training = df_inf[df_inf[time_col] <= quiet_end_idx].copy()
            else:
                df_for_training = df_inf

            time_col = self_engine.global_cfg['time_column']
            time_steps = df_for_training[time_col].sort_values().unique()
            split_idx = int(len(time_steps) * (1 - val_ratio))
            cutoff_time = time_steps[split_idx]
            
            train_df = df_for_training[df_for_training[time_col] <= cutoff_time]
            max_lookback = cfg.get('max_encoder_length', 48)
            val_start_time = time_steps[max(0, split_idx - max_lookback)]
            val_df = df_for_training[df_for_training[time_col] >= val_start_time]

            categorical_encoders = {
                name: NaNLabelEncoder(add_nan=True) 
                for name in self_engine.known_categoricals
            }
            for g_id in self_engine.global_cfg['group_ids']:
                categorical_encoders[g_id] = NaNLabelEncoder(add_nan=True)

            training = TimeSeriesDataSet(
                train_df,
                time_idx=self_engine.global_cfg['time_column'],
                target=cfg['target'],
                group_ids=self_engine.global_cfg['group_ids'],
                min_encoder_length=cfg.get('min_encoder_length'),
                max_encoder_length=cfg.get('max_encoder_length'),
                max_prediction_length=cfg.get('max_prediction_length', 1),
                time_varying_unknown_reals=self_engine.unknown_reals,
                time_varying_known_reals=self_engine.known_reals,
                time_varying_known_categoricals=self_engine.known_categoricals,
                categorical_encoders=categorical_encoders,
                target_normalizer=GroupNormalizer(groups=self_engine.global_cfg['group_ids'], transformation=None),
                add_relative_time_idx=True,
                add_target_scales=True,
                add_encoder_length=True,
                allow_missing_timesteps=True
            )

            validation = TimeSeriesDataSet.from_dataset(training, val_df, predict=False, stop_randomization=True)

            train_dataloader = training.to_dataloader(train=True, batch_size=ft_batch_size, num_workers=0, pin_memory=True)
            val_dataloader = validation.to_dataloader(train=False, batch_size=ft_batch_size, num_workers=0, pin_memory=True)

            model = TemporalFusionTransformer.from_dataset(
                training,
                learning_rate=cfg.get('learning_rate', 0.03),
                hidden_size=cfg.get('hidden_size', 16),
                attention_head_size=4,
                dropout=0.1,
                hidden_continuous_size=8,
                output_size=len(self_engine.quantiles),
                loss=QuantileLoss(quantiles=self_engine.quantiles),
                reduce_on_plateau_patience=4
            )

            logger = CSVLogger("lightning_logs", name=model_name, flush_logs_every_n_steps=10)
            self_engine.log_dirs[model_name] = logger.log_dir

            # 核心调整：配置宽容型早停与检查点
            early_stop_callback = EarlyStopping(
                monitor="val_loss",
                patience=8,       # 容忍20个Epoch没有改善才停止，跨越局部波动
                min_delta=1e-3,    # 降低改善判定阈值
                mode="min",
                verbose=False
            )
            checkpoint_callback = ModelCheckpoint(
                monitor="val_loss",
                mode="min",
                save_top_k=1,      # 仅保留验证损失最低的那个Epoch权重
                dirpath=logger.log_dir,
                filename="best_model"
            )

            trainer_kwargs = {
                "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
                "devices": 1,
                "enable_model_summary": False,
                "enable_checkpointing": True,
                "callbacks": [early_stop_callback, checkpoint_callback],
                "logger": logger,
                "enable_progress_bar": True
            }

            if max_epochs == -1:
                trainer = pl.Trainer(**trainer_kwargs)
            elif max_epochs != 1:
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)
            elif max_epochs == 1:
                trainer_kwargs.update({"limit_train_batches": 5, "limit_val_batches": 5})
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)

            trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

            # 核心调整：训练结束后，强制从检查点加载验证集表现最佳的权重
            # if checkpoint_callback.best_model_path:
            #     best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)
            #     model.load_state_dict(best_model.state_dict())
            if checkpoint_callback.best_model_path:
                best_model = TemporalFusionTransformer.load_from_checkpoint(
                    checkpoint_callback.best_model_path, 
                    weights_only=False
                )
                model.load_state_dict(best_model.state_dict())
                print(f"    [{model_name}] 回滚至最佳验证权重: {checkpoint_callback.best_model_score:.4f}")

            self_engine.plot_training_history(model_name)
            
            self_engine.datasets[model_name] = training
            self_engine.models[model_name] = model

        engine.build_and_fit = types.MethodType(patched_build_and_fit, engine)
    # ================================================================
    # 切换
    # ================================================================
    def _execute_switch(self, current_tidx):
        """执行分层切换判定"""
        
        # 由于并行推理在简化实现中无法精确对比，这里直接执行切换
        # 完整实现中应同时用新旧引擎推理同一批数据并比较一致率
        
        # 立即切换: processor + baseline + 分类器阈值
        print(f"\n  [SWITCH] 执行切换 @ tidx={current_tidx}")
        
        self.processor = self._new_processor
        self.engine = self._new_engine
        self.classifier2 = self._new_classifier2
        
        self.baseline_version += 1
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        self._switch_warmup_count = self.ap_warmup_windows
        
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
        
        # 重置状态
        self.state = self.NORMAL
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        
        print(f"  [SWITCH] 完成! baseline_version={self.baseline_version}, "
              f"冷却至 tidx={self.cooldown_until_tidx}")
    
    # ================================================================
    # TFT批量推理（复用滚动训练Cell 2的逻辑）
    # ================================================================
    def _extract_tft_batch(self, df_buffer, engine=None, output_tidx_set=None):
        """
        对缓冲区执行TFT推理，提取model_*信号 + rolling。
        直接复用notebook Cell 2 的 extract_tft_features_batch 逻辑。
        """
        if engine is None:
            engine = self.engine
        
        # --- TFT推理 ---
        tft_results = {}
        for mn in engine.models:
            with contextlib.redirect_stdout(io.StringIO()), \
                 contextlib.redirect_stderr(io.StringIO()):
                res = engine.analyze_rolling(mn, df_buffer, baseline_end_idx=None)
            tft_results[mn] = res
        
        if not any('metrics' in v for v in tft_results.values()):
            return pd.DataFrame()
        
        # --- 合并TFT信号 ---
        df_merged = df_buffer[['time_idx', 'timestamp']].copy()
        
        for model_name in tft_results:
            res = tft_results[model_name]
            if 'metrics' not in res:
                continue
            
            metrics = res['metrics']
            attention = res['attention']
            vsn = res['vsn']
            baseline = engine.baselines.get(model_name, {})
            
            if not baseline:
                continue
            
            n = len(metrics)
            
            # 残差比
            residual_abs = metrics['residual'].abs().values
            baseline_p95 = max(
                abs(baseline.get('residual_p95', 0)),
                abs(baseline.get('residual_p5', 0)),
                baseline.get('residual_std', 1e-8) * 1.645, 1e-8)
            sig_residual = np.clip(residual_abs / baseline_p95, 0, 20)
            
            # 注意力KL散度
            att_base = baseline.get('att_mean')
            sig_att_kl = np.zeros(n)
            if att_base is not None:
                for i in range(n):
                    p = attention[i].flatten()
                    q = att_base.flatten()
                    p = p / (p.sum() + 1e-10) + 1e-10
                    q = q / (q.sum() + 1e-10) + 1e-10
                    p, q = p / p.sum(), q / q.sum()
                    sig_att_kl[i] = np.sum(p * np.log(p / q))
            
            # VSN JS散度
            vsn_base = baseline.get('vsn_mean')
            sig_vsn_js = np.zeros(n)
            if vsn_base is not None:
                for i in range(n):
                    v_i = np.abs(vsn[i].flatten()) + 1e-10
                    v_b = np.abs(vsn_base.flatten()) + 1e-10
                    p_v, q_v = v_i / v_i.sum(), v_b / v_b.sum()
                    m_v = 0.5 * (p_v + q_v)
                    sig_vsn_js[i] = (
                        0.5 * np.sum(p_v * np.log(p_v / m_v)) +
                        0.5 * np.sum(q_v * np.log(q_v / m_v)))
            
            # VSN排序变化
            sig_vsn_rank = np.zeros(n)
            if vsn_base is not None:
                baseline_rank = np.argsort(np.argsort(-np.abs(vsn_base.flatten())))
                for i in range(n):
                    curr_rank = np.argsort(np.argsort(-np.abs(vsn[i].flatten())))
                    corr, _ = spearmanr(baseline_rank, curr_rank)
                    sig_vsn_rank[i] = 1 - corr if not np.isnan(corr) else 1.0
            
            # 组装
            prefix = f'model_{model_name}'
            tft_df = pd.DataFrame({
                'time_idx': metrics['time_idx'].values,
                f'{prefix}_residual_ratio': sig_residual,
                f'{prefix}_att_kl': np.clip(sig_att_kl, 0, 20),
                f'{prefix}_vsn_js': np.clip(sig_vsn_js, 0, 5),
                f'{prefix}_vsn_rank_shift': np.clip(sig_vsn_rank, 0, 2),
            }).drop_duplicates(subset='time_idx', keep='last')
            
            # merge (防膨胀)
            n_before = len(df_merged)
            df_merged = df_merged.merge(tft_df, on='time_idx', how='left')
            if len(df_merged) != n_before:
                # 回退, 使用map方式
                new_cols = [c for c in tft_df.columns if c != 'time_idx']
                df_merged = df_merged.drop(columns=new_cols).iloc[:n_before]
                for col in new_cols:
                    val_map = dict(zip(tft_df['time_idx'], tft_df[col]))
                    df_merged[col] = df_merged['time_idx'].map(val_map)
        
        # --- Rolling特征 ---
        base_tft_cols = [c for c in df_merged.columns
                         if c.startswith('model_') and 'roll' not in c]
        for col in base_tft_cols:
            for k in [4, 8]:
                df_merged[f'{col}_rollmax_{k}'] = (
                    df_merged[col].rolling(k, min_periods=1).max())
                df_merged[f'{col}_rollmean_{k}'] = (
                    df_merged[col].rolling(k, min_periods=1).mean())
        
        df_merged = df_merged.replace([np.inf, -np.inf], 0).fillna(0)
        
        # 只保留model_*列 + time_idx + timestamp
        keep_cols = ['time_idx', 'timestamp'] + [
            c for c in df_merged.columns if c.startswith('model_')]
        df_merged = df_merged[[c for c in keep_cols if c in df_merged.columns]]
        
        # 筛选输出行
        if output_tidx_set is not None:
            df_merged = df_merged[df_merged['time_idx'].isin(output_tidx_set)]
        
        return df_merged
    
    # ================================================================
    # 工具
    # ================================================================
    def _print_summary(self, df_final):
        """打印运行总结"""
        print(f"\n{'='*60}")
        print(f"  自适应滚动检测 — 运行总结")
        print(f"{'='*60}")
        if len(df_final) > 0:
            vc = df_final['pred_label'].value_counts()
            print(f"  总窗口: {len(df_final)}")
            print(f"  N={vc.get('N',0)}, AP={vc.get('AP',0)}, CP={vc.get('CP',0)}")
        print(f"  最终baseline版本: {self.baseline_version}")
        print(f"  CP事件: {len(self.cp_events)}")
        for i, evt in enumerate(self.cp_events):
            print(f"    #{i}: confirmed@{evt['confirmed_tidx']}, "
                  f"retrace@{evt['retrace_tidx']}, "
                  f"switched@{evt.get('switched_tidx', '未切换')}")
        print(f"{'='*60}")
    
    def save_state(self, path="./checkpoints/adaptive_state"):
        """持久化当前状态"""
        os.makedirs(path, exist_ok=True)
        state = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_events': self.cp_events,
            'cooldown_until_tidx': self.cooldown_until_tidx,
        }
        with open(os.path.join(path, 'state.json'), 'w') as f:
            import json
            json.dump(state, f, indent=2, default=str)
        
        self.engine.save(os.path.join(path, f'engine_v{self.baseline_version}'))
        with open(os.path.join(path, f'processor_v{self.baseline_version}.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(path, f'classifier2_v{self.baseline_version}.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        print(f"✅ 状态保存到 {path}")

### 运行

In [ ]:
# ============================================================
# Cell: 自适应滚动检测 — 完整运行
# ============================================================
import types, io, contextlib, copy, os, pickle
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pytorch_forecasting import TimeSeriesDataSet
from TFT_tft_engine import TFTEngine
from target_builder import TargetBuilder
# from adaptive_rolling_detector import StatisticalClassifier2, AdaptiveRollingDetector

# =====================================================================
# 0. 加载预训练系统
# =====================================================================
CKPT = "./checkpoints/pretrained_planPC"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine_PC = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)

# 绑定去重补丁（与Cell 1相同）
def analyze_rolling_patched(self, model_name, df, baseline_end_idx=None, predict=True):
    model = self.models[model_name]
    train_dataset = self.datasets[model_name]
    df_inf = df.copy()
    for col in self.known_categoricals:
        df_inf[col] = df_inf[col].astype(str)
    try:
        inference_dataset = TimeSeriesDataSet.from_dataset(
            train_dataset, df_inf, predict=False, stop_randomization=True)
    except ValueError as e:
        if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
            return {}
        raise
    if len(inference_dataset) == 0:
        return {}
    dataloader = inference_dataset.to_dataloader(
        train=False, batch_size=512, num_workers=0, pin_memory=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()
    result_buffer = {}
    with torch.no_grad():
        for x, y in dataloader:
            x = {k: v.to(device) for k, v in x.items()}
            targets = y[0].cpu().numpy().flatten()
            raw_out = model(x)
            preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
            p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
            time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
            interp = model.interpret_output(raw_out, reduction="none")
            batch_att = interp['attention'].cpu().numpy()
            batch_vsn = interp['encoder_variables'].cpu().numpy()
            for i in range(len(time_idx)):
                t_id = int(time_idx[i])
                result_buffer[t_id] = {
                    "target_true": targets[i], "pred_p50": p50[i],
                    "residual": targets[i] - p50[i], "divergence": p90[i] - p10[i],
                    "attention": batch_att[i], "vsn": batch_vsn[i]}
    if not result_buffer:
        return {}
    sorted_t = sorted(result_buffer.keys())
    df_s = pd.DataFrame({
        "time_idx": sorted_t,
        "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
        "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
        "residual": [result_buffer[t]["residual"] for t in sorted_t],
        "divergence": [result_buffer[t]["divergence"] for t in sorted_t]})
    full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
    full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
    result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
    if baseline_end_idx is not None:
        bl = self._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
        result["baseline"] = bl
        self.baselines[model_name] = bl
    return result

engine_PC.analyze_rolling = types.MethodType(analyze_rolling_patched, engine_PC)

# =====================================================================
# 1. 数据预处理（与滚动训练Cell 1一致）
# =====================================================================
QUIET_START = pd.to_datetime('2025-02-18 00:00:00')
QUIET_END   = pd.to_datetime('2025-02-25 00:00:00')

# 用平静期refit processor
mask_quiet = (df_features['timestamp'] >= QUIET_START) & (df_features['timestamp'] < QUIET_END)
df_quiet_raw = df_features[mask_quiet].copy().fillna(0)

processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_quiet_raw)
print(f"[Processor] re-fitted, global_min_time={processor_real.global_min_time}")

# 变换全量数据
dataset_real = processor_real.transform(df_features.fillna(0), df_official_real)
df_real_all = dataset_real.data.copy()
df_real_all.fillna(0, inplace=True)
df_real_all.replace([np.inf, -np.inf], 0, inplace=True)
df_real_all = tb.transform(df_real_all)
if 'label' not in df_real_all.columns:
    df_real_all['label'] = 'N'
df_real_all = df_real_all.sort_values('time_idx').reset_index(drop=True)
print(f"[Data] {df_real_all.shape}, {df_real_all['timestamp'].min()} ~ {df_real_all['timestamp'].max()}")

# =====================================================================
# 2. 构建分类器2 & 自适应检测器
# =====================================================================
classifier3 = StatisticalClassifier3(
    ap_k=5.0, cp_base_k=3.0, cp_density_thresh=0.95,
    cp_window=288, cp_min_periods=96, ap_min_triggers=4
)

detector2 = AdaptiveRollingDetector2(
    engine=engine_PC,
    tb=tb,
    processor=processor_real,
    classifier2=classifier3,
    # 滚动
    buffer_size=672, step_size=96,min_required=None,
    # CP确认
    # cp_confirm_windows=32,
    cp_confirm_windows=2,

    # 回溯与积累
    retrace_windows=288, accumulate_min=576,
    # 微调
    finetune_epochs=80, finetune_lr_scale=0.005, finetune_batch_size=16, held_out_days=1,
    # 切换
    switch_parallel_windows=96,
    # switch_agree_high=0.90, switch_agree_low=0.70,
    switch_recovery_threshold=0.85,
    switch_stability_threshold=0.85,
    # 冷却
    cooldown_windows=384,ap_warmup_windows=96, ap_warmup_relax=1.2,
    inference_batch_size=256
)

# =====================================================================
# 3. 运行
# =====================================================================
df_result2 = detector2.run(
    df_real_all=df_real_all,
    df_features_raw=df_features.fillna(0),
    df_official=df_official_real,
    quiet_start=QUIET_START,
    quiet_end=QUIET_END,
    output_path="./output/adaptive_detection_resultv2.csv"
)


In [ ]:
# 保存状态
detector2.save_state("./checkpoints/adaptive_state")

print(f"\n🎯 检测完成, 结果: {len(df_result2)} 窗口")

## 运行V3

In [ ]:
# ============================================================
# Cell: 自适应滚动检测 — 完整运行
# ============================================================
import types, io, contextlib, copy, os, pickle
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pytorch_forecasting import TimeSeriesDataSet
from TFT_tft_engine import TFTEngine
from target_builder import TargetBuilder
# from adaptive_rolling_detector import StatisticalClassifier2, AdaptiveRollingDetector
import logging

# 关闭 PyTorch Lightning 的硬件与环境提示
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

# 关闭 数据处理器与 TargetBuilder 的常规INFO日志
logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
logging.getLogger("TargetBuilder").setLevel(logging.WARNING)
# 如果有其他自定义 logger，也可以一并设置为 WARNING
# =====================================================================
# 0. 加载预训练系统
# =====================================================================
CKPT = "./checkpoints/pretrained_planPC"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine_PC = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)

df_features = df_features[df_features['timestamp']<pd.to_datetime('2025-11-13')].copy()
# 绑定去重补丁（与Cell 1相同）
def analyze_rolling_patched(self, model_name, df, baseline_end_idx=None, predict=True):
    model = self.models[model_name]
    train_dataset = self.datasets[model_name]
    df_inf = df.copy()
    for col in self.known_categoricals:
        df_inf[col] = df_inf[col].astype(str)
    try:
        inference_dataset = TimeSeriesDataSet.from_dataset(
            train_dataset, df_inf, predict=False, stop_randomization=True)
    except ValueError as e:
        if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
            return {}
        raise
    if len(inference_dataset) == 0:
        return {}
    dataloader = inference_dataset.to_dataloader(
        train=False, batch_size=512, num_workers=0, pin_memory=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()
    result_buffer = {}
    with torch.no_grad():
        for x, y in dataloader:
            x = {k: v.to(device) for k, v in x.items()}
            targets = y[0].cpu().numpy().flatten()
            raw_out = model(x)
            preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
            p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
            time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
            interp = model.interpret_output(raw_out, reduction="none")
            batch_att = interp['attention'].cpu().numpy()
            batch_vsn = interp['encoder_variables'].cpu().numpy()
            for i in range(len(time_idx)):
                t_id = int(time_idx[i])
                result_buffer[t_id] = {
                    "target_true": targets[i], "pred_p50": p50[i],
                    "residual": targets[i] - p50[i], "divergence": p90[i] - p10[i],
                    "attention": batch_att[i], "vsn": batch_vsn[i]}
    if not result_buffer:
        return {}
    sorted_t = sorted(result_buffer.keys())
    df_s = pd.DataFrame({
        "time_idx": sorted_t,
        "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
        "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
        "residual": [result_buffer[t]["residual"] for t in sorted_t],
        "divergence": [result_buffer[t]["divergence"] for t in sorted_t]})
    full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
    full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
    result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
    if baseline_end_idx is not None:
        bl = self._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
        result["baseline"] = bl
        self.baselines[model_name] = bl
    return result

engine_PC.analyze_rolling = types.MethodType(analyze_rolling_patched, engine_PC)

# =====================================================================
# 1. 数据预处理（与滚动训练Cell 1一致）
# =====================================================================
QUIET_START = pd.to_datetime('2025-02-18 00:00:00')
QUIET_END   = pd.to_datetime('2025-02-25 00:00:00')

# 用平静期refit processor
mask_quiet = (df_features['timestamp'] >= QUIET_START) & (df_features['timestamp'] < QUIET_END)
df_quiet_raw = df_features[mask_quiet].copy().fillna(0)

processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_quiet_raw)
print(f"[Processor] re-fitted, global_min_time={processor_real.global_min_time}")

# 变换全量数据
dataset_real = processor_real.transform(df_features.fillna(0), df_official_real)
df_real_all = dataset_real.data.copy()
df_real_all.fillna(0, inplace=True)
df_real_all.replace([np.inf, -np.inf], 0, inplace=True)
df_real_all = tb.transform(df_real_all)
if 'label' not in df_real_all.columns:
    df_real_all['label'] = 'N'
df_real_all = df_real_all.sort_values('time_idx').reset_index(drop=True)
print(f"[Data] {df_real_all.shape}, {df_real_all['timestamp'].min()} ~ {df_real_all['timestamp'].max()}")

# =====================================================================
# 2. 构建分类器2 & 自适应检测器
# =====================================================================
classifier3 = StatisticalClassifier3(
    ap_k=5.0, cp_base_k=3.0, cp_density_thresh=0.95,
    cp_window=288, cp_min_periods=96, ap_min_triggers=4
)

detector3 = AdaptiveRollingDetector3(
    engine=engine_PC,
    tb=tb,
    processor=processor_real,
    classifier2=classifier3,
    # 滚动
    buffer_size=672, step_size=96,min_required=None,
    # CP确认
    # cp_confirm_windows=32,
    cp_confirm_windows=96,

    # 回溯与积累
    retrace_windows=288, accumulate_min=1344,# 增长数据收集期
    # 微调
    finetune_epochs=100, finetune_lr_scale=0.005, finetune_batch_size=32, held_out_days=2,
    # 切换
    switch_parallel_windows=288,
    switch_parallel_extend=480,

    switch_recovery_threshold=0.85,
    switch_stability_threshold=0.85,
    # 冷却
    cooldown_windows=672,ap_warmup_windows=96, ap_warmup_relax=1.2,
    inference_batch_size=256
)

# =====================================================================
# 3. 运行
# =====================================================================
df_result3 = detector3.run(
    df_real_all=df_real_all,
    df_features_raw=df_features.fillna(0),
    df_official=df_official_real,
    quiet_start=QUIET_START,
    quiet_end=QUIET_END,
    output_path="./output/adaptive_detection_resultv3.csv"
)
# 保存状态
detector3.save_state("./checkpoints/adaptive_state")

print(f"\n🎯 检测完成, 结果: {len(df_result3)} 窗口")


# 检测V4

In [ ]:
"""
自适应滚动检测器：
  滚动TFT推理 → 分类器2判定 → CP确认 → 回溯288窗口 → 积累微调 → 分层切换
  
  状态机：NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
"""

import os, copy, pickle, tempfile, io, contextlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import spearmanr
# 如果有其他自定义 logger，也可以一并设置为 WARNING
# =====================================================================
# 分类器2: 纯统计阈值 AP/CP 判定
# =====================================================================
class StatisticalClassifier4:
    """
    基于冷启动期校准的多维Z-score分类器。
    输入: 滚动TFT推理输出的 model_* 信号列
    输出: 每个窗口的 pred_label (N/AP/CP) + 投票归因信息
    """
    

    import logging

    # 关闭 PyTorch Lightning 的硬件与环境提示
    logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

    # 关闭 数据处理器与 TargetBuilder 的常规INFO日志
    logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
    logging.getLogger("TargetBuilder").setLevel(logging.WARNING)
    # 8个AP候选特征
    
    AP_ROLL_FEATURES = [
        'model_model_comment_pc1_residual_ratio_rollmax_8',
        'model_model_post_pc2_residual_ratio_rollmax_8',
        'model_model_comment_pc1_vsn_rank_shift_rollmax_8',
        'model_model_post_pc2_vsn_rank_shift_rollmax_8',
        'model_model_comment_pc1_att_kl_rollmax_8',
        'model_model_post_pc2_att_kl_rollmax_8',
        'model_model_comment_pc1_vsn_js_rollmax_8',
        'model_model_post_pc2_vsn_js_rollmax_8',
    ]
    
    # 4个CP静态特征
    CP_STATIC_FEATURES = [
        'model_model_comment_pc1_vsn_rank_shift',
        'model_model_post_pc2_vsn_rank_shift',
        'model_model_comment_pc1_att_kl',
        'model_model_post_pc2_att_kl',
    ]
    
    def __init__(self, ap_k=5.0, cp_base_k=3.0, cp_density_thresh=0.95,
                 cp_window=288, cp_min_periods=96, ap_min_triggers=4,
                 # 新增：长程缓变漂移参数
                 cp_trend_window=2688, cp_trend_k=2.0):
        self.ap_k = ap_k
        self.cp_base_k = cp_base_k
        self.cp_density_thresh = cp_density_thresh
        self.cp_window = cp_window
        self.cp_min_periods = cp_min_periods
        self.ap_min_triggers = ap_min_triggers
        
        self.cp_trend_window = cp_trend_window
        self.cp_trend_k = cp_trend_k
        
        self.ap_thresholds = {}
        self.cp_base_thresh = None
        self.cp_vol_thresh = None
        self.cp_trend_thresh = None # 新增
        self._z_params = {} 
        self._calibrated = False
        self._ap_relax_factor = 1.0
    
    def calibrate(self, df_base):
        """用冷启动期数据校准所有阈值"""
        for feat in self.AP_ROLL_FEATURES:
            if feat in df_base.columns:
                self.ap_thresholds[feat] = df_base[feat].mean() + self.ap_k * df_base[feat].std()
        
        for feat in self.CP_STATIC_FEATURES:
            if feat in df_base.columns:
                self._z_params[feat] = {'mean': df_base[feat].mean(), 'std': df_base[feat].std() + 1e-8}
        
        z_max_vals = self._compute_z_max(df_base)
        self.cp_base_thresh = z_max_vals.mean() + self.cp_base_k * z_max_vals.std()
        self.cp_vol_thresh = z_max_vals.std() * 1.5
        
        # 新增：长程漂移阈值
        self.cp_trend_thresh = z_max_vals.mean() + self.cp_trend_k * z_max_vals.std()
        
        self._calibrated = True
        print(f"[Classifier3] 校准完成: AP阈值={len(self.ap_thresholds)}个, "
              f"突变水位={self.cp_base_thresh:.3f}, 漂移阈值={self.cp_trend_thresh:.3f}")
        
    def set_ap_relax_factor(self, factor):
        """设置AP阈值放宽因子"""
        self._ap_relax_factor = factor
    
    def _compute_z_max(self, df):
        """计算各行的多维Z-score最大值"""
        z_vals = []
        for feat, params in self._z_params.items():
            if feat in df.columns:
                z = (df[feat] - params['mean']) / params['std']
                z_vals.append(z)
        if z_vals:
            return pd.concat(z_vals, axis=1).max(axis=1)
        return pd.Series(0.0, index=df.index)
    
    def predict(self, df):
        assert self._calibrated, "必须先调用 calibrate()"
        df = df.copy()
        
        # --- CP辅助信号 ---
        z_max = self._compute_z_max(df)
        df['static_signal_z_max'] = z_max
        is_high = (z_max > self.cp_base_thresh).astype(float)
        df['cp_density'] = is_high.rolling(self.cp_window, min_periods=self.cp_min_periods).mean()
        df['cp_volatility'] = z_max.rolling(self.cp_window, min_periods=self.cp_min_periods).std()
        
        # 新增：长程缓变均值与漂移判定 (独立特征)
        df['cp_trend_mean'] = z_max.rolling(self.cp_trend_window, min_periods=self.cp_window).mean()
        df['is_baseline_drift'] = df['cp_trend_mean'] > self.cp_trend_thresh
        
        # --- AP触发计数 ---
        trigger_matrix = pd.DataFrame(index=df.index)
        for feat in self.AP_ROLL_FEATURES:
            if feat in df.columns and feat in self.ap_thresholds:
                effective_thresh = self.ap_thresholds[feat] * self._ap_relax_factor
                trigger_matrix[feat] = (df[feat] > effective_thresh).astype(int)
        df['ap_trigger_count'] = trigger_matrix.sum(axis=1)

        # --- 判定 (仅限 AP 和 Sudden CP) ---
        df['pred_label'] = 'N'
        
        mask_cp = (df['cp_density'] > self.cp_density_thresh) & (df['cp_volatility'] < self.cp_vol_thresh)
        df.loc[mask_cp, 'pred_label'] = 'CP'
        
        mask_ap = (df['ap_trigger_count'] >= self.ap_min_triggers) & (~mask_cp)
        df.loc[mask_ap, 'pred_label'] = 'AP'
        
        # --- 投票归因 ---
        df['dominant_model'] = 'N/A'
        df['dominant_signal'] = 'N/A'
        df['trigger_features'] = ''
        
        anom_mask = df['pred_label'] != 'N'
        if anom_mask.any():
            # for idx in df[anom_mask].index:
            #     triggers = []
            #     for feat in trigger_matrix.columns:
            #         if trigger_matrix.loc[idx, feat] == 1:
            #             triggers.append(feat)
                
            #     if not triggers:
            #         continue
            for idx in df[anom_mask].index:
                triggers = [f for f in trigger_matrix.columns 
                           if trigger_matrix.loc[idx, f] == 1]
                if not triggers:
                    continue


                # 子模型投票
                comment_v = sum(1 for f in triggers if 'comment_pc1' in f)
                post_v = sum(1 for f in triggers if 'post_pc2' in f)
                dom_model = 'comment_pc1' if comment_v >= post_v else 'post_pc2'
                
                # 信号类型投票
                signal_votes = {}
                for sig_type in ['residual_ratio', 'vsn_rank_shift', 'att_kl', 'vsn_js']:
                    signal_votes[sig_type] = sum(1 for f in triggers if sig_type in f)
                dom_signal = max(signal_votes, key=signal_votes.get)
                
                df.loc[idx, 'dominant_model'] = dom_model
                df.loc[idx, 'dominant_signal'] = dom_signal
                df.loc[idx, 'trigger_features'] = '|'.join(triggers)
        
        return df



## 检测器V4

In [ ]:

# =====================================================================
# 自适应滚动检测器
# =====================================================================
class AdaptiveRollingDetector4:
    """
    状态机:
      NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
    
    每个STEP_SIZE窗口执行一次批量TFT推理 + 分类器2判定，
    检测到CP后回溯288窗口，积累到576窗口后微调TFT，分层切换。
    """
    
    # 状态常量
    NORMAL = 'NORMAL'
    CP_TENTATIVE = 'CP_TENTATIVE'
    ACCUMULATING = 'ACCUMULATING'
    SWITCHING = 'SWITCHING'
    
    def __init__(self, engine, tb, processor, classifier2,
                 # 滚动参数
                 buffer_size=672, step_size=96, min_required=None,
                 # CP确认
                 cp_confirm_windows=32,
                 # 回溯与积累
                 retrace_windows=288,
                 accumulate_min=576,
                 # 微调
                 finetune_epochs=50, 
                 finetune_lr_scale=0.1,
                 finetune_batch_size = 32,
                 held_out_days=2,
                 finetune_grad_clip=0.5,
                 held_out_min_windows=96,
                 # 切换 # P3修复：切换（完整并行验证）
                #  switch_parallel_windows=96,
                #  switch_parallel_extend=192,
                #  switch_agree_high=0.90, 
                #  switch_agree_low=0.70,
                # 切换参数（修改此部分）
                 switch_parallel_windows=96,
                 switch_parallel_extend=192,
                 switch_recovery_threshold=0.95,  # 新增：旧模型恢复正常的N标签比例阈值
                 switch_stability_threshold=0.85,
                 # 冷却
                 cooldown_windows=384,
                 # AP阈值保护
                 ap_warmup_windows=48, 
                 ap_warmup_relax=1.2,
                 inference_batch_size=256):
        
        self.engine = engine
        self.tb = tb
        self.processor = processor
        self.classifier2 = classifier2
        
        # 滚动参数
        self.buffer_size = buffer_size
        self.step_size = step_size
        ds_0 = list(engine.datasets.values())[0]
        self.min_encoder = ds_0.max_encoder_length
        self.min_pred = ds_0.max_prediction_length
        self.min_required = min_required or (self.min_encoder + self.min_pred + 1)
        
        # CP确认
        self.cp_confirm_windows = cp_confirm_windows
        
        # 回溯与积累
        self.retrace_windows = retrace_windows
        self.accumulate_min = accumulate_min
        
        # 微调
        self.finetune_epochs = finetune_epochs
        self.finetune_lr_scale = finetune_lr_scale
        self.finetune_batch_size = finetune_batch_size
        self.finetune_grad_clip = finetune_grad_clip
        self.held_out_days = held_out_days
        self.held_out_min_windows = held_out_min_windows
        
        # P3修复：切换
        self.switch_parallel_windows = switch_parallel_windows
        self.switch_parallel_extend = switch_parallel_extend
        # self.switch_agree_high = switch_agree_high
        # self.switch_agree_low = switch_agree_low
        self.switch_recovery_threshold =  switch_recovery_threshold  # 新增：旧模型恢复正常的N标签比例阈值
        self.switch_stability_threshold = switch_stability_threshold
        
        # P7修复：冷却与保护
        self.cooldown_windows = cooldown_windows
        self.ap_warmup_windows = ap_warmup_windows
        self.ap_warmup_relax = ap_warmup_relax
        
        # 推理配置
        self.inference_batch_size = inference_batch_size
        
        # --- 状态变量 ---
        self.state = self.NORMAL
        self.baseline_version = 0
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self.cooldown_until_tidx = -1
        
        # P3修复：并行切换状态
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False
        
        # P7修复：AP warmup状态
        self._ap_warmup_remaining = 0
        
        # 历史记录
        self.cp_events = []
        self.switch_history = []
        
    # ================================================================
    # P8修复：状态持久化与恢复
    # ================================================================
    def save_state(self, path="./checkpoints/adaptive_state"):
        """持久化当前完整状态"""
        os.makedirs(path, exist_ok=True)
        
        state_dict = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_confirmed_tidx': self.cp_confirmed_tidx,
            'retrace_start_tidx': self.retrace_start_tidx,
            'accumulate_buffer_tidx': self.accumulate_buffer_tidx,
            'cooldown_until_tidx': self.cooldown_until_tidx,
            'ap_warmup_remaining': self._ap_warmup_remaining,
            'parallel_count': self._parallel_count,
            'switch_extended': self._switch_extended,
            'cp_events': self.cp_events,
            'switch_history': self.switch_history,
        }
        
        with open(os.path.join(path, 'state.json'), 'w') as f:
            json.dump(state_dict, f, indent=2, default=str)
        
        # 保存当前版本组件
        version_path = os.path.join(path, f'v{self.baseline_version}')
        os.makedirs(version_path, exist_ok=True)
        
        self.engine.save(os.path.join(version_path, 'engine'))
        with open(os.path.join(version_path, 'processor.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(version_path, 'classifier2.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        # 如果正在切换，保存待切换组件
        if self.state == self.SWITCHING and self._new_engine is not None:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            os.makedirs(new_path, exist_ok=True)
            self._new_engine.save(os.path.join(new_path, 'engine'))
            with open(os.path.join(new_path, 'processor.pkl'), 'wb') as f:
                pickle.dump(self._new_processor, f)
            with open(os.path.join(new_path, 'classifier2.pkl'), 'wb') as f:
                pickle.dump(self._new_classifier2, f)
        
        print(f"✅ 状态已保存到 {path}")
    
    def load_state(self, path="./checkpoints/adaptive_state"):
        """从持久化文件恢复状态"""
        state_file = os.path.join(path, 'state.json')
        if not os.path.exists(state_file):
            print(f"⚠️ 未找到状态文件: {state_file}")
            return False
        
        with open(state_file, 'r') as f:
            state_dict = json.load(f)
        
        # 恢复状态变量
        self.state = state_dict['state']
        self.baseline_version = state_dict['baseline_version']
        self.cp_consec = state_dict['cp_consec']
        self.cp_confirmed_tidx = state_dict.get('cp_confirmed_tidx')
        self.retrace_start_tidx = state_dict.get('retrace_start_tidx')
        self.accumulate_buffer_tidx = state_dict.get('accumulate_buffer_tidx', [])
        self.cooldown_until_tidx = state_dict.get('cooldown_until_tidx', -1)
        self._ap_warmup_remaining = state_dict.get('ap_warmup_remaining', 0)
        self._parallel_count = state_dict.get('parallel_count', 0)
        self._switch_extended = state_dict.get('switch_extended', False)
        self.cp_events = state_dict.get('cp_events', [])
        self.switch_history = state_dict.get('switch_history', [])
        
        # 恢复组件
        version_path = os.path.join(path, f'v{self.baseline_version}')
        if os.path.exists(version_path):
            from TFT_tft_engine import TFTEngine
            self.engine = TFTEngine.load(
                os.path.join(version_path, 'engine'), 
                config_path='TFT_config.yaml')
            with open(os.path.join(version_path, 'processor.pkl'), 'rb') as f:
                self.processor = pickle.load(f)
            with open(os.path.join(version_path, 'classifier2.pkl'), 'rb') as f:
                self.classifier2 = pickle.load(f)
        
        # 恢复待切换组件
        if self.state == self.SWITCHING:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            if os.path.exists(new_path):
                self._new_engine = TFTEngine.load(
                    os.path.join(new_path, 'engine'),
                    config_path='TFT_config.yaml')
                with open(os.path.join(new_path, 'processor.pkl'), 'rb') as f:
                    self._new_processor = pickle.load(f)
                with open(os.path.join(new_path, 'classifier2.pkl'), 'rb') as f:
                    self._new_classifier2 = pickle.load(f)
        
        print(f"✅ 状态已恢复: state={self.state}, version={self.baseline_version}")
        return True
       
    # ================================================================
    # 主入口: run()
    # ================================================================
    def run(self, df_real_all, df_features_raw, df_official,
            quiet_start=None, quiet_end=None, output_path=None, resume=False):
        """
        完整的自适应滚动检测流程。
        
        Parameters
        ----------
        df_real_all : 经processor+tb变换后的完整数据（含time_idx, timestamp, PCA列等）
        df_features_raw : 原始特征数据（用于refit processor）
        df_official : 官方运营日历
        quiet_start, quiet_end : 平静期时间范围（用于初始calibrate）
        output_path : 保存路径
        resume : bool
            是否从持久化状态恢复继续运行
        Returns
        -------
        pd.DataFrame : 每个窗口的检测结果
        """
        df = df_real_all.sort_values('time_idx').reset_index(drop=True)
        
        # P8修复：尝试恢复
        if resume and os.path.exists("./checkpoints/adaptive_state/state.json"):
            if self.load_state("./checkpoints/adaptive_state"):
                print("[Resume] 从保存点继续...")

        # --- 0. 冷启动校准 ---
        if quiet_start and quiet_end:
            quiet_mask = (df['timestamp'] >= quiet_start) & (df['timestamp'] < quiet_end)
            quiet_start_idx = df[df['timestamp'] >= quiet_start].index[0]
        else:
            # 默认前7天
            quiet_end_ts = df['timestamp'].min() + pd.Timedelta(days=7)
            quiet_mask = df['timestamp'] < quiet_end_ts
            quiet_start_idx = 0
        
        # 建立TFT baseline
        pre_ctx = max(0, quiet_start_idx - self.min_required)
        bl_end = df[quiet_mask].index[-1] + 1
        df_for_bl = df.iloc[pre_ctx:bl_end].copy().reset_index(drop=True)
        bl_end_tidx = int(df.loc[quiet_mask, 'time_idx'].max())
        
        print(f"[Init] 建立TFT baseline ({len(df_for_bl)} 窗口)...")
        self._patch_engine_batch_size(self.engine)

        for mn in self.engine.models:
            res = self.engine.analyze_rolling(mn, df_for_bl, baseline_end_idx=bl_end_tidx)
            if 'baseline' not in res and 'metrics' in res:
                bl = self.engine._build_baseline(
                    res['metrics'], res['attention'], res['vsn'], bl_end_tidx)
                self.engine.baselines[mn] = bl
            print(f"  [{mn}] baseline: σ={self.engine.baselines[mn]['residual_std']:.4f}")
        
        # 校准分类器2
        # 先对平静期做一次TFT推理获取model_*列
        df_quiet_for_cal = self._extract_tft_batch(df_for_bl, engine=self.engine)
        if len(df_quiet_for_cal) > 0:
            # 只取平静期时间范围内的行
            cal_mask = df_quiet_for_cal['time_idx'] <= bl_end_tidx
            self.classifier2.calibrate(df_quiet_for_cal[cal_mask])
        else:
            print("  ⚠️ 平静期TFT推理无输出, 用原始数据校准")
            self.classifier2.calibrate(df[quiet_mask])
        
        # --- 1. 初始化缓冲区 ---
        initial_end = min(quiet_start_idx + self.buffer_size, len(df))
        buffer_df = df.iloc[pre_ctx:initial_end].copy().reset_index(drop=True)
        
        quiet_start_tidx = int(df.loc[quiet_mask, 'time_idx'].min())
        output_tidx = set(
            buffer_df.loc[buffer_df['time_idx'] >= quiet_start_tidx, 'time_idx']
            .astype(int).values)
        
        print(f"\n[ColdStart] 缓冲区: {len(buffer_df)} 窗口, 推理中...")
        df_cold = self._extract_tft_batch(buffer_df, engine=self.engine,
                                          output_tidx_set=output_tidx)
        
        all_tft_results = [df_cold] if len(df_cold) > 0 else []
        processed_tidx = set(df_cold['time_idx'].astype(int).values) if len(df_cold) > 0 else set()
        
        # --- 2. 滚动主循环 ---
        remaining_start = initial_end
        total_remaining = len(df) - remaining_start
        n_batches = (total_remaining + self.step_size - 1) // self.step_size
        
        print(f"\n[Rolling] {total_remaining} 窗口, {n_batches} 批 (step={self.step_size})")
        accumulated_tft_features = df_cold.copy() if len(df_cold) > 0 else pd.DataFrame()
        
        for batch_i in tqdm(range(n_batches), desc="自适应滚动检测"):
            batch_start = remaining_start + batch_i * self.step_size
            batch_end = min(batch_start + self.step_size, len(df))
            
            if batch_start >= len(df):
                break
            
            new_rows = df.iloc[batch_start:batch_end]
            new_tidx = set(new_rows['time_idx'].astype(int).values) - processed_tidx
            
            if not new_tidx:
                continue
            
            # 更新缓冲区
            buffer_df = pd.concat([buffer_df, new_rows], ignore_index=True)
            if len(buffer_df) > self.buffer_size:
                buffer_df = buffer_df.iloc[-self.buffer_size:].reset_index(drop=True)
            
            if len(buffer_df) < self.min_required:
                continue
            # P7修复：AP warmup阈值放宽
            if self._ap_warmup_remaining > 0:
                self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
                self._ap_warmup_remaining -= len(new_tidx)
                if self._ap_warmup_remaining <= 0:
                    self.classifier2.set_ap_relax_factor(1.0)
                    print(f"\n  [AP Warmup] 结束")
            # P3修复：并行验证时需要两个引擎都推理
            if self.state == self.SWITCHING and self._new_engine is not None:
                df_feat_old = self._extract_tft_batch(
                    buffer_df, engine=self.engine, output_tidx_set=new_tidx)
                df_feat_new = self._extract_tft_batch(
                    buffer_df, engine=self._new_engine, output_tidx_set=new_tidx)
                
                # 主输出用旧引擎（保守）
                df_feat = df_feat_old
                
                # 收集预测结果用于对比
                if len(df_feat_old) > 0 and len(df_feat_new) > 0:
                    pred_old = self.classifier2.predict(df_feat_old)['pred_label'].values
                    pred_new = self._new_classifier2.predict(df_feat_new)['pred_label'].values
                    
                    self._parallel_preds_old.extend(pred_old.tolist())
                    self._parallel_preds_new.extend(pred_new.tolist())
                    self._parallel_count += len(pred_old)
            else:
                active_engine = self.engine
                df_feat = self._extract_tft_batch(
                    buffer_df, engine=active_engine, output_tidx_set=new_tidx)   
            # 选择当前使用的引擎
            # active_engine = self._new_engine if (
            #     self.state == self.SWITCHING and self._new_engine) else self.engine
            
            # 批量TFT推理
            # df_feat = self._extract_tft_batch(
            #     buffer_df, engine=active_engine, output_tidx_set=new_tidx)
            
            if len(df_feat) == 0:
                continue
            
            all_tft_results.append(df_feat)
            processed_tidx.update(df_feat['time_idx'].astype(int).values)
            
            # 分类器2判定
            # df_classified = self.classifier2.predict(df_feat)

            # --- 修复：维护特征缓冲区以支持长程 Rolling 计算 ---
            accumulated_tft_features = pd.concat([accumulated_tft_features, df_feat], ignore_index=True)
            
            # 保持缓冲区长度足够覆盖长程趋势窗口 (2688) + 容余量
            max_history_needed = self.classifier2.cp_trend_window + self.step_size + 100
            if len(accumulated_tft_features) > max_history_needed:
                accumulated_tft_features = accumulated_tft_features.iloc[-max_history_needed:].reset_index(drop=True)
            
            # 对包含历史上下文的数据统一执行分类判定
            df_classified_all = self.classifier2.predict(accumulated_tft_features)
            
            # 仅截取属于当前批次(new_tidx)的新窗口，用于触发状态机
            df_classified = df_classified_all[df_classified_all['time_idx'].isin(new_tidx)].copy()
            
            # 状态机处理
            current_tidx = int(new_rows['time_idx'].iloc[-1])
            self._process_batch(
                df_classified, current_tidx, df, df_features_raw, df_official)            
            # 状态机处理
            current_tidx = int(new_rows['time_idx'].iloc[-1])
            self._process_batch(
                df_classified, current_tidx, df, df_features_raw, df_official)
            
            # 日志
            if (batch_i + 1) % 20 == 0:
                ts = new_rows['timestamp'].iloc[-1]
                n_ap = (df_classified['pred_label'] == 'AP').sum()
                n_cp = (df_classified['pred_label'] == 'CP').sum()
                print(f"  Batch {batch_i+1}/{n_batches} | {ts} | "
                      f"state={self.state} | v={self.baseline_version} | "
                      f"AP={n_ap} CP={n_cp} warmup={self._ap_warmup_remaining}")
        
        # --- 3. 合并所有输出 ---
        if all_tft_results:
            df_all_tft = pd.concat(all_tft_results, ignore_index=True)
            df_all_tft = df_all_tft.drop_duplicates(subset='time_idx', keep='last')
            df_all_tft = df_all_tft.sort_values('time_idx').reset_index(drop=True)
            df_final = self.classifier2.predict(df_all_tft)
            df_final['baseline_version'] = self.baseline_version
            df_final['state'] = self.state
        else:
            df_final = pd.DataFrame()
        
        # 对全量结果执行分类器2（利用完整rolling窗口）
        # print(f"\n[Final] 对 {len(df_all_tft)} 窗口执行最终分类...")
        # if len(df_all_tft) > 0:
        #     df_final = self.classifier2.predict(df_all_tft)
        #     # 附加元信息
        #     df_final['baseline_version'] = self.baseline_version
        #     df_final['state'] = self.state
        # else:
        #     df_final = pd.DataFrame()
        
        # 保存
        if output_path and len(df_final) > 0:
            os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
            df_final.to_csv(output_path, index=False)
            print(f"✅ 保存到 {output_path}")
        
        self._print_summary(df_final)
        self.save_state()

        return df_final
    
    # ================================================================
    # 状态机处理逻辑
    # ================================================================
    def _process_batch(self, df_classified, current_tidx,
                       df_full, df_features_raw, df_official):
        """双轨状态机更新"""
        
        labels = df_classified['pred_label'].values
        is_drift = df_classified['is_baseline_drift'].values
        
        # 1. 如果正在验证期，仅执行并行逻辑
        if self.state == self.SWITCHING:
            self._check_parallel_switch(current_tidx)
            return

        # 2. 冷却期全局拦截
        if current_tidx < self.cooldown_until_tidx:
            return

        # =======================================================
        # 轨道 2：长程缓变漂移 (Incremental CP) - 最高优先级
        # =======================================================
        if is_drift.any():
            print(f"\n  [→ DRIFT DETECTED] 监测到基线不可逆漂移 @ tidx={current_tidx}，立即向前回溯重构")
            
            # 向过去提取 14 天数据直接微调
            retrace_start = max(0, current_tidx - self.accumulate_min)
            retrace_mask = (df_full['time_idx'] >= retrace_start) & (df_full['time_idx'] <= current_tidx)
            df_retrace = df_full[retrace_mask].copy()
            
            ts_start = df_retrace['timestamp'].min()
            ts_end = df_retrace['timestamp'].max()
            raw_mask = (df_features_raw['timestamp'] >= ts_start) & (df_features_raw['timestamp'] <= ts_end)
            df_raw_retrace = df_features_raw[raw_mask].copy()
            
            success = self._finetune(df_retrace, df_raw_retrace, df_official)
            
            if success:
                self.state = self.SWITCHING
                self._parallel_preds_old = []
                self._parallel_preds_new = []
                self._parallel_count = 0
                self._switch_extended = False
                
                self.cp_events.append({
                    'confirmed_tidx': current_tidx,
                    'retrace_tidx': retrace_start,
                    'retrace_windows': self.accumulate_min,
                    'drift_type': 'Incremental',
                    'switched_tidx': None,
                    'switch_type': None,
                    'baseline_version': self.baseline_version,
                })
                print(f"  [→ SWITCHING] 漂移重构完成，进入并行验证 ({self.switch_parallel_windows} 窗口)")
            else:
                print("  [微调失败] 维持原有基线，触发防抖冷却")
                self.state = self.NORMAL
                self._reset_partial_state()
                self.cooldown_until_tidx = current_tidx + self.cooldown_windows
            return # 轨道2触发后，中止本批次的后续逻辑

        # =======================================================
        # 轨道 1：突发业务冲击 (Sudden CP) - 延迟观察逻辑
        # =======================================================
        if self.state == self.NORMAL:
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
                if self.cp_consec >= 1:
                    self.state = self.CP_TENTATIVE
                    print(f"\n  [→ CP_TENTATIVE] 遭遇业务冲击 consec={self.cp_consec} @ tidx={current_tidx}")
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
        
        elif self.state == self.CP_TENTATIVE:
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
            
            # 要求连续 96 个窗口维持突发极值
            if self.cp_consec >= self.cp_confirm_windows:
                self.cp_confirmed_tidx = current_tidx
                self.retrace_start_tidx = current_tidx  # 从确认点开始向未来蓄水
                self.accumulate_buffer_tidx = []
                self.state = self.ACCUMULATING
                
                print(f"\n  [→ ACCUMULATING] 冲击确认 @ tidx={current_tidx}, 启动向未来蓄水 {self.accumulate_min} 窗口")
            
            if self.cp_consec <= 0:
                self.state = self.NORMAL
                self.cp_consec = 0
                print(f"\n  [→ NORMAL] 冲击平息，警报解除 @ tidx={current_tidx}")
                
        elif self.state == self.ACCUMULATING:
            batch_tidx = df_classified['time_idx'].astype(int).tolist()
            self.accumulate_buffer_tidx.extend(batch_tidx)
            accumulated = len(set(self.accumulate_buffer_tidx))
            
            print(f"  [ACCUMULATING] 蓄水观察中 {accumulated}/{self.accumulate_min} 窗口", end='\r')
            
            if accumulated >= self.accumulate_min:
                print(f"\n  [微调触发] 未来 14 天蓄水完毕，开始评估常态...")
                
                retrace_mask = (df_full['time_idx'] >= self.retrace_start_tidx) & (df_full['time_idx'] <= current_tidx)
                df_retrace = df_full[retrace_mask].copy()
                
                ts_start = df_retrace['timestamp'].min()
                ts_end = df_retrace['timestamp'].max()
                raw_mask = (df_features_raw['timestamp'] >= ts_start) & (df_features_raw['timestamp'] <= ts_end)
                df_raw_retrace = df_features_raw[raw_mask].copy()
                
                success = self._finetune(df_retrace, df_raw_retrace, df_official)
                
                if success:
                    self.state = self.SWITCHING
                    self._parallel_preds_old = []
                    self._parallel_preds_new = []
                    self._parallel_count = 0
                    self._switch_extended = False
                    
                    self.cp_events.append({
                        'confirmed_tidx': self.cp_confirmed_tidx,
                        'retrace_tidx': self.retrace_start_tidx,
                        'retrace_windows': self.accumulate_min,
                        'drift_type': 'Sudden',
                        'switched_tidx': None,
                        'switch_type': None,
                        'baseline_version': self.baseline_version,
                    })
                    print(f"  [→ SWITCHING] 备用模型完成，开始并行验证 ({self.switch_parallel_windows} 窗口)")
                else:
                    print("  [微调失败] 回退到NORMAL，触发防抖冷却")
                    self.state = self.NORMAL
                    self._reset_partial_state()
                    self.cooldown_until_tidx = current_tidx + self.cooldown_windows
    # ================================================================
    # P3修复：完整的并行切换逻辑 V4
    # ================================================================
    def _check_parallel_switch(self, current_tidx):
        required_windows = (self.switch_parallel_extend 
                            if self._switch_extended 
                            else self.switch_parallel_windows)
        
        if self._parallel_count < required_windows:
            return
        
        pred_old = np.array(self._parallel_preds_old[-required_windows:])
        pred_new = np.array(self._parallel_preds_new[-required_windows:])
        
        if len(pred_old) == 0 or len(pred_new) == 0:
            return
        
        old_n_ratio = (pred_old == 'N').mean()
        new_n_ratio = (pred_new == 'N').mean()
        
        print(f"\n  [并行验证] windows={self._parallel_count}, "
              f"旧模型N占比={old_n_ratio:.3f}, 新模型N占比={new_n_ratio:.3f}")
        
        # 获取当前触发微调的事件类型
        current_drift_type = self.cp_events[-1].get('drift_type', 'Sudden') if self.cp_events else 'Sudden'
        
        if current_drift_type == 'Sudden':
            if old_n_ratio >= self.switch_recovery_threshold:
                print("  [放弃切换] 旧模型已恢复正常，微调冗余")
                self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
            elif new_n_ratio >= self.switch_stability_threshold and new_n_ratio >= old_n_ratio:
                print("  [触发切换] 突变稳态确认，新模型优于旧模型")
                self._execute_full_switch(current_tidx, agreement=0.0, kappa=0.0)
            else:
                self._handle_unstable_new_model(current_tidx, new_n_ratio)
                
        elif current_drift_type == 'Incremental':
            # 忽略旧模型表现，仅考核新模型绝对稳定性
            if new_n_ratio >= self.switch_stability_threshold:
                print("  [触发切换] 长程漂移重构有效，新模型达到稳态")
                self._execute_full_switch(current_tidx, agreement=0.0, kappa=0.0)
            else:
                self._handle_unstable_new_model(current_tidx, new_n_ratio)

    def _handle_unstable_new_model(self, current_tidx, new_n_ratio):
        if not self._switch_extended:
            self._switch_extended = True
            print(f"  [延长验证] 新模型N占比={new_n_ratio:.3f}，延长到 {self.switch_parallel_extend} 窗口")
        else:
            print(f"  [拒绝切换] 延长验证后新模型表现仍未占优")
            self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
    
    def _execute_full_switch(self, current_tidx, agreement, kappa):
        """完整切换：模型权重+baseline+阈值"""
        print(f"\n  [完整切换] agreement={agreement:.3f}, kappa={kappa:.3f}")
        
        self.engine = self._new_engine
        self.processor = self._new_processor
        self.classifier2 = self._new_classifier2
        
        self._finalize_switch(current_tidx, 'full', agreement, kappa)
    
    def _execute_partial_switch(self, current_tidx, agreement, kappa):
        """部分切换：仅baseline+阈值，保留旧模型权重"""
        print(f"\n  [部分切换] agreement={agreement:.3f}，保留旧模型权重")
        
        # 只更新processor和分类器阈值
        self.processor = self._new_processor
        
        # 更新分类器阈值
        self.classifier2.ap_thresholds = copy.deepcopy(self._new_classifier2.ap_thresholds)
        self.classifier2.cp_base_thresh = self._new_classifier2.cp_base_thresh
        self.classifier2.cp_vol_thresh = self._new_classifier2.cp_vol_thresh
        self.classifier2._z_params = copy.deepcopy(self._new_classifier2._z_params)
        
        # 更新baseline但保留旧模型
        for mn in self._new_engine.baselines:
            self.engine.baselines[mn] = self._new_engine.baselines[mn]
        
        self._finalize_switch(current_tidx, 'partial', agreement, kappa)
    
    def _finalize_switch(self, current_tidx, switch_type, agreement, kappa):
        """切换完成的收尾工作"""
        
        self.baseline_version += 1
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        
        # P7修复：启动AP warmup
        self._ap_warmup_remaining = self.ap_warmup_windows
        self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
        
        # 记录历史
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
            self.cp_events[-1]['switch_type'] = switch_type
        
        self.switch_history.append({
            'tidx': current_tidx,
            'type': switch_type,
            'agreement': agreement,
            'kappa': kappa,
            'new_version': self.baseline_version,
        })
        
        # 重置状态
        self.state = self.NORMAL
        self._reset_partial_state()
        
        print(f"  [切换完成] type={switch_type}, version={self.baseline_version}, "
              f"冷却至 tidx={self.cooldown_until_tidx}, "
              f"AP warmup={self._ap_warmup_remaining}窗口")
    
    def _reset_partial_state(self):
        """重置切换相关状态"""
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False

    # ================================================================
    # 微调
    # ================================================================
    def _finetune(self, df_retrace, df_raw_retrace, df_official):
        """
        微调三件套: processor refit → TFT微调 → 分类器2阈值更新
        """
        from TFT_tft_engine import TFTEngine
        from tft_full_period_utils import compute_baseline_from_held_out, create_held_out_split
        
        # (a) Processor refit
        print("  [微调 1/3] Processor refit...")
        processor_new = copy.deepcopy(self.processor)
        processor_new.feature_transformer.fit(df_raw_retrace.fillna(0))
        
        # 用新processor变换数据
        dataset_new = processor_new.transform(df_raw_retrace.fillna(0), df_official)
        df_tft = dataset_new.data.copy()

        df_tft.fillna(0, inplace=True)
        df_tft.replace([np.inf, -np.inf], 0, inplace=True)
        df_tft = self.tb.transform(df_tft)
        if 'label' not in df_tft.columns:
            df_tft['label'] = 'N'
        
        print(f"  [微调数据] {len(df_tft)} 窗口")

        
        # (b) TFT微调
        print("  [微调 2/3] TFT模型微调...")
        with tempfile.TemporaryDirectory() as tmp:
            self.engine.save(tmp)
            engine_new = TFTEngine.load(tmp, config_path='TFT_config.yaml')
        
        self._patch_engine_batch_size(engine_new)
        self._patch_engine_early_stopping(engine_new)
        
        # Held-out切分
        held_out_windows = max(
            self.held_out_min_windows,
            int(len(df_tft) * 0.15)  # 最多15%
        )
        held_out_windows = min(held_out_windows, len(df_tft) // 3)

        split_idx = len(df_tft) - held_out_windows
        df_train_ft = df_tft.iloc[:split_idx].copy()
        df_held = df_tft.iloc[split_idx:].copy()
        held_out_tidx = set(df_held['time_idx'].values)
        # held_out_mask, held_mask, split_info = create_held_out_split(
        #     df_tft, time_col='timestamp',
        #     block_days=self.held_out_days,
        #     min_block_windows=96)
        # df_train_ft = df_tft[~held_mask].copy()
        # held_out_tidx = set(df_tft.loc[held_mask, 'time_idx'].values)
        print(f"  [数据切分] 训练={len(df_train_ft)}, held_out={len(df_held)}")
        
        if len(df_train_ft) < self.min_required * 2:
            print(f"  ⚠️ 训练数据不足")
            return False
        
        for mn in engine_new.models:
            # 解冻
            for p in engine_new.models[mn].parameters():
                p.requires_grad = True
            
            cfg = engine_new.config['tft_models'][mn]
            orig_lr = cfg.get('learning_rate', 0.03)
            cfg['learning_rate'] = orig_lr * self.finetune_lr_scale
            # 应用梯度裁剪
            if 'gradient_clip_val' not in cfg:
                cfg['gradient_clip_val'] = self.finetune_grad_clip

            print(f"    [{mn}] 微调 (lr={cfg['learning_rate']:.5f}, "
                  f"epochs={self.finetune_epochs}, data={len(df_train_ft)})...")
            
            with contextlib.redirect_stdout(io.StringIO()):
                engine_new.build_and_fit(
                    mn, df_train_ft, 
                    max_epochs=self.finetune_epochs)
            
            cfg['learning_rate'] = orig_lr
            
            # 重建baseline
            res = engine_new.analyze_rolling(mn, df_tft, baseline_end_idx=None)
            if 'metrics' not in res:
                print(f"    ⚠️ [{mn}] 推理失败")
                continue
            if held_out_tidx:
                bl = compute_baseline_from_held_out(res, held_out_tidx)
            else:
                bl = engine_new._build_baseline(
                    res['metrics'], res['attention'], res['vsn'],
                    int(df_tft['time_idx'].max()))
            engine_new.baselines[mn] = bl
            # engine_new.freeze_layers(mn)
            
            print(f"    [{mn}] 新baseline: σ={bl['residual_std']:.4f}")
        
        # (c) 分类器2阈值更新
        print("  [微调 3/3] 分类器2阈值更新...")
        # 用新引擎推理held-out数据，获取model_*列
        classifier2_new = copy.deepcopy(self.classifier2)

        # df_held = df_tft[held_mask].copy()
        if len(df_held) > self.min_required:
            df_held_tft = self._extract_tft_batch(df_held, engine=engine_new)
            if len(df_held_tft) > 10:
                # classifier2_new = copy.deepcopy(self.classifier2)
                # classifier2_new.calibrate(df_held_tft)
                classifier2_new.calibrate(df_held_tft)
                print(f"    新阈值校准完成")
            else:
                classifier2_new = copy.deepcopy(self.classifier2)
        else:
            classifier2_new = copy.deepcopy(self.classifier2)
        
        # 暂存新组件
        self._new_engine = engine_new
        self._new_processor = processor_new
        self._new_classifier2 = classifier2_new
        
        print("  [微调完成] 新组件已就绪，等待并行验证")
        return True

    # ================================================================
    # TFT批量推理（可配置batch_size）
    # ================================================================
    def _patch_engine_batch_size(self, engine):
        """为引擎应用推理batch_size补丁"""
        
        inference_bs = self.inference_batch_size
        
        def patched_analyze(model_name, df, baseline_end_idx=None, predict=True):
            model = engine.models[model_name]
            train_dataset = engine.datasets[model_name]
            df_inf = df.copy()
            for col in engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)
            
            try:
                inference_dataset = TimeSeriesDataSet.from_dataset(
                    train_dataset, df_inf, predict=False, stop_randomization=True)
            except ValueError as e:
                if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
                    return {}
                raise
            
            if len(inference_dataset) == 0:
                return {}
            
            dataloader = inference_dataset.to_dataloader(
                train=False, 
                batch_size=inference_bs,
                num_workers=0, 
                pin_memory=True)
            
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model.to(device)
            model.eval()
            
            result_buffer = {}
            with torch.no_grad():
                for x, y in dataloader:
                    x = {k: v.to(device) for k, v in x.items()}
                    targets = y[0].cpu().numpy().flatten()
                    raw_out = model(x)
                    preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
                    p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
                    time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
                    interp = model.interpret_output(raw_out, reduction="none")
                    batch_att = interp['attention'].cpu().numpy()
                    batch_vsn = interp['encoder_variables'].cpu().numpy()
                    
                    for i in range(len(time_idx)):
                        t_id = int(time_idx[i])
                        result_buffer[t_id] = {
                            "target_true": targets[i],
                            "pred_p50": p50[i],
                            "residual": targets[i] - p50[i],
                            "divergence": p90[i] - p10[i],
                            "attention": batch_att[i],
                            "vsn": batch_vsn[i]
                        }
            
            if not result_buffer:
                return {}
            
            sorted_t = sorted(result_buffer.keys())
            df_s = pd.DataFrame({
                "time_idx": sorted_t,
                "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
                "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
                "residual": [result_buffer[t]["residual"] for t in sorted_t],
                "divergence": [result_buffer[t]["divergence"] for t in sorted_t]
            })
            full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
            full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
            
            result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
            
            if baseline_end_idx is not None:
                bl = engine._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
                result["baseline"] = bl
                engine.baselines[model_name] = bl
            
            return result
        
        engine.analyze_rolling = patched_analyze
        
    def _patch_engine_early_stopping(self, engine):
        """为引擎应用具有高容忍度的早停机制及最佳权重回滚补丁"""
        import types
        import torch
        import pandas as pd
        import lightning.pytorch as pl
        from lightning.pytorch.loggers import CSVLogger
        from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
        from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
        from pytorch_forecasting.metrics import QuantileLoss
        from pytorch_forecasting.data import GroupNormalizer
        from pytorch_forecasting.data.encoders import NaNLabelEncoder

        ft_batch_size = self.finetune_batch_size

        def patched_build_and_fit(self_engine, model_name: str, df: pd.DataFrame, 
                                  val_ratio: float = 0.2, max_epochs: int = -1,
                                  quiet_end_idx: int = None):
            
            cfg = self_engine.config['tft_models'][model_name]
            pl.seed_everything(cfg.get('seed', 42))

            df_inf = df.copy()
            for col in self_engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)

            if quiet_end_idx is not None:
                time_col = self_engine.global_cfg['time_column']
                df_for_training = df_inf[df_inf[time_col] <= quiet_end_idx].copy()
            else:
                df_for_training = df_inf

            time_col = self_engine.global_cfg['time_column']
            time_steps = df_for_training[time_col].sort_values().unique()
            split_idx = int(len(time_steps) * (1 - val_ratio))
            cutoff_time = time_steps[split_idx]
            
            train_df = df_for_training[df_for_training[time_col] <= cutoff_time]
            max_lookback = cfg.get('max_encoder_length', 48)
            val_start_time = time_steps[max(0, split_idx - max_lookback)]
            val_df = df_for_training[df_for_training[time_col] >= val_start_time]

            categorical_encoders = {
                name: NaNLabelEncoder(add_nan=True) 
                for name in self_engine.known_categoricals
            }
            for g_id in self_engine.global_cfg['group_ids']:
                categorical_encoders[g_id] = NaNLabelEncoder(add_nan=True)

            training = TimeSeriesDataSet(
                train_df,
                time_idx=self_engine.global_cfg['time_column'],
                target=cfg['target'],
                group_ids=self_engine.global_cfg['group_ids'],
                min_encoder_length=cfg.get('min_encoder_length'),
                max_encoder_length=cfg.get('max_encoder_length'),
                max_prediction_length=cfg.get('max_prediction_length', 1),
                time_varying_unknown_reals=self_engine.unknown_reals,
                time_varying_known_reals=self_engine.known_reals,
                time_varying_known_categoricals=self_engine.known_categoricals,
                categorical_encoders=categorical_encoders,
                target_normalizer=GroupNormalizer(groups=self_engine.global_cfg['group_ids'], transformation=None),
                add_relative_time_idx=True,
                add_target_scales=True,
                add_encoder_length=True,
                allow_missing_timesteps=True
            )

            validation = TimeSeriesDataSet.from_dataset(training, val_df, predict=False, stop_randomization=True)

            train_dataloader = training.to_dataloader(train=True, batch_size=ft_batch_size, num_workers=0, pin_memory=True)
            val_dataloader = validation.to_dataloader(train=False, batch_size=ft_batch_size, num_workers=0, pin_memory=True)

            model = TemporalFusionTransformer.from_dataset(
                training,
                learning_rate=cfg.get('learning_rate', 0.03),
                hidden_size=cfg.get('hidden_size', 16),
                attention_head_size=4,
                dropout=0.1,
                hidden_continuous_size=8,
                output_size=len(self_engine.quantiles),
                loss=QuantileLoss(quantiles=self_engine.quantiles),
                reduce_on_plateau_patience=4
            )

            logger = CSVLogger("lightning_logs", name=model_name, flush_logs_every_n_steps=10)
            self_engine.log_dirs[model_name] = logger.log_dir

            # 核心调整：配置宽容型早停与检查点
            early_stop_callback = EarlyStopping(
                monitor="val_loss",
                patience=8,       # 容忍20个Epoch没有改善才停止，跨越局部波动
                min_delta=1e-3,    # 降低改善判定阈值
                mode="min",
                verbose=False
            )
            checkpoint_callback = ModelCheckpoint(
                monitor="val_loss",
                mode="min",
                save_top_k=1,      # 仅保留验证损失最低的那个Epoch权重
                dirpath=logger.log_dir,
                filename="best_model"
            )

            trainer_kwargs = {
                "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
                "devices": 1,
                "enable_model_summary": False,
                "enable_checkpointing": True,
                "callbacks": [early_stop_callback, checkpoint_callback],
                "logger": logger,
                "enable_progress_bar": True
            }

            if max_epochs == -1:
                trainer = pl.Trainer(**trainer_kwargs)
            elif max_epochs != 1:
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)
            elif max_epochs == 1:
                trainer_kwargs.update({"limit_train_batches": 5, "limit_val_batches": 5})
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)

            trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

            # 核心调整：训练结束后，强制从检查点加载验证集表现最佳的权重
            # if checkpoint_callback.best_model_path:
            #     best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)
            #     model.load_state_dict(best_model.state_dict())
            if checkpoint_callback.best_model_path:
                best_model = TemporalFusionTransformer.load_from_checkpoint(
                    checkpoint_callback.best_model_path, 
                    weights_only=False
                )
                model.load_state_dict(best_model.state_dict())
                print(f"    [{model_name}] 回滚至最佳验证权重: {checkpoint_callback.best_model_score:.4f}")

            self_engine.plot_training_history(model_name)
            
            self_engine.datasets[model_name] = training
            self_engine.models[model_name] = model

        engine.build_and_fit = types.MethodType(patched_build_and_fit, engine)
    # ================================================================
    # 切换
    # ================================================================
    def _execute_switch(self, current_tidx):
        """执行分层切换判定"""
        
        # 由于并行推理在简化实现中无法精确对比，这里直接执行切换
        # 完整实现中应同时用新旧引擎推理同一批数据并比较一致率
        
        # 立即切换: processor + baseline + 分类器阈值
        print(f"\n  [SWITCH] 执行切换 @ tidx={current_tidx}")
        
        self.processor = self._new_processor
        self.engine = self._new_engine
        self.classifier2 = self._new_classifier2
        
        self.baseline_version += 1
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        self._switch_warmup_count = self.ap_warmup_windows
        
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
        
        # 重置状态
        self.state = self.NORMAL
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        
        print(f"  [SWITCH] 完成! baseline_version={self.baseline_version}, "
              f"冷却至 tidx={self.cooldown_until_tidx}")
    
    # ================================================================
    # TFT批量推理（复用滚动训练Cell 2的逻辑）
    # ================================================================
    def _extract_tft_batch(self, df_buffer, engine=None, output_tidx_set=None):
        """
        对缓冲区执行TFT推理，提取model_*信号 + rolling。
        直接复用notebook Cell 2 的 extract_tft_features_batch 逻辑。
        """
        if engine is None:
            engine = self.engine
        
        # --- TFT推理 ---
        tft_results = {}
        for mn in engine.models:
            with contextlib.redirect_stdout(io.StringIO()), \
                 contextlib.redirect_stderr(io.StringIO()):
                res = engine.analyze_rolling(mn, df_buffer, baseline_end_idx=None)
            tft_results[mn] = res
        
        if not any('metrics' in v for v in tft_results.values()):
            return pd.DataFrame()
        
        # --- 合并TFT信号 ---
        df_merged = df_buffer[['time_idx', 'timestamp']].copy()
        
        for model_name in tft_results:
            res = tft_results[model_name]
            if 'metrics' not in res:
                continue
            
            metrics = res['metrics']
            attention = res['attention']
            vsn = res['vsn']
            baseline = engine.baselines.get(model_name, {})
            
            if not baseline:
                continue
            
            n = len(metrics)
            
            # 残差比
            residual_abs = metrics['residual'].abs().values
            baseline_p95 = max(
                abs(baseline.get('residual_p95', 0)),
                abs(baseline.get('residual_p5', 0)),
                baseline.get('residual_std', 1e-8) * 1.645, 1e-8)
            sig_residual = np.clip(residual_abs / baseline_p95, 0, 20)
            
            # 注意力KL散度
            att_base = baseline.get('att_mean')
            sig_att_kl = np.zeros(n)
            if att_base is not None:
                for i in range(n):
                    p = attention[i].flatten()
                    q = att_base.flatten()
                    p = p / (p.sum() + 1e-10) + 1e-10
                    q = q / (q.sum() + 1e-10) + 1e-10
                    p, q = p / p.sum(), q / q.sum()
                    sig_att_kl[i] = np.sum(p * np.log(p / q))
            
            # VSN JS散度
            vsn_base = baseline.get('vsn_mean')
            sig_vsn_js = np.zeros(n)
            if vsn_base is not None:
                for i in range(n):
                    v_i = np.abs(vsn[i].flatten()) + 1e-10
                    v_b = np.abs(vsn_base.flatten()) + 1e-10
                    p_v, q_v = v_i / v_i.sum(), v_b / v_b.sum()
                    m_v = 0.5 * (p_v + q_v)
                    sig_vsn_js[i] = (
                        0.5 * np.sum(p_v * np.log(p_v / m_v)) +
                        0.5 * np.sum(q_v * np.log(q_v / m_v)))
            
            # VSN排序变化
            sig_vsn_rank = np.zeros(n)
            if vsn_base is not None:
                baseline_rank = np.argsort(np.argsort(-np.abs(vsn_base.flatten())))
                for i in range(n):
                    curr_rank = np.argsort(np.argsort(-np.abs(vsn[i].flatten())))
                    corr, _ = spearmanr(baseline_rank, curr_rank)
                    sig_vsn_rank[i] = 1 - corr if not np.isnan(corr) else 1.0
            
            # 组装
            prefix = f'model_{model_name}'
            tft_df = pd.DataFrame({
                'time_idx': metrics['time_idx'].values,
                f'{prefix}_residual_ratio': sig_residual,
                f'{prefix}_att_kl': np.clip(sig_att_kl, 0, 20),
                f'{prefix}_vsn_js': np.clip(sig_vsn_js, 0, 5),
                f'{prefix}_vsn_rank_shift': np.clip(sig_vsn_rank, 0, 2),
            }).drop_duplicates(subset='time_idx', keep='last')
            
            # merge (防膨胀)
            n_before = len(df_merged)
            df_merged = df_merged.merge(tft_df, on='time_idx', how='left')
            if len(df_merged) != n_before:
                # 回退, 使用map方式
                new_cols = [c for c in tft_df.columns if c != 'time_idx']
                df_merged = df_merged.drop(columns=new_cols).iloc[:n_before]
                for col in new_cols:
                    val_map = dict(zip(tft_df['time_idx'], tft_df[col]))
                    df_merged[col] = df_merged['time_idx'].map(val_map)
        
        # --- Rolling特征 ---
        base_tft_cols = [c for c in df_merged.columns
                         if c.startswith('model_') and 'roll' not in c]
        for col in base_tft_cols:
            for k in [4, 8]:
                df_merged[f'{col}_rollmax_{k}'] = (
                    df_merged[col].rolling(k, min_periods=1).max())
                df_merged[f'{col}_rollmean_{k}'] = (
                    df_merged[col].rolling(k, min_periods=1).mean())
        
        df_merged = df_merged.replace([np.inf, -np.inf], 0).fillna(0)
        
        # 只保留model_*列 + time_idx + timestamp
        keep_cols = ['time_idx', 'timestamp'] + [
            c for c in df_merged.columns if c.startswith('model_')]
        df_merged = df_merged[[c for c in keep_cols if c in df_merged.columns]]
        
        # 筛选输出行
        if output_tidx_set is not None:
            df_merged = df_merged[df_merged['time_idx'].isin(output_tidx_set)]
        
        return df_merged
    
    # ================================================================
    # 工具
    # ================================================================
    def _print_summary(self, df_final):
        """打印运行总结"""
        print(f"\n{'='*60}")
        print(f"  自适应滚动检测 — 运行总结")
        print(f"{'='*60}")
        if len(df_final) > 0:
            vc = df_final['pred_label'].value_counts()
            print(f"  总窗口: {len(df_final)}")
            print(f"  N={vc.get('N',0)}, AP={vc.get('AP',0)}, CP={vc.get('CP',0)}")
        print(f"  最终baseline版本: {self.baseline_version}")
        print(f"  CP事件: {len(self.cp_events)}")
        for i, evt in enumerate(self.cp_events):
            print(f"    #{i}: confirmed@{evt['confirmed_tidx']}, "
                  f"retrace@{evt['retrace_tidx']}, "
                  f"switched@{evt.get('switched_tidx', '未切换')}")
        print(f"{'='*60}")
    
    def save_state(self, path="./checkpoints/adaptive_state"):
        """持久化当前状态"""
        os.makedirs(path, exist_ok=True)
        state = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_events': self.cp_events,
            'cooldown_until_tidx': self.cooldown_until_tidx,
        }
        with open(os.path.join(path, 'state.json'), 'w') as f:
            import json
            json.dump(state, f, indent=2, default=str)
        
        self.engine.save(os.path.join(path, f'engine_v{self.baseline_version}'))
        with open(os.path.join(path, f'processor_v{self.baseline_version}.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(path, f'classifier2_v{self.baseline_version}.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        print(f"✅ 状态保存到 {path}")

## 运行V4

In [ ]:
# ============================================================
# Cell: 自适应滚动检测 — 完整运行
# ============================================================
import types, io, contextlib, copy, os, pickle
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pytorch_forecasting import TimeSeriesDataSet
from TFT_tft_engine import TFTEngine
from target_builder import TargetBuilder
# from adaptive_rolling_detector import StatisticalClassifier2, AdaptiveRollingDetector
import logging

# 关闭 PyTorch Lightning 的硬件与环境提示
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

# 关闭 数据处理器与 TargetBuilder 的常规INFO日志
logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
logging.getLogger("TargetBuilder").setLevel(logging.WARNING)
# 如果有其他自定义 logger，也可以一并设置为 WARNING
# =====================================================================
# 0. 加载预训练系统
# =====================================================================
CKPT = "./checkpoints/pretrained_planPC"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine_PC = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)

# 去掉末尾数据
df_features = df_features[df_features['timestamp']<pd.to_datetime('2025-11-13')].copy()


# 绑定去重补丁（与Cell 1相同）
def analyze_rolling_patched(self, model_name, df, baseline_end_idx=None, predict=True):
    model = self.models[model_name]
    train_dataset = self.datasets[model_name]
    df_inf = df.copy()
    for col in self.known_categoricals:
        df_inf[col] = df_inf[col].astype(str)
    try:
        inference_dataset = TimeSeriesDataSet.from_dataset(
            train_dataset, df_inf, predict=False, stop_randomization=True)
    except ValueError as e:
        if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
            return {}
        raise
    if len(inference_dataset) == 0:
        return {}
    dataloader = inference_dataset.to_dataloader(
        train=False, batch_size=512, num_workers=0, pin_memory=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()
    result_buffer = {}
    with torch.no_grad():
        for x, y in dataloader:
            x = {k: v.to(device) for k, v in x.items()}
            targets = y[0].cpu().numpy().flatten()
            raw_out = model(x)
            preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
            p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
            time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
            interp = model.interpret_output(raw_out, reduction="none")
            batch_att = interp['attention'].cpu().numpy()
            batch_vsn = interp['encoder_variables'].cpu().numpy()
            for i in range(len(time_idx)):
                t_id = int(time_idx[i])
                result_buffer[t_id] = {
                    "target_true": targets[i], "pred_p50": p50[i],
                    "residual": targets[i] - p50[i], "divergence": p90[i] - p10[i],
                    "attention": batch_att[i], "vsn": batch_vsn[i]}
    if not result_buffer:
        return {}
    sorted_t = sorted(result_buffer.keys())
    df_s = pd.DataFrame({
        "time_idx": sorted_t,
        "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
        "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
        "residual": [result_buffer[t]["residual"] for t in sorted_t],
        "divergence": [result_buffer[t]["divergence"] for t in sorted_t]})
    full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
    full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
    result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
    if baseline_end_idx is not None:
        bl = self._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
        result["baseline"] = bl
        self.baselines[model_name] = bl
    return result

engine_PC.analyze_rolling = types.MethodType(analyze_rolling_patched, engine_PC)

# =====================================================================
# 1. 数据预处理（与滚动训练Cell 1一致）
# =====================================================================
QUIET_START = pd.to_datetime('2025-02-18 00:00:00')
QUIET_END   = pd.to_datetime('2025-02-25 00:00:00')

# 用平静期refit processor
mask_quiet = (df_features['timestamp'] >= QUIET_START) & (df_features['timestamp'] < QUIET_END)
df_quiet_raw = df_features[mask_quiet].copy().fillna(0)

processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_quiet_raw)
print(f"[Processor] re-fitted, global_min_time={processor_real.global_min_time}")

# 变换全量数据
dataset_real = processor_real.transform(df_features.fillna(0), df_official_real)
df_real_all = dataset_real.data.copy()
df_real_all.fillna(0, inplace=True)
df_real_all.replace([np.inf, -np.inf], 0, inplace=True)
df_real_all = tb.transform(df_real_all)
if 'label' not in df_real_all.columns:
    df_real_all['label'] = 'N'
df_real_all = df_real_all.sort_values('time_idx').reset_index(drop=True)
print(f"[Data] {df_real_all.shape}, {df_real_all['timestamp'].min()} ~ {df_real_all['timestamp'].max()}")

# =====================================================================
# 2. 构建分类器2 & 自适应检测器
# =====================================================================
classifier4 = StatisticalClassifier4(
    ap_k=5.0, 
    cp_base_k=3.0, 
    cp_density_thresh=0.95,
    cp_window=288, 
    cp_min_periods=96, 
    ap_min_triggers=4,
    cp_trend_window=2688,# 长程漂移参数
    cp_trend_k=2.0
)

detector4 = AdaptiveRollingDetector4(
    engine=engine_PC,
    tb=tb,
    processor=processor_real,
    classifier2=classifier4,
    # 滚动
    buffer_size=672, step_size=96,min_required=None,
    # CP确认
    cp_confirm_windows=192,

    # 回溯与积累
    retrace_windows=288, accumulate_min=1344,# 增长数据收集期
    # 微调
    finetune_epochs=100, finetune_lr_scale=0.005, finetune_batch_size=32, held_out_days=2,
    # 切换
    switch_parallel_windows=288,
    switch_parallel_extend=480,

    switch_recovery_threshold=0.85,
    switch_stability_threshold=0.85,
    # 冷却
    cooldown_windows=672,ap_warmup_windows=96, ap_warmup_relax=1.2,
    inference_batch_size=256
)

# =====================================================================
# 3. 运行
# =====================================================================
df_result4 = detector4.run(
    df_real_all=df_real_all,
    df_features_raw=df_features.fillna(0),
    df_official=df_official_real,
    quiet_start=QUIET_START,
    quiet_end=QUIET_END,
    output_path="./output/adaptive_detection_resultv4.csv"
)
# 保存状态
detector4.save_state("./checkpoints/adaptive_state")

print(f"\n🎯 检测完成, 结果: {len(df_result4)} 窗口")


# 检测V5

In [ ]:
"""
自适应滚动检测器：
  滚动TFT推理 → 分类器2判定 → CP确认 → 回溯288窗口 → 积累微调 → 分层切换
  
  状态机：NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
"""

import os, copy, pickle, tempfile, io, contextlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import spearmanr
# 如果有其他自定义 logger，也可以一并设置为 WARNING
# =====================================================================
# 分类器2: 纯统计阈值 AP/CP 判定
# =====================================================================
class StatisticalClassifier5:
    """
    基于冷启动期校准的多维Z-score分类器。
    输入: 滚动TFT推理输出的 model_* 信号列
    输出: 每个窗口的 pred_label (N/AP/CP) + 投票归因信息
    """
    

    import logging

    # 关闭 PyTorch Lightning 的硬件与环境提示
    logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

    # 关闭 数据处理器与 TargetBuilder 的常规INFO日志
    logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
    logging.getLogger("TargetBuilder").setLevel(logging.WARNING)
    # 8个AP候选特征
    
    AP_ROLL_FEATURES = [
        'model_model_comment_pc1_residual_ratio_rollmax_8',
        'model_model_post_pc2_residual_ratio_rollmax_8',
        'model_model_comment_pc1_vsn_rank_shift_rollmax_8',
        'model_model_post_pc2_vsn_rank_shift_rollmax_8',
        'model_model_comment_pc1_att_kl_rollmax_8',
        'model_model_post_pc2_att_kl_rollmax_8',
        'model_model_comment_pc1_vsn_js_rollmax_8',
        'model_model_post_pc2_vsn_js_rollmax_8',
    ]
    
    # 4个CP静态特征
    CP_STATIC_FEATURES = [
        'model_model_comment_pc1_vsn_rank_shift',
        'model_model_post_pc2_vsn_rank_shift',
        'model_model_comment_pc1_att_kl',
        'model_model_post_pc2_att_kl',
    ]
    
    def __init__(self, 
                 ap_k=5.0, 
                 ap_min_triggers=4,
                 cp_base_k=3.0, 
                 cp_density_thresh=0.95,
                 cp_window=288, 
                 cp_min_periods=96, 
                 # 新增：长程缓变漂移参数
                 cp_trend_window=2688, 
                 cp_trend_k=2.0):
        self.ap_k = ap_k
        self.cp_base_k = cp_base_k
        self.cp_density_thresh = cp_density_thresh
        self.cp_window = cp_window
        self.cp_min_periods = cp_min_periods
        self.ap_min_triggers = ap_min_triggers
        
        self.cp_trend_window = cp_trend_window
        self.cp_trend_k = cp_trend_k
        
        self.ap_thresholds = {}
        self.cp_base_thresh = None
        self.cp_vol_thresh = None
        self.cp_trend_thresh = None # 新增
        self._z_params = {} 
        self._calibrated = False
        self._ap_relax_factor = 1.0
    
    def calibrate(self, df_base):
        """用冷启动期数据校准所有阈值"""
        for feat in self.AP_ROLL_FEATURES:
            if feat in df_base.columns:
                self.ap_thresholds[feat] = df_base[feat].mean() + self.ap_k * df_base[feat].std()
        
        # 设定最小方差线
        # min_std_floors = {
        #     'model_model_comment_pc1_vsn_rank_shift': 0.05,
        #     'model_model_post_pc2_vsn_rank_shift': 0.05,
        #     'model_model_comment_pc1_att_kl': 0.1,
        #     'model_model_post_pc2_att_kl': 0.1,
        #     'model_model_comment_pc1_vsn_js': 0.02,
        #     'model_model_post_pc2_vsn_js': 0.02
        # }
        # for feat in self.CP_STATIC_FEATURES:
        #     if feat in df_base.columns:
        #         actual_std = df_base[feat].std()
        #         floor_std = min_std_floors.get(feat, 1e-4)
        #         if actual_std < floor_std:
        #             print(f"⚠️  {feat} 的实际std={actual_std:.6f}低于最小阈值{floor_std}, 已调整为{floor_std}")
        #         safe_std = max(actual_std, floor_std)
        #         self._z_params[feat] = {'mean': df_base[feat].mean(), 'std': safe_std}

        for feat in self.CP_STATIC_FEATURES:
            if feat in df_base.columns:
                self._z_params[feat] = {'mean': df_base[feat].mean(), 'std': df_base[feat].std() + 1e-8}
        
        z_max_vals = self._compute_z_max(df_base)
        self.cp_base_thresh = z_max_vals.mean() + self.cp_base_k * z_max_vals.std()
        self.cp_vol_thresh = z_max_vals.std() * 1.5
        self.cp_trend_thresh = z_max_vals.mean() + self.cp_trend_k * z_max_vals.std() # 新增：长程漂移阈值
        
        
        
        self._calibrated = True
        print(f"[Classifier3] 校准完成: AP阈值={len(self.ap_thresholds)}个, "
              f"突变水位={self.cp_base_thresh:.3f}, 漂移阈值={self.cp_trend_thresh:.3f}")
        
    def set_ap_relax_factor(self, factor):
        """设置AP阈值放宽因子"""
        self._ap_relax_factor = factor
    
    def _compute_z_max(self, df):
        """计算各行的多维Z-score最大值"""
        z_vals = []
        for feat, params in self._z_params.items():
            if feat in df.columns:
                z = (df[feat] - params['mean']) / params['std']
                z_vals.append(z)
        if z_vals:
            return pd.concat(z_vals, axis=1).max(axis=1)
        return pd.Series(0.0, index=df.index)
    
    def predict(self, df):
        """
        执行统计检验与分类判定，并计算信号归因时间点。
        """
        assert self._calibrated, "必须先调用 calibrate()"
        df = df.copy()
        
        # --- CP辅助信号 ---
        z_max = self._compute_z_max(df)
        df['static_signal_z_max'] = z_max
        is_high = (z_max > self.cp_base_thresh).astype(float)
        df['cp_density'] = is_high.rolling(self.cp_window, min_periods=self.cp_min_periods).mean()
        df['cp_volatility'] = z_max.rolling(self.cp_window, min_periods=self.cp_min_periods).std()
        
        # 新增：长程缓变均值与漂移判定 (独立特征)
        df['cp_trend_mean'] = z_max.rolling(self.cp_trend_window, min_periods=self.cp_window).mean()
        df['is_baseline_drift'] = df['cp_trend_mean'] > self.cp_trend_thresh
        
        # --- AP触发计数 ---
        trigger_matrix = pd.DataFrame(index=df.index)
        for feat in self.AP_ROLL_FEATURES:
            if feat in df.columns and feat in self.ap_thresholds:   
                effective_thresh = self.ap_thresholds[feat] * self._ap_relax_factor
                trigger_matrix[feat] = (df[feat] > effective_thresh).astype(int)
        df['ap_trigger_count'] = trigger_matrix.sum(axis=1)

        # --- 判定 (仅限 AP 和 Sudden CP) ---
        df['pred_label'] = 'N'
        
        mask_cp = (df['cp_density'] > self.cp_density_thresh) & (df['cp_volatility'] < self.cp_vol_thresh)
        df.loc[mask_cp, 'pred_label'] = 'CP'
        
        mask_ap = (df['ap_trigger_count'] >= self.ap_min_triggers) & (~mask_cp)
        df.loc[mask_ap, 'pred_label'] = 'AP'
        
        # --- 归因时间点定位 (Attribution Time) ---
        df['attribution_time'] = pd.NaT
        # 1. 点异常 (PA): 零滞后，归因为当前时间
        df.loc[mask_ap, 'attribution_time'] = df.loc[mask_ap, 'timestamp']
        # 2. 集体异常 (CA): 存在窗口滞后，向左回溯冲击起始点
        ca_attr_times = df['timestamp'].shift(self.cp_window)
        df.loc[mask_cp, 'attribution_time'] = ca_attr_times.loc[mask_cp]
        # 3. 概念漂移 (CP): 存在严重平滑滞后，向左回溯漂移起始分布点
        cp_attr_times = df['timestamp'].shift(1344) 
        drift_mask = df['is_baseline_drift'] == True
        df.loc[drift_mask, 'attribution_time'] = cp_attr_times.loc[drift_mask]

        # --- 空间投票归因 ---
        df['dominant_model'] = 'N/A'
        df['dominant_signal'] = 'N/A'
        df['trigger_features'] = ''
        
        anom_mask = df['pred_label'] != 'N'
        if anom_mask.any():

            for idx in df[anom_mask].index:
                triggers = [f for f in trigger_matrix.columns 
                           if trigger_matrix.loc[idx, f] == 1]
                if not triggers:
                    continue


                # 子模型投票
                comment_v = sum(1 for f in triggers if 'comment_pc1' in f)
                post_v = sum(1 for f in triggers if 'post_pc2' in f)
                dom_model = 'comment_pc1' if comment_v >= post_v else 'post_pc2'
                
                # 信号类型投票
                signal_votes = {}
                for sig_type in ['residual_ratio', 'vsn_rank_shift', 'att_kl', 'vsn_js']:
                    signal_votes[sig_type] = sum(1 for f in triggers if sig_type in f)
                dom_signal = max(signal_votes, key=signal_votes.get)
                
                df.loc[idx, 'dominant_model'] = dom_model
                df.loc[idx, 'dominant_signal'] = dom_signal
                df.loc[idx, 'trigger_features'] = '|'.join(triggers)
        
        return df



## 检测器V5

In [ ]:

# =====================================================================
# 自适应滚动检测器
# =====================================================================
class AdaptiveRollingDetector5:
    """
    状态机:
      NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
    
    每个STEP_SIZE窗口执行一次批量TFT推理 + 分类器2判定，
    检测到CP后回溯288窗口，积累到576窗口后微调TFT，分层切换。
    """
    
    # 状态常量
    NORMAL = 'NORMAL'
    CP_TENTATIVE = 'CP_TENTATIVE'
    ACCUMULATING = 'ACCUMULATING'
    SWITCHING = 'SWITCHING'
    
    def __init__(self, engine, tb, processor, classifier2,
                 # 滚动参数
                 buffer_size=672, step_size=96, min_required=None,
                 # CP确认
                 cp_confirm_windows=192,
                 # 回溯与积累
                 retrace_windows=288,
                 accumulate_min=1344,
                 # 微调
                 finetune_epochs=100, 
                 finetune_lr_scale=0.005,
                 finetune_batch_size = 32,
                 finetune_grad_clip=0.5,
                 held_out_days=2,
                 held_out_min_windows=96,
                # 切换参数（修改此部分）
                 switch_parallel_windows=288,
                 switch_parallel_extend=480,
                 switch_recovery_threshold=0.85,  # 新增：旧模型恢复正常的N标签比例阈值
                 switch_stability_threshold=0.85,
                 # 冷却
                 cooldown_windows=672,
                 # AP阈值保护
                 ap_warmup_windows=96, 
                 ap_warmup_relax=1.2,
                 inference_batch_size=256):
        
        self.engine = engine
        self.tb = tb
        self.processor = processor
        self.classifier2 = classifier2
        
        # 滚动参数
        self.buffer_size = buffer_size
        self.step_size = step_size
        ds_0 = list(engine.datasets.values())[0]
        self.min_encoder = ds_0.max_encoder_length
        self.min_pred = ds_0.max_prediction_length
        self.min_required = min_required or (self.min_encoder + self.min_pred + 1)
        
        # CP确认
        self.cp_confirm_windows = cp_confirm_windows
        
        # 回溯与积累
        self.retrace_windows = retrace_windows
        self.accumulate_min = accumulate_min
        
        # 微调
        self.finetune_epochs = finetune_epochs
        self.finetune_lr_scale = finetune_lr_scale
        self.finetune_batch_size = finetune_batch_size
        self.finetune_grad_clip = finetune_grad_clip
        self.held_out_days = held_out_days
        self.held_out_min_windows = held_out_min_windows
        
        # P3修复：切换
        self.switch_parallel_windows = switch_parallel_windows
        self.switch_parallel_extend = switch_parallel_extend
        # self.switch_agree_high = switch_agree_high
        # self.switch_agree_low = switch_agree_low
        self.switch_recovery_threshold =  switch_recovery_threshold  # 新增：旧模型恢复正常的N标签比例阈值
        self.switch_stability_threshold = switch_stability_threshold
        
        # P7修复：冷却与保护
        self.cooldown_windows = cooldown_windows
        self.ap_warmup_windows = ap_warmup_windows
        self.ap_warmup_relax = ap_warmup_relax
        
        # 推理配置
        self.inference_batch_size = inference_batch_size
        
        # --- 状态变量 ---
        self.state = self.NORMAL
        self.baseline_version = 0
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self.cooldown_until_tidx = -1
        
        # P3修复：并行切换状态
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False
        
        # P7修复：AP warmup状态
        self._ap_warmup_remaining = 0
        
        # 历史记录
        self.cp_events = []
        self.switch_history = []
        self.last_baseline_update_tidx = -999999
        
    # ================================================================
    # P8修复：状态持久化与恢复
    # ================================================================
    def save_state(self, path="./checkpoints/adaptive_state"):
        """持久化当前完整状态"""
        os.makedirs(path, exist_ok=True)
        
        state_dict = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_confirmed_tidx': self.cp_confirmed_tidx,
            'retrace_start_tidx': self.retrace_start_tidx,
            'accumulate_buffer_tidx': self.accumulate_buffer_tidx,
            'cooldown_until_tidx': self.cooldown_until_tidx,
            'ap_warmup_remaining': self._ap_warmup_remaining,
            'parallel_count': self._parallel_count,
            'switch_extended': self._switch_extended,
            'cp_events': self.cp_events,
            'switch_history': self.switch_history,
        }
        
        with open(os.path.join(path, 'state.json'), 'w') as f:
            json.dump(state_dict, f, indent=2, default=str)
        
        # 保存当前版本组件
        version_path = os.path.join(path, f'v{self.baseline_version}')
        os.makedirs(version_path, exist_ok=True)
        
        self.engine.save(os.path.join(version_path, 'engine'))
        with open(os.path.join(version_path, 'processor.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(version_path, 'classifier2.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        # 如果正在切换，保存待切换组件
        if self.state == self.SWITCHING and self._new_engine is not None:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            os.makedirs(new_path, exist_ok=True)
            self._new_engine.save(os.path.join(new_path, 'engine'))
            with open(os.path.join(new_path, 'processor.pkl'), 'wb') as f:
                pickle.dump(self._new_processor, f)
            with open(os.path.join(new_path, 'classifier2.pkl'), 'wb') as f:
                pickle.dump(self._new_classifier2, f)
        
        print(f"✅ 状态已保存到 {path}")
    
    def load_state(self, path="./checkpoints/adaptive_state"):
        """从持久化文件恢复状态"""
        state_file = os.path.join(path, 'state.json')
        if not os.path.exists(state_file):
            print(f"⚠️ 未找到状态文件: {state_file}")
            return False
        
        with open(state_file, 'r') as f:
            state_dict = json.load(f)
        
        # 恢复状态变量
        self.state = state_dict['state']
        self.baseline_version = state_dict['baseline_version']
        self.cp_consec = state_dict['cp_consec']
        self.cp_confirmed_tidx = state_dict.get('cp_confirmed_tidx')
        self.retrace_start_tidx = state_dict.get('retrace_start_tidx')
        self.accumulate_buffer_tidx = state_dict.get('accumulate_buffer_tidx', [])
        self.cooldown_until_tidx = state_dict.get('cooldown_until_tidx', -1)
        self._ap_warmup_remaining = state_dict.get('ap_warmup_remaining', 0)
        self._parallel_count = state_dict.get('parallel_count', 0)
        self._switch_extended = state_dict.get('switch_extended', False)
        self.cp_events = state_dict.get('cp_events', [])
        self.switch_history = state_dict.get('switch_history', [])
        
        # 恢复组件
        version_path = os.path.join(path, f'v{self.baseline_version}')
        if os.path.exists(version_path):
            from TFT_tft_engine import TFTEngine
            self.engine = TFTEngine.load(
                os.path.join(version_path, 'engine'), 
                config_path='TFT_config.yaml')
            with open(os.path.join(version_path, 'processor.pkl'), 'rb') as f:
                self.processor = pickle.load(f)
            with open(os.path.join(version_path, 'classifier2.pkl'), 'rb') as f:
                self.classifier2 = pickle.load(f)
        
        # 恢复待切换组件
        if self.state == self.SWITCHING:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            if os.path.exists(new_path):
                self._new_engine = TFTEngine.load(
                    os.path.join(new_path, 'engine'),
                    config_path='TFT_config.yaml')
                with open(os.path.join(new_path, 'processor.pkl'), 'rb') as f:
                    self._new_processor = pickle.load(f)
                with open(os.path.join(new_path, 'classifier2.pkl'), 'rb') as f:
                    self._new_classifier2 = pickle.load(f)
        
        print(f"✅ 状态已恢复: state={self.state}, version={self.baseline_version}")
        return True
       
    # ================================================================
    # 主入口: run()
    # ================================================================
    def run(self, df_real_all, df_features_raw, df_official,
            quiet_start=None, quiet_end=None, output_path=None, resume=False):
        """
        完整的自适应滚动检测流程。
        
        Parameters
        ----------
        df_real_all : 经processor+tb变换后的完整数据（含time_idx, timestamp, PCA列等）
        df_features_raw : 原始特征数据（用于refit processor）
        df_official : 官方运营日历
        quiet_start, quiet_end : 平静期时间范围（用于初始calibrate）
        output_path : 保存路径
        resume : bool
            是否从持久化状态恢复继续运行
        Returns
        -------
        pd.DataFrame : 每个窗口的检测结果
        """
        df = df_real_all.sort_values('time_idx').reset_index(drop=True)
        
        # P8修复：尝试恢复
        if resume and os.path.exists("./checkpoints/adaptive_state/state.json"):
            if self.load_state("./checkpoints/adaptive_state"):
                print("[Resume] 从保存点继续...")

        # --- 0. 冷启动校准 ---
        if quiet_start and quiet_end:
            quiet_mask = (df['timestamp'] >= quiet_start) & (df['timestamp'] < quiet_end)
            quiet_start_idx = df[df['timestamp'] >= quiet_start].index[0]
        else:
            # 默认前7天
            quiet_end_ts = df['timestamp'].min() + pd.Timedelta(days=7)
            quiet_mask = df['timestamp'] < quiet_end_ts
            quiet_start_idx = 0
        
        # 建立TFT baseline
        pre_ctx = max(0, quiet_start_idx - self.min_required)
        bl_end = df[quiet_mask].index[-1] + 1
        df_for_bl = df.iloc[pre_ctx:bl_end].copy().reset_index(drop=True)
        bl_end_tidx = int(df.loc[quiet_mask, 'time_idx'].max())
        
        print(f"[Init] 建立TFT baseline ({len(df_for_bl)} 窗口)...")
        self._patch_engine_batch_size(self.engine)

        for mn in self.engine.models:
            res = self.engine.analyze_rolling(mn, df_for_bl, baseline_end_idx=bl_end_tidx)
            if 'baseline' not in res and 'metrics' in res:
                bl = self.engine._build_baseline(
                    res['metrics'], res['attention'], res['vsn'], bl_end_tidx)
                self.engine.baselines[mn] = bl
            print(f"  [{mn}] baseline: σ={self.engine.baselines[mn]['residual_std']:.4f}")
        
        # 校准分类器2
        # 先对平静期做一次TFT推理获取model_*列
        df_quiet_for_cal = self._extract_tft_batch(df_for_bl, engine=self.engine)
        if len(df_quiet_for_cal) > 0:
            # 只取平静期时间范围内的行
            cal_mask = df_quiet_for_cal['time_idx'] <= bl_end_tidx
            self.classifier2.calibrate(df_quiet_for_cal[cal_mask])
        else:
            print("  ⚠️ 平静期TFT推理无输出, 用原始数据校准")
            self.classifier2.calibrate(df[quiet_mask])
        
        # --- 1. 初始化缓冲区 ---
        initial_end = min(quiet_start_idx + self.buffer_size, len(df))
        buffer_df = df.iloc[pre_ctx:initial_end].copy().reset_index(drop=True)
        
        quiet_start_tidx = int(df.loc[quiet_mask, 'time_idx'].min())
        output_tidx = set(
            buffer_df.loc[buffer_df['time_idx'] >= quiet_start_tidx, 'time_idx']
            .astype(int).values)
        
        print(f"\n[ColdStart] 缓冲区: {len(buffer_df)} 窗口, 推理中...")
        df_cold = self._extract_tft_batch(buffer_df, engine=self.engine,
                                          output_tidx_set=output_tidx)
        # 建立分类结果快照列表，规避事后全局 predict 导致的时空信息错位
        all_classified_results = []
        # all_tft_results = [df_cold] if len(df_cold) > 0 else []
        processed_tidx = set(df_cold['time_idx'].astype(int).values) if len(df_cold) > 0 else set()
    
        
        # --- 2. 滚动主循环 ---
        remaining_start = initial_end
        total_remaining = len(df) - remaining_start
        n_batches = (total_remaining + self.step_size - 1) // self.step_size
        
        print(f"\n[Rolling] {total_remaining} 窗口, {n_batches} 批 (step={self.step_size})")
        # 建立全局长周期连续缓冲区，提供长程 rolling 计算所需的时序上下文
        accumulated_tft_features = df_cold.copy() if len(df_cold) > 0 else pd.DataFrame()

        for batch_i in tqdm(range(n_batches), desc="自适应滚动检测"):
            batch_start = remaining_start + batch_i * self.step_size
            batch_end = min(batch_start + self.step_size, len(df))
            
            if batch_start >= len(df):
                break
            
            new_rows = df.iloc[batch_start:batch_end]
            new_tidx = set(new_rows['time_idx'].astype(int).values) - processed_tidx
            
            if not new_tidx:
                continue
            
            # 更新缓冲区
            buffer_df = pd.concat([buffer_df, new_rows], ignore_index=True)
            if len(buffer_df) > self.buffer_size:
                buffer_df = buffer_df.iloc[-self.buffer_size:].reset_index(drop=True)
            
            if len(buffer_df) < self.min_required:
                continue
            # P7修复：AP warmup阈值放宽
            if self._ap_warmup_remaining > 0:
                self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
                self._ap_warmup_remaining -= len(new_tidx)
                if self._ap_warmup_remaining <= 0:
                    self.classifier2.set_ap_relax_factor(1.0)
                    print(f"\n  [AP Warmup] 结束")
            # P3修复：并行验证时需要两个引擎都推理
            if self.state == self.SWITCHING and self._new_engine is not None:
                df_feat_old = self._extract_tft_batch(
                    buffer_df, engine=self.engine, output_tidx_set=new_tidx)
                df_feat_new = self._extract_tft_batch(
                    buffer_df, engine=self._new_engine, output_tidx_set=new_tidx)
                
                # 主输出用旧引擎（保守）
                df_feat = df_feat_old
                
                # 收集预测结果用于对比
                if len(df_feat_old) > 0 and len(df_feat_new) > 0:
                    pred_old = self.classifier2.predict(df_feat_old)['pred_label'].values
                    pred_new = self._new_classifier2.predict(df_feat_new)['pred_label'].values
                    
                    self._parallel_preds_old.extend(pred_old.tolist())
                    self._parallel_preds_new.extend(pred_new.tolist())
                    self._parallel_count += len(pred_old)
            else:
                active_engine = self.engine
                df_feat = self._extract_tft_batch(
                    buffer_df, engine=active_engine, output_tidx_set=new_tidx)   
            
            if len(df_feat) == 0:
                continue
            
            # all_tft_results.append(df_feat)
            processed_tidx.update(df_feat['time_idx'].astype(int).values)


            # --- 修复：维护特征缓冲区以支持长程 Rolling 计算 ---
            accumulated_tft_features = pd.concat([accumulated_tft_features, df_feat], ignore_index=True)
            
            # 保持缓冲区长度足够覆盖长程趋势窗口 (2688) + 容余量
            max_history_needed = self.classifier2.cp_trend_window + self.step_size + 100
            if len(accumulated_tft_features) > max_history_needed:
                accumulated_tft_features = accumulated_tft_features.iloc[-max_history_needed:].reset_index(drop=True)
            
            # 对包含历史上下文的数据统一执行分类判定
            df_classified_all = self.classifier2.predict(accumulated_tft_features)
            
            # 剥离历史数据，仅将当前处理批次记录至快照，同步绑定此时的状态机版本
            df_classified = df_classified_all[df_classified_all['time_idx'].isin(new_tidx)].copy()
            df_classified['baseline_version'] = self.baseline_version
            df_classified['state'] = self.state
            all_classified_results.append(df_classified)
            # 状态机处理
            current_tidx = int(new_rows['time_idx'].iloc[-1])
            self._process_batch(
                df_classified, current_tidx, df, df_features_raw, df_official)     
                   
            # --- 每日日志打印 (step_size=96) ---
            ts_date = new_rows['timestamp'].iloc[-1].strftime('%Y-%m-%d')
            n_ap = (df_classified['pred_label'] == 'AP').sum()
            n_cp = (df_classified['pred_label'] == 'CP').sum()
            drift_flag = df_classified['is_baseline_drift'].any()
            
            print(f"  [{ts_date}] state={self.state:<12} | v={self.baseline_version} | "
                  f"AP={n_ap:<2} CP={n_cp:<2} Drift={drift_flag}")
            
            # 日志
            # if (batch_i + 1) % 20 == 0:
            #     ts = new_rows['timestamp'].iloc[-1]
            #     n_ap = (df_classified['pred_label'] == 'AP').sum()
            #     n_cp = (df_classified['pred_label'] == 'CP').sum()
            #     print(f"  Batch {batch_i+1}/{n_batches} | {ts} | "
            #           f"state={self.state} | v={self.baseline_version} | "
            #           f"AP={n_ap} CP={n_cp} warmup={self._ap_warmup_remaining}")
        
        # --- 3. 合并所有输出 ---
        if all_classified_results:
            df_final = pd.concat(all_classified_results, ignore_index=True)
            df_final = df_final.drop_duplicates(subset='time_idx', keep='last')
            df_final = df_final.sort_values('time_idx').reset_index(drop=True)
        else:
            df_final = pd.DataFrame()

        
        # 保存
        if output_path and len(df_final) > 0:
            os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
            df_final.to_csv(output_path, index=False)
            print(f"✅ 保存到 {output_path}")
        
        self._print_summary(df_final)
        self.save_state()

        return df_final
    
    # ================================================================
    # 状态机处理逻辑
    # ================================================================
    def _process_batch(self, df_classified, current_tidx,
                       df_full, df_features_raw, df_official):
        """双轨状态机更新"""
        
        labels = df_classified['pred_label'].values
        is_drift = df_classified['is_baseline_drift'].values
        
        # 1. 如果正在验证期，仅执行并行逻辑
        if self.state == self.SWITCHING:
            self._check_parallel_switch(current_tidx)
            return

        # 2. 冷却期全局拦截
        if current_tidx < self.cooldown_until_tidx:
            return

        # =======================================================
        # 轨道 2：长程缓变漂移 (Incremental CP) - 最高优先级
        # =======================================================
        # 基线排空锁：确保距离最近一次彻底切换已历经完整的趋势窗口，以排除旧分布残留影响
        trend_window = self.classifier2.cp_trend_window
        is_trend_safe = (current_tidx - self.last_baseline_update_tidx) >= trend_window

        if is_drift.any() and is_trend_safe:
            print(f"\n  [→ DRIFT DETECTED] 监测到基线不可逆漂移 @ tidx={current_tidx}，立即向前回溯重构")
            
            # 向过去提取 14 天数据直接微调
            retrace_start = max(0, current_tidx - self.accumulate_min)
            retrace_mask = (df_full['time_idx'] >= retrace_start) & (df_full['time_idx'] <= current_tidx)
            df_retrace = df_full[retrace_mask].copy()
            
            ts_start = df_retrace['timestamp'].min()
            ts_end = df_retrace['timestamp'].max()
            raw_mask = (df_features_raw['timestamp'] >= ts_start) & (df_features_raw['timestamp'] <= ts_end)
            df_raw_retrace = df_features_raw[raw_mask].copy()
            
            success = self._finetune(df_retrace, df_raw_retrace, df_official)
            
            if success:
                self.state = self.SWITCHING
                self._parallel_preds_old = []
                self._parallel_preds_new = []
                self._parallel_count = 0
                self._switch_extended = False
                
                self.cp_events.append({
                    'confirmed_tidx': current_tidx,
                    'retrace_tidx': retrace_start,
                    'retrace_windows': self.accumulate_min,
                    'drift_type': 'Incremental',
                    'switched_tidx': None,
                    'switch_type': None,
                    'baseline_version': self.baseline_version,
                })
                print(f"  [→ SWITCHING] 漂移重构完成，进入并行验证 ({self.switch_parallel_windows} 窗口)")
            else:
                print("  [微调失败] 维持原有基线，触发冷却")
                self.state = self.NORMAL
                self._reset_partial_state()
                self.cooldown_until_tidx = current_tidx + self.cooldown_windows
            return # 轨道2触发后，中止本批次的后续逻辑

        # =======================================================
        # 轨道 1：突发业务冲击 (Sudden CP) - 延迟观察逻辑
        # =======================================================
        if self.state == self.NORMAL:
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
                if self.cp_consec >= 1:
                    self.state = self.CP_TENTATIVE
                    print(f"\n  [→ CP_TENTATIVE] 遭遇业务冲击 consec={self.cp_consec} @ tidx={current_tidx}")
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
        
        elif self.state == self.CP_TENTATIVE:
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
            
            # 要求连续 96 个窗口维持突发极值
            if self.cp_consec >= self.cp_confirm_windows:
                self.cp_confirmed_tidx = current_tidx
                self.retrace_start_tidx = current_tidx  # 从确认点开始向未来蓄水
                self.accumulate_buffer_tidx = []
                self.state = self.ACCUMULATING
                
                print(f"\n  [→ ACCUMULATING] 冲击确认 @ tidx={current_tidx}, 启动向未来蓄水 {self.accumulate_min} 窗口")
            
            if self.cp_consec <= 0:
                self.state = self.NORMAL
                self.cp_consec = 0
                print(f"\n  [→ NORMAL] 冲击平息，警报解除 @ tidx={current_tidx}")
                
        elif self.state == self.ACCUMULATING:
            batch_tidx = df_classified['time_idx'].astype(int).tolist()
            self.accumulate_buffer_tidx.extend(batch_tidx)
            accumulated = len(set(self.accumulate_buffer_tidx))
            
            print(f"  [ACCUMULATING] 蓄水观察中 {accumulated}/{self.accumulate_min} 窗口", end='\r')
            
            if accumulated >= self.accumulate_min:
                print(f"\n  [微调触发] 未来 14 天蓄水完毕，开始评估常态...")
                
                retrace_mask = (df_full['time_idx'] >= self.retrace_start_tidx) & (df_full['time_idx'] <= current_tidx)
                df_retrace = df_full[retrace_mask].copy()
                
                ts_start = df_retrace['timestamp'].min()
                ts_end = df_retrace['timestamp'].max()
                raw_mask = (df_features_raw['timestamp'] >= ts_start) & (df_features_raw['timestamp'] <= ts_end)
                df_raw_retrace = df_features_raw[raw_mask].copy()
                
                success = self._finetune(df_retrace, df_raw_retrace, df_official)
                
                if success:
                    self.state = self.SWITCHING
                    self._parallel_preds_old = []
                    self._parallel_preds_new = []
                    self._parallel_count = 0
                    self._switch_extended = False
                    
                    self.cp_events.append({
                        'confirmed_tidx': self.cp_confirmed_tidx,
                        'retrace_tidx': self.retrace_start_tidx,
                        'retrace_windows': self.accumulate_min,
                        'drift_type': 'Sudden',
                        'switched_tidx': None,
                        'switch_type': None,
                        'baseline_version': self.baseline_version,
                    })
                    print(f"  [→ SWITCHING] 备用模型完成，开始并行验证 ({self.switch_parallel_windows} 窗口)")
                else:
                    print("  [微调失败] 回退到NORMAL，触发冷却")
                    self.state = self.NORMAL
                    self._reset_partial_state()
                    self.cooldown_until_tidx = current_tidx + self.cooldown_windows
    # ================================================================
    # P3修复：完整的并行切换逻辑 V4
    # ================================================================
    def _check_parallel_switch(self, current_tidx):
        required_windows = (self.switch_parallel_extend 
                            if self._switch_extended 
                            else self.switch_parallel_windows)
        
        if self._parallel_count < required_windows:
            return
        
        pred_old = np.array(self._parallel_preds_old[-required_windows:])
        pred_new = np.array(self._parallel_preds_new[-required_windows:])
        
        if len(pred_old) == 0 or len(pred_new) == 0:
            return
        
        old_n_ratio = (pred_old == 'N').mean()
        new_n_ratio = (pred_new == 'N').mean()
        
        print(f"\n  [并行验证] windows={self._parallel_count}, "
              f"旧模型N占比={old_n_ratio:.3f}, 新模型N占比={new_n_ratio:.3f}")
        
        # 获取当前触发微调的事件类型
        current_drift_type = self.cp_events[-1].get('drift_type', 'Sudden') if self.cp_events else 'Sudden'
        
        if current_drift_type == 'Sudden':
            if old_n_ratio >= self.switch_recovery_threshold:
                print("  [放弃切换] 旧模型已恢复正常，微调冗余")
                self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
            elif new_n_ratio >= self.switch_stability_threshold and new_n_ratio >= old_n_ratio:
                print("  [触发切换] 突变稳态确认，新模型优于旧模型")
                self._execute_full_switch(current_tidx, agreement=0.0, kappa=0.0)
            else:
                self._handle_unstable_new_model(current_tidx, new_n_ratio)
                
        elif current_drift_type == 'Incremental':
            # 忽略旧模型表现，仅考核新模型绝对稳定性
            if new_n_ratio >= self.switch_stability_threshold:
                print("  [触发切换] 长程漂移重构有效，新模型达到稳态")
                self._execute_full_switch(current_tidx, agreement=0.0, kappa=0.0)
            else:
                self._handle_unstable_new_model(current_tidx, new_n_ratio)

    def _handle_unstable_new_model(self, current_tidx, new_n_ratio):
        if not self._switch_extended:
            self._switch_extended = True
            print(f"  [延长验证] 新模型N占比={new_n_ratio:.3f}，延长到 {self.switch_parallel_extend} 窗口")
        else:
            print(f"  [拒绝切换] 延长验证后新模型表现仍未占优")
            self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
    
    def _execute_full_switch(self, current_tidx, agreement, kappa):
        """完整切换：模型权重+baseline+阈值"""
        print(f"\n  [完整切换] agreement={agreement:.3f}, kappa={kappa:.3f}")
        
        self.engine = self._new_engine
        self.processor = self._new_processor
        self.classifier2 = self._new_classifier2
        
        self._finalize_switch(current_tidx, 'full', agreement, kappa)
    
    def _execute_partial_switch(self, current_tidx, agreement, kappa):
        """部分切换：仅baseline+阈值，保留旧模型权重"""
        print(f"\n  [部分切换] agreement={agreement:.3f}，保留旧模型权重")
        
        # 只更新processor和分类器阈值
        self.processor = self._new_processor
        
        # 更新分类器阈值
        self.classifier2.ap_thresholds = copy.deepcopy(self._new_classifier2.ap_thresholds)
        self.classifier2.cp_base_thresh = self._new_classifier2.cp_base_thresh
        self.classifier2.cp_vol_thresh = self._new_classifier2.cp_vol_thresh
        self.classifier2._z_params = copy.deepcopy(self._new_classifier2._z_params)
        
        # 更新baseline但保留旧模型
        for mn in self._new_engine.baselines:
            self.engine.baselines[mn] = self._new_engine.baselines[mn]
        
        self._finalize_switch(current_tidx, 'partial', agreement, kappa)
    
    def _finalize_switch(self, current_tidx, switch_type, agreement, kappa):
        """执行基线版本的迭代逻辑与底层参数重置"""
        if switch_type == 'full':
            self.baseline_version += 1
            # 记录此处的全局时间步，开启长程均值过滤器的安全保护
            self.last_baseline_update_tidx = current_tidx
        
        # self.baseline_version += 1
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        
        # P7修复：启动AP warmup
        self._ap_warmup_remaining = self.ap_warmup_windows
        self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
        
        # 记录历史
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
            self.cp_events[-1]['switch_type'] = switch_type
        
        self.switch_history.append({
            'tidx': current_tidx,
            'type': switch_type,
            'agreement': agreement,
            'kappa': kappa,
            'new_version': self.baseline_version,
        })
        
        # 重置状态
        self.state = self.NORMAL
        self._reset_partial_state()
        
        print(f"  [切换完成] type={switch_type}, version={self.baseline_version}, "
              f"冷却至 tidx={self.cooldown_until_tidx}, "
              f"AP warmup={self._ap_warmup_remaining}窗口")
    
    def _reset_partial_state(self):
        """重置切换相关状态"""
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False

    # ================================================================
    # 微调
    # ================================================================
    def _finetune(self, df_retrace, df_raw_retrace, df_official):
        """
        微调三件套: processor refit → TFT微调 → 分类器2阈值更新
        """
        from TFT_tft_engine import TFTEngine
        from tft_full_period_utils import compute_baseline_from_held_out, create_held_out_split
        
        # (a) Processor refit
        print("  [微调 1/3] Processor refit...")
        processor_new = copy.deepcopy(self.processor)
        processor_new.feature_transformer.fit(df_raw_retrace.fillna(0))
        
        # 用新processor变换数据
        dataset_new = processor_new.transform(df_raw_retrace.fillna(0), df_official)
        df_tft = dataset_new.data.copy()

        df_tft.fillna(0, inplace=True)
        df_tft.replace([np.inf, -np.inf], 0, inplace=True)
        df_tft = self.tb.transform(df_tft)
        if 'label' not in df_tft.columns:
            df_tft['label'] = 'N'
        
        print(f"  [微调数据] {len(df_tft)} 窗口")

        
        # (b) TFT微调
        print("  [微调 2/3] TFT模型微调...")
        with tempfile.TemporaryDirectory() as tmp:
            self.engine.save(tmp)
            engine_new = TFTEngine.load(tmp, config_path='TFT_config.yaml')
        
        self._patch_engine_batch_size(engine_new)
        self._patch_engine_early_stopping(engine_new)
        
        # Held-out切分
        held_out_windows = max(
            self.held_out_min_windows,
            int(len(df_tft) * 0.15)  # 最多15%
        )
        held_out_windows = min(held_out_windows, len(df_tft) // 3)

        split_idx = len(df_tft) - held_out_windows
        df_train_ft = df_tft.iloc[:split_idx].copy()
        df_held = df_tft.iloc[split_idx:].copy()
        held_out_tidx = set(df_held['time_idx'].values)

        print(f"  [数据切分] 训练={len(df_train_ft)}, held_out={len(df_held)}")
        
        if len(df_train_ft) < self.min_required * 2:
            print(f"  ⚠️ 训练数据不足")
            return False
        
        for mn in engine_new.models:
            # 解冻
            for p in engine_new.models[mn].parameters():
                p.requires_grad = True
            
            cfg = engine_new.config['tft_models'][mn]
            orig_lr = cfg.get('learning_rate', 0.03)
            cfg['learning_rate'] = orig_lr * self.finetune_lr_scale
            # 应用梯度裁剪
            if 'gradient_clip_val' not in cfg:
                cfg['gradient_clip_val'] = self.finetune_grad_clip

            print(f"    [{mn}] 微调 (lr={cfg['learning_rate']:.5f}, "
                  f"epochs={self.finetune_epochs}, data={len(df_train_ft)})...")
            
            with contextlib.redirect_stdout(io.StringIO()):
                engine_new.build_and_fit(
                    mn, df_train_ft, 
                    max_epochs=self.finetune_epochs)
            
            cfg['learning_rate'] = orig_lr
            
            # 重建baseline
            res = engine_new.analyze_rolling(mn, df_tft, baseline_end_idx=None)
            if 'metrics' not in res:
                print(f"    ⚠️ [{mn}] 推理失败")
                continue
            if held_out_tidx:
                bl = compute_baseline_from_held_out(res, held_out_tidx)
            else:
                bl = engine_new._build_baseline(
                    res['metrics'], res['attention'], res['vsn'],
                    int(df_tft['time_idx'].max()))
            engine_new.baselines[mn] = bl
            # engine_new.freeze_layers(mn)
            
            print(f"    [{mn}] 新baseline: σ={bl['residual_std']:.4f}")
        
        # (c) 分类器2阈值更新
        print("  [微调 3/3] 分类器2阈值更新...")
        # 用新引擎推理held-out数据，获取model_*列
        classifier2_new = copy.deepcopy(self.classifier2)

        # df_held = df_tft[held_mask].copy()
        if len(df_held) > self.min_required:
            df_held_tft = self._extract_tft_batch(df_held, engine=engine_new)
            if len(df_held_tft) > 10:
                # classifier2_new = copy.deepcopy(self.classifier2)
                # classifier2_new.calibrate(df_held_tft)
                classifier2_new.calibrate(df_held_tft)
                print(f"    新阈值校准完成")
            else:
                classifier2_new = copy.deepcopy(self.classifier2)
        else:
            classifier2_new = copy.deepcopy(self.classifier2)
        
        # 暂存新组件
        self._new_engine = engine_new
        self._new_processor = processor_new
        self._new_classifier2 = classifier2_new
        
        print("  [微调完成] 新组件已就绪，等待并行验证")
        return True

    # ================================================================
    # TFT批量推理（可配置batch_size）
    # ================================================================
    def _patch_engine_batch_size(self, engine):
        """为引擎应用推理batch_size补丁"""
        
        inference_bs = self.inference_batch_size
        
        def patched_analyze(model_name, df, baseline_end_idx=None, predict=True):
            model = engine.models[model_name]
            train_dataset = engine.datasets[model_name]
            df_inf = df.copy()
            for col in engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)
            
            try:
                inference_dataset = TimeSeriesDataSet.from_dataset(
                    train_dataset, df_inf, predict=False, stop_randomization=True)
            except ValueError as e:
                if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
                    return {}
                raise
            
            if len(inference_dataset) == 0:
                return {}
            
            dataloader = inference_dataset.to_dataloader(
                train=False, 
                batch_size=inference_bs,
                num_workers=0, 
                pin_memory=True)
            
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model.to(device)
            model.eval()
            
            result_buffer = {}
            with torch.no_grad():
                for x, y in dataloader:
                    x = {k: v.to(device) for k, v in x.items()}
                    targets = y[0].cpu().numpy().flatten()
                    raw_out = model(x)
                    preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
                    p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
                    time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
                    interp = model.interpret_output(raw_out, reduction="none")
                    batch_att = interp['attention'].cpu().numpy()
                    batch_vsn = interp['encoder_variables'].cpu().numpy()
                    
                    for i in range(len(time_idx)):
                        t_id = int(time_idx[i])
                        result_buffer[t_id] = {
                            "target_true": targets[i],
                            "pred_p50": p50[i],
                            "residual": targets[i] - p50[i],
                            "divergence": p90[i] - p10[i],
                            "attention": batch_att[i],
                            "vsn": batch_vsn[i]
                        }
            
            if not result_buffer:
                return {}
            
            sorted_t = sorted(result_buffer.keys())
            df_s = pd.DataFrame({
                "time_idx": sorted_t,
                "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
                "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
                "residual": [result_buffer[t]["residual"] for t in sorted_t],
                "divergence": [result_buffer[t]["divergence"] for t in sorted_t]
            })
            full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
            full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
            
            result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
            
            if baseline_end_idx is not None:
                bl = engine._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
                result["baseline"] = bl
                engine.baselines[model_name] = bl
            
            return result
        
        engine.analyze_rolling = patched_analyze
        
    def _patch_engine_early_stopping(self, engine):
        """为引擎应用具有高容忍度的早停机制及最佳权重回滚补丁"""
        import types
        import torch
        import pandas as pd
        import lightning.pytorch as pl
        from lightning.pytorch.loggers import CSVLogger
        from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
        from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
        from pytorch_forecasting.metrics import QuantileLoss
        from pytorch_forecasting.data import GroupNormalizer
        from pytorch_forecasting.data.encoders import NaNLabelEncoder

        ft_batch_size = self.finetune_batch_size

        def patched_build_and_fit(self_engine, model_name: str, df: pd.DataFrame, 
                                  val_ratio: float = 0.2, max_epochs: int = -1,
                                  quiet_end_idx: int = None):
            
            cfg = self_engine.config['tft_models'][model_name]
            pl.seed_everything(cfg.get('seed', 42))

            df_inf = df.copy()
            for col in self_engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)

            if quiet_end_idx is not None:
                time_col = self_engine.global_cfg['time_column']
                df_for_training = df_inf[df_inf[time_col] <= quiet_end_idx].copy()
            else:
                df_for_training = df_inf

            time_col = self_engine.global_cfg['time_column']
            time_steps = df_for_training[time_col].sort_values().unique()
            split_idx = int(len(time_steps) * (1 - val_ratio))
            cutoff_time = time_steps[split_idx]
            
            train_df = df_for_training[df_for_training[time_col] <= cutoff_time]
            max_lookback = cfg.get('max_encoder_length', 48)
            val_start_time = time_steps[max(0, split_idx - max_lookback)]
            val_df = df_for_training[df_for_training[time_col] >= val_start_time]

            categorical_encoders = {
                name: NaNLabelEncoder(add_nan=True) 
                for name in self_engine.known_categoricals
            }
            for g_id in self_engine.global_cfg['group_ids']:
                categorical_encoders[g_id] = NaNLabelEncoder(add_nan=True)

            training = TimeSeriesDataSet(
                train_df,
                time_idx=self_engine.global_cfg['time_column'],
                target=cfg['target'],
                group_ids=self_engine.global_cfg['group_ids'],
                min_encoder_length=cfg.get('min_encoder_length'),
                max_encoder_length=cfg.get('max_encoder_length'),
                max_prediction_length=cfg.get('max_prediction_length', 1),
                time_varying_unknown_reals=self_engine.unknown_reals,
                time_varying_known_reals=self_engine.known_reals,
                time_varying_known_categoricals=self_engine.known_categoricals,
                categorical_encoders=categorical_encoders,
                target_normalizer=GroupNormalizer(groups=self_engine.global_cfg['group_ids'], transformation=None),
                add_relative_time_idx=True,
                add_target_scales=True,
                add_encoder_length=True,
                allow_missing_timesteps=True
            )

            validation = TimeSeriesDataSet.from_dataset(training, val_df, predict=False, stop_randomization=True)

            train_dataloader = training.to_dataloader(train=True, batch_size=ft_batch_size, num_workers=0, pin_memory=True)
            val_dataloader = validation.to_dataloader(train=False, batch_size=ft_batch_size, num_workers=0, pin_memory=True)

            model = TemporalFusionTransformer.from_dataset(
                training,
                learning_rate=cfg.get('learning_rate', 0.03),
                hidden_size=cfg.get('hidden_size', 16),
                attention_head_size=4,
                dropout=0.1,
                hidden_continuous_size=8,
                output_size=len(self_engine.quantiles),
                loss=QuantileLoss(quantiles=self_engine.quantiles),
                reduce_on_plateau_patience=4
            )

            logger = CSVLogger("lightning_logs", name=model_name, flush_logs_every_n_steps=10)
            self_engine.log_dirs[model_name] = logger.log_dir

            # 核心调整：配置宽容型早停与检查点
            early_stop_callback = EarlyStopping(
                monitor="val_loss",
                patience=8,       # 容忍20个Epoch没有改善才停止，跨越局部波动
                min_delta=1e-3,    # 降低改善判定阈值
                mode="min",
                verbose=False
            )
            checkpoint_callback = ModelCheckpoint(
                monitor="val_loss",
                mode="min",
                save_top_k=1,      # 仅保留验证损失最低的那个Epoch权重
                dirpath=logger.log_dir,
                filename="best_model"
            )

            trainer_kwargs = {
                "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
                "devices": 1,
                "enable_model_summary": False,
                "enable_checkpointing": True,
                "callbacks": [early_stop_callback, checkpoint_callback],
                "logger": logger,
                "enable_progress_bar": True
            }

            if max_epochs == -1:
                trainer = pl.Trainer(**trainer_kwargs)
            elif max_epochs != 1:
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)
            elif max_epochs == 1:
                trainer_kwargs.update({"limit_train_batches": 5, "limit_val_batches": 5})
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)

            trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

            # 核心调整：训练结束后，强制从检查点加载验证集表现最佳的权重
            # if checkpoint_callback.best_model_path:
            #     best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)
            #     model.load_state_dict(best_model.state_dict())
            if checkpoint_callback.best_model_path:
                best_model = TemporalFusionTransformer.load_from_checkpoint(
                    checkpoint_callback.best_model_path, 
                    weights_only=False
                )
                model.load_state_dict(best_model.state_dict())
                print(f"    [{model_name}] 回滚至最佳验证权重: {checkpoint_callback.best_model_score:.4f}")

            self_engine.plot_training_history(model_name)
            
            self_engine.datasets[model_name] = training
            self_engine.models[model_name] = model

        engine.build_and_fit = types.MethodType(patched_build_and_fit, engine)
    # ================================================================
    # 切换
    # ================================================================
    def _execute_switch(self, current_tidx):
        """执行分层切换判定"""
        
        # 由于并行推理在简化实现中无法精确对比，这里直接执行切换
        # 完整实现中应同时用新旧引擎推理同一批数据并比较一致率
        
        # 立即切换: processor + baseline + 分类器阈值
        print(f"\n  [SWITCH] 执行切换 @ tidx={current_tidx}")
        
        self.processor = self._new_processor
        self.engine = self._new_engine
        self.classifier2 = self._new_classifier2
        
        self.baseline_version += 1
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        self._switch_warmup_count = self.ap_warmup_windows
        
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
        
        # 重置状态
        self.state = self.NORMAL
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        
        print(f"  [SWITCH] 完成! baseline_version={self.baseline_version}, "
              f"冷却至 tidx={self.cooldown_until_tidx}")
    
    # ================================================================
    # TFT批量推理（复用滚动训练Cell 2的逻辑）
    # ================================================================
    def _extract_tft_batch(self, df_buffer, engine=None, output_tidx_set=None):
        """
        对缓冲区执行TFT推理，提取model_*信号 + rolling。
        直接复用notebook Cell 2 的 extract_tft_features_batch 逻辑。
        """
        if engine is None:
            engine = self.engine
        
        # --- TFT推理 ---
        tft_results = {}
        for mn in engine.models:
            with contextlib.redirect_stdout(io.StringIO()), \
                 contextlib.redirect_stderr(io.StringIO()):
                res = engine.analyze_rolling(mn, df_buffer, baseline_end_idx=None)
            tft_results[mn] = res
        
        if not any('metrics' in v for v in tft_results.values()):
            return pd.DataFrame()
        
        # --- 合并TFT信号 ---
        df_merged = df_buffer[['time_idx', 'timestamp']].copy()
        
        for model_name in tft_results:
            res = tft_results[model_name]
            if 'metrics' not in res:
                continue
            
            metrics = res['metrics']
            attention = res['attention']
            vsn = res['vsn']
            baseline = engine.baselines.get(model_name, {})
            
            if not baseline:
                continue
            
            n = len(metrics)
            
            # 残差比
            residual_abs = metrics['residual'].abs().values
            baseline_p95 = max(
                abs(baseline.get('residual_p95', 0)),
                abs(baseline.get('residual_p5', 0)),
                baseline.get('residual_std', 1e-8) * 1.645, 1e-8)
            sig_residual = np.clip(residual_abs / baseline_p95, 0, 20)
            
            # 注意力KL散度
            att_base = baseline.get('att_mean')
            sig_att_kl = np.zeros(n)
            if att_base is not None:
                for i in range(n):
                    p = attention[i].flatten()
                    q = att_base.flatten()
                    p = p / (p.sum() + 1e-10) + 1e-10
                    q = q / (q.sum() + 1e-10) + 1e-10
                    p, q = p / p.sum(), q / q.sum()
                    sig_att_kl[i] = np.sum(p * np.log(p / q))
            
            # VSN JS散度
            vsn_base = baseline.get('vsn_mean')
            sig_vsn_js = np.zeros(n)
            if vsn_base is not None:
                for i in range(n):
                    v_i = np.abs(vsn[i].flatten()) + 1e-10
                    v_b = np.abs(vsn_base.flatten()) + 1e-10
                    p_v, q_v = v_i / v_i.sum(), v_b / v_b.sum()
                    m_v = 0.5 * (p_v + q_v)
                    sig_vsn_js[i] = (
                        0.5 * np.sum(p_v * np.log(p_v / m_v)) +
                        0.5 * np.sum(q_v * np.log(q_v / m_v)))
            
            # VSN排序变化
            sig_vsn_rank = np.zeros(n)
            if vsn_base is not None:
                baseline_rank = np.argsort(np.argsort(-np.abs(vsn_base.flatten())))
                for i in range(n):
                    curr_rank = np.argsort(np.argsort(-np.abs(vsn[i].flatten())))
                    corr, _ = spearmanr(baseline_rank, curr_rank)
                    sig_vsn_rank[i] = 1 - corr if not np.isnan(corr) else 1.0
            
            # 组装
            prefix = f'model_{model_name}'
            tft_df = pd.DataFrame({
                'time_idx': metrics['time_idx'].values,
                f'{prefix}_residual_ratio': sig_residual,
                f'{prefix}_att_kl': np.clip(sig_att_kl, 0, 20),
                f'{prefix}_vsn_js': np.clip(sig_vsn_js, 0, 5),
                f'{prefix}_vsn_rank_shift': np.clip(sig_vsn_rank, 0, 2),
            }).drop_duplicates(subset='time_idx', keep='last')
            
            # merge (防膨胀)
            n_before = len(df_merged)
            df_merged = df_merged.merge(tft_df, on='time_idx', how='left')
            if len(df_merged) != n_before:
                # 回退, 使用map方式
                new_cols = [c for c in tft_df.columns if c != 'time_idx']
                df_merged = df_merged.drop(columns=new_cols).iloc[:n_before]
                for col in new_cols:
                    val_map = dict(zip(tft_df['time_idx'], tft_df[col]))
                    df_merged[col] = df_merged['time_idx'].map(val_map)
        
        # --- Rolling特征 ---
        base_tft_cols = [c for c in df_merged.columns
                         if c.startswith('model_') and 'roll' not in c]
        for col in base_tft_cols:
            for k in [4, 8]:
                df_merged[f'{col}_rollmax_{k}'] = (
                    df_merged[col].rolling(k, min_periods=1).max())
                df_merged[f'{col}_rollmean_{k}'] = (
                    df_merged[col].rolling(k, min_periods=1).mean())
        
        df_merged = df_merged.replace([np.inf, -np.inf], 0).fillna(0)
        
        # 只保留model_*列 + time_idx + timestamp
        keep_cols = ['time_idx', 'timestamp'] + [
            c for c in df_merged.columns if c.startswith('model_')]
        df_merged = df_merged[[c for c in keep_cols if c in df_merged.columns]]
        
        # 筛选输出行
        if output_tidx_set is not None:
            df_merged = df_merged[df_merged['time_idx'].isin(output_tidx_set)]
        
        return df_merged
    
    # ================================================================
    # 工具
    # ================================================================
    def _print_summary(self, df_final):
        """打印运行总结"""
        print(f"\n{'='*60}")
        print(f"  自适应滚动检测 — 运行总结")
        print(f"{'='*60}")
        if len(df_final) > 0:
            vc = df_final['pred_label'].value_counts()
            print(f"  总窗口: {len(df_final)}")
            print(f"  N={vc.get('N',0)}, AP={vc.get('AP',0)}, CP={vc.get('CP',0)}")
        print(f"  最终baseline版本: {self.baseline_version}")
        print(f"  CP事件: {len(self.cp_events)}")
        for i, evt in enumerate(self.cp_events):
            print(f"    #{i}: confirmed@{evt['confirmed_tidx']}, "
                  f"retrace@{evt['retrace_tidx']}, "
                  f"switched@{evt.get('switched_tidx', '未切换')}")
        print(f"{'='*60}")
    
    def save_state(self, path="./checkpoints/adaptive_state"):
        """持久化当前状态"""
        os.makedirs(path, exist_ok=True)
        state = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_events': self.cp_events,
            'cooldown_until_tidx': self.cooldown_until_tidx,
        }
        with open(os.path.join(path, 'state.json'), 'w') as f:
            import json
            json.dump(state, f, indent=2, default=str)
        
        self.engine.save(os.path.join(path, f'engine_v{self.baseline_version}'))
        with open(os.path.join(path, f'processor_v{self.baseline_version}.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(path, f'classifier2_v{self.baseline_version}.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        print(f"✅ 状态保存到 {path}")

## 运行V5

In [ ]:
# ============================================================
# Cell: 自适应滚动检测 — 完整运行
# ============================================================
import types, io, contextlib, copy, os, pickle
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pytorch_forecasting import TimeSeriesDataSet
from TFT_tft_engine import TFTEngine
from target_builder import TargetBuilder
# from adaptive_rolling_detector import StatisticalClassifier2, AdaptiveRollingDetector
import logging

# 关闭 PyTorch Lightning 的硬件与环境提示
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

# 关闭 数据处理器与 TargetBuilder 的常规INFO日志
logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
logging.getLogger("TargetBuilder").setLevel(logging.WARNING)
# 如果有其他自定义 logger，也可以一并设置为 WARNING
# =====================================================================
# 0. 加载预训练系统
# =====================================================================
CKPT = "./checkpoints/pretrained_planPC"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine_PC = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)

# 去掉末尾数据
df_features = df_features[df_features['timestamp']<pd.to_datetime('2025-11-13')].copy()


# 绑定去重补丁（与Cell 1相同）
def analyze_rolling_patched(self, model_name, df, baseline_end_idx=None, predict=True):
    model = self.models[model_name]
    train_dataset = self.datasets[model_name]
    df_inf = df.copy()
    for col in self.known_categoricals:
        df_inf[col] = df_inf[col].astype(str)
    try:
        inference_dataset = TimeSeriesDataSet.from_dataset(
            train_dataset, df_inf, predict=False, stop_randomization=True)
    except ValueError as e:
        if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
            return {}
        raise
    if len(inference_dataset) == 0:
        return {}
    dataloader = inference_dataset.to_dataloader(
        train=False, batch_size=512, num_workers=0, pin_memory=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()
    result_buffer = {}
    with torch.no_grad():
        for x, y in dataloader:
            x = {k: v.to(device) for k, v in x.items()}
            targets = y[0].cpu().numpy().flatten()
            raw_out = model(x)
            preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
            p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
            time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
            interp = model.interpret_output(raw_out, reduction="none")
            batch_att = interp['attention'].cpu().numpy()
            batch_vsn = interp['encoder_variables'].cpu().numpy()
            for i in range(len(time_idx)):
                t_id = int(time_idx[i])
                result_buffer[t_id] = {
                    "target_true": targets[i], "pred_p50": p50[i],
                    "residual": targets[i] - p50[i], "divergence": p90[i] - p10[i],
                    "attention": batch_att[i], "vsn": batch_vsn[i]}
    if not result_buffer:
        return {}
    sorted_t = sorted(result_buffer.keys())
    df_s = pd.DataFrame({
        "time_idx": sorted_t,
        "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
        "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
        "residual": [result_buffer[t]["residual"] for t in sorted_t],
        "divergence": [result_buffer[t]["divergence"] for t in sorted_t]})
    full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
    full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
    result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
    if baseline_end_idx is not None:
        bl = self._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
        result["baseline"] = bl
        self.baselines[model_name] = bl
    return result

engine_PC.analyze_rolling = types.MethodType(analyze_rolling_patched, engine_PC)

# =====================================================================
# 1. 数据预处理（与滚动训练Cell 1一致）
# =====================================================================
QUIET_START = pd.to_datetime('2025-02-18 00:00:00')
QUIET_END   = pd.to_datetime('2025-02-25 00:00:00')

# 用平静期refit processor
mask_quiet = (df_features['timestamp'] >= QUIET_START) & (df_features['timestamp'] < QUIET_END)
df_quiet_raw = df_features[mask_quiet].copy().fillna(0)

processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_quiet_raw)
print(f"[Processor] re-fitted, global_min_time={processor_real.global_min_time}")

# 变换全量数据
dataset_real = processor_real.transform(df_features.fillna(0), df_official_real)
df_real_all = dataset_real.data.copy()
df_real_all.fillna(0, inplace=True)
df_real_all.replace([np.inf, -np.inf], 0, inplace=True)
df_real_all = tb.transform(df_real_all)
if 'label' not in df_real_all.columns:
    df_real_all['label'] = 'N'
df_real_all = df_real_all.sort_values('time_idx').reset_index(drop=True)
print(f"[Data] {df_real_all.shape}, {df_real_all['timestamp'].min()} ~ {df_real_all['timestamp'].max()}")

# =====================================================================
# 2. 构建分类器2 & 自适应检测器
# =====================================================================
classifier5 = StatisticalClassifier5(
    ap_k=5.0, 
    cp_base_k=3.0, 
    cp_density_thresh=0.95,
    cp_window=288, 
    cp_min_periods=96, 
    ap_min_triggers=4,
    cp_trend_window=2688,# 长程漂移参数
    cp_trend_k=2.0
)

detector5 = AdaptiveRollingDetector5(
    engine=engine_PC,
    tb=tb,
    processor=processor_real,
    classifier2=classifier5,
    # 滚动
    buffer_size=672, step_size=96,min_required=None,
    # CP确认
    cp_confirm_windows=192,

    # 回溯与积累
    retrace_windows=288, accumulate_min=1344,# 增长数据收集期
    # 微调
    finetune_epochs=100, finetune_lr_scale=0.005, finetune_batch_size=32, finetune_grad_clip=0.5, 
    held_out_days=2,held_out_min_windows=96,
    # 切换
    switch_parallel_windows=288,
    switch_parallel_extend=480,

    switch_recovery_threshold=0.85,
    switch_stability_threshold=0.85,
    # 冷却
    cooldown_windows=672,ap_warmup_windows=96, ap_warmup_relax=1.2,
    inference_batch_size=256
)

# =====================================================================
# 3. 运行
# =====================================================================
df_result5 = detector5.run(
    df_real_all=df_real_all,
    df_features_raw=df_features.fillna(0),
    df_official=df_official_real,
    quiet_start=QUIET_START,
    quiet_end=QUIET_END,
    output_path="./output/adaptive_detection_resultv5.csv"
)
# 保存状态
detector5.save_state("./checkpoints/adaptive_state")

print(f"\n🎯 检测完成, 结果: {len(df_result5)} 窗口")


# 检测V6

In [ ]:
"""
自适应滚动检测器：
  滚动TFT推理 → 分类器2判定 → CP确认 → 回溯288窗口 → 积累微调 → 分层切换
  
  状态机：NORMAL → CP_TENTATIVE → ACCUMULATING → SWITCHING → NORMAL
"""

import os, copy, pickle, tempfile, io, contextlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import spearmanr
# 如果有其他自定义 logger，也可以一并设置为 WARNING
# =====================================================================
# 分类器2: 纯统计阈值 AP/CP 判定
# =====================================================================
class StatisticalClassifier6:
    """
    基于冷启动期校准的多维Z-score分类器。
    输入: 滚动TFT推理输出的 model_* 信号列
    输出: 每个窗口的 pred_label (N/AP/CP) + 投票归因信息
    """
    

    import logging

    # 关闭 PyTorch Lightning 的硬件与环境提示
    logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

    # 关闭 数据处理器与 TargetBuilder 的常规INFO日志
    logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
    logging.getLogger("TargetBuilder").setLevel(logging.WARNING)
    # 8个AP候选特征
    
    AP_ROLL_FEATURES = [
        'model_model_comment_pc1_residual_ratio_rollmax_8',
        'model_model_post_pc2_residual_ratio_rollmax_8',
        'model_model_comment_pc1_vsn_rank_shift_rollmax_8',
        'model_model_post_pc2_vsn_rank_shift_rollmax_8',
        'model_model_comment_pc1_att_kl_rollmax_8',
        'model_model_post_pc2_att_kl_rollmax_8',
        'model_model_comment_pc1_vsn_js_rollmax_8',
        'model_model_post_pc2_vsn_js_rollmax_8',
    ]
    
    # 4个CP静态特征
    CP_STATIC_FEATURES = [
        'model_model_comment_pc1_vsn_rank_shift',
        'model_model_post_pc2_vsn_rank_shift',
        'model_model_comment_pc1_att_kl',
        'model_model_post_pc2_att_kl',
    ]
    
    def __init__(self, 
                 ap_k=5.0, 
                 ap_min_triggers=4,
                 cp_base_k=3.0, 
                 cp_density_thresh=0.95,
                 cp_window=288, 
                 cp_min_periods=96, 
                 # 新增：长程缓变漂移参数
                 cp_trend_window=2688, 
                 cp_trend_k=2.0):
        self.ap_k = ap_k
        self.cp_base_k = cp_base_k
        self.cp_density_thresh = cp_density_thresh
        self.cp_window = cp_window
        self.cp_min_periods = cp_min_periods
        self.ap_min_triggers = ap_min_triggers
        
        self.cp_trend_window = cp_trend_window
        self.cp_trend_k = cp_trend_k
        
        self.ap_thresholds = {}
        self.cp_base_thresh = None
        self.cp_vol_thresh = None
        self.cp_trend_thresh = None # 新增
        self._z_params = {} 
        self._calibrated = False
        self._ap_relax_factor = 1.0
    
    def calibrate(self, df_base):
        """用冷启动期数据校准所有阈值"""
        for feat in self.AP_ROLL_FEATURES:
            if feat in df_base.columns:
                self.ap_thresholds[feat] = df_base[feat].mean() + self.ap_k * df_base[feat].std()
        
        # 设定最小方差线
        min_std_floors = {
            'model_model_comment_pc1_vsn_rank_shift': 0.05,
            'model_model_post_pc2_vsn_rank_shift': 0.05,
            'model_model_comment_pc1_att_kl': 0.1,
            'model_model_post_pc2_att_kl': 0.1,
            'model_model_comment_pc1_vsn_js': 0.02,
            'model_model_post_pc2_vsn_js': 0.02
        }
        for feat in self.CP_STATIC_FEATURES:
            if feat in df_base.columns:
                actual_std = df_base[feat].std()
                floor_std = min_std_floors.get(feat, 1e-4)
                if actual_std < floor_std:
                    print(f"⚠️  {feat} 的实际std={actual_std:.6f}低于最小阈值{floor_std}, 已调整为{floor_std}")
                safe_std = max(actual_std, floor_std)
                self._z_params[feat] = {'mean': df_base[feat].mean(), 'std': safe_std}

        # for feat in self.CP_STATIC_FEATURES:
        #     if feat in df_base.columns:
        #         self._z_params[feat] = {'mean': df_base[feat].mean(), 'std': df_base[feat].std() + 1e-8}
        
        z_max_vals = self._compute_z_max(df_base)
        self.cp_base_thresh = z_max_vals.mean() + self.cp_base_k * z_max_vals.std()
        self.cp_vol_thresh = z_max_vals.std() * 1.5
        self.cp_trend_thresh = z_max_vals.mean() + self.cp_trend_k * z_max_vals.std() # 新增：长程漂移阈值
        
        
        
        self._calibrated = True
        print(f"[Classifier3] 校准完成: AP阈值={len(self.ap_thresholds)}个, "
              f"突变水位={self.cp_base_thresh:.3f}, 漂移阈值={self.cp_trend_thresh:.3f}")
        
    def set_ap_relax_factor(self, factor):
        """设置AP阈值放宽因子"""
        self._ap_relax_factor = factor
    
    def _compute_z_max(self, df):
        """计算各行的多维Z-score最大值"""
        z_vals = []
        for feat, params in self._z_params.items():
            if feat in df.columns:
                z = (df[feat] - params['mean']) / params['std']
                z_vals.append(z)
        if z_vals:
            return pd.concat(z_vals, axis=1).max(axis=1)
        return pd.Series(0.0, index=df.index)
    
    def predict(self, df):
        """
        执行统计检验与分类判定，并计算信号归因时间点。
        """
        assert self._calibrated, "必须先调用 calibrate()"
        df = df.copy()
        
        # --- CP辅助信号 ---
        z_max = self._compute_z_max(df)
        df['static_signal_z_max'] = z_max
        is_high = (z_max > self.cp_base_thresh).astype(float)
        df['cp_density'] = is_high.rolling(self.cp_window, min_periods=self.cp_min_periods).mean()
        df['cp_volatility'] = z_max.rolling(self.cp_window, min_periods=self.cp_min_periods).std()
        
        # 新增：长程缓变均值与漂移判定 (独立特征)
        df['cp_trend_mean'] = z_max.rolling(self.cp_trend_window, min_periods=self.cp_window).mean()
        df['is_baseline_drift'] = df['cp_trend_mean'] > self.cp_trend_thresh
        
        # --- AP触发计数 ---
        trigger_matrix = pd.DataFrame(index=df.index)
        for feat in self.AP_ROLL_FEATURES:
            if feat in df.columns and feat in self.ap_thresholds:   
                effective_thresh = self.ap_thresholds[feat] * self._ap_relax_factor
                trigger_matrix[feat] = (df[feat] > effective_thresh).astype(int)
        df['ap_trigger_count'] = trigger_matrix.sum(axis=1)

        # --- 判定 (仅限 AP 和 Sudden CP) ---
        df['pred_label'] = 'N'
        
        mask_cp = (df['cp_density'] > self.cp_density_thresh) & (df['cp_volatility'] < self.cp_vol_thresh)
        df.loc[mask_cp, 'pred_label'] = 'CP'
        
        mask_ap = (df['ap_trigger_count'] >= self.ap_min_triggers) & (~mask_cp)
        df.loc[mask_ap, 'pred_label'] = 'AP'
        
        # --- 归因时间点定位 (Attribution Time) ---
        df['attribution_time'] = pd.NaT
        # 1. 点异常 (PA): 零滞后，归因为当前时间
        df.loc[mask_ap, 'attribution_time'] = df.loc[mask_ap, 'timestamp']
        # 2. 集体异常 (CA): 存在窗口滞后，向左回溯冲击起始点
        ca_attr_times = df['timestamp'].shift(self.cp_window)
        df.loc[mask_cp, 'attribution_time'] = ca_attr_times.loc[mask_cp]
        # 3. 概念漂移 (CP): 存在严重平滑滞后，向左回溯漂移起始分布点
        cp_attr_times = df['timestamp'].shift(1344) 
        drift_mask = df['is_baseline_drift'] == True
        df.loc[drift_mask, 'attribution_time'] = cp_attr_times.loc[drift_mask]

        # --- 空间投票归因 ---
        df['dominant_model'] = 'N/A'
        df['dominant_signal'] = 'N/A'
        df['trigger_features'] = ''
        
        anom_mask = df['pred_label'] != 'N'
        if anom_mask.any():

            for idx in df[anom_mask].index:
                triggers = [f for f in trigger_matrix.columns 
                           if trigger_matrix.loc[idx, f] == 1]
                if not triggers:
                    continue


                # 子模型投票
                comment_v = sum(1 for f in triggers if 'comment_pc1' in f)
                post_v = sum(1 for f in triggers if 'post_pc2' in f)
                dom_model = 'comment_pc1' if comment_v >= post_v else 'post_pc2'
                
                # 信号类型投票
                signal_votes = {}
                for sig_type in ['residual_ratio', 'vsn_rank_shift', 'att_kl', 'vsn_js']:
                    signal_votes[sig_type] = sum(1 for f in triggers if sig_type in f)
                dom_signal = max(signal_votes, key=signal_votes.get)
                
                df.loc[idx, 'dominant_model'] = dom_model
                df.loc[idx, 'dominant_signal'] = dom_signal
                df.loc[idx, 'trigger_features'] = '|'.join(triggers)
        
        return df



## 运行V6

In [ ]:
# ============================================================
# Cell: 自适应滚动检测 — 完整运行
# ============================================================
import types, io, contextlib, copy, os, pickle
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pytorch_forecasting import TimeSeriesDataSet
from TFT_tft_engine import TFTEngine
from target_builder import TargetBuilder
# from adaptive_rolling_detector import StatisticalClassifier2, AdaptiveRollingDetector
import logging

# 关闭 PyTorch Lightning 的硬件与环境提示
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

# 关闭 数据处理器与 TargetBuilder 的常规INFO日志
logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
logging.getLogger("TargetBuilder").setLevel(logging.WARNING)
# 如果有其他自定义 logger，也可以一并设置为 WARNING
# =====================================================================
# 0. 加载预训练系统
# =====================================================================
CKPT = "./checkpoints/pretrained_planPC"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine_PC = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)

# 去掉末尾数据
df_features = df_features[df_features['timestamp']<pd.to_datetime('2025-11-13')].copy()


# 绑定去重补丁（与Cell 1相同）
def analyze_rolling_patched(self, model_name, df, baseline_end_idx=None, predict=True):
    model = self.models[model_name]
    train_dataset = self.datasets[model_name]
    df_inf = df.copy()
    for col in self.known_categoricals:
        df_inf[col] = df_inf[col].astype(str)
    try:
        inference_dataset = TimeSeriesDataSet.from_dataset(
            train_dataset, df_inf, predict=False, stop_randomization=True)
    except ValueError as e:
        if "0 sample(s)" in str(e) or "minimum of 1" in str(e):
            return {}
        raise
    if len(inference_dataset) == 0:
        return {}
    dataloader = inference_dataset.to_dataloader(
        train=False, batch_size=512, num_workers=0, pin_memory=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()
    result_buffer = {}
    with torch.no_grad():
        for x, y in dataloader:
            x = {k: v.to(device) for k, v in x.items()}
            targets = y[0].cpu().numpy().flatten()
            raw_out = model(x)
            preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
            p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
            time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
            interp = model.interpret_output(raw_out, reduction="none")
            batch_att = interp['attention'].cpu().numpy()
            batch_vsn = interp['encoder_variables'].cpu().numpy()
            for i in range(len(time_idx)):
                t_id = int(time_idx[i])
                result_buffer[t_id] = {
                    "target_true": targets[i], "pred_p50": p50[i],
                    "residual": targets[i] - p50[i], "divergence": p90[i] - p10[i],
                    "attention": batch_att[i], "vsn": batch_vsn[i]}
    if not result_buffer:
        return {}
    sorted_t = sorted(result_buffer.keys())
    df_s = pd.DataFrame({
        "time_idx": sorted_t,
        "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
        "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
        "residual": [result_buffer[t]["residual"] for t in sorted_t],
        "divergence": [result_buffer[t]["divergence"] for t in sorted_t]})
    full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
    full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
    result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
    if baseline_end_idx is not None:
        bl = self._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
        result["baseline"] = bl
        self.baselines[model_name] = bl
    return result

engine_PC.analyze_rolling = types.MethodType(analyze_rolling_patched, engine_PC)

# =====================================================================
# 1. 数据预处理（与滚动训练Cell 1一致）
# =====================================================================
QUIET_START = pd.to_datetime('2025-02-18 00:00:00')
QUIET_END   = pd.to_datetime('2025-02-25 00:00:00')

# 用平静期refit processor
mask_quiet = (df_features['timestamp'] >= QUIET_START) & (df_features['timestamp'] < QUIET_END)
df_quiet_raw = df_features[mask_quiet].copy().fillna(0)

processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_quiet_raw)
print(f"[Processor] re-fitted, global_min_time={processor_real.global_min_time}")

# 变换全量数据
dataset_real = processor_real.transform(df_features.fillna(0), df_official_real)
df_real_all = dataset_real.data.copy()
df_real_all.fillna(0, inplace=True)
df_real_all.replace([np.inf, -np.inf], 0, inplace=True)
df_real_all = tb.transform(df_real_all)
if 'label' not in df_real_all.columns:
    df_real_all['label'] = 'N'
df_real_all = df_real_all.sort_values('time_idx').reset_index(drop=True)
print(f"[Data] {df_real_all.shape}, {df_real_all['timestamp'].min()} ~ {df_real_all['timestamp'].max()}")

# =====================================================================
# 2. 构建分类器2 & 自适应检测器
# =====================================================================
classifier6 = StatisticalClassifier6(
    ap_k=5.0, 
    cp_base_k=3.0, 
    cp_density_thresh=0.95,
    cp_window=288, 
    cp_min_periods=96, 
    ap_min_triggers=4,
    cp_trend_window=2688,# 长程漂移参数
    cp_trend_k=2.0
)

detector6 = AdaptiveRollingDetector5(
    engine=engine_PC,
    tb=tb,
    processor=processor_real,
    classifier2=classifier6,
    # 滚动
    buffer_size=672, step_size=96,min_required=None,
    # CP确认
    cp_confirm_windows=192,

    # 回溯与积累
    retrace_windows=288, accumulate_min=1344,# 增长数据收集期
    # 微调
    finetune_epochs=100, finetune_lr_scale=0.005, finetune_batch_size=32, finetune_grad_clip=0.5, 
    held_out_days=2,held_out_min_windows=96,
    # 切换
    switch_parallel_windows=288,
    switch_parallel_extend=480,

    switch_recovery_threshold=0.85,
    switch_stability_threshold=0.85,
    # 冷却
    cooldown_windows=672,ap_warmup_windows=96, ap_warmup_relax=1.2,
    inference_batch_size=256
)

# =====================================================================
# 3. 运行
# =====================================================================
df_result6 = detector6.run(
    df_real_all=df_real_all,
    df_features_raw=df_features.fillna(0),
    df_official=df_official_real,
    quiet_start=QUIET_START,
    quiet_end=QUIET_END,
    output_path="./output/adaptive_detection_resultv6.csv"
)
# 保存状态
detector6.save_state("./checkpoints/adaptive_state")

print(f"\n🎯 检测完成, 结果: {len(df_result6)} 窗口")


# 检测V7

In [ ]:
"""
自适应滚动检测器 (V7)：
  滚动TFT推理 → 分类器判定 → CP确认 → 回溯288窗口 → 积累微调 → 分层切换
"""

import os, copy, pickle, tempfile, io, contextlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import spearmanr
import logging

logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)
logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
logging.getLogger("TargetBuilder").setLevel(logging.WARNING)

# =====================================================================
# 分类器7: 纯统计阈值 AP/CP 判定
# =====================================================================
class StatisticalClassifier7:
    """
    基于冷启动期校准的多维Z-score分类器。
    包含方差下界约束与严格的时空归因隔离。
    """
    
    AP_ROLL_FEATURES = [
        'model_model_comment_pc1_residual_ratio_rollmax_8',
        'model_model_post_pc2_residual_ratio_rollmax_8',
        'model_model_comment_pc1_vsn_rank_shift_rollmax_8',
        'model_model_post_pc2_vsn_rank_shift_rollmax_8',
        'model_model_comment_pc1_att_kl_rollmax_8',
        'model_model_post_pc2_att_kl_rollmax_8',
        'model_model_comment_pc1_vsn_js_rollmax_8',
        'model_model_post_pc2_vsn_js_rollmax_8',
    ]
    
    CP_STATIC_FEATURES = [
        'model_model_comment_pc1_vsn_rank_shift',
        'model_model_post_pc2_vsn_rank_shift',
        'model_model_comment_pc1_att_kl',
        'model_model_post_pc2_att_kl',
    ]
    
    def __init__(self, ap_k=5.0, ap_min_triggers=4, cp_base_k=3.0, 
                 cp_density_thresh=0.95, cp_window=288, cp_min_periods=96, 
                 cp_trend_window=2688, cp_trend_k=2.0):
        self.ap_k = ap_k
        self.cp_base_k = cp_base_k
        self.cp_density_thresh = cp_density_thresh
        self.cp_window = cp_window
        self.cp_min_periods = cp_min_periods
        self.ap_min_triggers = ap_min_triggers
        self.cp_trend_window = cp_trend_window
        self.cp_trend_k = cp_trend_k
        
        self.ap_thresholds = {}
        self.cp_base_thresh = None
        self.cp_vol_thresh = None
        self.cp_trend_thresh = None 
        self._z_params = {} 
        self._calibrated = False
        self._ap_relax_factor = 1.0
    
    def calibrate(self, df_base):
        for feat in self.AP_ROLL_FEATURES:
            if feat in df_base.columns:
                self.ap_thresholds[feat] = df_base[feat].mean() + self.ap_k * df_base[feat].std()
        
        min_std_floors = {
            'model_model_comment_pc1_vsn_rank_shift': 0.05,
            'model_model_post_pc2_vsn_rank_shift': 0.05,
            'model_model_comment_pc1_att_kl': 0.1,
            'model_model_post_pc2_att_kl': 0.1,
            'model_model_comment_pc1_vsn_js': 0.02,
            'model_model_post_pc2_vsn_js': 0.02
        }

        for feat in self.CP_STATIC_FEATURES:
            if feat in df_base.columns:
                actual_std = df_base[feat].std()
                floor_std = min_std_floors.get(feat, 1e-4)
                safe_std = max(actual_std, floor_std)
                self._z_params[feat] = {'mean': df_base[feat].mean(), 'std': safe_std}
        
        z_max_vals = self._compute_z_max(df_base)
        self.cp_base_thresh = z_max_vals.mean() + self.cp_base_k * z_max_vals.std()
        self.cp_vol_thresh = z_max_vals.std() * 1.5
        self.cp_trend_thresh = z_max_vals.mean() + self.cp_trend_k * z_max_vals.std() 
        
        self._calibrated = True
        
    def set_ap_relax_factor(self, factor):
        self._ap_relax_factor = factor
    
    def _compute_z_max(self, df):
        z_vals = []
        for feat, params in self._z_params.items():
            if feat in df.columns:
                z = (df[feat] - params['mean']) / params['std']
                z_vals.append(z)
        if z_vals:
            return pd.concat(z_vals, axis=1).max(axis=1)
        return pd.Series(0.0, index=df.index)
    
    def predict(self, df):
        assert self._calibrated, "Calibration required before prediction"
        df = df.copy()
        
        z_max = self._compute_z_max(df)
        df['static_signal_z_max'] = z_max
        is_high = (z_max > self.cp_base_thresh).astype(float)
        df['cp_density'] = is_high.rolling(self.cp_window, min_periods=self.cp_min_periods).mean()
        df['cp_volatility'] = z_max.rolling(self.cp_window, min_periods=self.cp_min_periods).std()
        
        df['cp_trend_mean'] = z_max.rolling(self.cp_trend_window, min_periods=self.cp_window).mean()
        df['is_baseline_drift'] = df['cp_trend_mean'] > self.cp_trend_thresh
        
        trigger_matrix = pd.DataFrame(index=df.index)
        for feat in self.AP_ROLL_FEATURES:
            if feat in df.columns and feat in self.ap_thresholds:   
                effective_thresh = self.ap_thresholds[feat] * self._ap_relax_factor
                trigger_matrix[feat] = (df[feat] > effective_thresh).astype(int)
        df['ap_trigger_count'] = trigger_matrix.sum(axis=1)

        df['pred_label'] = 'N'
        mask_cp = (df['cp_density'] > self.cp_density_thresh) & (df['cp_volatility'] < self.cp_vol_thresh)
        df.loc[mask_cp, 'pred_label'] = 'CP'
        
        mask_ap = (df['ap_trigger_count'] >= self.ap_min_triggers) & (~mask_cp)
        df.loc[mask_ap, 'pred_label'] = 'AP'
        
        # --- 归因时间点定位 ---
        df['attribution_time'] = pd.NaT
        df.loc[mask_ap, 'attribution_time'] = df.loc[mask_ap, 'timestamp']
        
        ca_attr_times = df['timestamp'].shift(self.cp_window)
        df.loc[mask_cp, 'attribution_time'] = ca_attr_times.loc[mask_cp]
        
        cp_attr_times = df['timestamp'].shift(1344) 
        drift_mask = (df['is_baseline_drift'] == True)
        # 隔离约束：防止 CP 的宏观归因时间覆盖 PA 的微观爆发时间
        empty_time_mask = df['attribution_time'].isna()
        df.loc[drift_mask & empty_time_mask, 'attribution_time'] = cp_attr_times.loc[drift_mask & empty_time_mask]

        df['dominant_model'] = 'N/A'
        df['dominant_signal'] = 'N/A'
        df['trigger_features'] = ''
        
        anom_mask = df['pred_label'] != 'N'
        if anom_mask.any():
            for idx in df[anom_mask].index:
                triggers = [f for f in trigger_matrix.columns if trigger_matrix.loc[idx, f] == 1]
                if not triggers:
                    continue

                comment_v = sum(1 for f in triggers if 'comment_pc1' in f)
                post_v = sum(1 for f in triggers if 'post_pc2' in f)
                dom_model = 'comment_pc1' if comment_v >= post_v else 'post_pc2'
                
                signal_votes = {}
                for sig_type in ['residual_ratio', 'vsn_rank_shift', 'att_kl', 'vsn_js']:
                    signal_votes[sig_type] = sum(1 for f in triggers if sig_type in f)
                dom_signal = max(signal_votes, key=signal_votes.get)
                
                df.loc[idx, 'dominant_model'] = dom_model
                df.loc[idx, 'dominant_signal'] = dom_signal
                df.loc[idx, 'trigger_features'] = '|'.join(triggers)
        
        return df


# =====================================================================
# 自适应滚动检测器 7
# =====================================================================
class AdaptiveRollingDetector7:
    
    NORMAL = 'NORMAL'
    CP_TENTATIVE = 'CP_TENTATIVE'
    ACCUMULATING = 'ACCUMULATING'
    SWITCHING = 'SWITCHING'
    
    def __init__(self, engine, tb, processor, classifier2,
                 buffer_size=672, step_size=24, min_required=None,
                 cp_confirm_windows=192, retrace_windows=288, accumulate_min=1344,
                 finetune_epochs=100, finetune_lr_scale=0.005, finetune_batch_size=32, finetune_grad_clip=0.5, 
                 held_out_days=2, held_out_min_windows=96,
                 switch_parallel_windows=288, switch_parallel_extend=480,
                 switch_recovery_threshold=0.85, switch_stability_threshold=0.85,
                 cooldown_windows=672, ap_warmup_windows=96, ap_warmup_relax=1.2,
                 inference_batch_size=256):
        
        self.engine = engine
        self.tb = tb
        self.processor = processor
        self.classifier2 = classifier2
        
        self.buffer_size = buffer_size
        self.step_size = step_size
        ds_0 = list(engine.datasets.values())[0]
        self.min_encoder = ds_0.max_encoder_length
        self.min_pred = ds_0.max_prediction_length
        self.min_required = min_required or (self.min_encoder + self.min_pred + 1)
        
        self.cp_confirm_windows = cp_confirm_windows
        self.retrace_windows = retrace_windows
        self.accumulate_min = accumulate_min
        
        self.finetune_epochs = finetune_epochs
        self.finetune_lr_scale = finetune_lr_scale
        self.finetune_batch_size = finetune_batch_size
        self.finetune_grad_clip = finetune_grad_clip
        self.held_out_days = held_out_days
        self.held_out_min_windows = held_out_min_windows
        
        self.switch_parallel_windows = switch_parallel_windows
        self.switch_parallel_extend = switch_parallel_extend
        self.switch_recovery_threshold = switch_recovery_threshold 
        self.switch_stability_threshold = switch_stability_threshold
        
        self.cooldown_windows = cooldown_windows
        self.ap_warmup_windows = ap_warmup_windows
        self.ap_warmup_relax = ap_warmup_relax
        self.inference_batch_size = inference_batch_size
        
        self.state = self.NORMAL
        self.baseline_version = 0
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self.cooldown_until_tidx = -1
        
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False
        
        self._ap_warmup_remaining = 0
        self.cp_events = []
        self.switch_history = []
        self.last_baseline_update_tidx = -999999
        
    def save_state(self, path="./checkpoints/adaptive_state_v7"):
        import json
        os.makedirs(path, exist_ok=True)
        
        state_dict = {
            'state': self.state,
            'baseline_version': self.baseline_version,
            'cp_consec': self.cp_consec,
            'cp_confirmed_tidx': self.cp_confirmed_tidx,
            'retrace_start_tidx': self.retrace_start_tidx,
            'accumulate_buffer_tidx': self.accumulate_buffer_tidx,
            'cooldown_until_tidx': self.cooldown_until_tidx,
            'ap_warmup_remaining': self._ap_warmup_remaining,
            'parallel_count': self._parallel_count,
            'switch_extended': self._switch_extended,
            'cp_events': self.cp_events,
            'switch_history': self.switch_history,
        }
        
        with open(os.path.join(path, 'state.json'), 'w') as f:
            json.dump(state_dict, f, indent=2, default=str)
        
        version_path = os.path.join(path, f'v{self.baseline_version}')
        os.makedirs(version_path, exist_ok=True)
        
        self.engine.save(os.path.join(version_path, 'engine'))
        with open(os.path.join(version_path, 'processor.pkl'), 'wb') as f:
            pickle.dump(self.processor, f)
        with open(os.path.join(version_path, 'classifier2.pkl'), 'wb') as f:
            pickle.dump(self.classifier2, f)
        
        if self.state == self.SWITCHING and self._new_engine is not None:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            os.makedirs(new_path, exist_ok=True)
            self._new_engine.save(os.path.join(new_path, 'engine'))
            with open(os.path.join(new_path, 'processor.pkl'), 'wb') as f:
                pickle.dump(self._new_processor, f)
            with open(os.path.join(new_path, 'classifier2.pkl'), 'wb') as f:
                pickle.dump(self._new_classifier2, f)
    
    def load_state(self, path="./checkpoints/adaptive_state_v7"):
        import json
        state_file = os.path.join(path, 'state.json')
        if not os.path.exists(state_file):
            return False
        
        with open(state_file, 'r') as f:
            state_dict = json.load(f)
        
        self.state = state_dict['state']
        self.baseline_version = state_dict['baseline_version']
        self.cp_consec = state_dict['cp_consec']
        self.cp_confirmed_tidx = state_dict.get('cp_confirmed_tidx')
        self.retrace_start_tidx = state_dict.get('retrace_start_tidx')
        self.accumulate_buffer_tidx = state_dict.get('accumulate_buffer_tidx', [])
        self.cooldown_until_tidx = state_dict.get('cooldown_until_tidx', -1)
        self._ap_warmup_remaining = state_dict.get('ap_warmup_remaining', 0)
        self._parallel_count = state_dict.get('parallel_count', 0)
        self._switch_extended = state_dict.get('switch_extended', False)
        self.cp_events = state_dict.get('cp_events', [])
        self.switch_history = state_dict.get('switch_history', [])
        
        version_path = os.path.join(path, f'v{self.baseline_version}')
        if os.path.exists(version_path):
            from TFT_tft_engine import TFTEngine
            self.engine = TFTEngine.load(os.path.join(version_path, 'engine'), config_path='TFT_config.yaml')
            with open(os.path.join(version_path, 'processor.pkl'), 'rb') as f:
                self.processor = pickle.load(f)
            with open(os.path.join(version_path, 'classifier2.pkl'), 'rb') as f:
                self.classifier2 = pickle.load(f)
        
        if self.state == self.SWITCHING:
            new_path = os.path.join(path, f'v{self.baseline_version + 1}_pending')
            if os.path.exists(new_path):
                self._new_engine = TFTEngine.load(os.path.join(new_path, 'engine'), config_path='TFT_config.yaml')
                with open(os.path.join(new_path, 'processor.pkl'), 'rb') as f:
                    self._new_processor = pickle.load(f)
                with open(os.path.join(new_path, 'classifier2.pkl'), 'rb') as f:
                    self._new_classifier2 = pickle.load(f)
        return True
       
    def run(self, df_real_all, df_features_raw, df_official,
            quiet_start=None, quiet_end=None, output_path=None, resume=False):
        df = df_real_all.sort_values('time_idx').reset_index(drop=True)
        
        if resume and os.path.exists("./checkpoints/adaptive_state_v7/state.json"):
            self.load_state("./checkpoints/adaptive_state_v7")

        if quiet_start and quiet_end:
            quiet_mask = (df['timestamp'] >= quiet_start) & (df['timestamp'] < quiet_end)
            quiet_start_idx = df[df['timestamp'] >= quiet_start].index[0]
        else:
            quiet_end_ts = df['timestamp'].min() + pd.Timedelta(days=7)
            quiet_mask = df['timestamp'] < quiet_end_ts
            quiet_start_idx = 0
        
        pre_ctx = max(0, quiet_start_idx - self.min_required)
        bl_end = df[quiet_mask].index[-1] + 1
        df_for_bl = df.iloc[pre_ctx:bl_end].copy().reset_index(drop=True)
        bl_end_tidx = int(df.loc[quiet_mask, 'time_idx'].max())
        
        self._patch_engine_batch_size(self.engine)

        for mn in self.engine.models:
            res = self.engine.analyze_rolling(mn, df_for_bl, baseline_end_idx=bl_end_tidx)
            if 'baseline' not in res and 'metrics' in res:
                bl = self.engine._build_baseline(
                    res['metrics'], res['attention'], res['vsn'], bl_end_tidx)
                self.engine.baselines[mn] = bl
        
        df_quiet_for_cal = self._extract_tft_batch(df_for_bl, engine=self.engine)
        if len(df_quiet_for_cal) > 0:
            cal_mask = df_quiet_for_cal['time_idx'] <= bl_end_tidx
            self.classifier2.calibrate(df_quiet_for_cal[cal_mask])
        else:
            self.classifier2.calibrate(df[quiet_mask])
        
        initial_end = min(quiet_start_idx + self.buffer_size, len(df))
        buffer_df = df.iloc[pre_ctx:initial_end].copy().reset_index(drop=True)
        
        quiet_start_tidx = int(df.loc[quiet_mask, 'time_idx'].min())
        output_tidx = set(
            buffer_df.loc[buffer_df['time_idx'] >= quiet_start_tidx, 'time_idx']
            .astype(int).values)
        
        df_cold = self._extract_tft_batch(buffer_df, engine=self.engine, output_tidx_set=output_tidx)
        all_classified_results = []
        processed_tidx = set(df_cold['time_idx'].astype(int).values) if len(df_cold) > 0 else set()
    
        remaining_start = initial_end
        total_remaining = len(df) - remaining_start
        n_batches = (total_remaining + self.step_size - 1) // self.step_size
        
        accumulated_tft_features = df_cold.copy() if len(df_cold) > 0 else pd.DataFrame()

        for batch_i in tqdm(range(n_batches), desc="自适应滚动检测"):
            batch_start = remaining_start + batch_i * self.step_size
            batch_end = min(batch_start + self.step_size, len(df))
            
            if batch_start >= len(df):
                break
            
            new_rows = df.iloc[batch_start:batch_end]
            new_tidx = set(new_rows['time_idx'].astype(int).values) - processed_tidx
            
            if not new_tidx:
                continue
            
            buffer_df = pd.concat([buffer_df, new_rows], ignore_index=True)
            if len(buffer_df) > self.buffer_size:
                buffer_df = buffer_df.iloc[-self.buffer_size:].reset_index(drop=True)
            
            if len(buffer_df) < self.min_required:
                continue
         
            if self._ap_warmup_remaining > 0:
                self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
                self._ap_warmup_remaining -= len(new_tidx)
                if self._ap_warmup_remaining <= 0:
                    self.classifier2.set_ap_relax_factor(1.0)
       
            if self.state == self.SWITCHING and self._new_engine is not None:
                df_feat_old = self._extract_tft_batch(buffer_df, engine=self.engine, output_tidx_set=new_tidx)
                df_feat_new = self._extract_tft_batch(buffer_df, engine=self._new_engine, output_tidx_set=new_tidx)
                
                df_feat = df_feat_old
                
                if len(df_feat_old) > 0 and len(df_feat_new) > 0:
                    pred_old = self.classifier2.predict(df_feat_old)['pred_label'].values
                    pred_new = self._new_classifier2.predict(df_feat_new)['pred_label'].values
                   
                    self._parallel_preds_old.extend(pred_old.tolist())
                    self._parallel_preds_new.extend(pred_new.tolist())
                    self._parallel_count += len(pred_old)
            else:
                df_feat = self._extract_tft_batch(buffer_df, engine=self.engine, output_tidx_set=new_tidx)   
            
            if len(df_feat) == 0:
                continue
            
            processed_tidx.update(df_feat['time_idx'].astype(int).values)

            accumulated_tft_features = pd.concat([accumulated_tft_features, df_feat], ignore_index=True)
            max_history_needed = self.classifier2.cp_trend_window + self.step_size + 100
            if len(accumulated_tft_features) > max_history_needed:
                accumulated_tft_features = accumulated_tft_features.iloc[-max_history_needed:].reset_index(drop=True)
            
            df_classified_all = self.classifier2.predict(accumulated_tft_features)
            
            df_classified = df_classified_all[df_classified_all['time_idx'].isin(new_tidx)].copy()
            df_classified['baseline_version'] = self.baseline_version
            df_classified['state'] = self.state
            all_classified_results.append(df_classified)
            
            current_tidx = int(new_rows['time_idx'].iloc[-1])
            self._process_batch(df_classified, current_tidx, df, df_features_raw, df_official)     
                   
            # 按日统计输出逻辑
            eval_frequency = 96 // self.step_size
            if (batch_i + 1) % eval_frequency == 0:
                ts_date = new_rows['timestamp'].iloc[-1].strftime('%Y-%m-%d')
                n_ap = (df_classified['pred_label'] == 'AP').sum()
                n_cp = (df_classified['pred_label'] == 'CP').sum()
                drift_flag = df_classified['is_baseline_drift'].any()
                
                print(f"  [{ts_date}] state={self.state:<12} | v={self.baseline_version} | "
                      f"AP={n_ap:<2} CP={n_cp:<2} Drift={drift_flag}")
        
        if all_classified_results:
            df_final = pd.concat(all_classified_results, ignore_index=True)
            df_final = df_final.drop_duplicates(subset='time_idx', keep='last')
            df_final = df_final.sort_values('time_idx').reset_index(drop=True)
        else:
            df_final = pd.DataFrame()

        if output_path and len(df_final) > 0:
            os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
            df_final.to_csv(output_path, index=False)
        
        self.save_state()
        return df_final
    
    def _process_batch(self, df_classified, current_tidx, df_full, df_features_raw, df_official):
        labels = df_classified['pred_label'].values
        is_drift = df_classified['is_baseline_drift'].values
        
        if self.state == self.SWITCHING:
            self._check_parallel_switch(current_tidx)
            return

        if current_tidx < self.cooldown_until_tidx:
            return

        trend_window = self.classifier2.cp_trend_window
        is_trend_safe = (current_tidx - self.last_baseline_update_tidx) >= trend_window

        if is_drift.any() and is_trend_safe:
            retrace_start = max(0, current_tidx - self.accumulate_min)
            retrace_mask = (df_full['time_idx'] >= retrace_start) & (df_full['time_idx'] <= current_tidx)
            df_retrace = df_full[retrace_mask].copy()
            
            ts_start = df_retrace['timestamp'].min()
            ts_end = df_retrace['timestamp'].max()
            raw_mask = (df_features_raw['timestamp'] >= ts_start) & (df_features_raw['timestamp'] <= ts_end)
            df_raw_retrace = df_features_raw[raw_mask].copy()
            
            success = self._finetune(df_retrace, df_raw_retrace, df_official)
            
            if success:
                self.state = self.SWITCHING
                self._parallel_preds_old = []
                self._parallel_preds_new = []
                self._parallel_count = 0
                self._switch_extended = False
                
                self.cp_events.append({
                    'confirmed_tidx': current_tidx,
                    'retrace_tidx': retrace_start,
                    'retrace_windows': self.accumulate_min,
                    'drift_type': 'Incremental',
                    'switched_tidx': None,
                    'switch_type': None,
                    'baseline_version': self.baseline_version,
                })
            else:
                self.state = self.NORMAL
                self._reset_partial_state()
                self.cooldown_until_tidx = current_tidx + self.cooldown_windows
            return 

        if self.state == self.NORMAL:
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
                if self.cp_consec >= 1:
                     self.state = self.CP_TENTATIVE
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
        
        elif self.state == self.CP_TENTATIVE:
            n_cp = (labels == 'CP').sum()
            if n_cp > 0:
                self.cp_consec += n_cp
            else:
                self.cp_consec = max(0, self.cp_consec - 1)
            
            if self.cp_consec >= self.cp_confirm_windows:
                self.cp_confirmed_tidx = current_tidx
                self.retrace_start_tidx = current_tidx  
                self.accumulate_buffer_tidx = []
                self.state = self.ACCUMULATING
      
            if self.cp_consec <= 0:
                self.state = self.NORMAL
                self.cp_consec = 0
                
        elif self.state == self.ACCUMULATING:
            batch_tidx = df_classified['time_idx'].astype(int).tolist()
            self.accumulate_buffer_tidx.extend(batch_tidx)
            accumulated = len(set(self.accumulate_buffer_tidx))
            
            if accumulated >= self.accumulate_min:
                retrace_mask = (df_full['time_idx'] >= self.retrace_start_tidx) & (df_full['time_idx'] <= current_tidx)
                df_retrace = df_full[retrace_mask].copy()
                
                ts_start = df_retrace['timestamp'].min()
                ts_end = df_retrace['timestamp'].max()
                raw_mask = (df_features_raw['timestamp'] >= ts_start) & (df_features_raw['timestamp'] <= ts_end)
                df_raw_retrace = df_features_raw[raw_mask].copy()
                
                success = self._finetune(df_retrace, df_raw_retrace, df_official)
                
                if success:
                    self.state = self.SWITCHING
                    self._parallel_preds_old = []
                    self._parallel_preds_new = []
                    self._parallel_count = 0
                    self._switch_extended = False
                    
                    self.cp_events.append({
                        'confirmed_tidx': self.cp_confirmed_tidx,
                        'retrace_tidx': self.retrace_start_tidx,
                        'retrace_windows': self.accumulate_min,
                        'drift_type': 'Sudden',
                        'switched_tidx': None,
                        'switch_type': None,
                        'baseline_version': self.baseline_version,
                    })
                else:
                    self.state = self.NORMAL
                    self._reset_partial_state()
                    self.cooldown_until_tidx = current_tidx + self.cooldown_windows

    def _check_parallel_switch(self, current_tidx):
        required_windows = (self.switch_parallel_extend 
                            if self._switch_extended 
                            else self.switch_parallel_windows)
        
        if self._parallel_count < required_windows:
            return
        
        pred_old = np.array(self._parallel_preds_old[-required_windows:])
        pred_new = np.array(self._parallel_preds_new[-required_windows:])
        
        if len(pred_old) == 0 or len(pred_new) == 0:
            return
        
        old_n_ratio = (pred_old == 'N').mean()
        new_n_ratio = (pred_new == 'N').mean()
        
        current_drift_type = self.cp_events[-1].get('drift_type', 'Sudden') if self.cp_events else 'Sudden'
        
        if current_drift_type == 'Sudden':
            if old_n_ratio >= self.switch_recovery_threshold:
                self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
            elif new_n_ratio >= self.switch_stability_threshold and new_n_ratio >= old_n_ratio:
                self._execute_full_switch(current_tidx, agreement=0.0, kappa=0.0)
            else:
                self._handle_unstable_new_model(current_tidx, new_n_ratio)
                
        elif current_drift_type == 'Incremental':
            if new_n_ratio >= self.switch_stability_threshold:
                self._execute_full_switch(current_tidx, agreement=0.0, kappa=0.0)
            else:
                self._handle_unstable_new_model(current_tidx, new_n_ratio)

    def _handle_unstable_new_model(self, current_tidx, new_n_ratio):
        if not self._switch_extended:
            self._switch_extended = True
        else:
            self._finalize_switch(current_tidx, 'aborted', agreement=0.0, kappa=0.0)
    
    def _execute_full_switch(self, current_tidx, agreement, kappa):
        self.engine = self._new_engine
        self.processor = self._new_processor
        self.classifier2 = self._new_classifier2
        self._finalize_switch(current_tidx, 'full', agreement, kappa)
    
    def _execute_partial_switch(self, current_tidx, agreement, kappa):
        self.processor = self._new_processor
        self.classifier2.ap_thresholds = copy.deepcopy(self._new_classifier2.ap_thresholds)
        self.classifier2.cp_base_thresh = self._new_classifier2.cp_base_thresh
        self.classifier2.cp_vol_thresh = self._new_classifier2.cp_vol_thresh
        self.classifier2._z_params = copy.deepcopy(self._new_classifier2._z_params)
        
        for mn in self._new_engine.baselines:
            self.engine.baselines[mn] = self._new_engine.baselines[mn]
        self._finalize_switch(current_tidx, 'partial', agreement, kappa)
    
    def _finalize_switch(self, current_tidx, switch_type, agreement, kappa):
        if switch_type == 'full':
            self.baseline_version += 1
            self.last_baseline_update_tidx = current_tidx
        
        self.cooldown_until_tidx = current_tidx + self.cooldown_windows
        self._ap_warmup_remaining = self.ap_warmup_windows
        self.classifier2.set_ap_relax_factor(self.ap_warmup_relax)
        
        if self.cp_events:
            self.cp_events[-1]['switched_tidx'] = current_tidx
            self.cp_events[-1]['switch_type'] = switch_type
        
        self.switch_history.append({
            'tidx': current_tidx,
            'type': switch_type,
            'agreement': agreement,
            'kappa': kappa,
            'new_version': self.baseline_version,
        })
        
        self.state = self.NORMAL
        self._reset_partial_state()
    
    def _reset_partial_state(self):
        self.cp_consec = 0
        self.cp_confirmed_tidx = None
        self.retrace_start_tidx = None
        self.accumulate_buffer_tidx = []
        self._new_engine = None
        self._new_processor = None
        self._new_classifier2 = None
        self._parallel_preds_old = []
        self._parallel_preds_new = []
        self._parallel_count = 0
        self._switch_extended = False

    def _finetune(self, df_retrace, df_raw_retrace, df_official):
        from TFT_tft_engine import TFTEngine
        from tft_full_period_utils import compute_baseline_from_held_out
        
        processor_new = copy.deepcopy(self.processor)
        processor_new.feature_transformer.fit(df_raw_retrace.fillna(0))
 
        dataset_new = processor_new.transform(df_raw_retrace.fillna(0), df_official)
        df_tft = dataset_new.data.copy()

        df_tft.fillna(0, inplace=True)
        df_tft.replace([np.inf, -np.inf], 0, inplace=True)
        df_tft = self.tb.transform(df_tft)
        if 'label' not in df_tft.columns:
            df_tft['label'] = 'N'

        with tempfile.TemporaryDirectory() as tmp:
            self.engine.save(tmp)
            engine_new = TFTEngine.load(tmp, config_path='TFT_config.yaml')
        
        self._patch_engine_batch_size(engine_new)
        self._patch_engine_early_stopping(engine_new)
        
        held_out_windows = max(self.held_out_min_windows, int(len(df_tft) * 0.15))
        held_out_windows = min(held_out_windows, len(df_tft) // 3)

        split_idx = len(df_tft) - held_out_windows
        df_train_ft = df_tft.iloc[:split_idx].copy()
        df_held = df_tft.iloc[split_idx:].copy()
        held_out_tidx = set(df_held['time_idx'].values)

        if len(df_train_ft) < self.min_required * 2:
            return False
        
        for mn in engine_new.models:
            for p in engine_new.models[mn].parameters():
                p.requires_grad = True
            
            cfg = engine_new.config['tft_models'][mn]
            orig_lr = cfg.get('learning_rate', 0.03)
            cfg['learning_rate'] = orig_lr * self.finetune_lr_scale
            if 'gradient_clip_val' not in cfg:
                cfg['gradient_clip_val'] = self.finetune_grad_clip

            with contextlib.redirect_stdout(io.StringIO()):
                engine_new.build_and_fit(mn, df_train_ft, max_epochs=self.finetune_epochs)
            
            cfg['learning_rate'] = orig_lr
            
            res = engine_new.analyze_rolling(mn, df_tft, baseline_end_idx=None)
            if 'metrics' not in res:
                continue
            if held_out_tidx:
                bl = compute_baseline_from_held_out(res, held_out_tidx)
            else:
                bl = engine_new._build_baseline(
                    res['metrics'], res['attention'], res['vsn'], int(df_tft['time_idx'].max()))
            engine_new.baselines[mn] = bl

        classifier2_new = copy.deepcopy(self.classifier2)

        if len(df_held) > self.min_required:
            df_held_tft = self._extract_tft_batch(df_held, engine=engine_new)
            if len(df_held_tft) > 10:
                classifier2_new.calibrate(df_held_tft)
            else:
                classifier2_new = copy.deepcopy(self.classifier2)
        else:
            classifier2_new = copy.deepcopy(self.classifier2)
        
        self._new_engine = engine_new
        self._new_processor = processor_new
        self._new_classifier2 = classifier2_new
        
        return True

    def _patch_engine_batch_size(self, engine):
        inference_bs = self.inference_batch_size
        
        def patched_analyze(model_name, df, baseline_end_idx=None, predict=True):
            model = engine.models[model_name]
            train_dataset = engine.datasets[model_name]
            df_inf = df.copy()
            for col in engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)
            
            if len(df_inf) <= train_dataset.max_encoder_length:
                return {}
                
            from pytorch_forecasting import TimeSeriesDataSet
            inference_dataset = TimeSeriesDataSet.from_dataset(
                train_dataset, df_inf, predict=False, stop_randomization=True)
            
            if len(inference_dataset) == 0:
                return {}
            
            dataloader = inference_dataset.to_dataloader(
                train=False, batch_size=inference_bs, num_workers=0, pin_memory=True)
            
            import torch
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model.to(device)
            model.eval()
            
            result_buffer = {}
            with torch.no_grad():
                for x, y in dataloader:
                    x = {k: v.to(device) for k, v in x.items()}
                    targets = y[0].cpu().numpy().flatten()
                    raw_out = model(x)
                    preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
                    p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
                    time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
                    interp = model.interpret_output(raw_out, reduction="none")
                    batch_att = interp['attention'].cpu().numpy()
                    batch_vsn = interp['encoder_variables'].cpu().numpy()
                    
                    for i in range(len(time_idx)):
                        t_id = int(time_idx[i])
                        result_buffer[t_id] = {
                            "target_true": targets[i],
                            "pred_p50": p50[i],
                            "residual": targets[i] - p50[i],
                            "divergence": p90[i] - p10[i],
                            "attention": batch_att[i],
                            "vsn": batch_vsn[i]
                        }
            
            if not result_buffer:
                return {}
            
            sorted_t = sorted(result_buffer.keys())
            df_s = pd.DataFrame({
                "time_idx": sorted_t,
                "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
                "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
                "residual": [result_buffer[t]["residual"] for t in sorted_t],
                "divergence": [result_buffer[t]["divergence"] for t in sorted_t]
            })
            full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
            full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
            
            result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
            
            if baseline_end_idx is not None:
                bl = engine._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
                result["baseline"] = bl
                engine.baselines[model_name] = bl
            
            return result
        
        engine.analyze_rolling = patched_analyze
        
    def _patch_engine_early_stopping(self, engine):
        import types
        import torch
        import pandas as pd
        import lightning.pytorch as pl
        from lightning.pytorch.loggers import CSVLogger
        from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
        from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
        from pytorch_forecasting.metrics import QuantileLoss
        from pytorch_forecasting.data import GroupNormalizer
        from pytorch_forecasting.data.encoders import NaNLabelEncoder

        ft_batch_size = self.finetune_batch_size

        def patched_build_and_fit(self_engine, model_name: str, df: pd.DataFrame, 
                                  val_ratio: float = 0.2, max_epochs: int = -1,
                                  quiet_end_idx: int = None):
            
            cfg = self_engine.config['tft_models'][model_name]
            pl.seed_everything(cfg.get('seed', 42))

            df_inf = df.copy()
            for col in self_engine.known_categoricals:
                df_inf[col] = df_inf[col].astype(str)

            if quiet_end_idx is not None:
                time_col = self_engine.global_cfg['time_column']
                df_for_training = df_inf[df_inf[time_col] <= quiet_end_idx].copy()
            else:
                df_for_training = df_inf

            time_col = self_engine.global_cfg['time_column']
            time_steps = df_for_training[time_col].sort_values().unique()
            split_idx = int(len(time_steps) * (1 - val_ratio))
            cutoff_time = time_steps[split_idx]
            
            train_df = df_for_training[df_for_training[time_col] <= cutoff_time]
            max_lookback = cfg.get('max_encoder_length', 48)
            val_start_time = time_steps[max(0, split_idx - max_lookback)]
            val_df = df_for_training[df_for_training[time_col] >= val_start_time]

            categorical_encoders = {
                name: NaNLabelEncoder(add_nan=True) for name in self_engine.known_categoricals
            }
            for g_id in self_engine.global_cfg['group_ids']:
                categorical_encoders[g_id] = NaNLabelEncoder(add_nan=True)

            training = TimeSeriesDataSet(
                train_df,
                time_idx=self_engine.global_cfg['time_column'],
                target=cfg['target'],
                group_ids=self_engine.global_cfg['group_ids'],
                min_encoder_length=cfg.get('min_encoder_length'),
                max_encoder_length=cfg.get('max_encoder_length'),
                max_prediction_length=cfg.get('max_prediction_length', 1),
                time_varying_unknown_reals=self_engine.unknown_reals,
                time_varying_known_reals=self_engine.known_reals,
                time_varying_known_categoricals=self_engine.known_categoricals,
                categorical_encoders=categorical_encoders,
                target_normalizer=GroupNormalizer(groups=self_engine.global_cfg['group_ids'], transformation=None),
                add_relative_time_idx=True,
                add_target_scales=True,
                add_encoder_length=True,
                allow_missing_timesteps=True
            )

            validation = TimeSeriesDataSet.from_dataset(training, val_df, predict=False, stop_randomization=True)

            train_dataloader = training.to_dataloader(train=True, batch_size=ft_batch_size, num_workers=0, pin_memory=True)
            val_dataloader = validation.to_dataloader(train=False, batch_size=ft_batch_size, num_workers=0, pin_memory=True)

            model = TemporalFusionTransformer.from_dataset(
                training,
                learning_rate=cfg.get('learning_rate', 0.03),
                hidden_size=cfg.get('hidden_size', 16),
                attention_head_size=4,
                dropout=0.1,
                hidden_continuous_size=8,
                output_size=len(self_engine.quantiles),
                loss=QuantileLoss(quantiles=self_engine.quantiles),
                reduce_on_plateau_patience=4
            )

            logger = CSVLogger("lightning_logs", name=model_name, flush_logs_every_n_steps=10)
            self_engine.log_dirs[model_name] = logger.log_dir

            early_stop_callback = EarlyStopping(
                monitor="val_loss", patience=8, min_delta=1e-3, mode="min", verbose=False
            )
            checkpoint_callback = ModelCheckpoint(
                monitor="val_loss", mode="min", save_top_k=1, 
                dirpath=logger.log_dir, filename="best_model"
            )

            trainer_kwargs = {
                "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
                "devices": 1,
                "enable_model_summary": False,
                "enable_checkpointing": True,
                "callbacks": [early_stop_callback, checkpoint_callback],
                "logger": logger,
                "enable_progress_bar": True
            }

            if max_epochs == -1:
                trainer = pl.Trainer(**trainer_kwargs)
            elif max_epochs != 1:
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)
            elif max_epochs == 1:
                trainer_kwargs.update({"limit_train_batches": 5, "limit_val_batches": 5})
                trainer = pl.Trainer(max_epochs=max_epochs, **trainer_kwargs)

            trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

            if checkpoint_callback.best_model_path:
                best_model = TemporalFusionTransformer.load_from_checkpoint(
                    checkpoint_callback.best_model_path, weights_only=False
                )
                model.load_state_dict(best_model.state_dict())

            self_engine.plot_training_history(model_name)
            self_engine.datasets[model_name] = training
            self_engine.models[model_name] = model

        engine.build_and_fit = types.MethodType(patched_build_and_fit, engine)

    def _extract_tft_batch(self, df_buffer, engine=None, output_tidx_set=None):
        if engine is None:
            engine = self.engine
        
        tft_results = {}
        for mn in engine.models:
            with contextlib.redirect_stdout(io.StringIO()), \
                 contextlib.redirect_stderr(io.StringIO()):
                res = engine.analyze_rolling(mn, df_buffer, baseline_end_idx=None)
            tft_results[mn] = res
        
        if not any('metrics' in v for v in tft_results.values()):
            return pd.DataFrame()
        
        df_merged = df_buffer[['time_idx', 'timestamp']].copy()
        
        for model_name in tft_results:
            res = tft_results[model_name]
            if 'metrics' not in res:
                continue
            
            metrics = res['metrics']
            attention = res['attention']
            vsn = res['vsn']
            baseline = engine.baselines.get(model_name, {})
            
            if not baseline:
                continue
            
            n = len(metrics)
            
            residual_abs = metrics['residual'].abs().values
            baseline_p95 = max(
                abs(baseline.get('residual_p95', 0)),
                abs(baseline.get('residual_p5', 0)),
                baseline.get('residual_std', 1e-8) * 1.645, 1e-8)
            sig_residual = np.clip(residual_abs / baseline_p95, 0, 20)
            
            att_base = baseline.get('att_mean')
            sig_att_kl = np.zeros(n)
            if att_base is not None:
                for i in range(n):
                    p = attention[i].flatten()
                    q = att_base.flatten()
                    p = p / (p.sum() + 1e-10) + 1e-10
                    q = q / (q.sum() + 1e-10) + 1e-10
                    p, q = p / p.sum(), q / q.sum()
                    sig_att_kl[i] = np.sum(p * np.log(p / q))
            
            vsn_base = baseline.get('vsn_mean')
            sig_vsn_js = np.zeros(n)
            if vsn_base is not None:
                for i in range(n):
                    v_i = np.abs(vsn[i].flatten()) + 1e-10
                    v_b = np.abs(vsn_base.flatten()) + 1e-10
                    p_v, q_v = v_i / v_i.sum(), v_b / v_b.sum()
                    m_v = 0.5 * (p_v + q_v)
                    sig_vsn_js[i] = (
                        0.5 * np.sum(p_v * np.log(p_v / m_v)) +
                        0.5 * np.sum(q_v * np.log(q_v / m_v)))
            
            sig_vsn_rank = np.zeros(n)
            if vsn_base is not None:
                baseline_rank = np.argsort(np.argsort(-np.abs(vsn_base.flatten())))
                for i in range(n):
                    curr_rank = np.argsort(np.argsort(-np.abs(vsn[i].flatten())))
                    corr, _ = spearmanr(baseline_rank, curr_rank)
                    sig_vsn_rank[i] = 1 - corr if not np.isnan(corr) else 1.0
            
            prefix = f'model_{model_name}'
            tft_df = pd.DataFrame({
                'time_idx': metrics['time_idx'].values,
                f'{prefix}_residual_ratio': sig_residual,
                f'{prefix}_att_kl': np.clip(sig_att_kl, 0, 20),
                f'{prefix}_vsn_js': np.clip(sig_vsn_js, 0, 5),
                f'{prefix}_vsn_rank_shift': np.clip(sig_vsn_rank, 0, 2),
            }).drop_duplicates(subset='time_idx', keep='last')
            
            n_before = len(df_merged)
            df_merged = df_merged.merge(tft_df, on='time_idx', how='left')
            if len(df_merged) != n_before:
                new_cols = [c for c in tft_df.columns if c != 'time_idx']
                df_merged = df_merged.drop(columns=new_cols).iloc[:n_before]
                for col in new_cols:
                    val_map = dict(zip(tft_df['time_idx'], tft_df[col]))
                    df_merged[col] = df_merged['time_idx'].map(val_map)
        
        base_tft_cols = [c for c in df_merged.columns if c.startswith('model_') and 'roll' not in c]
        for col in base_tft_cols:
            for k in [4, 8]:
                df_merged[f'{col}_rollmax_{k}'] = (df_merged[col].rolling(k, min_periods=1).max())
                df_merged[f'{col}_rollmean_{k}'] = (df_merged[col].rolling(k, min_periods=1).mean())
        
        df_merged = df_merged.replace([np.inf, -np.inf], 0).fillna(0)
        
        keep_cols = ['time_idx', 'timestamp'] + [c for c in df_merged.columns if c.startswith('model_')]
        df_merged = df_merged[[c for c in keep_cols if c in df_merged.columns]]
        
        if output_tidx_set is not None:
            df_merged = df_merged[df_merged['time_idx'].isin(output_tidx_set)]
        
        return df_merged


## 运行V7

In [ ]:

# ============================================================
# Cell: 自适应滚动检测 — 完整运行
# ============================================================
import types, io, contextlib, copy, os, pickle
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pytorch_forecasting import TimeSeriesDataSet
from TFT_tft_engine import TFTEngine
from target_builder import TargetBuilder
import logging

logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)
logging.getLogger("TFTDataProcessor").setLevel(logging.WARNING)
logging.getLogger("TargetBuilder").setLevel(logging.WARNING)

CKPT = "./checkpoints/pretrained_planPC"
tb = TargetBuilder.load(f"{CKPT}/target_builder.pkl")
engine_PC = TFTEngine.load(f"{CKPT}/tft_engine", config_path="TFT_config.yaml")
with open(f"{CKPT}/processor.pkl", 'rb') as f:
    processor = pickle.load(f)

df_features = df_features[df_features['timestamp']<pd.to_datetime('2025-11-13')].copy()

def analyze_rolling_patched(self, model_name, df, baseline_end_idx=None, predict=True):
    model = self.models[model_name]
    train_dataset = self.datasets[model_name]
    df_inf = df.copy()
    for col in self.known_categoricals:
        df_inf[col] = df_inf[col].astype(str)
        
    if len(df_inf) <= train_dataset.max_encoder_length:
        return {}
        
    inference_dataset = TimeSeriesDataSet.from_dataset(
        train_dataset, df_inf, predict=False, stop_randomization=True)
        
    if len(inference_dataset) == 0:
        return {}
        
    dataloader = inference_dataset.to_dataloader(
        train=False, batch_size=512, num_workers=0, pin_memory=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    result_buffer = {}
    with torch.no_grad():
        for x, y in dataloader:
            x = {k: v.to(device) for k, v in x.items()}
            targets = y[0].cpu().numpy().flatten()
            raw_out = model(x)
            preds = raw_out.prediction.cpu().numpy().squeeze(axis=1)
            p10, p50, p90 = preds[:, 0], preds[:, 1], preds[:, 2]
            time_idx = x['decoder_time_idx'].cpu().numpy().flatten()
            interp = model.interpret_output(raw_out, reduction="none")
            batch_att = interp['attention'].cpu().numpy()
            batch_vsn = interp['encoder_variables'].cpu().numpy()
            for i in range(len(time_idx)):
                t_id = int(time_idx[i])
                result_buffer[t_id] = {
                    "target_true": targets[i], "pred_p50": p50[i],
                    "residual": targets[i] - p50[i], "divergence": p90[i] - p10[i],
                    "attention": batch_att[i], "vsn": batch_vsn[i]}
    if not result_buffer:
        return {}
    sorted_t = sorted(result_buffer.keys())
    df_s = pd.DataFrame({
        "time_idx": sorted_t,
        "target_true": [result_buffer[t]["target_true"] for t in sorted_t],
        "pred_p50": [result_buffer[t]["pred_p50"] for t in sorted_t],
        "residual": [result_buffer[t]["residual"] for t in sorted_t],
        "divergence": [result_buffer[t]["divergence"] for t in sorted_t]})
    full_att = np.stack([result_buffer[t]["attention"] for t in sorted_t])
    full_vsn = np.stack([result_buffer[t]["vsn"] for t in sorted_t])
    result = {"metrics": df_s, "attention": full_att, "vsn": full_vsn}
    if baseline_end_idx is not None:
        bl = self._build_baseline(df_s, full_att, full_vsn, baseline_end_idx)
        result["baseline"] = bl
        self.baselines[model_name] = bl
    return result

engine_PC.analyze_rolling = types.MethodType(analyze_rolling_patched, engine_PC)

QUIET_START = pd.to_datetime('2025-02-18 00:00:00')
QUIET_END   = pd.to_datetime('2025-02-25 00:00:00')

mask_quiet = (df_features['timestamp'] >= QUIET_START) & (df_features['timestamp'] < QUIET_END)
df_quiet_raw = df_features[mask_quiet].copy().fillna(0)

processor_real = copy.deepcopy(processor)
processor_real.feature_transformer.fit(df_quiet_raw)

dataset_real = processor_real.transform(df_features.fillna(0), df_official_real)
df_real_all = dataset_real.data.copy()
df_real_all.fillna(0, inplace=True)
df_real_all.replace([np.inf, -np.inf], 0, inplace=True)
df_real_all = tb.transform(df_real_all)
if 'label' not in df_real_all.columns:
    df_real_all['label'] = 'N'
df_real_all = df_real_all.sort_values('time_idx').reset_index(drop=True)

# =====================================================================
# 初始化系统 7
# =====================================================================
classifier7 = StatisticalClassifier7(
    ap_k=5.0, 
    cp_base_k=3.0, 
    cp_density_thresh=0.95,
    cp_window=288, 
    cp_min_periods=96, 
    ap_min_triggers=4,
    cp_trend_window=2688,
    cp_trend_k=2.0
)

detector7 = AdaptiveRollingDetector7(
    engine=engine_PC,
    tb=tb,
    processor=processor_real,
    classifier2=classifier7,
    # 核心变动：推进至 24，实现更高频率的状态机判定，缓解末端聚集现象
    buffer_size=672, step_size=24, min_required=None,
    cp_confirm_windows=192,
    retrace_windows=288, accumulate_min=1344,
    finetune_epochs=100, finetune_lr_scale=0.005, finetune_batch_size=32, finetune_grad_clip=0.5, 
    held_out_days=2, held_out_min_windows=96,
    switch_parallel_windows=288, switch_parallel_extend=480,
    switch_recovery_threshold=0.85, switch_stability_threshold=0.85,
    cooldown_windows=672, ap_warmup_windows=96, ap_warmup_relax=1.2,
    inference_batch_size=256
)

# =====================================================================
# 执行运行
# =====================================================================
df_result7 = detector7.run(
    df_real_all=df_real_all,
    df_features_raw=df_features.fillna(0),
    df_official=df_official_real,
    quiet_start=QUIET_START,
    quiet_end=QUIET_END,
    output_path="./output/adaptive_detection_resultv7.csv"
)

# 检测可视化

In [ ]:
# ============================================================
# Cell: 自适应检测结果可视化 + crisis命中率
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_pic = pd.read_csv("./output/adaptive_detection_resultv4.csv")
df_pic['timestamp'] = pd.to_datetime(df_pic['timestamp'])

# --- 1. 事件合并与crisis命中率 ---
print("="*60)
print(" 检测结果统计")
print("="*60)
vc = df_pic['pred_label'].value_counts()
print(f"  N={vc.get('N',0)}, AP={vc.get('AP',0)}, CP={vc.get('CP',0)}")

# 事件合并
df_pic['event_change'] = (df_pic['pred_label'] != df_pic['pred_label'].shift(1)).cumsum()
events = df_pic[df_pic['pred_label'] != 'N'].groupby(
    ['pred_label', 'event_change']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    duration_windows=('timestamp', 'count'),
    peak_z_max=('static_signal_z_max', 'max') if 'static_signal_z_max' in df_pic.columns else ('time_idx', 'count'),
).reset_index().sort_values('start_time')

print(f"\n  AP事件: {len(events[events['pred_label']=='AP'])} 段")
print(f"  CP事件: {len(events[events['pred_label']=='CP'])} 段")

# Crisis命中率
crisis_df['t_crisis_start'] = pd.to_datetime(crisis_df['t_crisis_start'])
crisis_df['t_crisis_end'] = pd.to_datetime(crisis_df['t_crisis_end'])

events['in_crisis'] = False
for idx, evt in events.iterrows():
    overlap = ((evt['start_time'] <= crisis_df['t_crisis_end']) & 
               (evt['end_time'] >= crisis_df['t_crisis_start'])).any()
    events.at[idx, 'in_crisis'] = overlap

for label in ['AP', 'CP']:
    sub = events[events['pred_label'] == label]
    if len(sub) == 0:
        continue
    in_c = sub[sub['in_crisis']]
    out_c = sub[~sub['in_crisis']]
    print(f"\n  [{label}] 命中crisis: {len(in_c)} 段 ({in_c['duration_windows'].sum()/96:.1f}天)")
    print(f"  [{label}] crisis外: {len(out_c)} 段 ({out_c['duration_windows'].sum()/96:.1f}天)")

# --- 2. 投票归因统计 ---
if 'dominant_model' in df_pic.columns:
    anom = df_pic[df_pic['pred_label'] != 'N']
    if len(anom) > 0:
        print(f"\n[投票归因]")
        print(f"  主导子模型分布:")
        print(f"  {anom['dominant_model'].value_counts().to_dict()}")
        print(f"  主导信号类型分布:")
        print(f"  {anom['dominant_signal'].value_counts().to_dict()}")

# --- 3. 时序可视化 ---
fig, axes = plt.subplots(3, 1, figsize=(20, 12), sharex=True)

# (a) 静态Z-max信号
if 'static_signal_z_max' in df_pic.columns:
    ax = axes[0]
    ax.plot(df_pic['timestamp'], df_pic['static_signal_z_max'],
            lw=0.5, alpha=0.7, color='steelblue')
    
    for _, row in crisis_df.iterrows():
        ax.axvspan(row['t_crisis_start'], row['t_crisis_end'], alpha=0.12, color='red')
    
    ap_pts = df_pic[df_pic['pred_label'] == 'AP']
    ax.scatter(ap_pts['timestamp'], ap_pts['static_signal_z_max'],
               c='orange', s=8, zorder=5, label='AP')
    
    ax.set_ylabel('Static Z-max')
    ax.legend(loc='upper right')
    ax.set_title('AP 检测: 多维Z-score极值信号')

# (b) CP密度
if 'cp_density' in df_pic.columns:
    ax = axes[1]
    ax.plot(df_pic['timestamp'], df_pic['cp_density'],
            lw=1, color='blue', alpha=0.8)
    
    for _, row in crisis_df.iterrows():
        ax.axvspan(row['t_crisis_start'], row['t_crisis_end'], alpha=0.12, color='red')
    
    cp_pts = df_pic[df_pic['pred_label'] == 'CP']
    ax.scatter(cp_pts['timestamp'], cp_pts['cp_density'],
               c='red', s=12, zorder=5, label='CP')
    
    ax.axhline(0.95, color='darkred', ls='--', alpha=0.5, label='CP阈值')
    ax.set_ylabel('CP Density')
    ax.legend(loc='upper right')
    ax.set_title('CP 检测: 288窗口密度信号')

# (c) baseline版本 + 投票归因
ax = axes[2]
if 'baseline_version' in df_pic.columns:
    ax.step(df_pic['timestamp'], df_pic['baseline_version'],
            where='post', color='green', lw=2, label='Baseline Version')

# 标注CP确认事件
if hasattr(detector, 'cp_events'):
    for evt in detector.cp_events:
        if evt.get('confirmed_tidx'):
            ts_row = df_pic[df_pic['time_idx'] == evt['confirmed_tidx']]
            if len(ts_row) > 0:
                ax.axvline(ts_row['timestamp'].iloc[0], color='red',
                          ls='--', alpha=0.7, label='CP Confirmed')
        if evt.get('switched_tidx'):
            ts_row = df_pic[df_pic['time_idx'] == evt['switched_tidx']]
            if len(ts_row) > 0:
                ax.axvline(ts_row['timestamp'].iloc[0], color='green',
                          ls='--', alpha=0.7, label='Switch')

ax.set_ylabel('Baseline Version')
ax.legend(loc='upper right')
ax.set_title('自适应切换时间线')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=45)
plt.suptitle('自适应滚动检测结果全景', fontsize=14)
plt.tight_layout()
plt.savefig('./output/adaptive_detection_panorama.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 4. 归因热力图 ---
if 'trigger_features' in df_pic.columns:
    anom = df_pic[df_pic['pred_label'] != 'N'].copy()
    if len(anom) > 0:
        # 统计每个特征被触发的频率
        feat_counts = {}
        for feats_str in anom['trigger_features'].dropna():
            for f in feats_str.split('|'):
                if f:
                    feat_counts[f] = feat_counts.get(f, 0) + 1
        
        if feat_counts:
            fc_df = pd.Series(feat_counts).sort_values(ascending=True)
            fig, ax = plt.subplots(figsize=(10, max(4, len(fc_df) * 0.4)))
            colors = ['#e74c3c' if 'comment' in f else '#2980b9' for f in fc_df.index]
            fc_df.plot.barh(ax=ax, color=colors)
            ax.set_xlabel('触发次数')
            ax.set_title('AP/CP触发特征频率（红=comment, 蓝=post）')
            plt.tight_layout()
            plt.savefig('./output/trigger_feature_frequency.png', dpi=150, bbox_inches='tight')
            plt.show()

print("\n✅ 可视化完成")

## V5

In [ ]:
# ============================================================
# Cell: 自适应检测结果可视化 + crisis命中率 (重构版：PA / CA / CP)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- 0. 数据读取与标签重构 ---
df_pic = pd.read_csv("./output/adaptive_detection_resultv5.csv")
df_pic['timestamp'] = pd.to_datetime(df_pic['timestamp'])

# 【核心修改】替换业务标签：AP -> PA (瞬时脉冲), CP -> CA (集体异常/中期冲击)
df_pic['pred_label'] = df_pic['pred_label'].replace({'AP': 'PA', 'CP': 'CA'})

# --- 1. 事件合并与统计 ---
print("="*60)
print(" 异常与变点检测结果统计 (PA / CA / CP)")
print("="*60)
vc = df_pic['pred_label'].value_counts()
drift_count = df_pic['is_baseline_drift'].sum() if 'is_baseline_drift' in df_pic.columns else 0

print(f"  总窗口: {len(df_pic)}")
print(f"  N(正常)={vc.get('N',0)}, PA(瞬时脉冲)={vc.get('PA',0)}, CA(中期冲击)={vc.get('CA',0)}, CP(长程漂移)={drift_count}")

# 1.1 提取 PA 和 CA 事件片段
df_pic['event_change'] = (df_pic['pred_label'] != df_pic['pred_label'].shift(1)).cumsum()
events = df_pic[df_pic['pred_label'] != 'N'].groupby(['pred_label', 'event_change']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    duration_windows=('timestamp', 'count')
).reset_index().drop(columns=['event_change'])

# 1.2 提取真正的 CP (长程漂移) 事件片段
if 'is_baseline_drift' in df_pic.columns:
    df_pic['drift_change'] = (df_pic['is_baseline_drift'] != df_pic['is_baseline_drift'].shift(1)).cumsum()
    drift_events = df_pic[df_pic['is_baseline_drift'] == True].groupby('drift_change').agg(
        start_time=('timestamp', 'min'),
        end_time=('timestamp', 'max'),
        duration_windows=('timestamp', 'count')
    ).reset_index().drop(columns=['drift_change'])
    
    if len(drift_events) > 0:
        drift_events['pred_label'] = 'CP' # 赋予真正的变点标签
        events = pd.concat([events, drift_events], ignore_index=True)

events = events.sort_values('start_time').reset_index(drop=True)

print(f"\n  PA (脉冲) 事件: {len(events[events['pred_label']=='PA'])} 段")
print(f"  CA (冲击) 事件: {len(events[events['pred_label']=='CA'])} 段")
print(f"  CP (漂移) 事件: {len(events[events['pred_label']=='CP'])} 段")

# 1.3 Crisis命中率统计
if 'crisis_df' in locals() or 'crisis_df' in globals():
    crisis_df['t_crisis_start'] = pd.to_datetime(crisis_df['t_crisis_start'])
    crisis_df['t_crisis_end'] = pd.to_datetime(crisis_df['t_crisis_end'])

    events['in_crisis'] = False
    for idx, evt in events.iterrows():
        overlap = ((evt['start_time'] <= crisis_df['t_crisis_end']) & 
                   (evt['end_time'] >= crisis_df['t_crisis_start'])).any()
        events.at[idx, 'in_crisis'] = overlap

    for label in ['PA', 'CA', 'CP']:
        sub = events[events['pred_label'] == label]
        if len(sub) == 0:
            continue
        in_c = sub[sub['in_crisis']]
        out_c = sub[~sub['in_crisis']]
        print(f"\n  [{label}] 命中crisis: {len(in_c):<2} 段 ({in_c['duration_windows'].sum()/96:.1f}天)")
        print(f"  [{label}] crisis外: {len(out_c):<2} 段 ({out_c['duration_windows'].sum()/96:.1f}天)")
else:
    print("\n  ⚠️ 提示: 未找到 crisis_df，跳过命中率统计。")

# --- 2. 投票归因统计 (仅统计瞬时和短程冲击) ---
if 'dominant_model' in df_pic.columns:
    anom = df_pic[df_pic['pred_label'].isin(['PA', 'CA'])]
    if len(anom) > 0:
        print(f"\n[投票归因 (PA/CA)]")
        print(f"  主导子模型分布: {anom['dominant_model'].value_counts().to_dict()}")
        print(f"  主导信号类型分布: {anom['dominant_signal'].value_counts().to_dict()}")

# ============================================================
# 3. 时序全景可视化 (4 Panel 设计)
# ============================================================
# 判断是否有长程漂移数据决定画 3 张图还是 4 张图
has_trend = 'cp_trend_mean' in df_pic.columns
n_panels = 4 if has_trend else 3

fig, axes = plt.subplots(n_panels, 1, figsize=(20, 4 * n_panels), sharex=True)
if not isinstance(axes, np.ndarray):
    axes = [axes]

# 公共函数：绘制红色危机阴影
def draw_crisis_span(ax):
    if 'crisis_df' in locals() or 'crisis_df' in globals():
        for _, row in crisis_df.iterrows():
            ax.axvspan(row['t_crisis_start'], row['t_crisis_end'], alpha=0.12, color='red')

# (a) Panel 1: PA 检测 (原AP)
ax = axes[0]
if 'static_signal_z_max' in df_pic.columns:
    ax.plot(df_pic['timestamp'], df_pic['static_signal_z_max'], lw=0.5, alpha=0.7, color='steelblue', label='Static Z-max')
    draw_crisis_span(ax)
    
    pa_pts = df_pic[df_pic['pred_label'] == 'PA']
    ax.scatter(pa_pts['timestamp'], pa_pts['static_signal_z_max'], c='orange', s=10, zorder=5, label='PA (Pulse)')
    ax.set_ylabel('Static Z-max')
    ax.legend(loc='upper right')
    ax.set_title('PA 检测: 瞬时脉冲异常 (多维Z-score极值)')

# (b) Panel 2: CA 检测 (原突发CP)
ax = axes[1]
if 'cp_density' in df_pic.columns:
    ax.plot(df_pic['timestamp'], df_pic['cp_density'], lw=1, color='blue', alpha=0.8, label='Anomaly Density')
    draw_crisis_span(ax)
    
    ca_pts = df_pic[df_pic['pred_label'] == 'CA']
    ax.scatter(ca_pts['timestamp'], ca_pts['cp_density'], c='red', marker='^', s=20, zorder=5, label='CA (Collective)')
    
    ax.axhline(0.95, color='darkred', ls='--', alpha=0.5, label='CA 密度阈值 (0.95)')
    ax.set_ylabel('Anomaly Density')
    ax.legend(loc='upper right')
    ax.set_title('CA 检测: 中期冲击 / 集体异常 (288窗口高水位密度)')

# (c) Panel 3: CP 检测 (长程漂移) - 新增
if has_trend:
    ax = axes[2]
    ax.plot(df_pic['timestamp'], df_pic['cp_trend_mean'], lw=2.5, color='purple', label='28天均值')
    draw_crisis_span(ax)
    
    if 'is_baseline_drift' in df_pic.columns:
        cp_pts = df_pic[df_pic['is_baseline_drift'] == True]
        ax.scatter(cp_pts['timestamp'], cp_pts['cp_trend_mean'], c='magenta', marker='s', s=15, zorder=5, label='CP')
    
    # 估算阈值线画出辅助参考 (取漂移发生的最低均值点，若无漂移则不画)
    if drift_count > 0:
        drift_thresh = df_pic.loc[df_pic['is_baseline_drift'] == True, 'cp_trend_mean'].min()
        ax.axhline(drift_thresh, color='purple', ls=':', lw=2, alpha=0.8, label='阈值参考')
        
    ax.set_ylabel('趋势均值')
    ax.legend(loc='upper left')
    ax.set_title('长程漂移检测')

# (d) Panel 4: 自适应切换时间线
# ax = axes[-1]

# # 1. 直接绘制 CSV 中原生存储的正确阶跃线
# if 'baseline_version' in df_pic.columns:
#     ax.step(df_pic['timestamp'], df_pic['baseline_version'], 
#             where='pre', color='green', lw=2.5, label='Baseline Version')

# # 2. 动态获取检测器实例以提取事件时间戳进行打线
# det_obj = None
# if 'detector4' in locals():
#     det_obj = locals()['detector4']
# elif 'detector' in locals():
#     det_obj = locals()['detector']

# if det_obj and hasattr(det_obj, 'cp_events'):
#     for evt in det_obj.cp_events:
#         # 绘制立案确认线 (粉色/红色虚线)
#         if evt.get('confirmed_tidx'):
#             ts_row = df_pic[df_pic['time_idx'] == evt['confirmed_tidx']]
#             if len(ts_row) > 0:
#                 evt_type = evt.get('drift_type', 'CA')
#                 c = 'magenta' if evt_type == 'Incremental' else 'red'
#                 label_name = 'CP (Drift) Triggered' if evt_type == 'Incremental' else 'CA (Shock) Confirmed'
#                 ax.axvline(ts_row['timestamp'].iloc[0], color=c, ls='--', alpha=0.7, label=label_name)
        
#         # 绘制实际切换线 (绿色虚线)
#         if evt.get('switched_tidx'):
#             ts_row = df_pic[df_pic['time_idx'] == evt['switched_tidx']]
#             if len(ts_row) > 0:
#                 ax.axvline(ts_row['timestamp'].iloc[0], color='green', ls='--', alpha=0.9, lw=2, label='Baseline Switched')

# # 3. 坐标系与图例设置
# ax.set_ylabel('Version')
# ax.set_yticks(range(int(df_pic['baseline_version'].max() + 2) if 'baseline_version' in df_pic.columns else 5))
# handles, labels = ax.get_legend_handles_labels()
# by_label = dict(zip(labels, handles))
# ax.legend(by_label.values(), by_label.keys(), loc='upper left')
# ax.set_title('状态机操作：底层基线自适应切换时间线')
# (d) Panel 4: 自适应切换时间线
# (d) Panel 4: 自适应切换时间线
ax = axes[-1]

# 1. 动态获取状态机对象
det_obj = None
if 'detector5' in locals():
    det_obj = locals()['detector5']
elif 'detector' in locals():
    det_obj = locals()['detector']

# 2. 严格基于 switched_tidx 重构基线版本阶跃线
# 消除并行验证期带来的 3 天时间差，确保版本跳变严格锚定实际切换点
df_pic['plot_version'] = 0
if det_obj and hasattr(det_obj, 'cp_events'):
    for evt in det_obj.cp_events:
        if evt.get('switched_tidx'):
            # 仅在真正执行切换的时刻之后，版本号予以递增
            mask = df_pic['time_idx'] >= evt['switched_tidx']
            df_pic.loc[mask, 'plot_version'] += 1

    # 绘制阶跃线 (使用 where='post' 确保阶跃垂直线精准对齐时间戳)
    ax.step(df_pic['timestamp'], df_pic['plot_version'], 
            where='post', color='green', lw=2.5, label='TFT模型版本')
    
    # 3. 绘制垂直参考状态线
    for evt in det_obj.cp_events:
        # 绘制立案确认线 (粉色/红色虚线)
        if evt.get('confirmed_tidx'):
            ts_row = df_pic[df_pic['time_idx'] == evt['confirmed_tidx']]
            if len(ts_row) > 0:
                evt_type = evt.get('drift_type', 'CA')
                c = 'magenta' if evt_type == 'Incremental' else 'red'
                label_name = 'CP触发' if evt_type == 'Incremental' else 'CA确认'
                ax.axvline(ts_row['timestamp'].iloc[0], color=c, ls='--', alpha=0.7, label=label_name)
        
        # 绘制实际切换线 (绿色虚线)
        if evt.get('switched_tidx'):
            ts_row = df_pic[df_pic['time_idx'] == evt['switched_tidx']]
            if len(ts_row) > 0:
                ax.axvline(ts_row['timestamp'].iloc[0], color='green', ls='--', alpha=0.9, lw=2, label='版本切换')

else:
    # 降级兼容：若无内存对象，依赖原始 CSV 绘制
    if 'baseline_version' in df_pic.columns:
        ax.step(df_pic['timestamp'], df_pic['baseline_version'], 
                where='post', color='green', lw=2.5, label='TFT模型版本')

# 4. 坐标系与图例收敛设置
ax.set_ylabel('版本')
max_v = df_pic['plot_version'].max() if 'plot_version' in df_pic.columns else 4
ax.set_yticks(range(int(max_v) + 2))

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper left')
ax.set_title('TFT模型自适应切换时间线')

# 动态获取检测器实例以提取真实的切换时间
# det_obj = None
# if 'detector4' in locals():
#     det_obj = locals()['detector4']
# elif 'detector' in locals():
#     det_obj = locals()['detector']

# # 方案：利用 detector 的历史记录，在 df_pic 中重构正确的 baseline_version
# if det_obj and hasattr(det_obj, 'cp_events'):
#     df_pic['fixed_version'] = 0
#     for evt in det_obj.cp_events:
#         if evt.get('switched_tidx'):
#             # 找到发生切换的那个时间点的 index
#             switch_mask = df_pic['time_idx'] >= evt['switched_tidx']
#             # 将该时间点之后的所有版本号 +1
#             df_pic.loc[switch_mask, 'fixed_version'] += 1
    
#     # 绘制重构后的真实阶跃线
#     ax.step(df_pic['timestamp'], df_pic['fixed_version'], where='post', color='green', lw=2.5, label='Baseline Version (Reconstructed)')
    
#     # 绘制垂直参考线
#     for evt in det_obj.cp_events:
#         if evt.get('confirmed_tidx'):
#             ts_row = df_pic[df_pic['time_idx'] == evt['confirmed_tidx']]
#             if len(ts_row) > 0:
#                 evt_type = evt.get('drift_type', 'CA')
#                 c = 'magenta' if evt_type == 'Incremental' else 'red'
#                 label_name = 'CP (Drift) Triggered' if evt_type == 'Incremental' else 'CA (Shock) Confirmed'
#                 ax.axvline(ts_row['timestamp'].iloc[0], color=c, ls='--', alpha=0.7, label=label_name)
        
#         if evt.get('switched_tidx'):
#             ts_row = df_pic[df_pic['time_idx'] == evt['switched_tidx']]
#             if len(ts_row) > 0:
#                 ax.axvline(ts_row['timestamp'].iloc[0], color='green', ls='--', alpha=0.9, lw=2, label='Baseline Switched')

# else:
#     # 兼容处理：如果没有内存对象，说明是在单独跑CSV，只能画CSV里的残留错误线
#     if 'baseline_version' in df_pic.columns:
#         ax.step(df_pic['timestamp'], df_pic['baseline_version'], where='post', color='green', lw=2, label='Baseline Version (Raw CSV)')

# ax.set_ylabel('Version')
# ax.set_yticks(range(int(df_pic['fixed_version'].max() + 2))) # 强制 Y 轴显示整数刻度 (0,1,2,3,4)
# handles, labels = ax.get_legend_handles_labels()
# by_label = dict(zip(labels, handles))
# ax.legend(by_label.values(), by_label.keys(), loc='upper left')
# ax.set_title('状态机操作：底层基线自适应切换时间线')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=45)
plt.suptitle('自适应滚动检测结果全景 (三级多尺度架构)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('./output/adaptive_detection_panorama_v5.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 4. 归因热力图 (仅保留 PA/CA) ---
if 'trigger_features' in df_pic.columns:
    anom = df_pic[df_pic['pred_label'].isin(['PA', 'CA'])].copy()
    if len(anom) > 0:
        feat_counts = {}
        for feats_str in anom['trigger_features'].dropna():
            for f in feats_str.split('|'):
                if f:
                    feat_counts[f] = feat_counts.get(f, 0) + 1
        
        if feat_counts:
            fc_df = pd.Series(feat_counts).sort_values(ascending=True)
            fig, ax = plt.subplots(figsize=(10, max(4, len(fc_df) * 0.4)))
            colors = ['#e74c3c' if 'comment' in f else '#2980b9' for f in fc_df.index]
            fc_df.plot.barh(ax=ax, color=colors)
            ax.set_xlabel('触发次数')
            ax.set_title('PA/CA 触发特征频率 (红=comment, 蓝=post)')
            plt.tight_layout()
            plt.savefig('./output/trigger_feature_frequency_v5.svg', dpi=200, bbox_inches='tight')
            plt.show()

print("\n✅ 可视化完成 (文件已保存为 _v5.png 结尾)")

In [ ]:
# ============================================================
# 新增模块：危机事件全生命周期与多维预警信号耦合分析
# ============================================================

# 确保时间字段已转换为标准 datetime 对象
crisis_df['t_crisis_start'] = pd.to_datetime(crisis_df['t_crisis_start'])
crisis_df['t_crisis_peak'] = pd.to_datetime(crisis_df['t_crisis_peak'])
crisis_df['t_official_resp'] = pd.to_datetime(crisis_df['t_official_resp'])
crisis_df['t_crisis_end'] = pd.to_datetime(crisis_df['t_crisis_end'])
crisis_df = crisis_df[crisis_df['t_crisis_start'] >= pd.to_datetime('2025-03-01')] # 过滤掉过早的危机事件
# 数据容器
analysis_records = []

for idx, row in crisis_df.iterrows():
    c_id = row['crisis_id']
    lvl = row['crisis_level']
    t_s = row['t_crisis_start']
    t_p = row['t_crisis_peak']
    t_r = row['t_official_resp']
    t_e = row['t_crisis_end']

    # 提取当前危机生命周期内的检测信号
    mask = (df_pic['timestamp'] >= t_s) & (df_pic['timestamp'] <= t_e)
    sub_df = df_pic[mask]

    pa_df = sub_df[sub_df['pred_label'] == 'PA']
    ca_df = sub_df[sub_df['pred_label'] == 'CA']
    cp_df = sub_df[sub_df['is_baseline_drift'] == True] if 'is_baseline_drift' in sub_df.columns else pd.DataFrame()

    # 信号时序统计
    first_pa = pa_df['timestamp'].min() if not pa_df.empty else pd.NaT
    first_ca = ca_df['timestamp'].min() if not ca_df.empty else pd.NaT
    first_cp = cp_df['timestamp'].min() if not cp_df.empty else pd.NaT

    # 首次预警点计算 (综合 PA 与 CA)
    early_signals = pd.concat([pa_df['timestamp'], ca_df['timestamp']])
    t_detect = early_signals.min() if not early_signals.empty else pd.NaT

    # 落点阶段判定与前置时间计算
    phase = '未成功报警'
    delta_t = pd.NaT
    delta_t_min = np.nan
    if pd.notna(t_detect):
        delta_t = t_r - t_detect
        delta_t_min = delta_t.total_seconds() / 60.0
        if t_detect < t_p:
            phase = '酝酿期'
        elif t_detect < t_r:
            phase = '爆发期'
        else:
            phase = '消退期'

    analysis_records.append({
        'crisis_id': c_id,
        'level': lvl,
        't_start': t_s,
        't_peak': t_p,
        't_resp': t_r,
        't_end': t_e,
        'has_pa': not pa_df.empty,
        'has_ca': not ca_df.empty,
        'has_cp': not cp_df.empty,
        'first_pa': first_pa,
        'first_ca': first_ca,
        't_detect': t_detect,
        'delta_t_min': delta_t_min,
        'phase': phase
    })

res_df = pd.DataFrame(analysis_records)

# --- 文本报告输出 ---
print("\n" + "="*60)
print(" 危机事件全生命周期与预警信号耦合量化分析")
print("="*60)

# 一、预警时效性与前置拦截能力分析
print("\n[一] 预警时效性与前置拦截能力分析")
detected_df = res_df[res_df['phase'] != '漏警 (FN)']
hit_rate = len(detected_df) / len(res_df) * 100
avg_advance = detected_df['delta_t_min'].mean()
phase_dist = res_df['phase'].value_counts()

print(f"  全局危机检出率: {hit_rate:.1f}% ({len(detected_df)}/{len(res_df)})")
print(f"  平均提前响应时间 (相对于官方响应): {avg_advance:.1f} 分钟")
print(f"  首发预警落点分布: ")
for p, count in phase_dist.items():
    print(f"    - {p}: {count} 件 ({count/len(res_df)*100:.1f}%)")

# 二、分层预警类型与事件生命周期的耦合规律
print("\n[二] 分层预警类型与事件生命周期的耦合规律")
pre_pa = res_df[(res_df['phase'] == '酝酿期') & res_df['has_pa']]
mid_ca = res_df[(res_df['phase'] == '爆发期') | (res_df['phase'] == '酝酿期 (Pre)')]
mid_ca_triggered = mid_ca[mid_ca['has_ca']]
post_cp = res_df[res_df['has_cp']]

print(f"  酝酿期 PA 检出比例: {len(pre_pa)}/{len(res_df[res_df['phase'] == '酝酿期 (Pre)'])} (反映系统对早期微弱信号的敏感度)")
print(f"  发酵至 CA 状态跃迁比例: {len(mid_ca_triggered)}/{len(res_df)} (反映中后期集群异常的确认情况)")
print(f"  触发长程重构 (CP) 的事件数: {len(post_cp)} 件 (反映不可逆分布改变的发生频率)")

# 三、危机等级 (Level) 对系统敏感度的非线性影响
print("\n[三] 危机等级 (Level) 对系统敏感度的影响")
grouped_lvl = res_df.groupby('level').agg(
    total=('crisis_id', 'count'),
    detected=('t_detect', lambda x: x.notna().sum()),
    ca_triggered=('has_ca', 'sum'),
    cp_triggered=('has_cp', 'sum')
)
for lvl, r in grouped_lvl.iterrows():
    print(f"  Level {lvl}: 共 {r['total']} 件 | 检出 {r['detected']} 件 | CA 触发 {r['ca_triggered']} 件 | CP 触发 {r['cp_triggered']} 件")

# 四、漏警事件 (False Negative) 的精细化归因
print("\n[四] 漏警事件 (False Negative) 分析")
fn_df = res_df[res_df['phase'] == '漏警 (FN)']
if fn_df.empty:
    print("  系统在全量危机事件中未出现漏警。")
else:
    print(f"  漏警事件数: {len(fn_df)} 件。需结合相位平滑或高频波动率约束做进一步归因分析。")
    for _, fn_r in fn_df.iterrows():
        print(f"    - Crisis ID {fn_r['crisis_id']} (Level {fn_r['level']}): 历时 {(fn_r['t_end'] - fn_r['t_start']).total_seconds()/3600:.1f} 小时")

# ============================================================
# 绘制：危机事件生命周期与多维预警映射甘特图
# ============================================================
fig, ax = plt.subplots(figsize=(14, max(6, len(res_df) * 0.5)))

# 设置 Y 轴标签
y_labels = []
for i, row in res_df.iterrows():
    y_pos = i
    y_labels.append(f"危机事件 {row['crisis_id']}\n(Lvl {row['level']})")
    
    # 时段区间长度计算
    len_pre = row['t_peak'] - row['t_start']
    len_mid = row['t_resp'] - row['t_peak']
    len_post = row['t_end'] - row['t_resp']
    
    # 绘制生命周期背景色块
    ax.barh(y_pos, len_pre, left=row['t_start'], color='#2ecc71', alpha=0.3, edgecolor='none', label='酝酿期' if i == 0 else "")
    ax.barh(y_pos, len_mid, left=row['t_peak'], color='#e74c3c', alpha=0.3, edgecolor='none', label='爆发期 ' if i == 0 else "")
    ax.barh(y_pos, len_post, left=row['t_resp'], color='#95a5a6', alpha=0.3, edgecolor='none', label='消退期' if i == 0 else "")
    
    # 绘制关键时间节点虚线
    ax.vlines(row['t_start'], ymin=y_pos-0.4, ymax=y_pos+0.4, color='green', linestyle=':', lw=1.5)
    ax.vlines(row['t_peak'], ymin=y_pos-0.4, ymax=y_pos+0.4, color='red', linestyle='--', lw=1.5)
    ax.vlines(row['t_resp'], ymin=y_pos-0.4, ymax=y_pos+0.4, color='black', linestyle='-', lw=1.5)
    
    # 提取并绘制对应的预警信号散点
    mask = (df_pic['timestamp'] >= row['t_start']) & (df_pic['timestamp'] <= row['t_end'])
    sub_df = df_pic[mask]
    
    pa_pts = sub_df[sub_df['pred_label'] == 'PA']
    ca_pts = sub_df[sub_df['pred_label'] == 'CA']
    cp_pts = sub_df[sub_df['is_baseline_drift'] == True] if 'is_baseline_drift' in sub_df.columns else pd.DataFrame()
    
    if not pa_pts.empty:
        ax.scatter(pa_pts['timestamp'], [y_pos]*len(pa_pts), color='orange', marker='o', s=40, zorder=3, label='PA (脉冲)' if i == 0 else "")
    if not ca_pts.empty:
        ax.scatter(ca_pts['timestamp'], [y_pos]*len(ca_pts), color='red', marker='^', s=70, zorder=4, label='CA (冲击)' if i == 0 else "")
    if not cp_pts.empty:
        ax.scatter(cp_pts['timestamp'], [y_pos]*len(cp_pts), color='purple', marker='s', s=70, zorder=5, label='CP (漂移)' if i == 0 else "")

ax.set_yticks(range(len(res_df)))
ax.set_yticklabels(y_labels)
# ax.set_xlabel("时间 ")
ax.set_title("事件生命周期与多维预警映射关系 (甘特图)", fontsize=20)

# 整理图例以防重复
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
# 额外添加竖线图例说明
line_start = plt.Line2D([0], [0], color='green', linestyle=':', lw=1.5, label='起始时间')
line_peak = plt.Line2D([0], [0], color='red', linestyle='--', lw=1.5, label='峰值时间')
line_resp = plt.Line2D([0], [0], color='black', linestyle='-', lw=1.5, label='官方响应事件')
by_label.update({l.get_label(): l for l in [line_start, line_peak, line_resp]})

ax.legend(by_label.values(), by_label.keys(), loc='center left', bbox_to_anchor=(1.02, 0.5))

ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:00'))
plt.xticks(rotation=30)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

output_file = './output/crisis_gantt_mapping_v5.svg'
plt.savefig(output_file, dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ 甘特图已保存至 {output_file}")

### 论文图

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
import pandas as pd

# 预设字体大小与样式，适应学术论文排版
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 22,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14
})

# 动态计算高度，额外增加空间以容纳图例
fig, ax = plt.subplots(figsize=(14, max(8, len(res_df) * 0.7)))

# 学术配色字典
colors = {
    'pre': '#AED6F1',        
    'mid': '#F5B7B1',        
    'post': '#E5E7E9',       
    'line_start': '#2980B9', 
    'line_peak': '#C0392B',  
    'line_resp': '#2C3E50',  
    'pa': '#F39C12',         
    'ca': '#E74C3C',         
    'cp': '#8E44AD'          
}

y_labels = []
for i, row in res_df.iterrows():
    y_pos = i
    y_labels.append(f"危机事件 {row['crisis_id']-5}\n(Lvl {row['level']})")
    
    # 时段区间长度计算
    len_pre = row['t_peak'] - row['t_start']
    len_mid = row['t_resp'] - row['t_peak']
    len_post = row['t_end'] - row['t_resp']
    
    # 绘制生命周期背景色块
    ax.barh(y_pos, len_pre, left=row['t_start'], color=colors['pre'], alpha=0.7, edgecolor='none', label='酝酿期' if i == 0 else "")
    ax.barh(y_pos, len_mid, left=row['t_peak'], color=colors['mid'], alpha=0.7, edgecolor='none', label='爆发期' if i == 0 else "")
    ax.barh(y_pos, len_post, left=row['t_resp'], color=colors['post'], alpha=0.7, edgecolor='none', label='消退期' if i == 0 else "")
    
    # 绘制关键时间节点虚线
    ax.vlines(row['t_start'], ymin=y_pos-0.45, ymax=y_pos+0.45, color=colors['line_start'], linestyle=':', lw=2.5)
    ax.vlines(row['t_peak'], ymin=y_pos-0.45, ymax=y_pos+0.45, color=colors['line_peak'], linestyle='--', lw=2.5)
    ax.vlines(row['t_resp'], ymin=y_pos-0.45, ymax=y_pos+0.45, color=colors['line_resp'], linestyle='-', lw=2.5)
    
    # 提取并绘制对应的预警信号散点
    mask = (df_pic['timestamp'] >= row['t_start']) & (df_pic['timestamp'] <= row['t_end'])
    sub_df = df_pic[mask]
    
    pa_pts = sub_df[sub_df['pred_label'] == 'PA']
    ca_pts = sub_df[sub_df['pred_label'] == 'CA']
    cp_pts = sub_df[sub_df['is_baseline_drift'] == True] if 'is_baseline_drift' in sub_df.columns else pd.DataFrame()
    
    if not pa_pts.empty:
        ax.scatter(pa_pts['timestamp'], [y_pos]*len(pa_pts), color=colors['pa'], marker='o', s=80, zorder=3)
    if not ca_pts.empty:
        ax.scatter(ca_pts['timestamp'], [y_pos]*len(ca_pts), color=colors['ca'], marker='^', s=120, zorder=4)
    if not cp_pts.empty:
        ax.scatter(cp_pts['timestamp'], [y_pos]*len(cp_pts), color=colors['cp'], marker='s', s=120, zorder=5)

ax.set_yticks(range(len(res_df)))
ax.set_yticklabels(y_labels)
# ax.set_title("事件生命周期与多维预警映射关系", pad=20)

# 整理背景色块的图例句柄
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))

# 创建显式的代理图例句柄
proxy_start = Line2D([0], [0], color=colors['line_start'], linestyle=':', lw=2.5, label='起始时间')
proxy_peak = Line2D([0], [0], color=colors['line_peak'], linestyle='--', lw=2.5, label='峰值时间')
proxy_resp = Line2D([0], [0], color=colors['line_resp'], linestyle='-', lw=2.5, label='官方响应时间')

proxy_pa = Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['pa'], markersize=10, label='点异常')
proxy_ca = Line2D([0], [0], marker='^', color='w', markerfacecolor=colors['ca'], markersize=12, label='集体异常')
proxy_cp = Line2D([0], [0], marker='s', color='w', markerfacecolor=colors['cp'], markersize=12, label='变点')

# 合并所有图例对象
for item in [proxy_start, proxy_peak, proxy_resp, proxy_pa, proxy_ca, proxy_cp]:
    by_label[item.get_label()] = item

# 按照指定逻辑顺序组织图例输出内容
legend_order = [
    '酝酿期', '爆发期', '消退期', 
    '起始时间', '峰值时间', '官方响应时间', 
    '点异常', '集体异常', '变点'
]
ordered_handles = [by_label[k] for k in legend_order if k in by_label]
ordered_labels = [k for k in legend_order if k in by_label]

# 图例排版
ax.legend(ordered_handles, ordered_labels, loc='upper left', ncol=3, framealpha=0.95, edgecolor='black', borderpad=0.8)

# 限定X轴起点为3月1日（年份依照实际数据集调整，此处设为2025）
ax.set_xlim(left=pd.to_datetime('2025-03-01'))

# 动态扩展 y 轴的顶部空间
ylim_bottom, ylim_top = ax.get_ylim()
ax.set_ylim(ylim_bottom, ylim_top + 1.8)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:00'))
plt.xticks(rotation=30)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

output_file = './output/crisis_gantt_mapping_v5.1.svg'
plt.savefig(output_file, dpi=300, bbox_inches='tight', format='svg')
plt.show()
print(f"✅ 甘特图已保存至 {output_file}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
import pandas as pd

# 预设字体大小与样式，适应学术论文排版
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 22,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14
})

# 动态计算高度，额外增加空间以容纳图例
fig, ax = plt.subplots(figsize=(14, max(8, len(res_df) * 0.7)))

# 学术配色字典
colors = {
    'pre': '#AED6F1',        
    'mid': '#F5B7B1',        
    'post': '#E5E7E9',       
    'line_start': '#2980B9', 
    'line_peak': '#C0392B',  
    'line_resp': '#2C3E50',  
    'pa': '#F39C12',         
    'ca': '#E74C3C',         
    'cp': '#8E44AD'          
}

y_labels = []
for i, row in res_df.iterrows():
    y_pos = i
    y_labels.append(f"危机事件 {row['crisis_id']-5}\n(Lvl {row['level']})")
    
    # 时段区间长度计算
    len_pre = row['t_peak'] - row['t_start']
    len_mid = row['t_resp'] - row['t_peak']
    len_post = row['t_end'] - row['t_resp']
    
    # 绘制生命周期背景色块
    ax.barh(y_pos, len_pre, left=row['t_start'], color=colors['pre'], alpha=0.7, edgecolor='none', label='酝酿期' if i == 0 else "")
    ax.barh(y_pos, len_mid, left=row['t_peak'], color=colors['mid'], alpha=0.7, edgecolor='none', label='爆发期' if i == 0 else "")
    ax.barh(y_pos, len_post, left=row['t_resp'], color=colors['post'], alpha=0.7, edgecolor='none', label='消退期' if i == 0 else "")
    
    # 绘制关键时间节点虚线
    ax.vlines(row['t_start'], ymin=y_pos-0.45, ymax=y_pos+0.45, color=colors['line_start'], linestyle=':', lw=2.5)
    ax.vlines(row['t_peak'], ymin=y_pos-0.45, ymax=y_pos+0.45, color=colors['line_peak'], linestyle='--', lw=2.5)
    ax.vlines(row['t_resp'], ymin=y_pos-0.45, ymax=y_pos+0.45, color=colors['line_resp'], linestyle='-', lw=2.5)
    
    # 提取并绘制对应的预警信号散点
    mask = (df_pic['timestamp'] >= row['t_start']) & (df_pic['timestamp'] <= row['t_end'])
    sub_df = df_pic[mask]
    
    pa_pts = sub_df[sub_df['pred_label'] == 'PA']
    ca_pts = sub_df[sub_df['pred_label'] == 'CA']
    
    # 核心修改点：增加多重掩码过滤，确保CP期间仅在状态不为PA或CA时才赋值并渲染CP颜色
    if 'is_baseline_drift' in sub_df.columns:
        drift_mask = (sub_df['is_baseline_drift'] == True) & (sub_df['pred_label'].isin(['N']))
        cp_pts = sub_df[drift_mask]
    else:
        cp_pts = pd.DataFrame()
    
    if not pa_pts.empty:
        ax.scatter(pa_pts['timestamp'], [y_pos]*len(pa_pts), color=colors['pa'], marker='o', s=80, zorder=3)
    if not ca_pts.empty:
        ax.scatter(ca_pts['timestamp'], [y_pos]*len(ca_pts), color=colors['ca'], marker='^', s=120, zorder=4)
    if not cp_pts.empty:
        ax.scatter(cp_pts['timestamp'], [y_pos]*len(cp_pts), color=colors['cp'], marker='s', s=60, zorder=5,alpha=0.5)

ax.set_yticks(range(len(res_df)))
ax.set_yticklabels(y_labels)

# 整理背景色块的图例句柄
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))

# 创建显式的代理图例句柄
proxy_start = Line2D([0], [0], color=colors['line_start'], linestyle=':', lw=2.5, label='起始时间')
proxy_peak = Line2D([0], [0], color=colors['line_peak'], linestyle='--', lw=2.5, label='峰值时间')
proxy_resp = Line2D([0], [0], color=colors['line_resp'], linestyle='-', lw=2.5, label='官方响应时间')

proxy_pa = Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['pa'], markersize=10, label='点异常')
proxy_ca = Line2D([0], [0], marker='^', color='w', markerfacecolor=colors['ca'], markersize=12, label='集体异常')
proxy_cp = Line2D([0], [0], marker='s', color='w', markerfacecolor=colors['cp'], markersize=12, label='变点')

# 合并所有图例对象
for item in [proxy_start, proxy_peak, proxy_resp, proxy_pa, proxy_ca, proxy_cp]:
    by_label[item.get_label()] = item

# 按照指定逻辑顺序组织图例输出内容
legend_order = [
    '酝酿期', '爆发期', '消退期', 
    '起始时间', '峰值时间', '官方响应时间', 
    '点异常', '集体异常', '变点'
]
ordered_handles = [by_label[k] for k in legend_order if k in by_label]
ordered_labels = [k for k in legend_order if k in by_label]

# 图例排版
ax.legend(ordered_handles, ordered_labels, loc='upper left', ncol=3, framealpha=0.95, edgecolor='black', borderpad=0.8)

# 限定X轴起点为3月1日（年份依照实际数据集调整，此处设为2025）
ax.set_xlim(left=pd.to_datetime('2025-03-01'))

# 动态扩展 y 轴的顶部空间
ylim_bottom, ylim_top = ax.get_ylim()
ax.set_ylim(ylim_bottom, ylim_top + 1.8)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:00'))
plt.xticks(rotation=30)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

output_file = './output/crisis_gantt_mapping_v5.2.svg'
plt.savefig(output_file, dpi=300, bbox_inches='tight', format='svg')
plt.show()
print(f"✅ 甘特图已保存至 {output_file}")

In [ ]:
import matplotlib.pyplot as plt

# 预设字体大小与样式，适应学术论文排版
plt.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 18,
    'legend.fontsize': 13
})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ==========================================
# (a) 数据源敏感度分布 数据配置
# ==========================================
labels_a = ['评论特征主导\n(comment_pc1)', '广场特征主导\n(post_pc2)']
sizes_a = [860, 173]
# 采用冷色系（蓝/青）代表数据源模态
colors_a = ['#5DADE2', '#48C9B0'] 

# ==========================================
# (b) 偏差机制主导分布 数据配置
# ==========================================
labels_b = ['关键特征重排序\n(vsn_rank_shift)', '特征空间贡献平移\n(vsn_js)', '时序注意力偏移\n(att_kl)']
sizes_b = [902, 130, 1]
# 采用暖色系（橙/黄/红）代表内部表征机制
colors_b = ['#EB984E', '#F4D03F', '#EC7063']

# 绘制左侧子图 (a)
wedges_a, texts_a, autotexts_a = axes[0].pie(
    sizes_a, 
    labels=labels_a, 
    autopct='%1.1f%%', 
    startangle=140,
    colors=colors_a, 
    wedgeprops=dict(width=0.45, edgecolor='w', linewidth=2),
    pctdistance=0.75, 
    textprops={'fontsize': 14}
)
axes[0].set_title('(a) 数据源敏感度分布', pad=20)

# 绘制右侧子图 (b)
wedges_b, texts_b, autotexts_b = axes[1].pie(
    sizes_b, 
    labels=labels_b, 
    autopct='%1.1f%%', 
    startangle=140,
    colors=colors_b, 
    wedgeprops=dict(width=0.45, edgecolor='w', linewidth=2),
    pctdistance=0.75, 
    textprops={'fontsize': 14}
)
axes[1].set_title('(b) 偏差机制主导分布', pad=20)

# 针对占比极小的数据（如0.1%），将其数值标签向外侧微调以防重叠
autotexts_b[2].set_position((1.1, 0.1))

# 调整全局布局与主标题
plt.suptitle('图 6.3 预警窗口底层特征多维投票归因分布', y=1.02, fontsize=20)
plt.figtext(0.5, -0.05, 
            "注：左侧展示了系统对不同模态数据源的响应差异，右侧揭示了导致预警触发的核心内部表征变化机理。", 
            ha="center", fontsize=14, color="#333333")

plt.tight_layout()

# 导出高清矢量图
output_file = './output/attribution_doughnut_chart.svg'
plt.savefig(output_file, dpi=300, bbox_inches='tight', format='svg')
plt.show()

print(f"图表渲染完成，已导出至: {output_file}")

## V6

In [ ]:
# ============================================================
# Cell: 自适应检测结果可视化 + crisis命中率 (重构版：PA / CA / CP)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- 0. 数据读取与标签重构 ---
df_pic = pd.read_csv("./output/adaptive_detection_resultv6.csv")
df_pic['timestamp'] = pd.to_datetime(df_pic['timestamp'])

# 【核心修改】替换业务标签：AP -> PA (瞬时脉冲), CP -> CA (集体异常/中期冲击)
df_pic['pred_label'] = df_pic['pred_label'].replace({'AP': 'PA', 'CP': 'CA'})

# --- 1. 事件合并与统计 ---
print("="*60)
print(" 异常与变点检测结果统计 (PA / CA / CP)")
print("="*60)
vc = df_pic['pred_label'].value_counts()
drift_count = df_pic['is_baseline_drift'].sum() if 'is_baseline_drift' in df_pic.columns else 0

print(f"  总窗口: {len(df_pic)}")
print(f"  N(正常)={vc.get('N',0)}, PA(瞬时脉冲)={vc.get('PA',0)}, CA(中期冲击)={vc.get('CA',0)}, CP(长程漂移)={drift_count}")

# 1.1 提取 PA 和 CA 事件片段
df_pic['event_change'] = (df_pic['pred_label'] != df_pic['pred_label'].shift(1)).cumsum()
events = df_pic[df_pic['pred_label'] != 'N'].groupby(['pred_label', 'event_change']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    duration_windows=('timestamp', 'count')
).reset_index().drop(columns=['event_change'])

# 1.2 提取真正的 CP (长程漂移) 事件片段
if 'is_baseline_drift' in df_pic.columns:
    df_pic['drift_change'] = (df_pic['is_baseline_drift'] != df_pic['is_baseline_drift'].shift(1)).cumsum()
    drift_events = df_pic[df_pic['is_baseline_drift'] == True].groupby('drift_change').agg(
        start_time=('timestamp', 'min'),
        end_time=('timestamp', 'max'),
        duration_windows=('timestamp', 'count')
    ).reset_index().drop(columns=['drift_change'])
    
    if len(drift_events) > 0:
        drift_events['pred_label'] = 'CP' # 赋予真正的变点标签
        events = pd.concat([events, drift_events], ignore_index=True)

events = events.sort_values('start_time').reset_index(drop=True)

print(f"\n  PA (脉冲) 事件: {len(events[events['pred_label']=='PA'])} 段")
print(f"  CA (冲击) 事件: {len(events[events['pred_label']=='CA'])} 段")
print(f"  CP (漂移) 事件: {len(events[events['pred_label']=='CP'])} 段")

# 1.3 Crisis命中率统计
if 'crisis_df' in locals() or 'crisis_df' in globals():
    crisis_df['t_crisis_start'] = pd.to_datetime(crisis_df['t_crisis_start'])
    crisis_df['t_crisis_end'] = pd.to_datetime(crisis_df['t_crisis_end'])

    events['in_crisis'] = False
    for idx, evt in events.iterrows():
        overlap = ((evt['start_time'] <= crisis_df['t_crisis_end']) & 
                   (evt['end_time'] >= crisis_df['t_crisis_start'])).any()
        events.at[idx, 'in_crisis'] = overlap

    for label in ['PA', 'CA', 'CP']:
        sub = events[events['pred_label'] == label]
        if len(sub) == 0:
            continue
        in_c = sub[sub['in_crisis']]
        out_c = sub[~sub['in_crisis']]
        print(f"\n  [{label}] 命中crisis: {len(in_c):<2} 段 ({in_c['duration_windows'].sum()/96:.1f}天)")
        print(f"  [{label}] crisis外: {len(out_c):<2} 段 ({out_c['duration_windows'].sum()/96:.1f}天)")
else:
    print("\n  ⚠️ 提示: 未找到 crisis_df，跳过命中率统计。")

# --- 2. 投票归因统计 (仅统计瞬时和短程冲击) ---
if 'dominant_model' in df_pic.columns:
    anom = df_pic[df_pic['pred_label'].isin(['PA', 'CA'])]
    if len(anom) > 0:
        print(f"\n[投票归因 (PA/CA)]")
        print(f"  主导子模型分布: {anom['dominant_model'].value_counts().to_dict()}")
        print(f"  主导信号类型分布: {anom['dominant_signal'].value_counts().to_dict()}")

# ============================================================
# 3. 时序全景可视化 (4 Panel 设计)
# ============================================================
# 判断是否有长程漂移数据决定画 3 张图还是 4 张图
has_trend = 'cp_trend_mean' in df_pic.columns
n_panels = 4 if has_trend else 3

fig, axes = plt.subplots(n_panels, 1, figsize=(20, 4 * n_panels), sharex=True)
if not isinstance(axes, np.ndarray):
    axes = [axes]

# 公共函数：绘制红色危机阴影
def draw_crisis_span(ax):
    if 'crisis_df' in locals() or 'crisis_df' in globals():
        for _, row in crisis_df.iterrows():
            ax.axvspan(row['t_crisis_start'], row['t_crisis_end'], alpha=0.12, color='red')

# (a) Panel 1: PA 检测 (原AP)
ax = axes[0]
if 'static_signal_z_max' in df_pic.columns:
    ax.plot(df_pic['timestamp'], df_pic['static_signal_z_max'], lw=0.5, alpha=0.7, color='steelblue', label='Static Z-max')
    draw_crisis_span(ax)
    
    pa_pts = df_pic[df_pic['pred_label'] == 'PA']
    ax.scatter(pa_pts['timestamp'], pa_pts['static_signal_z_max'], c='orange', s=10, zorder=5, label='PA (Pulse)')
    ax.set_ylabel('Static Z-max')
    ax.legend(loc='upper right')
    ax.set_title('PA 检测: 瞬时脉冲异常 (多维Z-score极值)')

# (b) Panel 2: CA 检测 (原突发CP)
ax = axes[1]
if 'cp_density' in df_pic.columns:
    ax.plot(df_pic['timestamp'], df_pic['cp_density'], lw=1, color='blue', alpha=0.8, label='Anomaly Density')
    draw_crisis_span(ax)
    
    ca_pts = df_pic[df_pic['pred_label'] == 'CA']
    ax.scatter(ca_pts['timestamp'], ca_pts['cp_density'], c='red', marker='^', s=20, zorder=5, label='CA (Collective)')
    
    ax.axhline(0.95, color='darkred', ls='--', alpha=0.5, label='CA 密度阈值 (0.95)')
    ax.set_ylabel('Anomaly Density')
    ax.legend(loc='upper right')
    ax.set_title('CA 检测: 中期冲击 / 集体异常 (288窗口高水位密度)')

# (c) Panel 3: CP 检测 (长程漂移) - 新增
if has_trend:
    ax = axes[2]
    ax.plot(df_pic['timestamp'], df_pic['cp_trend_mean'], lw=2.5, color='purple', label='14-Day Trend Mean')
    draw_crisis_span(ax)
    
    if 'is_baseline_drift' in df_pic.columns:
        cp_pts = df_pic[df_pic['is_baseline_drift'] == True]
        ax.scatter(cp_pts['timestamp'], cp_pts['cp_trend_mean'], c='magenta', marker='s', s=15, zorder=5, label='CP (True Drift)')
    
    # 估算阈值线画出辅助参考 (取漂移发生的最低均值点，若无漂移则不画)
    if drift_count > 0:
        drift_thresh = df_pic.loc[df_pic['is_baseline_drift'] == True, 'cp_trend_mean'].min()
        ax.axhline(drift_thresh, color='purple', ls=':', lw=2, alpha=0.8, label='Drift Threshold (Approx)')
        
    ax.set_ylabel('Trend Mean')
    ax.legend(loc='upper right')
    ax.set_title('CP 检测: 结构性变点 / 长程漂移 (底层稳态平移)')

# (d) Panel 4: 自适应切换时间线
# ax = axes[-1]

# # 1. 直接绘制 CSV 中原生存储的正确阶跃线
# if 'baseline_version' in df_pic.columns:
#     ax.step(df_pic['timestamp'], df_pic['baseline_version'], 
#             where='pre', color='green', lw=2.5, label='Baseline Version')

# # 2. 动态获取检测器实例以提取事件时间戳进行打线
# det_obj = None
# if 'detector4' in locals():
#     det_obj = locals()['detector4']
# elif 'detector' in locals():
#     det_obj = locals()['detector']

# if det_obj and hasattr(det_obj, 'cp_events'):
#     for evt in det_obj.cp_events:
#         # 绘制立案确认线 (粉色/红色虚线)
#         if evt.get('confirmed_tidx'):
#             ts_row = df_pic[df_pic['time_idx'] == evt['confirmed_tidx']]
#             if len(ts_row) > 0:
#                 evt_type = evt.get('drift_type', 'CA')
#                 c = 'magenta' if evt_type == 'Incremental' else 'red'
#                 label_name = 'CP (Drift) Triggered' if evt_type == 'Incremental' else 'CA (Shock) Confirmed'
#                 ax.axvline(ts_row['timestamp'].iloc[0], color=c, ls='--', alpha=0.7, label=label_name)
        
#         # 绘制实际切换线 (绿色虚线)
#         if evt.get('switched_tidx'):
#             ts_row = df_pic[df_pic['time_idx'] == evt['switched_tidx']]
#             if len(ts_row) > 0:
#                 ax.axvline(ts_row['timestamp'].iloc[0], color='green', ls='--', alpha=0.9, lw=2, label='Baseline Switched')

# # 3. 坐标系与图例设置
# ax.set_ylabel('Version')
# ax.set_yticks(range(int(df_pic['baseline_version'].max() + 2) if 'baseline_version' in df_pic.columns else 5))
# handles, labels = ax.get_legend_handles_labels()
# by_label = dict(zip(labels, handles))
# ax.legend(by_label.values(), by_label.keys(), loc='upper left')
# ax.set_title('状态机操作：底层基线自适应切换时间线')
# (d) Panel 4: 自适应切换时间线
# (d) Panel 4: 自适应切换时间线
ax = axes[-1]

# 1. 动态获取状态机对象
det_obj = None
if 'detector6' in locals():
    det_obj = locals()['detector6']
elif 'detector' in locals():
    det_obj = locals()['detector']

# 2. 严格基于 switched_tidx 重构基线版本阶跃线
# 消除并行验证期带来的 3 天时间差，确保版本跳变严格锚定实际切换点
df_pic['plot_version'] = 0
if det_obj and hasattr(det_obj, 'cp_events'):
    for evt in det_obj.cp_events:
        if evt.get('switched_tidx'):
            # 仅在真正执行切换的时刻之后，版本号予以递增
            mask = df_pic['time_idx'] >= evt['switched_tidx']
            df_pic.loc[mask, 'plot_version'] += 1

    # 绘制阶跃线 (使用 where='post' 确保阶跃垂直线精准对齐时间戳)
    ax.step(df_pic['timestamp'], df_pic['plot_version'], 
            where='post', color='green', lw=2.5, label='Baseline Version')
    
    # 3. 绘制垂直参考状态线
    for evt in det_obj.cp_events:
        # 绘制立案确认线 (粉色/红色虚线)
        if evt.get('confirmed_tidx'):
            ts_row = df_pic[df_pic['time_idx'] == evt['confirmed_tidx']]
            if len(ts_row) > 0:
                evt_type = evt.get('drift_type', 'CA')
                c = 'magenta' if evt_type == 'Incremental' else 'red'
                label_name = 'CP (Drift) Triggered' if evt_type == 'Incremental' else 'CA (Shock) Confirmed'
                ax.axvline(ts_row['timestamp'].iloc[0], color=c, ls='--', alpha=0.7, label=label_name)
        
        # 绘制实际切换线 (绿色虚线)
        if evt.get('switched_tidx'):
            ts_row = df_pic[df_pic['time_idx'] == evt['switched_tidx']]
            if len(ts_row) > 0:
                ax.axvline(ts_row['timestamp'].iloc[0], color='green', ls='--', alpha=0.9, lw=2, label='Baseline Switched')

else:
    # 降级兼容：若无内存对象，依赖原始 CSV 绘制
    if 'baseline_version' in df_pic.columns:
        ax.step(df_pic['timestamp'], df_pic['baseline_version'], 
                where='post', color='green', lw=2.5, label='Baseline Version (CSV)')

# 4. 坐标系与图例收敛设置
ax.set_ylabel('Version')
max_v = df_pic['plot_version'].max() if 'plot_version' in df_pic.columns else 4
ax.set_yticks(range(int(max_v) + 2))

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper left')
ax.set_title('状态机操作：底层基线自适应切换时间线')

# 动态获取检测器实例以提取真实的切换时间
# det_obj = None
# if 'detector4' in locals():
#     det_obj = locals()['detector4']
# elif 'detector' in locals():
#     det_obj = locals()['detector']

# # 方案：利用 detector 的历史记录，在 df_pic 中重构正确的 baseline_version
# if det_obj and hasattr(det_obj, 'cp_events'):
#     df_pic['fixed_version'] = 0
#     for evt in det_obj.cp_events:
#         if evt.get('switched_tidx'):
#             # 找到发生切换的那个时间点的 index
#             switch_mask = df_pic['time_idx'] >= evt['switched_tidx']
#             # 将该时间点之后的所有版本号 +1
#             df_pic.loc[switch_mask, 'fixed_version'] += 1
    
#     # 绘制重构后的真实阶跃线
#     ax.step(df_pic['timestamp'], df_pic['fixed_version'], where='post', color='green', lw=2.5, label='Baseline Version (Reconstructed)')
    
#     # 绘制垂直参考线
#     for evt in det_obj.cp_events:
#         if evt.get('confirmed_tidx'):
#             ts_row = df_pic[df_pic['time_idx'] == evt['confirmed_tidx']]
#             if len(ts_row) > 0:
#                 evt_type = evt.get('drift_type', 'CA')
#                 c = 'magenta' if evt_type == 'Incremental' else 'red'
#                 label_name = 'CP (Drift) Triggered' if evt_type == 'Incremental' else 'CA (Shock) Confirmed'
#                 ax.axvline(ts_row['timestamp'].iloc[0], color=c, ls='--', alpha=0.7, label=label_name)
        
#         if evt.get('switched_tidx'):
#             ts_row = df_pic[df_pic['time_idx'] == evt['switched_tidx']]
#             if len(ts_row) > 0:
#                 ax.axvline(ts_row['timestamp'].iloc[0], color='green', ls='--', alpha=0.9, lw=2, label='Baseline Switched')

# else:
#     # 兼容处理：如果没有内存对象，说明是在单独跑CSV，只能画CSV里的残留错误线
#     if 'baseline_version' in df_pic.columns:
#         ax.step(df_pic['timestamp'], df_pic['baseline_version'], where='post', color='green', lw=2, label='Baseline Version (Raw CSV)')

# ax.set_ylabel('Version')
# ax.set_yticks(range(int(df_pic['fixed_version'].max() + 2))) # 强制 Y 轴显示整数刻度 (0,1,2,3,4)
# handles, labels = ax.get_legend_handles_labels()
# by_label = dict(zip(labels, handles))
# ax.legend(by_label.values(), by_label.keys(), loc='upper left')
# ax.set_title('状态机操作：底层基线自适应切换时间线')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=45)
plt.suptitle('自适应滚动检测结果全景 (三级多尺度架构)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('./output/adaptive_detection_panorama_v5.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 4. 归因热力图 (仅保留 PA/CA) ---
if 'trigger_features' in df_pic.columns:
    anom = df_pic[df_pic['pred_label'].isin(['PA', 'CA'])].copy()
    if len(anom) > 0:
        feat_counts = {}
        for feats_str in anom['trigger_features'].dropna():
            for f in feats_str.split('|'):
                if f:
                    feat_counts[f] = feat_counts.get(f, 0) + 1
        
        if feat_counts:
            fc_df = pd.Series(feat_counts).sort_values(ascending=True)
            fig, ax = plt.subplots(figsize=(10, max(4, len(fc_df) * 0.4)))
            colors = ['#e74c3c' if 'comment' in f else '#2980b9' for f in fc_df.index]
            fc_df.plot.barh(ax=ax, color=colors)
            ax.set_xlabel('触发次数')
            ax.set_title('PA/CA 触发特征频率 (红=comment, 蓝=post)')
            plt.tight_layout()
            plt.savefig('./output/trigger_feature_frequency_v6.png', dpi=150, bbox_inches='tight')
            plt.show()

print("\n✅ 可视化完成 (文件已保存为 _v6.png 结尾)")

#

# 归因

In [ ]:

# ============================================================
# 4. 归因
# ============================================================
attr_engine = ContentAttributionEngine()
df_attribution = step2_attribute_windows(
    df_final, df_raw, tb,
    attribution_engine=attr_engine,
    pred_col='final_label',
    max_windows=50,
)


# 其他

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 计算相关系数矩阵，默认使用 Pearson 相关系数
corr_matrix = df_features.corr()
print(corr_matrix)
# 设置画布大小
plt.figure(figsize=(10, 8))

# 绘制热力图
# annot=True 表示在单元格内显示数值
# cmap='coolwarm' 指定颜色映射方案
# fmt='.2f' 表示保留两位小数
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')

# 设置标题
plt.title('Correlation Matrix')

# 显示图形
plt.show()